<a href="https://colab.research.google.com/github/Levan-Danelia/FRTB/blob/main/FRTB_Jupyter_Notebooks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

FRTB ASA explai nstrcure here

In [1]:
# import libraries
import pandas as pd
import numpy as np
from tabulate import tabulate

In [2]:
# ============================================================
# FORMAT TABLE FUNCTION
# ============================================================

def format_table(df, columns, col_formats=None, alignments=None):
    """
    Format a DataFrame for display using tabulate.

    Parameters:
        df: pandas DataFrame
        columns: list of column names to display
        col_formats: dict mapping column names to format strings (e.g., ',.0f', ',.2f', '.4f')
        alignments: tuple of alignment strings ('left', 'right', 'center')

    Returns:
        Formatted table string
    """
    df_copy = df[columns].copy()

    if col_formats:
        for col, fmt in col_formats.items():
            if col in df_copy.columns:
                df_copy[col] = df_copy[col].apply(
                    lambda x: f"{x:{fmt}}" if isinstance(x, (int, float, np.integer, np.floating)) else x
                )

    return tabulate(
        df_copy,
        headers='keys',
        tablefmt='simple_outline',
        showindex=False,
        colalign=alignments
    )

# Sensitivity Based Method

## Delta

### IRDL

In [3]:
# ============================================================
# IRDL
# ============================================================


# TABLE OF CONTENTS
#   1) Gross sensitivities
#   2) Net sensitivities (s_k)
#   3) Weighted sensitivities (WS_k)
#   4) Intra-bucket correlation (rho_kl)
#   5) Intra-bucket aggregation (K_b)
#   6) Bucket sums (S_b)
#   7) Cross-bucket correlation (gamma_bc)
#   8) Cross-bucket aggregation
#   9) Correlation scenarios (high/low)
#  10) Capital
#  11) Printing
# ============================================================


# ============================================================
# 1) GROSS SENSITIVITIES
# ============================================================
# Position-level delta sensitivities to interest rate curves.

data = [
    {'trade_id': 1, 'bucket': 'eur', 'rfr': 'ois', 'maturity': '6m',  'gross_sk': -1300},
    {'trade_id': 2, 'bucket': 'eur', 'rfr': 'ois', 'maturity': '3m',  'gross_sk': 2100},
    {'trade_id': 3, 'bucket': 'eur', 'rfr': 'sov', 'maturity': '30y', 'gross_sk': -4900},
    {'trade_id': 4, 'bucket': 'gbp', 'rfr': 'ois', 'maturity': '3m',  'gross_sk': 1800},
]

irdl_gross_sk = pd.DataFrame(data)


# ============================================================
# 2) NET SENSITIVITIES (s_k)
# ============================================================
# Net sensitivity s_k = Σ gross sensitivities for each unique risk factor.
# Risk factor defined by: bucket + rfr + maturity.

irdl_risk_factor_cols = ['bucket', 'rfr', 'maturity']

irdl_net_sk = (
    irdl_gross_sk
    .groupby(irdl_risk_factor_cols)
    .agg(sk=('gross_sk', 'sum'))
    .reset_index()
)


# ============================================================
# 3) WEIGHTED SENSITIVITIES (WS_k)
# ============================================================
# WS_k = s_k × RW_k
# For liquid currencies: RW_k = RW_k(base) / √2

# Tenor string to years mapping
IRDL_TENOR_TO_YEARS = {
    '3m': 0.25,
    '6m': 0.5,
    '1y': 1,
    '2y': 2,
    '3y': 3,
    '5y': 5,
    '10y': 10,
    '15y': 15,
    '20y': 20,
    '30y': 30,
}

# Risk weights by tenor in years
IRDL_RISK_WEIGHTS = {
    0.25: 0.017,
    0.5: 0.017,
    1: 0.016,
    2: 0.013,
    3: 0.012,
    5: 0.011,
    10: 0.011,
    15: 0.011,
    20: 0.011,
    30: 0.011,
}

# Most liquid currencies receive RW / √2 adjustment (Article 325ae)
IRDL_LIQUID_CURRENCIES = ['eur', 'gbp', 'usd', 'jpy', 'chf', 'aud', 'cad', 'sek']

irdl_wsk = irdl_net_sk.copy()
irdl_wsk['tenor_years'] = irdl_wsk['maturity'].map(IRDL_TENOR_TO_YEARS)
irdl_wsk['rw_base'] = irdl_wsk['tenor_years'].map(IRDL_RISK_WEIGHTS)
irdl_wsk['rw'] = irdl_wsk.apply(
    lambda row: row['rw_base'] / np.sqrt(2) if row['bucket'] in IRDL_LIQUID_CURRENCIES else row['rw_base'],
    axis=1
)
irdl_wsk['wsk'] = irdl_wsk['sk'] * irdl_wsk['rw']

# Extract weighted sensitivities per bucket for aggregation
# After groupby, order is alphabetical: (bucket, rfr, maturity)
irdl_wsk_eur = irdl_wsk[irdl_wsk['bucket'] == 'eur']['wsk'].values
irdl_wsk_eur_1 = irdl_wsk_eur[0]  # EUR OIS 3M
irdl_wsk_eur_2 = irdl_wsk_eur[1]  # EUR OIS 6M
irdl_wsk_eur_3 = irdl_wsk_eur[2]  # EUR SOV 30Y

irdl_wsk_gbp = irdl_wsk[irdl_wsk['bucket'] == 'gbp']['wsk'].values
irdl_wsk_gbp_1 = irdl_wsk_gbp[0]  # GBP OIS 3M

# Extract tenor info for correlation calculations (matching wsk order)
irdl_tenor_eur_1 = 0.25  # 3M
irdl_tenor_eur_2 = 0.5   # 6M
irdl_tenor_eur_3 = 30    # 30Y
irdl_tenor_gbp_1 = 0.25  # 3M


# ============================================================
# 4) INTRA-BUCKET CORRELATION (rho_kl)
# ============================================================
# ρ_kl = ρ_tenor × ρ_curve
#
# Where:
#   ρ_tenor = max( exp(-θ × |T_k - T_l| / min(T_k, T_l)), 0.4 )
#   ρ_curve = 1.0 if same curve, else 0.999

IRDL_THETA = 0.03  # Tenor correlation parameter

# EUR position pairs
# Position 1 vs Position 2: OIS 3M vs OIS 6M (same curve, different tenor)
irdl_rho_tenor_12 = max(np.exp(-IRDL_THETA * abs(irdl_tenor_eur_1 - irdl_tenor_eur_2) / min(irdl_tenor_eur_1, irdl_tenor_eur_2)), 0.4)
irdl_rho_curve_12 = 1.0  # same curve (OIS)
irdl_rho_12 = irdl_rho_tenor_12 * irdl_rho_curve_12

# Position 1 vs Position 3: OIS 3M vs SOV 30Y (different curve, different tenor)
irdl_rho_tenor_13 = max(np.exp(-IRDL_THETA * abs(irdl_tenor_eur_1 - irdl_tenor_eur_3) / min(irdl_tenor_eur_1, irdl_tenor_eur_3)), 0.4)
irdl_rho_curve_13 = 0.999  # different curve
irdl_rho_13 = irdl_rho_tenor_13 * irdl_rho_curve_13

# Position 2 vs Position 3: OIS 6M vs SOV 30Y (different curve, different tenor)
irdl_rho_tenor_23 = max(np.exp(-IRDL_THETA * abs(irdl_tenor_eur_2 - irdl_tenor_eur_3) / min(irdl_tenor_eur_2, irdl_tenor_eur_3)), 0.4)
irdl_rho_curve_23 = 0.999  # different curve
irdl_rho_23 = irdl_rho_tenor_23 * irdl_rho_curve_23

# GBP: single risk factor, no intra-bucket correlation needed
irdl_rho_gbp = None


# ============================================================
# 5) INTRA-BUCKET AGGREGATION (K_b)
# ============================================================
# K_b = √( Σ WS_k² + Σ Σ ρ_kl × WS_k × WS_l )  for k ≠ l

# EUR bucket: 3 risk factors
irdl_sum_wsk_sq_eur = irdl_wsk_eur_1**2 + irdl_wsk_eur_2**2 + irdl_wsk_eur_3**2
irdl_cross_term_eur = (
    2 * irdl_rho_12 * irdl_wsk_eur_1 * irdl_wsk_eur_2
    + 2 * irdl_rho_13 * irdl_wsk_eur_1 * irdl_wsk_eur_3
    + 2 * irdl_rho_23 * irdl_wsk_eur_2 * irdl_wsk_eur_3
)
irdl_K_eur = np.sqrt(max(0, irdl_sum_wsk_sq_eur + irdl_cross_term_eur))

# GBP bucket: 1 risk factor → K_b = |WS_k|
irdl_K_gbp = abs(irdl_wsk_gbp_1)


# ============================================================
# 6) BUCKET SUMS (S_b)
# ============================================================
# S_b = Σ WS_k  (simple sum of weighted sensitivities in bucket)

irdl_S_eur = irdl_wsk[irdl_wsk['bucket'] == 'eur']['wsk'].sum()
irdl_S_gbp = irdl_wsk[irdl_wsk['bucket'] == 'gbp']['wsk'].sum()


# ============================================================
# 7) CROSS-BUCKET CORRELATION (gamma_bc)
# ============================================================
# γ = 0.5 for all GIRR bucket pairs

IRDL_GAMMA = 0.5
irdl_gamma = IRDL_GAMMA


# ============================================================
# 8) CROSS-BUCKET AGGREGATION
# ============================================================
# K = √( Σ K_b² + Σ Σ γ_bc × S_b × S_c )  for b ≠ c

irdl_sum_K_sq = irdl_K_eur**2 + irdl_K_gbp**2
irdl_cross_bucket = 2 * irdl_gamma * irdl_S_eur * irdl_S_gbp

irdl_k_squared = irdl_sum_K_sq + irdl_cross_bucket

# Apply the check at the overall level (Floor at 0)
irdl_k_squared = max(irdl_k_squared, 0)

irdl_K_medium = np.sqrt(irdl_k_squared)


# ============================================================
# 9) CORRELATION SCENARIOS (HIGH/LOW)
# ============================================================
# High: ρ_high = min(1.25 × ρ_medium, 1.0)
# Low:  ρ_low  = max(2 × ρ_medium - 1, 0.75 × ρ_medium)

# --- High scenario correlations --- #
irdl_rho_12_high = min(irdl_rho_12 * 1.25, 1.0)
irdl_rho_13_high = min(irdl_rho_13 * 1.25, 1.0)
irdl_rho_23_high = min(irdl_rho_23 * 1.25, 1.0)
irdl_gamma_high = min(irdl_gamma * 1.25, 1.0)

# --- Low scenario correlations --- #
irdl_rho_12_low = max(2 * irdl_rho_12 - 1.0, 0.75 * irdl_rho_12)
irdl_rho_13_low = max(2 * irdl_rho_13 - 1.0, 0.75 * irdl_rho_13)
irdl_rho_23_low = max(2 * irdl_rho_23 - 1.0, 0.75 * irdl_rho_23)
irdl_gamma_low = max(2 * irdl_gamma - 1.0, 0.75 * irdl_gamma)

# --- High scenario intra-bucket K_b --- #
irdl_cross_term_eur_high = (
    2 * irdl_rho_12_high * irdl_wsk_eur_1 * irdl_wsk_eur_2
    + 2 * irdl_rho_13_high * irdl_wsk_eur_1 * irdl_wsk_eur_3
    + 2 * irdl_rho_23_high * irdl_wsk_eur_2 * irdl_wsk_eur_3
)
irdl_K_eur_high = np.sqrt(max(0, irdl_sum_wsk_sq_eur + irdl_cross_term_eur_high))
irdl_K_gbp_high = irdl_K_gbp  # Single RF, unchanged

# --- Low scenario intra-bucket K_b --- #
irdl_cross_term_eur_low = (
    2 * irdl_rho_12_low * irdl_wsk_eur_1 * irdl_wsk_eur_2
    + 2 * irdl_rho_13_low * irdl_wsk_eur_1 * irdl_wsk_eur_3
    + 2 * irdl_rho_23_low * irdl_wsk_eur_2 * irdl_wsk_eur_3
)
irdl_K_eur_low = np.sqrt(max(0, irdl_sum_wsk_sq_eur + irdl_cross_term_eur_low))
irdl_K_gbp_low = irdl_K_gbp  # Single RF, unchanged

# --- High scenario cross-bucket --- #
irdl_k_sq_high = (
    irdl_K_eur_high**2 + irdl_K_gbp_high**2
    + 2 * irdl_gamma_high * irdl_S_eur * irdl_S_gbp
)
irdl_K_high = np.sqrt(max(irdl_k_sq_high, 0))

# --- Low scenario cross-bucket --- #
irdl_k_sq_low = (
    irdl_K_eur_low**2 + irdl_K_gbp_low**2
    + 2 * irdl_gamma_low * irdl_S_eur * irdl_S_gbp
)
irdl_K_low = np.sqrt(max(irdl_k_sq_low, 0))


# ============================================================
# 10) CAPITAL
# ============================================================
# Final capital = max(K_medium, K_high, K_low)

irdl_capital = max(irdl_K_medium, irdl_K_high, irdl_K_low)


# ============================================================
# 11) PRINTING
# ============================================================

# --- 11.1 Gross sensitivities --- #
irdl_gross_sk_print = format_table(
    irdl_gross_sk,
    columns=['trade_id', 'bucket', 'rfr', 'maturity', 'gross_sk'],
    col_formats={'gross_sk': ',.0f'},
    alignments=('left', 'left', 'left', 'left', 'right')
)

# --- 11.2 Net sensitivities (s_k) --- #
irdl_net_sk_print = format_table(
    irdl_net_sk,
    columns=['bucket', 'rfr', 'maturity', 'sk'],
    col_formats={'sk': ',.0f'},
    alignments=('left', 'left', 'left', 'right')
)

# --- 11.3 Weighted sensitivities (WS_k) --- #
irdl_wsk_print = format_table(
    irdl_wsk,
    columns=['bucket', 'rfr', 'maturity', 'sk', 'rw', 'wsk'],
    col_formats={'sk': ',.0f', 'rw': '.4f', 'wsk': ',.2f'},
    alignments=('left', 'left', 'left', 'right', 'right', 'right')
)

# --- 11.4 Intra-bucket correlation (rho_kl) --- #
# Standardized column name to 'rho_kl'
irdl_rho_print = format_table(
    pd.DataFrame([
        {'bucket': 'eur', 'pair': '1 vs 2', 'desc': 'OIS 3M vs OIS 6M', 'rho_tenor': irdl_rho_tenor_12, 'rho_curve': irdl_rho_curve_12, 'rho_kl': irdl_rho_12},
        {'bucket': 'eur', 'pair': '1 vs 3', 'desc': 'OIS 3M vs SOV 30Y', 'rho_tenor': irdl_rho_tenor_13, 'rho_curve': irdl_rho_curve_13, 'rho_kl': irdl_rho_13},
        {'bucket': 'eur', 'pair': '2 vs 3', 'desc': 'OIS 6M vs SOV 30Y', 'rho_tenor': irdl_rho_tenor_23, 'rho_curve': irdl_rho_curve_23, 'rho_kl': irdl_rho_23},
        {'bucket': 'gbp', 'pair': 'n/a', 'desc': 'single RF', 'rho_tenor': 'n/a', 'rho_curve': 'n/a', 'rho_kl': 'n/a'},
    ]),
    columns=['bucket', 'pair', 'desc', 'rho_tenor', 'rho_curve', 'rho_kl'],
    col_formats={'rho_tenor': '.4f', 'rho_curve': '.3f', 'rho_kl': '.4f'},
    alignments=('left', 'left', 'left', 'right', 'right', 'right')
)

# --- 11.5 Intra-bucket aggregation (K_b) --- #
# EUR bucket (3 risk factors) - order after groupby is: OIS 3M, OIS 6M, SOV 30Y
irdl_K_b_eur_print = format_table(
    pd.DataFrame([
        {'component': 'wsk_1 (OIS 3M)', 'value': irdl_wsk_eur_1},
        {'component': 'wsk_2 (OIS 6M)', 'value': irdl_wsk_eur_2},
        {'component': 'wsk_3 (SOV 30Y)', 'value': irdl_wsk_eur_3},
        {'component': 'rho_12', 'value': f"{irdl_rho_12:.4f}"},
        {'component': 'rho_13', 'value': f"{irdl_rho_13:.4f}"},
        {'component': 'rho_23', 'value': f"{irdl_rho_23:.4f}"},
        {'component': 'K_b', 'value': irdl_K_eur},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# GBP bucket (1 risk factor)
irdl_K_b_gbp_print = format_table(
    pd.DataFrame([
        {'component': 'wsk_1 (OIS 3M)', 'value': irdl_wsk_gbp_1},
        {'component': 'K_b', 'value': irdl_K_gbp},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# --- 11.6 Bucket sums (S_b) --- #
# Standardized columns to WS_k_1, WS_k_2, etc.
irdl_S_b_print = format_table(
    pd.DataFrame([
        {'bucket': 'eur', 'WS_k_1': irdl_wsk_eur_1, 'WS_k_2': irdl_wsk_eur_2, 'WS_k_3': irdl_wsk_eur_3, 'S_b': irdl_S_eur},
        {'bucket': 'gbp', 'WS_k_1': irdl_wsk_gbp_1, 'WS_k_2': 'n/a', 'WS_k_3': 'n/a', 'S_b': irdl_S_gbp},
    ]),
    columns=['bucket', 'WS_k_1', 'WS_k_2', 'WS_k_3', 'S_b'],
    col_formats={'WS_k_1': ',.2f', 'WS_k_2': ',.2f', 'WS_k_3': ',.2f', 'S_b': ',.2f'},
    alignments=('left', 'right', 'right', 'right', 'right')
)

# --- 11.7 Cross-bucket correlation (gamma_bc) --- #
# Standardized column name to 'gamma_bc'
irdl_gamma_print = format_table(
    pd.DataFrame([
        {'buckets': 'EUR vs GBP', 'gamma_bc': irdl_gamma},
    ]),
    columns=['buckets', 'gamma_bc'],
    col_formats={'gamma_bc': '.2f'},
    alignments=('left', 'right')
)

# --- 11.8 Cross-bucket aggregation --- #
irdl_cross_agg_print = format_table(
    pd.DataFrame([
        {'component': 'K_eur', 'value': irdl_K_eur},
        {'component': 'K_gbp', 'value': irdl_K_gbp},
        {'component': 'S_eur', 'value': irdl_S_eur},
        {'component': 'S_gbp', 'value': irdl_S_gbp},
        {'component': 'gamma_bc', 'value': f"{irdl_gamma:.2f}"},
        {'component': 'capital', 'value': irdl_K_medium},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# --- 11.9 Correlation scenarios --- #
irdl_corr_scenarios_print = format_table(
    pd.DataFrame([
        {'parameter': 'rho_12', 'medium': irdl_rho_12, 'high': irdl_rho_12_high, 'low': irdl_rho_12_low},
        {'parameter': 'rho_13', 'medium': irdl_rho_13, 'high': irdl_rho_13_high, 'low': irdl_rho_13_low},
        {'parameter': 'rho_23', 'medium': irdl_rho_23, 'high': irdl_rho_23_high, 'low': irdl_rho_23_low},
        {'parameter': 'gamma', 'medium': irdl_gamma, 'high': irdl_gamma_high, 'low': irdl_gamma_low},
    ]),
    columns=['parameter', 'medium', 'high', 'low'],
    col_formats={'medium': '.4f', 'high': '.4f', 'low': '.4f'},
    alignments=('left', 'right', 'right', 'right')
)

# --- 11.10 Capital --- #
irdl_capital_print = format_table(
    pd.DataFrame([
        {'scenario': 'medium', 'capital': irdl_K_medium},
        {'scenario': 'high', 'capital': irdl_K_high},
        {'scenario': 'low', 'capital': irdl_K_low},
    ]),
    columns=['scenario', 'capital'],
    col_formats={'capital': ',.2f'},
    alignments=('left', 'right')
)


# ============================================================
# DISPLAY ALL OUTPUTS
# ============================================================

print("=" * 60)
print("IRDL")
print("=" * 60)

print("\n1) GROSS SENSITIVITIES")
print(irdl_gross_sk_print)

print("\n2) NET SENSITIVITIES (s_k)")
print(irdl_net_sk_print)

print("\n3) WEIGHTED SENSITIVITIES (WS_k)")
print(irdl_wsk_print)

print("\n4) INTRA-BUCKET CORRELATION (rho_kl)")
print(irdl_rho_print)

print("\n5) INTRA-BUCKET AGGREGATION (K_b)")
print("\n--- EUR bucket ---")
print(irdl_K_b_eur_print)
print("\n--- GBP bucket ---")
print(irdl_K_b_gbp_print)

print("\n6) BUCKET SUMS (S_b)")
print(irdl_S_b_print)

print("\n7) CROSS-BUCKET CORRELATION (gamma_bc)")
print(irdl_gamma_print)

print("\n8) CROSS-BUCKET AGGREGATION")
print(irdl_cross_agg_print)

print("\n9) CORRELATION SCENARIOS")
print(irdl_corr_scenarios_print)

print("\n10) CAPITAL")
print(irdl_capital_print)

IRDL

1) GROSS SENSITIVITIES
┌────────────┬──────────┬───────┬────────────┬────────────┐
│ trade_id   │ bucket   │ rfr   │ maturity   │   gross_sk │
├────────────┼──────────┼───────┼────────────┼────────────┤
│ 1          │ eur      │ ois   │ 6m         │     -1,300 │
│ 2          │ eur      │ ois   │ 3m         │      2,100 │
│ 3          │ eur      │ sov   │ 30y        │     -4,900 │
│ 4          │ gbp      │ ois   │ 3m         │      1,800 │
└────────────┴──────────┴───────┴────────────┴────────────┘

2) NET SENSITIVITIES (s_k)
┌──────────┬───────┬────────────┬────────┐
│ bucket   │ rfr   │ maturity   │     sk │
├──────────┼───────┼────────────┼────────┤
│ eur      │ ois   │ 3m         │  2,100 │
│ eur      │ ois   │ 6m         │ -1,300 │
│ eur      │ sov   │ 30y        │ -4,900 │
│ gbp      │ ois   │ 3m         │  1,800 │
└──────────┴───────┴────────────┴────────┘

3) WEIGHTED SENSITIVITIES (WS_k)
┌──────────┬───────┬────────────┬────────┬────────┬────────┐
│ bucket   │ rfr   │ mat

### CRDL Non Sec

In [4]:
# ============================================================
# CRDL NON-SEC
# ============================================================


# TABLE OF CONTENTS
#   1) Gross sensitivities
#   2) Net sensitivities (s_k)
#   3) Weighted sensitivities (WS_k)
#   4) Intra-bucket correlation (rho_kl)
#   5) Intra-bucket aggregation (K_b)
#   6) Bucket sums (S_b)
#   7) Cross-bucket correlation (gamma_bc)
#   8) Cross-bucket aggregation
#   9) Correlation scenarios (high/low)
#  10) Capital
#  11) Printing
# ============================================================


# ============================================================
# 1) GROSS SENSITIVITIES
# ============================================================
# Position-level delta sensitivities to credit spreads.

data = [
    {'trade_id': 1, 'bucket': '6', 'rating': 'ig', 'sector': 'tech', 'curve': 'cds',  'maturity': '6m', 'issuer': 'issuer_1', 'gross_s_k':  1800},
    {'trade_id': 2, 'bucket': '6', 'rating': 'ig', 'sector': 'tech', 'curve': 'cds',  'maturity': '6m', 'issuer': 'issuer_1', 'gross_s_k':  1300},
    {'trade_id': 3, 'bucket': '6', 'rating': 'ig', 'sector': 'tech', 'curve': 'cds',  'maturity': '6m', 'issuer': 'issuer_2', 'gross_s_k':   980},
    {'trade_id': 4, 'bucket': '6', 'rating': 'ig', 'sector': 'tech', 'curve': 'cds',  'maturity': '6m', 'issuer': 'issuer_2', 'gross_s_k':   650},
    {'trade_id': 5, 'bucket': '3', 'rating': 'ig', 'sector': 'fin',  'curve': 'bond', 'maturity': '6m', 'issuer': 'issuer_3', 'gross_s_k': -1500},
    {'trade_id': 6, 'bucket': '3', 'rating': 'ig', 'sector': 'fin',  'curve': 'cds',  'maturity': '6m', 'issuer': 'issuer_3', 'gross_s_k': -2000},
    {'trade_id': 7, 'bucket': '3', 'rating': 'ig', 'sector': 'fin',  'curve': 'cds',  'maturity': '6m', 'issuer': 'issuer_3', 'gross_s_k':  1400}
]

crdl_non_sec_gross_sk = pd.DataFrame(data)


# ============================================================
# 2) NET SENSITIVITIES (s_k)
# ============================================================
# Net sensitivity s_k = Σ gross sensitivities for each unique risk factor.
# Risk factor defined by: bucket + rating + sector + curve + issuer + maturity.

crdl_non_sec_risk_factor_cols = ['bucket', 'rating', 'sector', 'curve', 'issuer', 'maturity']

crdl_non_sec_net_sk = (
    crdl_non_sec_gross_sk
    .groupby(crdl_non_sec_risk_factor_cols)
    .agg(s_k=('gross_s_k', 'sum'))
    .reset_index()
)


# ============================================================
# 3) WEIGHTED SENSITIVITIES (WS_k)
# ============================================================
# WS_k = s_k × RW_k

crdl_non_sec_risk_weights = {
    '3': 0.05,
    '6': 0.02
}

crdl_non_sec_wsk = crdl_non_sec_net_sk.copy()
crdl_non_sec_wsk['rw'] = crdl_non_sec_wsk['bucket'].map(crdl_non_sec_risk_weights)
crdl_non_sec_wsk['WS_k'] = crdl_non_sec_wsk['s_k'] * crdl_non_sec_wsk['rw']

# Extract weighted sensitivities per bucket for aggregation
# Bucket 3: Issuer 3 (Bond vs CDS)
crdl_non_sec_ws_b3 = crdl_non_sec_wsk[crdl_non_sec_wsk['bucket'] == '3']['WS_k'].values
crdl_non_sec_WS_k_b3_bond = crdl_non_sec_ws_b3[0]  # Issuer 3, Bond
crdl_non_sec_WS_k_b3_cds  = crdl_non_sec_ws_b3[1]  # Issuer 3, CDS

# Bucket 6: Issuer 1 vs Issuer 2 (Both CDS)
crdl_non_sec_ws_b6 = crdl_non_sec_wsk[crdl_non_sec_wsk['bucket'] == '6']['WS_k'].values
crdl_non_sec_WS_k_b6_issuer1 = crdl_non_sec_ws_b6[0]  # Issuer 1, CDS
crdl_non_sec_WS_k_b6_issuer2 = crdl_non_sec_ws_b6[1]  # Issuer 2, CDS


# ============================================================
# 4) INTRA-BUCKET CORRELATION (rho_kl)
# ============================================================
# ρ_kl = ρ_name × ρ_tenor × ρ_basis
# ρ_name = 1.0 (same issuer), else 0.35 (IG)
# ρ_tenor = 1.0 (same tenor), else 0.999
# ρ_basis = 1.0 (same curve), else 0.999

# --- Bucket 3 Pair: Issuer 3 Bond vs Issuer 3 CDS ---
crdl_non_sec_rho_name_b3 = 1.0     # Same issuer
crdl_non_sec_rho_tenor_b3 = 1.0    # Same tenor (6M)
crdl_non_sec_rho_basis_b3 = 0.999  # Different curve
crdl_non_sec_rho_kl_b3 = crdl_non_sec_rho_name_b3 * crdl_non_sec_rho_tenor_b3 * crdl_non_sec_rho_basis_b3

# --- Bucket 6 Pair: Issuer 1 CDS vs Issuer 2 CDS ---
crdl_non_sec_rho_name_b6 = 0.35    # Different issuer (IG)
crdl_non_sec_rho_tenor_b6 = 1.0    # Same tenor (6M)
crdl_non_sec_rho_basis_b6 = 1.0    # Same curve
crdl_non_sec_rho_kl_b6 = crdl_non_sec_rho_name_b6 * crdl_non_sec_rho_tenor_b6 * crdl_non_sec_rho_basis_b6


# ============================================================
# 5) INTRA-BUCKET AGGREGATION (K_b)
# ============================================================
# K_b = √( Σ WS_k² + Σ Σ ρ_kl × WS_k × WS_l )  for k ≠ l

# Bucket 3: 2 risk factors
crdl_non_sec_sum_sq_b3 = crdl_non_sec_WS_k_b3_bond**2 + crdl_non_sec_WS_k_b3_cds**2
crdl_non_sec_cross_b3 = 2 * crdl_non_sec_rho_kl_b3 * crdl_non_sec_WS_k_b3_bond * crdl_non_sec_WS_k_b3_cds
crdl_non_sec_K_b3 = np.sqrt(max(0, crdl_non_sec_sum_sq_b3 + crdl_non_sec_cross_b3))

# Bucket 6: 2 risk factors
crdl_non_sec_sum_sq_b6 = crdl_non_sec_WS_k_b6_issuer1**2 + crdl_non_sec_WS_k_b6_issuer2**2
crdl_non_sec_cross_b6 = 2 * crdl_non_sec_rho_kl_b6 * crdl_non_sec_WS_k_b6_issuer1 * crdl_non_sec_WS_k_b6_issuer2
crdl_non_sec_K_b6 = np.sqrt(max(0, crdl_non_sec_sum_sq_b6 + crdl_non_sec_cross_b6))


# ============================================================
# 6) BUCKET SUMS (S_b)
# ============================================================
# S_b = Σ WS_k

crdl_non_sec_S_b3 = crdl_non_sec_wsk[crdl_non_sec_wsk['bucket'] == '3']['WS_k'].sum()
crdl_non_sec_S_b6 = crdl_non_sec_wsk[crdl_non_sec_wsk['bucket'] == '6']['WS_k'].sum()


# ============================================================
# 7) CROSS-BUCKET CORRELATION (gamma_bc)
# ============================================================
# γ = 0.20 for CSR Non-Sec bucket pairs (Article 325aj)

# Component params
crdl_non_sec_gamma_rating = 1.0
crdl_non_sec_gamma_sector = 0.20
crdl_non_sec_gamma = crdl_non_sec_gamma_rating * crdl_non_sec_gamma_sector


# ============================================================
# 8) CROSS-BUCKET AGGREGATION
# ============================================================
# K = √( Σ K_b² + Σ Σ γ_bc × S_b × S_c )  for b ≠ c

crdl_non_sec_sum_K_sq = crdl_non_sec_K_b3**2 + crdl_non_sec_K_b6**2
crdl_non_sec_cross_bucket = 2 * crdl_non_sec_gamma * crdl_non_sec_S_b3 * crdl_non_sec_S_b6

crdl_non_sec_k_squared = crdl_non_sec_sum_K_sq + crdl_non_sec_cross_bucket

# Apply the check at the overall level (Floor at 0)
crdl_non_sec_k_squared = max(crdl_non_sec_k_squared, 0)

crdl_non_sec_K_medium = np.sqrt(crdl_non_sec_k_squared)


# ============================================================
# 9) CORRELATION SCENARIOS (HIGH/LOW)
# ============================================================
# High: ρ_high = min(1.25 × ρ_medium, 1.0)
# Low:  ρ_low  = max(2 × ρ_medium - 1, 0.75 × ρ_medium)

# --- High scenario correlations --- #
crdl_non_sec_rho_b3_high = min(crdl_non_sec_rho_kl_b3 * 1.25, 1.0)
crdl_non_sec_rho_b6_high = min(crdl_non_sec_rho_kl_b6 * 1.25, 1.0)
crdl_non_sec_gamma_high = min(crdl_non_sec_gamma * 1.25, 1.0)

# --- Low scenario correlations --- #
crdl_non_sec_rho_b3_low = max(2 * crdl_non_sec_rho_kl_b3 - 1.0, 0.75 * crdl_non_sec_rho_kl_b3)
crdl_non_sec_rho_b6_low = max(2 * crdl_non_sec_rho_kl_b6 - 1.0, 0.75 * crdl_non_sec_rho_kl_b6)
crdl_non_sec_gamma_low = max(2 * crdl_non_sec_gamma - 1.0, 0.75 * crdl_non_sec_gamma)

# --- High scenario intra-bucket K_b --- #
crdl_non_sec_cross_b3_high = 2 * crdl_non_sec_rho_b3_high * crdl_non_sec_WS_k_b3_bond * crdl_non_sec_WS_k_b3_cds
crdl_non_sec_K_b3_high = np.sqrt(max(0, crdl_non_sec_sum_sq_b3 + crdl_non_sec_cross_b3_high))

crdl_non_sec_cross_b6_high = 2 * crdl_non_sec_rho_b6_high * crdl_non_sec_WS_k_b6_issuer1 * crdl_non_sec_WS_k_b6_issuer2
crdl_non_sec_K_b6_high = np.sqrt(max(0, crdl_non_sec_sum_sq_b6 + crdl_non_sec_cross_b6_high))

# --- Low scenario intra-bucket K_b --- #
crdl_non_sec_cross_b3_low = 2 * crdl_non_sec_rho_b3_low * crdl_non_sec_WS_k_b3_bond * crdl_non_sec_WS_k_b3_cds
crdl_non_sec_K_b3_low = np.sqrt(max(0, crdl_non_sec_sum_sq_b3 + crdl_non_sec_cross_b3_low))

crdl_non_sec_cross_b6_low = 2 * crdl_non_sec_rho_b6_low * crdl_non_sec_WS_k_b6_issuer1 * crdl_non_sec_WS_k_b6_issuer2
crdl_non_sec_K_b6_low = np.sqrt(max(0, crdl_non_sec_sum_sq_b6 + crdl_non_sec_cross_b6_low))

# --- High scenario cross-bucket --- #
crdl_non_sec_k_sq_high = (
    crdl_non_sec_K_b3_high**2 + crdl_non_sec_K_b6_high**2
    + 2 * crdl_non_sec_gamma_high * crdl_non_sec_S_b3 * crdl_non_sec_S_b6
)
crdl_non_sec_K_high = np.sqrt(max(crdl_non_sec_k_sq_high, 0))

# --- Low scenario cross-bucket --- #
crdl_non_sec_k_sq_low = (
    crdl_non_sec_K_b3_low**2 + crdl_non_sec_K_b6_low**2
    + 2 * crdl_non_sec_gamma_low * crdl_non_sec_S_b3 * crdl_non_sec_S_b6
)
crdl_non_sec_K_low = np.sqrt(max(crdl_non_sec_k_sq_low, 0))


# ============================================================
# 10) CAPITAL
# ============================================================
# Final capital = max(K_medium, K_high, K_low)

crdl_non_sec_capital = max(crdl_non_sec_K_medium, crdl_non_sec_K_high, crdl_non_sec_K_low)


# ============================================================
# 11) PRINTING
# ============================================================

# --- 11.1 Gross sensitivities --- #
crdl_non_sec_gross_sk_print = format_table(
    crdl_non_sec_gross_sk,
    columns=['trade_id', 'bucket', 'rating', 'sector', 'curve', 'maturity', 'issuer', 'gross_s_k'],
    col_formats={'gross_s_k': ',.0f'},
    alignments=('left', 'left', 'left', 'left', 'left', 'left', 'left', 'right')
)

# --- 11.2 Net sensitivities (s_k) --- #
crdl_non_sec_net_sk_print = format_table(
    crdl_non_sec_net_sk,
    columns=['bucket', 'rating', 'sector', 'curve', 'maturity', 'issuer', 's_k'],
    col_formats={'s_k': ',.0f'},
    alignments=('left', 'left', 'left', 'left', 'left', 'left', 'right')
)

# --- 11.3 Weighted sensitivities (WS_k) --- #
crdl_non_sec_wsk_print = format_table(
    crdl_non_sec_wsk,
    columns=['bucket', 'rating', 'sector', 'curve', 'maturity', 'issuer', 's_k', 'rw', 'WS_k'],
    col_formats={'s_k': ',.0f', 'rw': '.4f', 'WS_k': ',.2f'},
    alignments=('left', 'left', 'left', 'left', 'left', 'left', 'right', 'right', 'right')
)

# --- 11.4 Intra-bucket correlation (rho_kl) --- #
crdl_non_sec_rho_print = format_table(
    pd.DataFrame([
        {'bucket': '3', 'pair': 'Bond vs CDS', 'rho_curve': crdl_non_sec_rho_basis_b3, 'rho_maturity': crdl_non_sec_rho_tenor_b3, 'rho_issuer': crdl_non_sec_rho_name_b3, 'rho_kl': crdl_non_sec_rho_kl_b3},
        {'bucket': '6', 'pair': 'Iss1 vs Iss2', 'rho_curve': crdl_non_sec_rho_basis_b6, 'rho_maturity': crdl_non_sec_rho_tenor_b6, 'rho_issuer': crdl_non_sec_rho_name_b6, 'rho_kl': crdl_non_sec_rho_kl_b6},
    ]),
    columns=['bucket', 'pair', 'rho_curve', 'rho_maturity', 'rho_issuer', 'rho_kl'],
    col_formats={'rho_curve': '.4f', 'rho_maturity': '.4f', 'rho_issuer': '.4f', 'rho_kl': '.4f'},
    alignments=('left', 'left', 'right', 'right', 'right', 'right')
)

# --- 11.5 Intra-bucket aggregation (K_b) --- #
# Prepare Formula Strings
crdl_non_sec_Kb3_formula_str = (
    f"K_b3 = sqrt( ({crdl_non_sec_WS_k_b3_bond:.2f}^2 + {crdl_non_sec_WS_k_b3_cds:.2f}^2) "
    f"+ (2 * {crdl_non_sec_rho_kl_b3:.4f} * {crdl_non_sec_WS_k_b3_bond:.2f} * {crdl_non_sec_WS_k_b3_cds:.2f}) )"
)

crdl_non_sec_Kb6_formula_str = (
    f"K_b6 = sqrt( ({crdl_non_sec_WS_k_b6_issuer1:.2f}^2 + {crdl_non_sec_WS_k_b6_issuer2:.2f}^2) "
    f"+ (2 * {crdl_non_sec_rho_kl_b6:.4f} * {crdl_non_sec_WS_k_b6_issuer1:.2f} * {crdl_non_sec_WS_k_b6_issuer2:.2f}) )"
)

# Bucket 3 Table
crdl_non_sec_K_b3_print = format_table(
    pd.DataFrame([
        {'component': 'WS_k_1 (Iss3 Bond)', 'value': crdl_non_sec_WS_k_b3_bond},
        {'component': 'WS_k_2 (Iss3 CDS)', 'value': crdl_non_sec_WS_k_b3_cds},
        {'component': 'rho_kl_12', 'value': f"{crdl_non_sec_rho_kl_b3:.4f}"},
        {'component': 'K_b', 'value': crdl_non_sec_K_b3},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# Bucket 6 Table
crdl_non_sec_K_b6_print = format_table(
    pd.DataFrame([
        {'component': 'WS_k_1 (Iss1 CDS)', 'value': crdl_non_sec_WS_k_b6_issuer1},
        {'component': 'WS_k_2 (Iss2 CDS)', 'value': crdl_non_sec_WS_k_b6_issuer2},
        {'component': 'rho_kl_12', 'value': f"{crdl_non_sec_rho_kl_b6:.4f}"},
        {'component': 'K_b', 'value': crdl_non_sec_K_b6},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# --- 11.6 Bucket sums (S_b) --- #
crdl_non_sec_S_b_print = format_table(
    pd.DataFrame([
        {'bucket': '3', 'WS_k_1': crdl_non_sec_WS_k_b3_bond, 'WS_k_2': crdl_non_sec_WS_k_b3_cds, 'S_b': crdl_non_sec_S_b3},
        {'bucket': '6', 'WS_k_1': crdl_non_sec_WS_k_b6_issuer1, 'WS_k_2': crdl_non_sec_WS_k_b6_issuer2, 'S_b': crdl_non_sec_S_b6},
    ]),
    columns=['bucket', 'WS_k_1', 'WS_k_2', 'S_b'],
    col_formats={'WS_k_1': ',.2f', 'WS_k_2': ',.2f', 'S_b': ',.2f'},
    alignments=('left', 'right', 'right', 'right')
)

# --- 11.7 Cross-bucket correlation (gamma_bc) --- #
crdl_non_sec_gamma_print = format_table(
    pd.DataFrame([
        {'buckets': '3 vs 6', 'rating': crdl_non_sec_gamma_rating, 'sector': crdl_non_sec_gamma_sector, 'gamma_bc': crdl_non_sec_gamma},
    ]),
    columns=['buckets', 'rating', 'sector', 'gamma_bc'],
    col_formats={'rating': '.2f', 'sector': '.2f', 'gamma_bc': '.2f'},
    alignments=('left', 'right', 'right', 'right')
)

# --- 11.8 Cross-bucket aggregation --- #
# Prepare Formula String
crdl_non_sec_K_formula_str = (
    f"Capital = sqrt( ({crdl_non_sec_K_b3:.2f}^2 + {crdl_non_sec_K_b6:.2f}^2) "
    f"+ (2 * {crdl_non_sec_gamma:.2f} * {crdl_non_sec_S_b3:.2f} * {crdl_non_sec_S_b6:.2f}) )"
)

crdl_non_sec_cross_agg_print = format_table(
    pd.DataFrame([
        {'component': 'K_b3', 'value': crdl_non_sec_K_b3},
        {'component': 'K_b6', 'value': crdl_non_sec_K_b6},
        {'component': 'S_b3', 'value': crdl_non_sec_S_b3},
        {'component': 'S_b6', 'value': crdl_non_sec_S_b6},
        {'component': 'gamma_bc', 'value': f"{crdl_non_sec_gamma:.2f}"},
        {'component': 'capital', 'value': crdl_non_sec_K_medium},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# --- 11.9 Correlation scenarios --- #
crdl_non_sec_scenarios_print = format_table(
    pd.DataFrame([
        {'parameter': 'rho_kl_b3', 'medium': crdl_non_sec_rho_kl_b3, 'high': crdl_non_sec_rho_b3_high, 'low': crdl_non_sec_rho_b3_low},
        {'parameter': 'rho_kl_b6', 'medium': crdl_non_sec_rho_kl_b6, 'high': crdl_non_sec_rho_b6_high, 'low': crdl_non_sec_rho_b6_low},
        {'parameter': 'gamma_bc', 'medium': crdl_non_sec_gamma, 'high': crdl_non_sec_gamma_high, 'low': crdl_non_sec_gamma_low},
    ]),
    columns=['parameter', 'medium', 'high', 'low'],
    col_formats={'medium': '.4f', 'high': '.4f', 'low': '.4f'},
    alignments=('left', 'right', 'right', 'right')
)

# --- 11.10 Capital --- #
crdl_non_sec_capital_print = format_table(
    pd.DataFrame([
        {'scenario': 'medium', 'capital': crdl_non_sec_K_medium},
        {'scenario': 'high', 'capital': crdl_non_sec_K_high},
        {'scenario': 'low', 'capital': crdl_non_sec_K_low},
    ]),
    columns=['scenario', 'capital'],
    col_formats={'capital': ',.2f'},
    alignments=('left', 'right')
)


# ============================================================
# DISPLAY ALL OUTPUTS
# ============================================================

print("=" * 60)
print("CRDL NON-SECURITISATION")
print("=" * 60)

print("\n1) GROSS SENSITIVITIES")
print(crdl_non_sec_gross_sk_print)

print("\n2) NET SENSITIVITIES (s_k)")
print(crdl_non_sec_net_sk_print)

print("\n3) WEIGHTED SENSITIVITIES (WS_k)")
print(crdl_non_sec_wsk_print)

print("\n4) INTRA-BUCKET CORRELATION (rho_kl)")
print(crdl_non_sec_rho_print)

print("\n5) INTRA-BUCKET AGGREGATION (K_b)")
print("\n--- Bucket 3 ---")
print(crdl_non_sec_Kb3_formula_str)
print(crdl_non_sec_K_b3_print)
print("\n--- Bucket 6 ---")
print(crdl_non_sec_Kb6_formula_str)
print(crdl_non_sec_K_b6_print)

print("\n6) BUCKET SUMS (S_b)")
print(crdl_non_sec_S_b_print)

print("\n7) CROSS-BUCKET CORRELATION (gamma_bc)")
print(crdl_non_sec_gamma_print)

print("\n8) CROSS-BUCKET AGGREGATION")
print(crdl_non_sec_K_formula_str)
print(crdl_non_sec_cross_agg_print)

print("\n9) CORRELATION SCENARIOS")
print(crdl_non_sec_scenarios_print)

print("\n10) CAPITAL")
print(crdl_non_sec_capital_print)

CRDL NON-SECURITISATION

1) GROSS SENSITIVITIES
┌────────────┬──────────┬──────────┬──────────┬─────────┬────────────┬──────────┬─────────────┐
│ trade_id   │ bucket   │ rating   │ sector   │ curve   │ maturity   │ issuer   │   gross_s_k │
├────────────┼──────────┼──────────┼──────────┼─────────┼────────────┼──────────┼─────────────┤
│ 1          │ 6        │ ig       │ tech     │ cds     │ 6m         │ issuer_1 │       1,800 │
│ 2          │ 6        │ ig       │ tech     │ cds     │ 6m         │ issuer_1 │       1,300 │
│ 3          │ 6        │ ig       │ tech     │ cds     │ 6m         │ issuer_2 │         980 │
│ 4          │ 6        │ ig       │ tech     │ cds     │ 6m         │ issuer_2 │         650 │
│ 5          │ 3        │ ig       │ fin      │ bond    │ 6m         │ issuer_3 │      -1,500 │
│ 6          │ 3        │ ig       │ fin      │ cds     │ 6m         │ issuer_3 │      -2,000 │
│ 7          │ 3        │ ig       │ fin      │ cds     │ 6m         │ issuer_3 │       

### CRDL Sec CTP

In [5]:
# ============================================================
# CRDL SEC CTP
# ============================================================


# TABLE OF CONTENTS
#   1) Gross sensitivities
#   2) Net sensitivities (s_k)
#   3) Weighted sensitivities (WS_k)
#   4) Intra-bucket correlation (rho_kl)
#   5) Intra-bucket aggregation (K_b)
#   6) Bucket sums (S_b)
#   7) Cross-bucket correlation (gamma_bc)
#   8) Cross-bucket aggregation
#   9) Correlation scenarios (high/low)
#  10) Capital
#  11) Printing
# ============================================================


# ============================================================
# 1) GROSS SENSITIVITIES
# ============================================================
# Position-level delta sensitivities to credit spreads.

data = [
    {'trade_id': 1, 'bucket': '13', 'rating': 'nig', 'sector': 'cons', 'curve': 'cds',  'maturity': '6m', 'issuer_underlying': 'issuer_1', 'gross_s_k':  2300},
    {'trade_id': 2, 'bucket': '13', 'rating': 'nig', 'sector': 'cons', 'curve': 'cds',  'maturity': '1y', 'issuer_underlying': 'issuer_1', 'gross_s_k': -1800},
    {'trade_id': 3, 'bucket': '13', 'rating': 'nig', 'sector': 'cons', 'curve': 'cds',  'maturity': '3y', 'issuer_underlying': 'issuer_1', 'gross_s_k': -5500}
]

crdl_sec_ctp_gross_sk = pd.DataFrame(data)


# ============================================================
# 2) NET SENSITIVITIES (s_k)
# ============================================================
# Net sensitivity s_k = Σ gross sensitivities for each unique risk factor.
# Risk factor defined by: bucket + rating + sector + curve + issuer_underlying + maturity.

crdl_sec_ctp_risk_factor_cols = ['bucket', 'rating', 'sector', 'curve', 'issuer_underlying', 'maturity']

crdl_sec_ctp_net_sk = (
    crdl_sec_ctp_gross_sk
    .groupby(crdl_sec_ctp_risk_factor_cols)
    .agg(s_k=('gross_s_k', 'sum'))
    .reset_index()
)


# ============================================================
# 3) WEIGHTED SENSITIVITIES (WS_k)
# ============================================================
# WS_k = s_k × RW_k

crdl_sec_ctp_risk_weights = {
    '13': 0.12
}

crdl_sec_ctp_wsk = crdl_sec_ctp_net_sk.copy()
crdl_sec_ctp_wsk['rw'] = crdl_sec_ctp_wsk['bucket'].map(crdl_sec_ctp_risk_weights)
crdl_sec_ctp_wsk['WS_k'] = crdl_sec_ctp_wsk['s_k'] * crdl_sec_ctp_wsk['rw']

# Extract weighted sensitivities per bucket for aggregation
# Bucket 13: 3 risk factors with different tenors (6M, 1Y, 3Y)
crdl_sec_ctp_ws_b13 = crdl_sec_ctp_wsk[crdl_sec_ctp_wsk['bucket'] == '13']['WS_k'].values
crdl_sec_ctp_WS_k_b13_6m = crdl_sec_ctp_ws_b13[0]  # 6M tenor (Position 1)
crdl_sec_ctp_WS_k_b13_1y = crdl_sec_ctp_ws_b13[1]  # 1Y tenor (Position 2)
crdl_sec_ctp_WS_k_b13_3y = crdl_sec_ctp_ws_b13[2]  # 3Y tenor (Position 3)


# ============================================================
# 4) INTRA-BUCKET CORRELATION (rho_kl)
# ============================================================
# ρ_kl = ρ_name × ρ_tenor × ρ_basis
# ρ_name = 1.0 (same issuer)
# ρ_tenor = 0.65 (different maturities for CTP)
# ρ_basis = 1.0 (same curve)

# Common parameters for this bucket (same issuer/curve, different tenors)
crdl_sec_ctp_rho_name_b13 = 1.0
crdl_sec_ctp_rho_tenor_b13 = 0.65
crdl_sec_ctp_rho_basis_b13 = 1.0

# Explicit calculation for each pair
# Pair 1 vs 2: 6M vs 1Y
crdl_sec_ctp_rho_12 = crdl_sec_ctp_rho_name_b13 * crdl_sec_ctp_rho_tenor_b13 * crdl_sec_ctp_rho_basis_b13

# Pair 1 vs 3: 6M vs 3Y
crdl_sec_ctp_rho_13 = crdl_sec_ctp_rho_name_b13 * crdl_sec_ctp_rho_tenor_b13 * crdl_sec_ctp_rho_basis_b13

# Pair 2 vs 3: 1Y vs 3Y
crdl_sec_ctp_rho_23 = crdl_sec_ctp_rho_name_b13 * crdl_sec_ctp_rho_tenor_b13 * crdl_sec_ctp_rho_basis_b13


# ============================================================
# 5) INTRA-BUCKET AGGREGATION (K_b)
# ============================================================
# K_b = √( Σ WS_k² + Σ Σ ρ_kl × WS_k × WS_l )  for k ≠ l

# Bucket 13: 3 risk factors
crdl_sec_ctp_sum_sq_b13 = crdl_sec_ctp_WS_k_b13_6m**2 + crdl_sec_ctp_WS_k_b13_1y**2 + crdl_sec_ctp_WS_k_b13_3y**2
crdl_sec_ctp_cross_b13 = (
    2 * crdl_sec_ctp_rho_12 * crdl_sec_ctp_WS_k_b13_6m * crdl_sec_ctp_WS_k_b13_1y
    + 2 * crdl_sec_ctp_rho_13 * crdl_sec_ctp_WS_k_b13_6m * crdl_sec_ctp_WS_k_b13_3y
    + 2 * crdl_sec_ctp_rho_23 * crdl_sec_ctp_WS_k_b13_1y * crdl_sec_ctp_WS_k_b13_3y
)
crdl_sec_ctp_K_b13 = np.sqrt(max(0, crdl_sec_ctp_sum_sq_b13 + crdl_sec_ctp_cross_b13))


# ============================================================
# 6) BUCKET SUMS (S_b)
# ============================================================
# S_b = Σ WS_k

crdl_sec_ctp_S_b13 = crdl_sec_ctp_wsk[crdl_sec_ctp_wsk['bucket'] == '13']['WS_k'].sum()


# ============================================================
# 7) CROSS-BUCKET CORRELATION (gamma_bc)
# ============================================================
# For single-bucket portfolios, cross-bucket correlation is not applicable.

crdl_sec_ctp_gamma = 0.0  # Not applicable for single bucket


# ============================================================
# 8) CROSS-BUCKET AGGREGATION
# ============================================================
# K = √( Σ K_b² + Σ Σ γ_bc × S_b × S_c )  for b ≠ c

# Standard structure: Sum of Squares + Cross Terms
crdl_sec_ctp_sum_K_sq = crdl_sec_ctp_K_b13**2
crdl_sec_ctp_cross_bucket = 0  # No other buckets

crdl_sec_ctp_k_squared = crdl_sec_ctp_sum_K_sq + crdl_sec_ctp_cross_bucket

# Apply the check at the overall level (Floor at 0)
crdl_sec_ctp_k_squared = max(crdl_sec_ctp_k_squared, 0)

crdl_sec_ctp_K_medium = np.sqrt(crdl_sec_ctp_k_squared)


# ============================================================
# 9) CORRELATION SCENARIOS (HIGH/LOW)
# ============================================================
# High: ρ_high = min(1.25 × ρ_medium, 1.0)
# Low:  ρ_low  = max(2 × ρ_medium - 1, 0.75 × ρ_medium)

# --- High scenario correlations --- #
crdl_sec_ctp_rho_12_high = min(crdl_sec_ctp_rho_12 * 1.25, 1.0)
crdl_sec_ctp_rho_13_high = min(crdl_sec_ctp_rho_13 * 1.25, 1.0)
crdl_sec_ctp_rho_23_high = min(crdl_sec_ctp_rho_23 * 1.25, 1.0)
crdl_sec_ctp_gamma_high = min(crdl_sec_ctp_gamma * 1.25, 1.0)

# --- Low scenario correlations --- #
crdl_sec_ctp_rho_12_low = max(2 * crdl_sec_ctp_rho_12 - 1.0, 0.75 * crdl_sec_ctp_rho_12)
crdl_sec_ctp_rho_13_low = max(2 * crdl_sec_ctp_rho_13 - 1.0, 0.75 * crdl_sec_ctp_rho_13)
crdl_sec_ctp_rho_23_low = max(2 * crdl_sec_ctp_rho_23 - 1.0, 0.75 * crdl_sec_ctp_rho_23)
crdl_sec_ctp_gamma_low = max(2 * crdl_sec_ctp_gamma - 1.0, 0.75 * crdl_sec_ctp_gamma)

# --- High scenario intra-bucket K_b --- #
crdl_sec_ctp_cross_b13_high = (
    2 * crdl_sec_ctp_rho_12_high * crdl_sec_ctp_WS_k_b13_6m * crdl_sec_ctp_WS_k_b13_1y
    + 2 * crdl_sec_ctp_rho_13_high * crdl_sec_ctp_WS_k_b13_6m * crdl_sec_ctp_WS_k_b13_3y
    + 2 * crdl_sec_ctp_rho_23_high * crdl_sec_ctp_WS_k_b13_1y * crdl_sec_ctp_WS_k_b13_3y
)
crdl_sec_ctp_K_b13_high = np.sqrt(max(0, crdl_sec_ctp_sum_sq_b13 + crdl_sec_ctp_cross_b13_high))

# --- Low scenario intra-bucket K_b --- #
crdl_sec_ctp_cross_b13_low = (
    2 * crdl_sec_ctp_rho_12_low * crdl_sec_ctp_WS_k_b13_6m * crdl_sec_ctp_WS_k_b13_1y
    + 2 * crdl_sec_ctp_rho_13_low * crdl_sec_ctp_WS_k_b13_6m * crdl_sec_ctp_WS_k_b13_3y
    + 2 * crdl_sec_ctp_rho_23_low * crdl_sec_ctp_WS_k_b13_1y * crdl_sec_ctp_WS_k_b13_3y
)
crdl_sec_ctp_K_b13_low = np.sqrt(max(0, crdl_sec_ctp_sum_sq_b13 + crdl_sec_ctp_cross_b13_low))

# --- High scenario cross-bucket --- #
# Standardized aggregation logic
crdl_sec_ctp_k_sq_high = (
    crdl_sec_ctp_K_b13_high**2
    + 2 * crdl_sec_ctp_gamma_high * 0  # No cross-terms
)
crdl_sec_ctp_k_high = np.sqrt(max(crdl_sec_ctp_k_sq_high, 0))

# --- Low scenario cross-bucket --- #
# Standardized aggregation logic
crdl_sec_ctp_k_sq_low = (
    crdl_sec_ctp_K_b13_low**2
    + 2 * crdl_sec_ctp_gamma_low * 0  # No cross-terms
)
crdl_sec_ctp_k_low = np.sqrt(max(crdl_sec_ctp_k_sq_low, 0))


# ============================================================
# 10) CAPITAL
# ============================================================
# Final capital = max(K_medium, K_high, K_low)

crdl_sec_ctp_capital = max(crdl_sec_ctp_K_medium, crdl_sec_ctp_k_high, crdl_sec_ctp_k_low)


# ============================================================
# 11) PRINTING
# ============================================================

# --- 11.1 Gross sensitivities --- #
crdl_sec_ctp_gross_sk_print = format_table(
    crdl_sec_ctp_gross_sk,
    columns=['trade_id', 'bucket', 'rating', 'sector', 'curve', 'maturity', 'issuer_underlying', 'gross_s_k'],
    col_formats={'gross_s_k': ',.0f'},
    alignments=('left', 'left', 'left', 'left', 'left', 'left', 'left', 'right')
)

# --- 11.2 Net sensitivities (s_k) --- #
crdl_sec_ctp_net_sk_print = format_table(
    crdl_sec_ctp_net_sk,
    columns=['bucket', 'rating', 'sector', 'curve', 'maturity', 'issuer_underlying', 's_k'],
    col_formats={'s_k': ',.0f'},
    alignments=('left', 'left', 'left', 'left', 'left', 'left', 'right')
)

# --- 11.3 Weighted sensitivities (WS_k) --- #
crdl_sec_ctp_wsk_print = format_table(
    crdl_sec_ctp_wsk,
    columns=['bucket', 'rating', 'sector', 'curve', 'maturity', 'issuer_underlying', 's_k', 'rw', 'WS_k'],
    col_formats={'s_k': ',.0f', 'rw': '.4f', 'WS_k': ',.2f'},
    alignments=('left', 'left', 'left', 'left', 'left', 'left', 'right', 'right', 'right')
)

# --- 11.4 Intra-bucket correlation (rho_kl) --- #
crdl_sec_ctp_rho_print = format_table(
    pd.DataFrame([
        {'bucket': '13', 'pair': '6m vs 1y', 'rho_curve': crdl_sec_ctp_rho_basis_b13, 'rho_maturity': crdl_sec_ctp_rho_tenor_b13, 'rho_issuer_underlying': crdl_sec_ctp_rho_name_b13, 'rho_kl': crdl_sec_ctp_rho_12},
        {'bucket': '13', 'pair': '6m vs 3y', 'rho_curve': crdl_sec_ctp_rho_basis_b13, 'rho_maturity': crdl_sec_ctp_rho_tenor_b13, 'rho_issuer_underlying': crdl_sec_ctp_rho_name_b13, 'rho_kl': crdl_sec_ctp_rho_13},
        {'bucket': '13', 'pair': '1y vs 3y', 'rho_curve': crdl_sec_ctp_rho_basis_b13, 'rho_maturity': crdl_sec_ctp_rho_tenor_b13, 'rho_issuer_underlying': crdl_sec_ctp_rho_name_b13, 'rho_kl': crdl_sec_ctp_rho_23},
    ]),
    columns=['bucket', 'pair', 'rho_curve', 'rho_maturity', 'rho_issuer_underlying', 'rho_kl'],
    col_formats={'rho_curve': '.4f', 'rho_maturity': '.4f', 'rho_issuer_underlying': '.4f', 'rho_kl': '.4f'},
    alignments=('left', 'left', 'right', 'right', 'right', 'right')
)

# --- 11.5 Intra-bucket aggregation (K_b) --- #
# Create formula string
crdl_sec_ctp_Kb_formula_str = (
    f"K_b = sqrt( ({crdl_sec_ctp_WS_k_b13_6m:.2f}^2 + {crdl_sec_ctp_WS_k_b13_1y:.2f}^2 + {crdl_sec_ctp_WS_k_b13_3y:.2f}^2) "
    f"+ (2 * {crdl_sec_ctp_rho_12:.4f} * {crdl_sec_ctp_WS_k_b13_6m:.2f} * {crdl_sec_ctp_WS_k_b13_1y:.2f}) "
    f"+ (2 * {crdl_sec_ctp_rho_13:.4f} * {crdl_sec_ctp_WS_k_b13_6m:.2f} * {crdl_sec_ctp_WS_k_b13_3y:.2f}) "
    f"+ (2 * {crdl_sec_ctp_rho_23:.4f} * {crdl_sec_ctp_WS_k_b13_1y:.2f} * {crdl_sec_ctp_WS_k_b13_3y:.2f}) )"
)

# Create table
crdl_sec_ctp_K_b13_print = format_table(
    pd.DataFrame([
        {'component': 'WS_k_1 (6m)', 'value': crdl_sec_ctp_WS_k_b13_6m},
        {'component': 'WS_k_2 (1y)', 'value': crdl_sec_ctp_WS_k_b13_1y},
        {'component': 'WS_k_3 (3y)', 'value': crdl_sec_ctp_WS_k_b13_3y},
        {'component': 'rho_kl_12', 'value': f"{crdl_sec_ctp_rho_12:.4f}"},
        {'component': 'rho_kl_13', 'value': f"{crdl_sec_ctp_rho_13:.4f}"},
        {'component': 'rho_kl_23', 'value': f"{crdl_sec_ctp_rho_23:.4f}"},
        {'component': 'K_b', 'value': crdl_sec_ctp_K_b13},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# --- 11.6 Bucket sums (S_b) --- #
crdl_sec_ctp_S_b_print = format_table(
    pd.DataFrame([
        {'bucket': '13', 'WS_k_1': crdl_sec_ctp_WS_k_b13_6m, 'WS_k_2': crdl_sec_ctp_WS_k_b13_1y, 'WS_k_3': crdl_sec_ctp_WS_k_b13_3y, 'S_b': crdl_sec_ctp_S_b13},
    ]),
    columns=['bucket', 'WS_k_1', 'WS_k_2', 'WS_k_3', 'S_b'],
    col_formats={'WS_k_1': ',.2f', 'WS_k_2': ',.2f', 'WS_k_3': ',.2f', 'S_b': ',.2f'},
    alignments=('left', 'right', 'right', 'right', 'right')
)

# --- 11.7 Cross-bucket correlation (gamma_bc) --- #
crdl_sec_ctp_gamma_print = format_table(
    pd.DataFrame([
        {'buckets': 'n/a (single)', 'gamma_bc': crdl_sec_ctp_gamma},
    ]),
    columns=['buckets', 'gamma_bc'],
    col_formats={'gamma_bc': '.2f'},
    alignments=('left', 'right')
)

# --- 11.8 Cross-bucket aggregation --- #
# Create formula string - Expanded to show components
crdl_sec_ctp_K_formula_str = (
    f"Capital = sqrt( {crdl_sec_ctp_K_b13:.2f}^2 + 0 )"
)

# Create table
crdl_sec_ctp_cross_agg_print = format_table(
    pd.DataFrame([
        {'component': 'K_b13', 'value': crdl_sec_ctp_K_b13},
        {'component': 'S_b13', 'value': crdl_sec_ctp_S_b13},
        {'component': 'gamma_bc', 'value': f"{crdl_sec_ctp_gamma:.2f}"},
        {'component': 'capital', 'value': crdl_sec_ctp_K_medium},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# --- 11.9 Correlation scenarios --- #
crdl_sec_ctp_scenarios_print = format_table(
    pd.DataFrame([
        {'parameter': 'rho_kl_12', 'medium': crdl_sec_ctp_rho_12, 'high': crdl_sec_ctp_rho_12_high, 'low': crdl_sec_ctp_rho_12_low},
        {'parameter': 'rho_kl_13', 'medium': crdl_sec_ctp_rho_13, 'high': crdl_sec_ctp_rho_13_high, 'low': crdl_sec_ctp_rho_13_low},
        {'parameter': 'rho_kl_23', 'medium': crdl_sec_ctp_rho_23, 'high': crdl_sec_ctp_rho_23_high, 'low': crdl_sec_ctp_rho_23_low},
        {'parameter': 'gamma_bc', 'medium': crdl_sec_ctp_gamma, 'high': crdl_sec_ctp_gamma_high, 'low': crdl_sec_ctp_gamma_low},
    ]),
    columns=['parameter', 'medium', 'high', 'low'],
    col_formats={'medium': '.4f', 'high': '.4f', 'low': '.4f'},
    alignments=('left', 'right', 'right', 'right')
)

# --- 11.10 Capital --- #
crdl_sec_ctp_capital_print = format_table(
    pd.DataFrame([
        {'scenario': 'medium', 'capital': crdl_sec_ctp_K_medium},
        {'scenario': 'high', 'capital': crdl_sec_ctp_k_high},
        {'scenario': 'low', 'capital': crdl_sec_ctp_k_low},
    ]),
    columns=['scenario', 'capital'],
    col_formats={'capital': ',.2f'},
    alignments=('left', 'right')
)


# ============================================================
# DISPLAY ALL OUTPUTS
# ============================================================

print("=" * 60)
print("CRDL SECURITISATION CTP")
print("=" * 60)

print("\n1) GROSS SENSITIVITIES")
print(crdl_sec_ctp_gross_sk_print)

print("\n2) NET SENSITIVITIES (s_k)")
print(crdl_sec_ctp_net_sk_print)

print("\n3) WEIGHTED SENSITIVITIES (WS_k)")
print(crdl_sec_ctp_wsk_print)

print("\n4) INTRA-BUCKET CORRELATION (rho_kl)")
print(crdl_sec_ctp_rho_print)

print("\n5) INTRA-BUCKET AGGREGATION (K_b)")
print("\n--- Bucket 13 ---")
print(crdl_sec_ctp_Kb_formula_str)
print(crdl_sec_ctp_K_b13_print)

print("\n6) BUCKET SUMS (S_b)")
print(crdl_sec_ctp_S_b_print)

print("\n7) CROSS-BUCKET CORRELATION (gamma_bc)")
print(crdl_sec_ctp_gamma_print)

print("\n8) CROSS-BUCKET AGGREGATION")
print(crdl_sec_ctp_K_formula_str)
print(crdl_sec_ctp_cross_agg_print)

print("\n9) CORRELATION SCENARIOS")
print(crdl_sec_ctp_scenarios_print)

print("\n10) CAPITAL")
print(crdl_sec_ctp_capital_print)

CRDL SECURITISATION CTP

1) GROSS SENSITIVITIES
┌────────────┬──────────┬──────────┬──────────┬─────────┬────────────┬─────────────────────┬─────────────┐
│ trade_id   │ bucket   │ rating   │ sector   │ curve   │ maturity   │ issuer_underlying   │   gross_s_k │
├────────────┼──────────┼──────────┼──────────┼─────────┼────────────┼─────────────────────┼─────────────┤
│ 1          │ 13       │ nig      │ cons     │ cds     │ 6m         │ issuer_1            │       2,300 │
│ 2          │ 13       │ nig      │ cons     │ cds     │ 1y         │ issuer_1            │      -1,800 │
│ 3          │ 13       │ nig      │ cons     │ cds     │ 3y         │ issuer_1            │      -5,500 │
└────────────┴──────────┴──────────┴──────────┴─────────┴────────────┴─────────────────────┴─────────────┘

2) NET SENSITIVITIES (s_k)
┌──────────┬──────────┬──────────┬─────────┬────────────┬─────────────────────┬────────┐
│ bucket   │ rating   │ sector   │ curve   │ maturity   │ issuer_underlying   │    s_k

In [6]:
# ============================================================
# CRDL NON-SECURITISATION
# ============================================================


# TABLE OF CONTENTS
#   1) Gross sensitivities
#   2) Net sensitivities by risk factor
#   3) Weighted sensitivities (WS_k)
#   4) Intra-bucket correlation
#   5) Intra-bucket aggregation (K_b)
#   6) Bucket sums (S_b)
#   7) Cross-bucket correlation
#   8) Cross-bucket aggregation
#   9) Correlation scenarios (high/low)
#  10) Capital
#  11) Printing
# ============================================================


# ============================================================
# 1) GROSS SENSITIVITIES
# ============================================================
# Position-level delta sensitivities to credit spreads.

data = [
    {'trade_id': 1, 'bucket': '6', 'rating': 'ig', 'sector': 'tech', 'curve': 'cds',  'maturity': '6m', 'issuer': 'issuer_1', 'gross_sk':  1800},
    {'trade_id': 2, 'bucket': '6', 'rating': 'ig', 'sector': 'tech', 'curve': 'cds',  'maturity': '6m', 'issuer': 'issuer_1', 'gross_sk':  1300},
    {'trade_id': 3, 'bucket': '6', 'rating': 'ig', 'sector': 'tech', 'curve': 'cds',  'maturity': '6m', 'issuer': 'issuer_2', 'gross_sk':   980},
    {'trade_id': 4, 'bucket': '6', 'rating': 'ig', 'sector': 'tech', 'curve': 'cds',  'maturity': '6m', 'issuer': 'issuer_2', 'gross_sk':   650},
    {'trade_id': 5, 'bucket': '3', 'rating': 'ig', 'sector': 'fin',  'curve': 'bond', 'maturity': '6m', 'issuer': 'issuer_3', 'gross_sk': -1500},
    {'trade_id': 6, 'bucket': '3', 'rating': 'ig', 'sector': 'fin',  'curve': 'cds',  'maturity': '6m', 'issuer': 'issuer_3', 'gross_sk': -2000},
    {'trade_id': 7, 'bucket': '3', 'rating': 'ig', 'sector': 'fin',  'curve': 'cds',  'maturity': '6m', 'issuer': 'issuer_3', 'gross_sk':  1400}
]

crdl_non_sec_gross_sk = pd.DataFrame(data)


# ============================================================
# 2) NET SENSITIVITIES BY RISK FACTOR
# ============================================================
# Net sensitivity s_k = Σ gross sensitivities for each unique risk factor.
# Risk factor defined by: bucket + rating + sector + curve + issuer + maturity.

crdl_non_sec_risk_factor_cols = ['bucket', 'rating', 'sector', 'curve', 'issuer', 'maturity']

crdl_non_sec_net_sk = (
    crdl_non_sec_gross_sk
    .groupby(crdl_non_sec_risk_factor_cols)
    .agg(sk=('gross_sk', 'sum'))
    .reset_index()
)


# ============================================================
# 3) WEIGHTED SENSITIVITIES (WS_k)
# ============================================================
# WS_k = s_k × RW_k

crdl_non_sec_risk_weights = {
    '3': 0.05,
    '6': 0.02
}

crdl_non_sec_wsk = crdl_non_sec_net_sk.copy()
crdl_non_sec_wsk['rw'] = crdl_non_sec_wsk['bucket'].map(crdl_non_sec_risk_weights)
crdl_non_sec_wsk['wsk'] = crdl_non_sec_wsk['sk'] * crdl_non_sec_wsk['rw']

# Extract weighted sensitivities per bucket for aggregation
# Bucket 3: Issuer 3 (Bond vs CDS)
crdl_non_sec_ws_b3 = crdl_non_sec_wsk[crdl_non_sec_wsk['bucket'] == '3']['wsk'].values
crdl_non_sec_wsk_b3_bond = crdl_non_sec_ws_b3[0]  # Issuer 3, Bond
crdl_non_sec_wsk_b3_cds  = crdl_non_sec_ws_b3[1]  # Issuer 3, CDS

# Bucket 6: Issuer 1 vs Issuer 2 (Both CDS)
crdl_non_sec_ws_b6 = crdl_non_sec_wsk[crdl_non_sec_wsk['bucket'] == '6']['wsk'].values
crdl_non_sec_wsk_b6_issuer1 = crdl_non_sec_ws_b6[0]  # Issuer 1, CDS
crdl_non_sec_wsk_b6_issuer2 = crdl_non_sec_ws_b6[1]  # Issuer 2, CDS


# ============================================================
# 4) INTRA-BUCKET CORRELATION
# ============================================================
# ρ_kl = ρ_name × ρ_tenor × ρ_basis
# ρ_name = 1.0 (same issuer), else 0.35 (IG)
# ρ_tenor = 1.0 (same tenor), else 0.999
# ρ_basis = 1.0 (same curve), else 0.999

# --- Bucket 3 Pair: Issuer 3 Bond vs Issuer 3 CDS ---
crdl_non_sec_rho_name_b3 = 1.0     # Same issuer
crdl_non_sec_rho_tenor_b3 = 1.0    # Same tenor (6M)
crdl_non_sec_rho_basis_b3 = 0.999  # Different curve
crdl_non_sec_rho_b3 = crdl_non_sec_rho_name_b3 * crdl_non_sec_rho_tenor_b3 * crdl_non_sec_rho_basis_b3

# --- Bucket 6 Pair: Issuer 1 CDS vs Issuer 2 CDS ---
crdl_non_sec_rho_name_b6 = 0.35    # Different issuer (IG)
crdl_non_sec_rho_tenor_b6 = 1.0    # Same tenor (6M)
crdl_non_sec_rho_basis_b6 = 1.0    # Same curve
crdl_non_sec_rho_b6 = crdl_non_sec_rho_name_b6 * crdl_non_sec_rho_tenor_b6 * crdl_non_sec_rho_basis_b6


# ============================================================
# 5) INTRA-BUCKET AGGREGATION (K_b)
# ============================================================
# K_b = √( Σ WS_k² + Σ Σ ρ_kl × WS_k × WS_l )  for k ≠ l

# Bucket 3: 2 risk factors
crdl_non_sec_sum_sq_b3 = crdl_non_sec_wsk_b3_bond**2 + crdl_non_sec_wsk_b3_cds**2
crdl_non_sec_cross_b3 = 2 * crdl_non_sec_rho_b3 * crdl_non_sec_wsk_b3_bond * crdl_non_sec_wsk_b3_cds
crdl_non_sec_K_b3 = np.sqrt(max(0, crdl_non_sec_sum_sq_b3 + crdl_non_sec_cross_b3))

# Bucket 6: 2 risk factors
crdl_non_sec_sum_sq_b6 = crdl_non_sec_wsk_b6_issuer1**2 + crdl_non_sec_wsk_b6_issuer2**2
crdl_non_sec_cross_b6 = 2 * crdl_non_sec_rho_b6 * crdl_non_sec_wsk_b6_issuer1 * crdl_non_sec_wsk_b6_issuer2
crdl_non_sec_K_b6 = np.sqrt(max(0, crdl_non_sec_sum_sq_b6 + crdl_non_sec_cross_b6))


# ============================================================
# 6) BUCKET SUMS (S_b)
# ============================================================
# S_b = Σ WS_k

crdl_non_sec_S_b3 = crdl_non_sec_wsk[crdl_non_sec_wsk['bucket'] == '3']['wsk'].sum()
crdl_non_sec_S_b6 = crdl_non_sec_wsk[crdl_non_sec_wsk['bucket'] == '6']['wsk'].sum()


# ============================================================
# 7) CROSS-BUCKET CORRELATION
# ============================================================
# γ = 0.20 for CSR Non-Sec bucket pairs (Article 325aj)

# Component params
crdl_non_sec_gamma_rating = 1.0
crdl_non_sec_gamma_sector = 0.20
crdl_non_sec_gamma = crdl_non_sec_gamma_rating * crdl_non_sec_gamma_sector


# ============================================================
# 8) CROSS-BUCKET AGGREGATION
# ============================================================
# K = √( Σ K_b² + Σ Σ γ_bc × S_b × S_c )  for b ≠ c

crdl_non_sec_sum_K_sq = crdl_non_sec_K_b3**2 + crdl_non_sec_K_b6**2
crdl_non_sec_cross_bucket = 2 * crdl_non_sec_gamma * crdl_non_sec_S_b3 * crdl_non_sec_S_b6

crdl_non_sec_k_squared = crdl_non_sec_sum_K_sq + crdl_non_sec_cross_bucket

# Apply S_b capping if aggregation goes negative
if crdl_non_sec_k_squared < 0:
    crdl_non_sec_S_b3_capped = np.clip(crdl_non_sec_S_b3, -crdl_non_sec_K_b3, crdl_non_sec_K_b3)
    crdl_non_sec_S_b6_capped = np.clip(crdl_non_sec_S_b6, -crdl_non_sec_K_b6, crdl_non_sec_K_b6)
    crdl_non_sec_k_squared = crdl_non_sec_sum_K_sq + 2 * crdl_non_sec_gamma * crdl_non_sec_S_b3_capped * crdl_non_sec_S_b6_capped
    crdl_non_sec_k_squared = max(crdl_non_sec_k_squared, 0)

crdl_non_sec_K_medium = np.sqrt(crdl_non_sec_k_squared)


# ============================================================
# 9) CORRELATION SCENARIOS (HIGH/LOW)
# ============================================================
# High: ρ_high = min(1.25 × ρ_medium, 1.0)
# Low:  ρ_low  = max(2 × ρ_medium - 1, 0.75 × ρ_medium)

# --- High scenario correlations --- #
crdl_non_sec_rho_b3_high = min(crdl_non_sec_rho_b3 * 1.25, 1.0)
crdl_non_sec_rho_b6_high = min(crdl_non_sec_rho_b6 * 1.25, 1.0)
crdl_non_sec_gamma_high = min(crdl_non_sec_gamma * 1.25, 1.0)

# --- Low scenario correlations --- #
crdl_non_sec_rho_b3_low = max(2 * crdl_non_sec_rho_b3 - 1.0, 0.75 * crdl_non_sec_rho_b3)
crdl_non_sec_rho_b6_low = max(2 * crdl_non_sec_rho_b6 - 1.0, 0.75 * crdl_non_sec_rho_b6)
crdl_non_sec_gamma_low = max(2 * crdl_non_sec_gamma - 1.0, 0.75 * crdl_non_sec_gamma)

# --- High scenario intra-bucket K_b --- #
crdl_non_sec_cross_b3_high = 2 * crdl_non_sec_rho_b3_high * crdl_non_sec_wsk_b3_bond * crdl_non_sec_wsk_b3_cds
crdl_non_sec_K_b3_high = np.sqrt(max(0, crdl_non_sec_sum_sq_b3 + crdl_non_sec_cross_b3_high))

crdl_non_sec_cross_b6_high = 2 * crdl_non_sec_rho_b6_high * crdl_non_sec_wsk_b6_issuer1 * crdl_non_sec_wsk_b6_issuer2
crdl_non_sec_K_b6_high = np.sqrt(max(0, crdl_non_sec_sum_sq_b6 + crdl_non_sec_cross_b6_high))

# --- Low scenario intra-bucket K_b --- #
crdl_non_sec_cross_b3_low = 2 * crdl_non_sec_rho_b3_low * crdl_non_sec_wsk_b3_bond * crdl_non_sec_wsk_b3_cds
crdl_non_sec_K_b3_low = np.sqrt(max(0, crdl_non_sec_sum_sq_b3 + crdl_non_sec_cross_b3_low))

crdl_non_sec_cross_b6_low = 2 * crdl_non_sec_rho_b6_low * crdl_non_sec_wsk_b6_issuer1 * crdl_non_sec_wsk_b6_issuer2
crdl_non_sec_K_b6_low = np.sqrt(max(0, crdl_non_sec_sum_sq_b6 + crdl_non_sec_cross_b6_low))

# --- High scenario cross-bucket --- #
crdl_non_sec_k_sq_high = crdl_non_sec_K_b3_high**2 + crdl_non_sec_K_b6_high**2 + 2 * crdl_non_sec_gamma_high * crdl_non_sec_S_b3 * crdl_non_sec_S_b6
if crdl_non_sec_k_sq_high < 0:
    crdl_non_sec_S_b3_cap_high = np.clip(crdl_non_sec_S_b3, -crdl_non_sec_K_b3_high, crdl_non_sec_K_b3_high)
    crdl_non_sec_S_b6_cap_high = np.clip(crdl_non_sec_S_b6, -crdl_non_sec_K_b6_high, crdl_non_sec_K_b6_high)
    crdl_non_sec_k_sq_high = crdl_non_sec_K_b3_high**2 + crdl_non_sec_K_b6_high**2 + 2 * crdl_non_sec_gamma_high * crdl_non_sec_S_b3_cap_high * crdl_non_sec_S_b6_cap_high
    crdl_non_sec_k_sq_high = max(crdl_non_sec_k_sq_high, 0)
crdl_non_sec_K_high = np.sqrt(crdl_non_sec_k_sq_high)

# --- Low scenario cross-bucket --- #
crdl_non_sec_k_sq_low = crdl_non_sec_K_b3_low**2 + crdl_non_sec_K_b6_low**2 + 2 * crdl_non_sec_gamma_low * crdl_non_sec_S_b3 * crdl_non_sec_S_b6
if crdl_non_sec_k_sq_low < 0:
    crdl_non_sec_S_b3_cap_low = np.clip(crdl_non_sec_S_b3, -crdl_non_sec_K_b3_low, crdl_non_sec_K_b3_low)
    crdl_non_sec_S_b6_cap_low = np.clip(crdl_non_sec_S_b6, -crdl_non_sec_K_b6_low, crdl_non_sec_K_b6_low)
    crdl_non_sec_k_sq_low = crdl_non_sec_K_b3_low**2 + crdl_non_sec_K_b6_low**2 + 2 * crdl_non_sec_gamma_low * crdl_non_sec_S_b3_cap_low * crdl_non_sec_S_b6_cap_low
    crdl_non_sec_k_sq_low = max(crdl_non_sec_k_sq_low, 0)
crdl_non_sec_K_low = np.sqrt(crdl_non_sec_k_sq_low)


# ============================================================
# 10) CAPITAL
# ============================================================
# Final capital = max(K_medium, K_high, K_low)

crdl_non_sec_capital = max(crdl_non_sec_K_medium, crdl_non_sec_K_high, crdl_non_sec_K_low)
crdl_non_sec_binding_scenario = (
    "medium" if crdl_non_sec_capital == crdl_non_sec_K_medium else
    "high" if crdl_non_sec_capital == crdl_non_sec_K_high else "low"
)


# ============================================================
# 11) PRINTING
# ============================================================

# --- 11.1 Gross sensitivities --- #
crdl_non_sec_gross_sk_print = format_table(
    crdl_non_sec_gross_sk,
    columns=['trade_id', 'bucket', 'rating', 'sector', 'curve', 'maturity', 'issuer', 'gross_sk'],
    col_formats={'gross_sk': ',.0f'},
    alignments=('left', 'left', 'left', 'left', 'left', 'left', 'left', 'right')
)

# --- 11.2 Net sensitivities --- #
crdl_non_sec_net_sk_print = format_table(
    crdl_non_sec_net_sk,
    columns=['bucket', 'rating', 'sector', 'curve', 'maturity', 'issuer', 'sk'],
    col_formats={'sk': ',.0f'},
    alignments=('left', 'left', 'left', 'left', 'left', 'left', 'right')
)

# --- 11.3 Weighted sensitivities --- #
crdl_non_sec_wsk_print = format_table(
    crdl_non_sec_wsk,
    columns=['bucket', 'rating', 'sector', 'curve', 'maturity', 'issuer', 'sk', 'rw', 'wsk'],
    col_formats={'sk': ',.0f', 'rw': '.4f', 'wsk': ',.2f'},
    alignments=('left', 'left', 'left', 'left', 'left', 'left', 'right', 'right', 'right')
)

# --- 11.4 Intra-bucket correlation --- #
crdl_non_sec_rho_print = format_table(
    pd.DataFrame([
        {'bucket': '3', 'pair': 'Bond vs CDS', 'rho_name': crdl_non_sec_rho_name_b3, 'rho_tenor': crdl_non_sec_rho_tenor_b3, 'rho_basis': crdl_non_sec_rho_basis_b3, 'rho': crdl_non_sec_rho_b3},
        {'bucket': '6', 'pair': 'Iss1 vs Iss2', 'rho_name': crdl_non_sec_rho_name_b6, 'rho_tenor': crdl_non_sec_rho_tenor_b6, 'rho_basis': crdl_non_sec_rho_basis_b6, 'rho': crdl_non_sec_rho_b6},
    ]),
    columns=['bucket', 'pair', 'rho_name', 'rho_tenor', 'rho_basis', 'rho'],
    col_formats={'rho_name': '.4f', 'rho_tenor': '.4f', 'rho_basis': '.4f', 'rho': '.4f'},
    alignments=('left', 'left', 'right', 'right', 'right', 'right')
)

# --- 11.5 Intra-bucket aggregation (K_b) --- #
crdl_non_sec_K_b_print = format_table(
    pd.DataFrame([
        {'bucket': '3', 'wsk_k': crdl_non_sec_wsk_b3_bond, 'wsk_l': crdl_non_sec_wsk_b3_cds, 'rho': crdl_non_sec_rho_b3, 'K_b': crdl_non_sec_K_b3},
        {'bucket': '6', 'wsk_k': crdl_non_sec_wsk_b6_issuer1, 'wsk_l': crdl_non_sec_wsk_b6_issuer2, 'rho': crdl_non_sec_rho_b6, 'K_b': crdl_non_sec_K_b6},
    ]),
    columns=['bucket', 'wsk_k', 'wsk_l', 'rho', 'K_b'],
    col_formats={'wsk_k': ',.2f', 'wsk_l': ',.2f', 'rho': '.4f', 'K_b': ',.2f'},
    alignments=('left', 'right', 'right', 'right', 'right')
)

# --- 11.6 Bucket sums (S_b) --- #
crdl_non_sec_S_b_print = format_table(
    pd.DataFrame([
        {'bucket': '3', 'S_b': crdl_non_sec_S_b3},
        {'bucket': '6', 'S_b': crdl_non_sec_S_b6},
    ]),
    columns=['bucket', 'S_b'],
    col_formats={'S_b': ',.2f'},
    alignments=('left', 'right')
)

# --- 11.7 Cross-bucket correlation --- #
crdl_non_sec_gamma_print = format_table(
    pd.DataFrame([
        {'buckets': '3 vs 6', 'rating': crdl_non_sec_gamma_rating, 'sector': crdl_non_sec_gamma_sector, 'gamma': crdl_non_sec_gamma},
    ]),
    columns=['buckets', 'rating', 'sector', 'gamma'],
    col_formats={'rating': '.2f', 'sector': '.2f', 'gamma': '.2f'},
    alignments=('left', 'right', 'right', 'right')
)

# --- 11.8 Cross-bucket aggregation --- #
crdl_non_sec_cross_agg_print = format_table(
    pd.DataFrame([
        {'component': 'K_b3', 'value': crdl_non_sec_K_b3},
        {'component': 'K_b6', 'value': crdl_non_sec_K_b6},
        {'component': 'S_b3', 'value': crdl_non_sec_S_b3},
        {'component': 'S_b6', 'value': crdl_non_sec_S_b6},
        {'component': 'gamma', 'value': f"{crdl_non_sec_gamma:.2f}"},
        {'component': 'capital', 'value': crdl_non_sec_K_medium},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# --- 11.9 Correlation scenarios --- #
crdl_non_sec_scenarios_print = format_table(
    pd.DataFrame([
        {'parameter': 'rho bucket 3', 'medium': crdl_non_sec_rho_b3, 'high': crdl_non_sec_rho_b3_high, 'low': crdl_non_sec_rho_b3_low},
        {'parameter': 'rho bucket 6', 'medium': crdl_non_sec_rho_b6, 'high': crdl_non_sec_rho_b6_high, 'low': crdl_non_sec_rho_b6_low},
        {'parameter': 'gamma', 'medium': crdl_non_sec_gamma, 'high': crdl_non_sec_gamma_high, 'low': crdl_non_sec_gamma_low},
    ]),
    columns=['parameter', 'medium', 'high', 'low'],
    col_formats={'medium': '.4f', 'high': '.4f', 'low': '.4f'},
    alignments=('left', 'right', 'right', 'right')
)

# --- 11.10 Capital --- #
crdl_non_sec_capital_print = format_table(
    pd.DataFrame([
        {'scenario': 'medium', 'capital': crdl_non_sec_K_medium},
        {'scenario': 'high', 'capital': crdl_non_sec_K_high},
        {'scenario': 'low', 'capital': crdl_non_sec_K_low},
        {'scenario': 'FINAL', 'capital': crdl_non_sec_capital},
    ]),
    columns=['scenario', 'capital'],
    col_formats={'capital': ',.2f'},
    alignments=('left', 'right')
)


# ============================================================
# DISPLAY ALL OUTPUTS
# ============================================================

print("=" * 60)
print("CRDL NON-SECURITISATION")
print("=" * 60)

print("\n1) GROSS SENSITIVITIES")
print(crdl_non_sec_gross_sk_print)

print("\n2) NET SENSITIVITIES")
print(crdl_non_sec_net_sk_print)

print("\n3) WEIGHTED SENSITIVITIES (WS_k)")
print(crdl_non_sec_wsk_print)

print("\n4) INTRA-BUCKET CORRELATION")
print(crdl_non_sec_rho_print)

print("\n5) INTRA-BUCKET AGGREGATION (K_b)")
print(crdl_non_sec_K_b_print)

print("\n6) BUCKET SUMS (S_b)")
print(crdl_non_sec_S_b_print)

print("\n7) CROSS-BUCKET CORRELATION")
print(crdl_non_sec_gamma_print)

print("\n8) CROSS-BUCKET AGGREGATION")
print(crdl_non_sec_cross_agg_print)

print("\n9) CORRELATION SCENARIOS")
print(crdl_non_sec_scenarios_print)

print("\n10) CAPITAL")
print(crdl_non_sec_capital_print)
print(f"\nBinding scenario: {crdl_non_sec_binding_scenario}")

CRDL NON-SECURITISATION

1) GROSS SENSITIVITIES
┌────────────┬──────────┬──────────┬──────────┬─────────┬────────────┬──────────┬────────────┐
│ trade_id   │ bucket   │ rating   │ sector   │ curve   │ maturity   │ issuer   │   gross_sk │
├────────────┼──────────┼──────────┼──────────┼─────────┼────────────┼──────────┼────────────┤
│ 1          │ 6        │ ig       │ tech     │ cds     │ 6m         │ issuer_1 │      1,800 │
│ 2          │ 6        │ ig       │ tech     │ cds     │ 6m         │ issuer_1 │      1,300 │
│ 3          │ 6        │ ig       │ tech     │ cds     │ 6m         │ issuer_2 │        980 │
│ 4          │ 6        │ ig       │ tech     │ cds     │ 6m         │ issuer_2 │        650 │
│ 5          │ 3        │ ig       │ fin      │ bond    │ 6m         │ issuer_3 │     -1,500 │
│ 6          │ 3        │ ig       │ fin      │ cds     │ 6m         │ issuer_3 │     -2,000 │
│ 7          │ 3        │ ig       │ fin      │ cds     │ 6m         │ issuer_3 │      1,400 │
└─

### CDRL Sec

In [7]:
# ============================================================
# CRDL SEC
# ============================================================


# TABLE OF CONTENTS
#   1) Gross sensitivities
#   2) Net sensitivities (s_k)
#   3) Weighted sensitivities (WS_k)
#   4) Intra-bucket correlation (rho_kl)
#   5) Intra-bucket aggregation (K_b)
#   6) Bucket sums (S_b)
#   7) Cross-bucket correlation (gamma_bc)
#   8) Cross-bucket aggregation
#   9) Correlation scenarios (high/low)
#  10) Capital
#  11) Printing
# ============================================================


# ============================================================
# 1) GROSS SENSITIVITIES
# ============================================================
# Position-level delta sensitivities to securitisation spreads.

data = [
    {'trade_id': 1, 'bucket': '14', 'curve': 'bond', 'maturity': '10y', 'tranche': 'tranche_a', 'gross_s_k': 1200},
    {'trade_id': 2, 'bucket': '14', 'curve': 'bond', 'maturity': '5y',  'tranche': 'tranche_a', 'gross_s_k': 750},
    {'trade_id': 3, 'bucket': '4',  'curve': 'cds',  'maturity': '1y',  'tranche': 'tranche_b', 'gross_s_k': 360},
    {'trade_id': 4, 'bucket': '4',  'curve': 'cds',  'maturity': '3y',  'tranche': 'tranche_b', 'gross_s_k': 900}
]

crdl_sec_gross_sk = pd.DataFrame(data)


# ============================================================
# 2) NET SENSITIVITIES (s_k)
# ============================================================
# Net sensitivity s_k = Σ gross sensitivities for each unique risk factor.
# Risk factor defined by: bucket + maturity + curve + tranche.

crdl_sec_risk_factor_cols = ['bucket', 'curve', 'tranche', 'maturity']

crdl_sec_net_sk = (
    crdl_sec_gross_sk
    .groupby(crdl_sec_risk_factor_cols)
    .agg(s_k=('gross_s_k', 'sum'))
    .reset_index()
)


# ============================================================
# 3) WEIGHTED SENSITIVITIES (WS_k)
# ============================================================
# WS_k = s_k × RW_k

crdl_sec_risk_weights = {
    '4':  0.020,
    '14': 0.015
}

crdl_sec_wsk = crdl_sec_net_sk.copy()
crdl_sec_wsk['rw'] = crdl_sec_wsk['bucket'].map(crdl_sec_risk_weights)
crdl_sec_wsk['WS_k'] = crdl_sec_wsk['s_k'] * crdl_sec_wsk['rw']

# Extract weighted sensitivities per bucket for aggregation
# Bucket 4: CMBS (1Y vs 3Y)
crdl_sec_ws_b4 = crdl_sec_wsk[crdl_sec_wsk['bucket'] == '4']['WS_k'].values
crdl_sec_WS_k_b4_1y = crdl_sec_ws_b4[0]  # 1Y maturity
crdl_sec_WS_k_b4_3y = crdl_sec_ws_b4[1]  # 3Y maturity

# Bucket 14: ABS Credit Cards (10Y vs 5Y)
crdl_sec_ws_b14 = crdl_sec_wsk[crdl_sec_wsk['bucket'] == '14']['WS_k'].values
crdl_sec_WS_k_b14_10y = crdl_sec_ws_b14[0]  # 10Y maturity
crdl_sec_WS_k_b14_5y  = crdl_sec_ws_b14[1]  # 5Y maturity


# ============================================================
# 4) INTRA-BUCKET CORRELATION (rho_kl)
# ============================================================
# ρ_kl = ρ_tranche × ρ_tenor × ρ_basis
# ρ_tranche = 1.0 (same tranche)
# ρ_tenor   = 0.80 (different maturities)
# ρ_basis   = 1.0 (same basis)

# Bucket 4: 1Y vs 3Y (different maturities)
crdl_sec_rho_tranche_b4 = 1.0
crdl_sec_rho_tenor_b4 = 0.80   # Different maturities
crdl_sec_rho_basis_b4 = 1.0
crdl_sec_rho_kl_b4 = crdl_sec_rho_tranche_b4 * crdl_sec_rho_tenor_b4 * crdl_sec_rho_basis_b4

# Bucket 14: 10Y vs 5Y (different maturities)
crdl_sec_rho_tranche_b14 = 1.0
crdl_sec_rho_tenor_b14 = 0.80  # Different maturities
crdl_sec_rho_basis_b14 = 1.0
crdl_sec_rho_kl_b14 = crdl_sec_rho_tranche_b14 * crdl_sec_rho_tenor_b14 * crdl_sec_rho_basis_b14


# ============================================================
# 5) INTRA-BUCKET AGGREGATION (K_b)
# ============================================================
# K_b = √( Σ WS_k² + Σ Σ ρ_kl × WS_k × WS_l )  for k ≠ l

# Bucket 4: 2 risk factors
crdl_sec_sum_sq_b4 = crdl_sec_WS_k_b4_1y**2 + crdl_sec_WS_k_b4_3y**2
crdl_sec_cross_b4 = 2 * crdl_sec_rho_kl_b4 * crdl_sec_WS_k_b4_1y * crdl_sec_WS_k_b4_3y
crdl_sec_K_b4 = np.sqrt(max(0, crdl_sec_sum_sq_b4 + crdl_sec_cross_b4))

# Bucket 14: 2 risk factors
crdl_sec_sum_sq_b14 = crdl_sec_WS_k_b14_10y**2 + crdl_sec_WS_k_b14_5y**2
crdl_sec_cross_b14 = 2 * crdl_sec_rho_kl_b14 * crdl_sec_WS_k_b14_10y * crdl_sec_WS_k_b14_5y
crdl_sec_K_b14 = np.sqrt(max(0, crdl_sec_sum_sq_b14 + crdl_sec_cross_b14))


# ============================================================
# 6) BUCKET SUMS (S_b)
# ============================================================
# S_b = Σ WS_k

crdl_sec_S_b4 = crdl_sec_wsk[crdl_sec_wsk['bucket'] == '4']['WS_k'].sum()
crdl_sec_S_b14 = crdl_sec_wsk[crdl_sec_wsk['bucket'] == '14']['WS_k'].sum()


# ============================================================
# 7) CROSS-BUCKET CORRELATION (gamma_bc)
# ============================================================
# γ = 0.0 for CSR Sec (non-CTP) bucket pairs

crdl_sec_gamma = 0.0


# ============================================================
# 8) CROSS-BUCKET AGGREGATION
# ============================================================
# K = √( Σ K_b² + Σ Σ γ_bc × S_b × S_c )  for b ≠ c

crdl_sec_sum_K_sq = crdl_sec_K_b4**2 + crdl_sec_K_b14**2
crdl_sec_cross_bucket = 2 * crdl_sec_gamma * crdl_sec_S_b4 * crdl_sec_S_b14

crdl_sec_k_squared = crdl_sec_sum_K_sq + crdl_sec_cross_bucket

# Apply the check at the overall level (Floor at 0)
crdl_sec_k_squared = max(crdl_sec_k_squared, 0)

crdl_sec_K_medium = np.sqrt(crdl_sec_k_squared)


# ============================================================
# 9) CORRELATION SCENARIOS (HIGH/LOW)
# ============================================================
# High: ρ_high = min(1.25 × ρ_medium, 1.0)
# Low:  ρ_low  = max(2 × ρ_medium - 1, 0.75 × ρ_medium)

# --- High scenario correlations --- #
crdl_sec_rho_b4_high = min(crdl_sec_rho_kl_b4 * 1.25, 1.0)
crdl_sec_rho_b14_high = min(crdl_sec_rho_kl_b14 * 1.25, 1.0)
crdl_sec_gamma_high = min(crdl_sec_gamma * 1.25, 1.0)

# --- Low scenario correlations --- #
crdl_sec_rho_b4_low = max(2 * crdl_sec_rho_kl_b4 - 1.0, 0.75 * crdl_sec_rho_kl_b4)
crdl_sec_rho_b14_low = max(2 * crdl_sec_rho_kl_b14 - 1.0, 0.75 * crdl_sec_rho_kl_b14)
crdl_sec_gamma_low = max(2 * crdl_sec_gamma - 1.0, 0.75 * crdl_sec_gamma)

# --- High scenario intra-bucket K_b --- #
crdl_sec_cross_b4_high = 2 * crdl_sec_rho_b4_high * crdl_sec_WS_k_b4_1y * crdl_sec_WS_k_b4_3y
crdl_sec_K_b4_high = np.sqrt(max(0, crdl_sec_sum_sq_b4 + crdl_sec_cross_b4_high))

crdl_sec_cross_b14_high = 2 * crdl_sec_rho_b14_high * crdl_sec_WS_k_b14_10y * crdl_sec_WS_k_b14_5y
crdl_sec_K_b14_high = np.sqrt(max(0, crdl_sec_sum_sq_b14 + crdl_sec_cross_b14_high))

# --- Low scenario intra-bucket K_b --- #
crdl_sec_cross_b4_low = 2 * crdl_sec_rho_b4_low * crdl_sec_WS_k_b4_1y * crdl_sec_WS_k_b4_3y
crdl_sec_K_b4_low = np.sqrt(max(0, crdl_sec_sum_sq_b4 + crdl_sec_cross_b4_low))

crdl_sec_cross_b14_low = 2 * crdl_sec_rho_b14_low * crdl_sec_WS_k_b14_10y * crdl_sec_WS_k_b14_5y
crdl_sec_K_b14_low = np.sqrt(max(0, crdl_sec_sum_sq_b14 + crdl_sec_cross_b14_low))

# --- High scenario cross-bucket --- #
crdl_sec_k_sq_high = (
    crdl_sec_K_b4_high**2 + crdl_sec_K_b14_high**2
    + 2 * crdl_sec_gamma_high * crdl_sec_S_b4 * crdl_sec_S_b14
)
crdl_sec_K_high = np.sqrt(max(crdl_sec_k_sq_high, 0))

# --- Low scenario cross-bucket --- #
crdl_sec_k_sq_low = (
    crdl_sec_K_b4_low**2 + crdl_sec_K_b14_low**2
    + 2 * crdl_sec_gamma_low * crdl_sec_S_b4 * crdl_sec_S_b14
)
crdl_sec_K_low = np.sqrt(max(crdl_sec_k_sq_low, 0))


# ============================================================
# 10) CAPITAL
# ============================================================
# Final capital = max(K_medium, K_high, K_low)

crdl_sec_capital = max(crdl_sec_K_medium, crdl_sec_K_high, crdl_sec_K_low)


# ============================================================
# 11) PRINTING
# ============================================================

# --- 11.1 Gross sensitivities --- #
crdl_sec_gross_sk_print = format_table(
    crdl_sec_gross_sk,
    columns=['trade_id', 'bucket', 'curve', 'maturity', 'tranche', 'gross_s_k'],
    col_formats={'gross_s_k': ',.0f'},
    alignments=('left', 'left', 'left', 'left', 'left', 'right')
)

# --- 11.2 Net sensitivities (s_k) --- #
crdl_sec_net_sk_print = format_table(
    crdl_sec_net_sk,
    columns=['bucket', 'curve', 'maturity', 'tranche', 's_k'],
    col_formats={'s_k': ',.0f'},
    alignments=('left', 'left', 'left', 'left', 'right')
)

# --- 11.3 Weighted sensitivities (WS_k) --- #
crdl_sec_wsk_print = format_table(
    crdl_sec_wsk,
    columns=['bucket', 'curve', 'maturity', 'tranche', 's_k', 'rw', 'WS_k'],
    col_formats={'s_k': ',.0f', 'rw': '.4f', 'WS_k': ',.2f'},
    alignments=('left', 'left', 'left', 'left', 'right', 'right', 'right')
)

# --- 11.4 Intra-bucket correlation (rho_kl) --- #
crdl_sec_rho_print = format_table(
    pd.DataFrame([
        {'bucket': '4', 'pair': '1y vs 3y', 'rho_curve': crdl_sec_rho_basis_b4, 'rho_maturity': crdl_sec_rho_tenor_b4, 'rho_tranche': crdl_sec_rho_tranche_b4, 'rho_kl': crdl_sec_rho_kl_b4},
        {'bucket': '14', 'pair': '10y vs 5y', 'rho_curve': crdl_sec_rho_basis_b14, 'rho_maturity': crdl_sec_rho_tenor_b14, 'rho_tranche': crdl_sec_rho_tranche_b14, 'rho_kl': crdl_sec_rho_kl_b14},
    ]),
    columns=['bucket', 'pair', 'rho_curve', 'rho_maturity', 'rho_tranche', 'rho_kl'],
    col_formats={'rho_curve': '.4f', 'rho_maturity': '.4f', 'rho_tranche': '.4f', 'rho_kl': '.4f'},
    alignments=('left', 'left', 'right', 'right', 'right', 'right')
)

# --- 11.5 Intra-bucket aggregation (K_b) --- #
# Prepare Formula Strings
crdl_sec_Kb4_formula_str = (
    f"K_b4 = sqrt( ({crdl_sec_WS_k_b4_1y:.2f}^2 + {crdl_sec_WS_k_b4_3y:.2f}^2) "
    f"+ (2 * {crdl_sec_rho_kl_b4:.4f} * {crdl_sec_WS_k_b4_1y:.2f} * {crdl_sec_WS_k_b4_3y:.2f}) )"
)

crdl_sec_Kb14_formula_str = (
    f"K_b14 = sqrt( ({crdl_sec_WS_k_b14_10y:.2f}^2 + {crdl_sec_WS_k_b14_5y:.2f}^2) "
    f"+ (2 * {crdl_sec_rho_kl_b14:.4f} * {crdl_sec_WS_k_b14_10y:.2f} * {crdl_sec_WS_k_b14_5y:.2f}) )"
)

# Bucket 4
crdl_sec_K_b4_print = format_table(
    pd.DataFrame([
        {'component': 'WS_k_1 (1y)', 'value': crdl_sec_WS_k_b4_1y},
        {'component': 'WS_k_2 (3y)', 'value': crdl_sec_WS_k_b4_3y},
        {'component': 'rho_kl', 'value': f"{crdl_sec_rho_kl_b4:.4f}"},
        {'component': 'K_b', 'value': crdl_sec_K_b4},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# Bucket 14
crdl_sec_K_b14_print = format_table(
    pd.DataFrame([
        {'component': 'WS_k_1 (10y)', 'value': crdl_sec_WS_k_b14_10y},
        {'component': 'WS_k_2 (5y)', 'value': crdl_sec_WS_k_b14_5y},
        {'component': 'rho_kl', 'value': f"{crdl_sec_rho_kl_b14:.4f}"},
        {'component': 'K_b', 'value': crdl_sec_K_b14},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# --- 11.6 Bucket sums (S_b) --- #
crdl_sec_S_b_print = format_table(
    pd.DataFrame([
        {'bucket': '4', 'WS_k_1': crdl_sec_WS_k_b4_1y, 'WS_k_2': crdl_sec_WS_k_b4_3y, 'S_b': crdl_sec_S_b4},
        {'bucket': '14', 'WS_k_1': crdl_sec_WS_k_b14_10y, 'WS_k_2': crdl_sec_WS_k_b14_5y, 'S_b': crdl_sec_S_b14},
    ]),
    columns=['bucket', 'WS_k_1', 'WS_k_2', 'S_b'],
    col_formats={'WS_k_1': ',.2f', 'WS_k_2': ',.2f', 'S_b': ',.2f'},
    alignments=('left', 'right', 'right', 'right')
)

# --- 11.7 Cross-bucket correlation (gamma_bc) --- #
crdl_sec_gamma_print = format_table(
    pd.DataFrame([
        {'buckets': '4 vs 14', 'gamma_bc': crdl_sec_gamma},
    ]),
    columns=['buckets', 'gamma_bc'],
    col_formats={'gamma_bc': '.2f'},
    alignments=('left', 'right')
)

# --- 11.8 Cross-bucket aggregation --- #
# Prepare Formula String
crdl_sec_K_formula_str = (
    f"Capital = sqrt( ({crdl_sec_K_b4:.2f}^2 + {crdl_sec_K_b14:.2f}^2) "
    f"+ (2 * {crdl_sec_gamma:.2f} * {crdl_sec_S_b4:.2f} * {crdl_sec_S_b14:.2f}) )"
)

crdl_sec_cross_agg_print = format_table(
    pd.DataFrame([
        {'component': 'K_b4', 'value': crdl_sec_K_b4},
        {'component': 'K_b14', 'value': crdl_sec_K_b14},
        {'component': 'S_b4', 'value': crdl_sec_S_b4},
        {'component': 'S_b14', 'value': crdl_sec_S_b14},
        {'component': 'gamma_bc', 'value': f"{crdl_sec_gamma:.2f}"},
        {'component': 'capital', 'value': crdl_sec_K_medium},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# --- 11.9 Correlation scenarios --- #
crdl_sec_scenarios_print = format_table(
    pd.DataFrame([
        {'parameter': 'rho_kl_b4', 'medium': crdl_sec_rho_kl_b4, 'high': crdl_sec_rho_b4_high, 'low': crdl_sec_rho_b4_low},
        {'parameter': 'rho_kl_b14', 'medium': crdl_sec_rho_kl_b14, 'high': crdl_sec_rho_b14_high, 'low': crdl_sec_rho_b14_low},
        {'parameter': 'gamma_bc', 'medium': crdl_sec_gamma, 'high': crdl_sec_gamma_high, 'low': crdl_sec_gamma_low},
    ]),
    columns=['parameter', 'medium', 'high', 'low'],
    col_formats={'medium': '.4f', 'high': '.4f', 'low': '.4f'},
    alignments=('left', 'right', 'right', 'right')
)

# --- 11.10 Capital --- #
crdl_sec_capital_print = format_table(
    pd.DataFrame([
        {'scenario': 'medium', 'capital': crdl_sec_K_medium},
        {'scenario': 'high', 'capital': crdl_sec_K_high},
        {'scenario': 'low', 'capital': crdl_sec_K_low},
    ]),
    columns=['scenario', 'capital'],
    col_formats={'capital': ',.2f'},
    alignments=('left', 'right')
)


# ============================================================
# DISPLAY ALL OUTPUTS
# ============================================================

print("=" * 60)
print("CRDL SECURITISATION")
print("=" * 60)

print("\n1) GROSS SENSITIVITIES")
print(crdl_sec_gross_sk_print)

print("\n2) NET SENSITIVITIES (s_k)")
print(crdl_sec_net_sk_print)

print("\n3) WEIGHTED SENSITIVITIES (WS_k)")
print(crdl_sec_wsk_print)

print("\n4) INTRA-BUCKET CORRELATION (rho_kl)")
print(crdl_sec_rho_print)

print("\n5) INTRA-BUCKET AGGREGATION (K_b)")
print("\n--- Bucket 4 ---")
print(crdl_sec_Kb4_formula_str)
print(crdl_sec_K_b4_print)
print("\n--- Bucket 14 ---")
print(crdl_sec_Kb14_formula_str)
print(crdl_sec_K_b14_print)

print("\n6) BUCKET SUMS (S_b)")
print(crdl_sec_S_b_print)

print("\n7) CROSS-BUCKET CORRELATION (gamma_bc)")
print(crdl_sec_gamma_print)

print("\n8) CROSS-BUCKET AGGREGATION")
print(crdl_sec_K_formula_str)
print(crdl_sec_cross_agg_print)

print("\n9) CORRELATION SCENARIOS")
print(crdl_sec_scenarios_print)

print("\n10) CAPITAL")
print(crdl_sec_capital_print)

CRDL SECURITISATION

1) GROSS SENSITIVITIES
┌────────────┬──────────┬─────────┬────────────┬───────────┬─────────────┐
│ trade_id   │ bucket   │ curve   │ maturity   │ tranche   │   gross_s_k │
├────────────┼──────────┼─────────┼────────────┼───────────┼─────────────┤
│ 1          │ 14       │ bond    │ 10y        │ tranche_a │       1,200 │
│ 2          │ 14       │ bond    │ 5y         │ tranche_a │         750 │
│ 3          │ 4        │ cds     │ 1y         │ tranche_b │         360 │
│ 4          │ 4        │ cds     │ 3y         │ tranche_b │         900 │
└────────────┴──────────┴─────────┴────────────┴───────────┴─────────────┘

2) NET SENSITIVITIES (s_k)
┌──────────┬─────────┬────────────┬───────────┬───────┐
│ bucket   │ curve   │ maturity   │ tranche   │   s_k │
├──────────┼─────────┼────────────┼───────────┼───────┤
│ 14       │ bond    │ 10y        │ tranche_a │ 1,200 │
│ 14       │ bond    │ 5y         │ tranche_a │   750 │
│ 4        │ cds     │ 1y         │ tranche_b │ 

### EQDL

In [8]:
# ============================================================
# EQDL
# ============================================================


# TABLE OF CONTENTS
#   1) Gross sensitivities
#   2) Net sensitivities (s_k)
#   3) Weighted sensitivities (WS_k)
#   4) Intra-bucket correlation (rho_kl)
#   5) Intra-bucket aggregation (K_b)
#   6) Bucket sums (S_b)
#   7) Cross-bucket correlation (gamma_bc)
#   8) Cross-bucket aggregation
#   9) Correlation scenarios (high/low)
#  10) Capital
#  11) Printing
# ============================================================


# ============================================================
# 1) GROSS SENSITIVITIES
# ============================================================
# Position-level delta sensitivities to equity spot.

data = [
    {'trade_id': 1, 'bucket': '6', 'spot': 'equity a', 'gross_s_k': -1800},
    {'trade_id': 2, 'bucket': '6', 'spot': 'equity b', 'gross_s_k': -700},
    {'trade_id': 3, 'bucket': '2', 'spot': 'equity c', 'gross_s_k': -400},
]

eqdl_gross_sk = pd.DataFrame(data)


# ============================================================
# 2) NET SENSITIVITIES (s_k)
# ============================================================
# Net sensitivity s_k = Σ gross sensitivities for each unique risk factor.
# Risk factor defined by: bucket + spot.

eqdl_risk_factor_cols = ['bucket', 'spot']

eqdl_net_sk = (
    eqdl_gross_sk
    .groupby(eqdl_risk_factor_cols)
    .agg(s_k=('gross_s_k', 'sum'))
    .reset_index()
)


# ============================================================
# 3) WEIGHTED SENSITIVITIES (WS_k)
# ============================================================
# WS_k = s_k × RW_k

# Risk weights (RW_k) from Article 325ap for each equity bucket
eqdl_risk_weights = {
    '2': 0.60,
    '6': 0.35,
}

eqdl_wsk = eqdl_net_sk.copy()
eqdl_wsk['rw'] = eqdl_wsk['bucket'].map(eqdl_risk_weights)
eqdl_wsk['WS_k'] = eqdl_wsk['s_k'] * eqdl_wsk['rw']

# Extract weighted sensitivities per bucket for aggregation formulas
# Bucket 2: Single Equity C
eqdl_ws_b2 = eqdl_wsk[eqdl_wsk['bucket'] == '2']['WS_k'].values
eqdl_WS_k_b2 = eqdl_ws_b2[0]  # Equity C

# Bucket 6: Equity A and Equity B
eqdl_ws_b6 = eqdl_wsk[eqdl_wsk['bucket'] == '6']['WS_k'].values
eqdl_WS_k_b6_eqA = eqdl_ws_b6[0]  # Equity A
eqdl_WS_k_b6_eqB = eqdl_ws_b6[1]  # Equity B


# ============================================================
# 4) INTRA-BUCKET CORRELATION (rho_kl)
# ============================================================
# Intra-bucket correlation ρ_kl from Article 325aq
# For Large Cap buckets: ρ = 0.25

# Bucket 2: only 1 risk factor
eqdl_rho_kl_b2 = None  # single position, no correlation needed

# Bucket 6: correlation between Equity A and Equity B
eqdl_rho_kl_b6 = 0.25  # same bucket, different underlyings


# ============================================================
# 5) INTRA-BUCKET AGGREGATION (K_b)
# ============================================================
# K_b = √( Σ WS_k² + Σ Σ ρ_kl × WS_k × WS_l )  for k ≠ l

# Bucket 2: 1 risk factor → K_b = |WS_k|
eqdl_K_b2 = abs(eqdl_WS_k_b2)

# Bucket 6: 2 risk factors
eqdl_sum_sq_b6 = eqdl_WS_k_b6_eqA**2 + eqdl_WS_k_b6_eqB**2
eqdl_cross_b6 = 2 * eqdl_rho_kl_b6 * eqdl_WS_k_b6_eqA * eqdl_WS_k_b6_eqB
eqdl_K_b6 = np.sqrt(max(0, eqdl_sum_sq_b6 + eqdl_cross_b6))

# --- Explicit K_b formula strings for printing ---
eqdl_Kb2_formula_str = f"K_b2 = abs({eqdl_WS_k_b2:.2f})"

eqdl_Kb6_formula_str = (
    f"K_b6 = sqrt( ({eqdl_WS_k_b6_eqA:.2f}^2 + {eqdl_WS_k_b6_eqB:.2f}^2) "
    f"+ (2 * {eqdl_rho_kl_b6:.4f} * {eqdl_WS_k_b6_eqA:.2f} * {eqdl_WS_k_b6_eqB:.2f}) )"
)


# ============================================================
# 6) BUCKET SUMS (S_b)
# ============================================================
# S_b = Σ WS_k

eqdl_S_b2 = eqdl_wsk[eqdl_wsk['bucket'] == '2']['WS_k'].sum()
eqdl_S_b6 = eqdl_wsk[eqdl_wsk['bucket'] == '6']['WS_k'].sum()


# ============================================================
# 7) CROSS-BUCKET CORRELATION (gamma_bc)
# ============================================================
# γ = 0.15 for buckets 1-10 (Article 325ar)

eqdl_gamma = 0.15


# ============================================================
# 8) CROSS-BUCKET AGGREGATION
# ============================================================
# K = √( Σ K_b² + Σ Σ γ_bc × S_b × S_c )  for b ≠ c

eqdl_sum_K_sq = eqdl_K_b2**2 + eqdl_K_b6**2
eqdl_cross_bucket = 2 * eqdl_gamma * eqdl_S_b2 * eqdl_S_b6

eqdl_k_squared = eqdl_sum_K_sq + eqdl_cross_bucket

# Apply the check at the overall level (Floor at 0)
eqdl_k_squared = max(eqdl_k_squared, 0)

eqdl_K_medium = np.sqrt(eqdl_k_squared)

# --- Explicit Cross-bucket formula string for printing ---
eqdl_K_formula_str = (
    f"Capital = sqrt( ({eqdl_K_b2:.2f}^2 + {eqdl_K_b6:.2f}^2) "
    f"+ (2 * {eqdl_gamma:.2f} * {eqdl_S_b2:.2f} * {eqdl_S_b6:.2f}) )"
)


# ============================================================
# 9) CORRELATION SCENARIOS (HIGH/LOW)
# ============================================================
# High: ρ_high = min(1.25 × ρ_medium, 1.0)
# Low:  ρ_low  = max(2 × ρ_medium - 1, 0.75 × ρ_medium)

# --- High scenario correlations --- #
eqdl_rho_b6_high = min(eqdl_rho_kl_b6 * 1.25, 1.0)
eqdl_gamma_high = min(eqdl_gamma * 1.25, 1.0)

# --- Low scenario correlations --- #
eqdl_rho_b6_low = max(2 * eqdl_rho_kl_b6 - 1.0, 0.75 * eqdl_rho_kl_b6)
eqdl_gamma_low = max(2 * eqdl_gamma - 1.0, 0.75 * eqdl_gamma)

# --- High scenario intra-bucket K_b --- #
eqdl_K_b2_high = abs(eqdl_WS_k_b2)  # Single RF, unchanged

eqdl_cross_b6_high = 2 * eqdl_rho_b6_high * eqdl_WS_k_b6_eqA * eqdl_WS_k_b6_eqB
eqdl_K_b6_high = np.sqrt(max(0, eqdl_sum_sq_b6 + eqdl_cross_b6_high))

# --- Low scenario intra-bucket K_b --- #
eqdl_K_b2_low = abs(eqdl_WS_k_b2)  # Single RF, unchanged

eqdl_cross_b6_low = 2 * eqdl_rho_b6_low * eqdl_WS_k_b6_eqA * eqdl_WS_k_b6_eqB
eqdl_K_b6_low = np.sqrt(max(0, eqdl_sum_sq_b6 + eqdl_cross_b6_low))

# --- High scenario cross-bucket --- #
eqdl_k_sq_high = (
    eqdl_K_b2_high**2 + eqdl_K_b6_high**2
    + 2 * eqdl_gamma_high * eqdl_S_b2 * eqdl_S_b6
)
eqdl_K_high = np.sqrt(max(eqdl_k_sq_high, 0))

# --- Low scenario cross-bucket --- #
eqdl_k_sq_low = (
    eqdl_K_b2_low**2 + eqdl_K_b6_low**2
    + 2 * eqdl_gamma_low * eqdl_S_b2 * eqdl_S_b6
)
eqdl_K_low = np.sqrt(max(eqdl_k_sq_low, 0))


# ============================================================
# 10) CAPITAL
# ============================================================
# Final capital = max(K_medium, K_high, K_low)

eqdl_capital = max(eqdl_K_medium, eqdl_K_high, eqdl_K_low)


# ============================================================
# 11) PRINTING
# ============================================================

# --- 11.1 Gross sensitivities --- #
eqdl_gross_sk_print = format_table(
    eqdl_gross_sk,
    columns=['trade_id', 'bucket', 'spot', 'gross_s_k'],
    col_formats={'gross_s_k': ',.0f'},
    alignments=('left', 'left', 'left', 'right')
)

# --- 11.2 Net sensitivities (s_k) --- #
eqdl_net_sk_print = format_table(
    eqdl_net_sk,
    columns=['bucket', 'spot', 's_k'],
    col_formats={'s_k': ',.0f'},
    alignments=('left', 'left', 'right')
)

# --- 11.3 Weighted sensitivities (WS_k) --- #
eqdl_wsk_print = format_table(
    eqdl_wsk,
    columns=['bucket', 'spot', 's_k', 'rw', 'WS_k'],
    col_formats={'s_k': ',.0f', 'rw': '.2f', 'WS_k': ',.0f'},
    alignments=('left', 'left', 'right', 'right', 'right')
)

# --- 11.4 Intra-bucket correlation (rho_kl) --- #
eqdl_rho_print = format_table(
    pd.DataFrame([
        {'bucket': '2', 'pair': 'n/a', 'rho_kl': 'n/a'},
        {'bucket': '6', 'pair': 'EqA vs EqB', 'rho_kl': eqdl_rho_kl_b6},
    ]),
    columns=['bucket', 'pair', 'rho_kl'],
    col_formats={'rho_kl': '.4f'},
    alignments=('left', 'left', 'right')
)

# --- 11.5 Intra-bucket aggregation (K_b) --- #
# Bucket 2
eqdl_K_b2_print = format_table(
    pd.DataFrame([
        {'component': 'WS_k_1 (EqC)', 'value': eqdl_WS_k_b2},
        {'component': 'K_b', 'value': eqdl_K_b2},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# Bucket 6
eqdl_K_b6_print = format_table(
    pd.DataFrame([
        {'component': 'WS_k_1 (EqA)', 'value': eqdl_WS_k_b6_eqA},
        {'component': 'WS_k_2 (EqB)', 'value': eqdl_WS_k_b6_eqB},
        {'component': 'rho_kl', 'value': f"{eqdl_rho_kl_b6:.4f}"},
        {'component': 'K_b', 'value': eqdl_K_b6},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# --- 11.6 Bucket sums (S_b) --- #
eqdl_S_b_print = format_table(
    pd.DataFrame([
        {'bucket': '2', 'WS_k_1': eqdl_WS_k_b2, 'WS_k_2': 'n/a', 'S_b': eqdl_S_b2},
        {'bucket': '6', 'WS_k_1': eqdl_WS_k_b6_eqA, 'WS_k_2': eqdl_WS_k_b6_eqB, 'S_b': eqdl_S_b6},
    ]),
    columns=['bucket', 'WS_k_1', 'WS_k_2', 'S_b'],
    col_formats={'WS_k_1': ',.2f', 'WS_k_2': ',.2f', 'S_b': ',.2f'},
    alignments=('left', 'right', 'right', 'right')
)

# --- 11.7 Cross-bucket correlation (gamma_bc) --- #
eqdl_gamma_print = format_table(
    pd.DataFrame([
        {'buckets': '2 vs 6', 'gamma_bc': eqdl_gamma},
    ]),
    columns=['buckets', 'gamma_bc'],
    col_formats={'gamma_bc': '.2f'},
    alignments=('left', 'right')
)

# --- 11.8 Cross-bucket aggregation --- #
eqdl_cross_agg_print = format_table(
    pd.DataFrame([
        {'component': 'K_b2', 'value': eqdl_K_b2},
        {'component': 'K_b6', 'value': eqdl_K_b6},
        {'component': 'S_b2', 'value': eqdl_S_b2},
        {'component': 'S_b6', 'value': eqdl_S_b6},
        {'component': 'gamma_bc', 'value': f"{eqdl_gamma:.2f}"},
        {'component': 'capital', 'value': eqdl_K_medium},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# --- 11.9 Correlation scenarios --- #
eqdl_scenarios_print = format_table(
    pd.DataFrame([
        {'parameter': 'rho_kl_b6', 'medium': eqdl_rho_kl_b6, 'high': eqdl_rho_b6_high, 'low': eqdl_rho_b6_low},
        {'parameter': 'gamma_bc', 'medium': eqdl_gamma, 'high': eqdl_gamma_high, 'low': eqdl_gamma_low},
    ]),
    columns=['parameter', 'medium', 'high', 'low'],
    col_formats={'medium': '.4f', 'high': '.4f', 'low': '.4f'},
    alignments=('left', 'right', 'right', 'right')
)

# --- 11.10 Capital --- #
eqdl_capital_print = format_table(
    pd.DataFrame([
        {'scenario': 'medium', 'capital': eqdl_K_medium},
        {'scenario': 'high', 'capital': eqdl_K_high},
        {'scenario': 'low', 'capital': eqdl_K_low},
    ]),
    columns=['scenario', 'capital'],
    col_formats={'capital': ',.2f'},
    alignments=('left', 'right')
)


# ============================================================
# DISPLAY ALL OUTPUTS
# ============================================================

print("=" * 60)
print("EQDL")
print("=" * 60)

print("\n1) GROSS SENSITIVITIES")
print(eqdl_gross_sk_print)

print("\n2) NET SENSITIVITIES (s_k)")
print(eqdl_net_sk_print)

print("\n3) WEIGHTED SENSITIVITIES (WS_k)")
print(eqdl_wsk_print)

print("\n4) INTRA-BUCKET CORRELATION (rho_kl)")
print(eqdl_rho_print)

print("\n5) INTRA-BUCKET AGGREGATION (K_b)")
print("\n--- Bucket 2 ---")
print(eqdl_Kb2_formula_str)
print(eqdl_K_b2_print)
print("\n--- Bucket 6 ---")
print(eqdl_Kb6_formula_str)
print(eqdl_K_b6_print)

print("\n6) BUCKET SUMS (S_b)")
print(eqdl_S_b_print)

print("\n7) CROSS-BUCKET CORRELATION (gamma_bc)")
print(eqdl_gamma_print)

print("\n8) CROSS-BUCKET AGGREGATION")
print(eqdl_K_formula_str)
print(eqdl_cross_agg_print)

print("\n9) CORRELATION SCENARIOS")
print(eqdl_scenarios_print)

print("\n10) CAPITAL")
print(eqdl_capital_print)

EQDL

1) GROSS SENSITIVITIES
┌────────────┬──────────┬──────────┬─────────────┐
│ trade_id   │ bucket   │ spot     │   gross_s_k │
├────────────┼──────────┼──────────┼─────────────┤
│ 1          │ 6        │ equity a │      -1,800 │
│ 2          │ 6        │ equity b │        -700 │
│ 3          │ 2        │ equity c │        -400 │
└────────────┴──────────┴──────────┴─────────────┘

2) NET SENSITIVITIES (s_k)
┌──────────┬──────────┬────────┐
│ bucket   │ spot     │    s_k │
├──────────┼──────────┼────────┤
│ 2        │ equity c │   -400 │
│ 6        │ equity a │ -1,800 │
│ 6        │ equity b │   -700 │
└──────────┴──────────┴────────┘

3) WEIGHTED SENSITIVITIES (WS_k)
┌──────────┬──────────┬────────┬──────┬────────┐
│ bucket   │ spot     │    s_k │   rw │   WS_k │
├──────────┼──────────┼────────┼──────┼────────┤
│ 2        │ equity c │   -400 │  0.6 │   -240 │
│ 6        │ equity a │ -1,800 │ 0.35 │   -630 │
│ 6        │ equity b │   -700 │ 0.35 │   -245 │
└──────────┴──────────┴────

### CMDL

In [9]:
# ============================================================
# CMDL
# ============================================================


# TABLE OF CONTENTS
#   1) Gross sensitivities
#   2) Net sensitivities (s_k)
#   3) Weighted sensitivities (WS_k)
#   4) Intra-bucket correlation (rho_kl)
#   5) Intra-bucket aggregation (K_b)
#   6) Bucket sums (S_b)
#   7) Cross-bucket correlation (gamma_bc)
#   8) Cross-bucket aggregation
#   9) Correlation scenarios (high/low)
#  10) Capital
#  11) Printing
# ============================================================


# ============================================================
# 1) GROSS SENSITIVITIES
# ============================================================
# Position-level delta sensitivities to commodity spot prices.

data = [
    {'trade_id': 1, 'bucket': '7', 'product': 'silver', 'maturity': '0M', 'location': 'none', 'gross_s_k':  1700},
    {'trade_id': 2, 'bucket': '7', 'product': 'silver', 'maturity': '0M', 'location': 'none', 'gross_s_k': -1900},
    {'trade_id': 3, 'bucket': '7', 'product': 'silver', 'maturity': '3M', 'location': 'none', 'gross_s_k':   800},
    {'trade_id': 4, 'bucket': '7', 'product': 'silver', 'maturity': '3M', 'location': 'none', 'gross_s_k': -1300},
    {'trade_id': 5, 'bucket': '6', 'product': 'gas',    'maturity': '3M', 'location': 'UK',   'gross_s_k':  1200},
    {'trade_id': 6, 'bucket': '6', 'product': 'gas',    'maturity': '0M', 'location': 'UK',   'gross_s_k':  1900}
]

cmdl_gross_sk = pd.DataFrame(data)


# ============================================================
# 2) NET SENSITIVITIES (s_k)
# ============================================================
# Net sensitivity s_k = Σ gross sensitivities for each unique risk factor.
# Risk factor defined by: bucket + product + maturity + location.

cmdl_risk_factor_cols = ['bucket', 'product', 'maturity', 'location']

cmdl_net_sk = (
    cmdl_gross_sk
    .groupby(cmdl_risk_factor_cols)
    .agg(s_k=('gross_s_k', 'sum'))
    .reset_index()
)


# ============================================================
# 3) WEIGHTED SENSITIVITIES (WS_k)
# ============================================================
# WS_k = s_k × RW_k

cmdl_risk_weights = {
    '6': 0.45,
    '7': 0.20
}

cmdl_wsk = cmdl_net_sk.copy()
cmdl_wsk['rw'] = cmdl_wsk['bucket'].map(cmdl_risk_weights)
cmdl_wsk['WS_k'] = cmdl_wsk['s_k'] * cmdl_wsk['rw']

# Extract weighted sensitivities per bucket for aggregation
# Bucket 6: Gas UK (0M vs 3M)
# After groupby, order is likely alphabetical/numerical. 0M comes before 3M.
cmdl_ws_b6 = cmdl_wsk[cmdl_wsk['bucket'] == '6'].sort_values('maturity')['WS_k'].values
cmdl_WS_k_b6_0m = cmdl_ws_b6[0]  # 0M
cmdl_WS_k_b6_3m = cmdl_ws_b6[1]  # 3M

# Bucket 7: Silver (0M vs 3M)
cmdl_ws_b7 = cmdl_wsk[cmdl_wsk['bucket'] == '7'].sort_values('maturity')['WS_k'].values
cmdl_WS_k_b7_0m = cmdl_ws_b7[0]  # 0M
cmdl_WS_k_b7_3m = cmdl_ws_b7[1]  # 3M


# ============================================================
# 4) INTRA-BUCKET CORRELATION (rho_kl)
# ============================================================
# ρ_kl = ρ_commodity × ρ_tenor × ρ_basis (Article 325au)
# ρ_commodity = 1.0 (same product)
# ρ_tenor     = 0.99 (different tenors)
# ρ_basis     = 1.0 (same location)

# Bucket 6: Gas UK (0M vs 3M)
cmdl_rho_commodity_b6 = 1.0
cmdl_rho_tenor_b6     = 0.99
cmdl_rho_basis_b6     = 1.0
cmdl_rho_kl_b6        = cmdl_rho_commodity_b6 * cmdl_rho_tenor_b6 * cmdl_rho_basis_b6

# Bucket 7: Silver (0M vs 3M)
cmdl_rho_commodity_b7 = 1.0
cmdl_rho_tenor_b7     = 0.99
cmdl_rho_basis_b7     = 1.0
cmdl_rho_kl_b7        = cmdl_rho_commodity_b7 * cmdl_rho_tenor_b7 * cmdl_rho_basis_b7


# ============================================================
# 5) INTRA-BUCKET AGGREGATION (K_b)
# ============================================================
# K_b = √( Σ WS_k² + Σ Σ ρ_kl × WS_k × WS_l )  for k ≠ l

# Bucket 6: 2 risk factors
cmdl_sum_sq_b6 = cmdl_WS_k_b6_0m**2 + cmdl_WS_k_b6_3m**2
cmdl_cross_b6 = 2 * cmdl_rho_kl_b6 * cmdl_WS_k_b6_0m * cmdl_WS_k_b6_3m
cmdl_K_b6 = np.sqrt(max(0, cmdl_sum_sq_b6 + cmdl_cross_b6))

# Bucket 7: 2 risk factors
cmdl_sum_sq_b7 = cmdl_WS_k_b7_0m**2 + cmdl_WS_k_b7_3m**2
cmdl_cross_b7 = 2 * cmdl_rho_kl_b7 * cmdl_WS_k_b7_0m * cmdl_WS_k_b7_3m
cmdl_K_b7 = np.sqrt(max(0, cmdl_sum_sq_b7 + cmdl_cross_b7))

# --- Explicit K_b formula strings for printing ---
cmdl_Kb6_formula_str = (
    f"K_b6 = sqrt( ({cmdl_WS_k_b6_0m:.2f}^2 + {cmdl_WS_k_b6_3m:.2f}^2) "
    f"+ (2 * {cmdl_rho_kl_b6:.4f} * {cmdl_WS_k_b6_0m:.2f} * {cmdl_WS_k_b6_3m:.2f}) )"
)

cmdl_Kb7_formula_str = (
    f"K_b7 = sqrt( ({cmdl_WS_k_b7_0m:.2f}^2 + {cmdl_WS_k_b7_3m:.2f}^2) "
    f"+ (2 * {cmdl_rho_kl_b7:.4f} * {cmdl_WS_k_b7_0m:.2f} * {cmdl_WS_k_b7_3m:.2f}) )"
)


# ============================================================
# 6) BUCKET SUMS (S_b)
# ============================================================
# S_b = Σ WS_k

cmdl_S_b6 = cmdl_wsk[cmdl_wsk['bucket'] == '6']['WS_k'].sum()
cmdl_S_b7 = cmdl_wsk[cmdl_wsk['bucket'] == '7']['WS_k'].sum()


# ============================================================
# 7) CROSS-BUCKET CORRELATION (gamma_bc)
# ============================================================
# γ = 0.20 for buckets 1–10

cmdl_gamma = 0.20


# ============================================================
# 8) CROSS-BUCKET AGGREGATION
# ============================================================
# K = √( Σ K_b² + Σ Σ γ_bc × S_b × S_c )  for b ≠ c

cmdl_sum_K_sq = cmdl_K_b6**2 + cmdl_K_b7**2
cmdl_cross_bucket = 2 * cmdl_gamma * cmdl_S_b6 * cmdl_S_b7

cmdl_k_squared = cmdl_sum_K_sq + cmdl_cross_bucket

# Apply the check at the overall level (Floor at 0)
cmdl_k_squared = max(cmdl_k_squared, 0)

cmdl_K_medium = np.sqrt(cmdl_k_squared)

# --- Explicit Cross-bucket formula string for printing ---
cmdl_K_formula_str = (
    f"Capital = sqrt( ({cmdl_K_b6:.2f}^2 + {cmdl_K_b7:.2f}^2) "
    f"+ (2 * {cmdl_gamma:.2f} * {cmdl_S_b6:.2f} * {cmdl_S_b7:.2f}) )"
)


# ============================================================
# 9) CORRELATION SCENARIOS (HIGH/LOW)
# ============================================================
# High: ρ_high = min(1.25 × ρ_medium, 1.0)
# Low:  ρ_low  = max(2 × ρ_medium - 1, 0.75 × ρ_medium)

# --- High scenario correlations --- #
cmdl_rho_b6_high = min(cmdl_rho_kl_b6 * 1.25, 1.0)
cmdl_rho_b7_high = min(cmdl_rho_kl_b7 * 1.25, 1.0)
cmdl_gamma_high = min(cmdl_gamma * 1.25, 1.0)

# --- Low scenario correlations --- #
cmdl_rho_b6_low = max(2 * cmdl_rho_kl_b6 - 1.0, 0.75 * cmdl_rho_kl_b6)
cmdl_rho_b7_low = max(2 * cmdl_rho_kl_b7 - 1.0, 0.75 * cmdl_rho_kl_b7)
cmdl_gamma_low = max(2 * cmdl_gamma - 1.0, 0.75 * cmdl_gamma)

# --- High scenario intra-bucket K_b --- #
cmdl_cross_b6_high = 2 * cmdl_rho_b6_high * cmdl_WS_k_b6_0m * cmdl_WS_k_b6_3m
cmdl_K_b6_high = np.sqrt(max(0, cmdl_sum_sq_b6 + cmdl_cross_b6_high))

cmdl_cross_b7_high = 2 * cmdl_rho_b7_high * cmdl_WS_k_b7_0m * cmdl_WS_k_b7_3m
cmdl_K_b7_high = np.sqrt(max(0, cmdl_sum_sq_b7 + cmdl_cross_b7_high))

# --- Low scenario intra-bucket K_b --- #
cmdl_cross_b6_low = 2 * cmdl_rho_b6_low * cmdl_WS_k_b6_0m * cmdl_WS_k_b6_3m
cmdl_K_b6_low = np.sqrt(max(0, cmdl_sum_sq_b6 + cmdl_cross_b6_low))

cmdl_cross_b7_low = 2 * cmdl_rho_b7_low * cmdl_WS_k_b7_0m * cmdl_WS_k_b7_3m
cmdl_K_b7_low = np.sqrt(max(0, cmdl_sum_sq_b7 + cmdl_cross_b7_low))

# --- High scenario cross-bucket --- #
cmdl_k_sq_high = (
    cmdl_K_b6_high**2 + cmdl_K_b7_high**2
    + 2 * cmdl_gamma_high * cmdl_S_b6 * cmdl_S_b7
)
cmdl_K_high = np.sqrt(max(cmdl_k_sq_high, 0))

# --- Low scenario cross-bucket --- #
cmdl_k_sq_low = (
    cmdl_K_b6_low**2 + cmdl_K_b7_low**2
    + 2 * cmdl_gamma_low * cmdl_S_b6 * cmdl_S_b7
)
cmdl_K_low = np.sqrt(max(cmdl_k_sq_low, 0))


# ============================================================
# 10) CAPITAL
# ============================================================
# Final capital = max(K_medium, K_high, K_low)

cmdl_capital = max(cmdl_K_medium, cmdl_K_high, cmdl_K_low)


# ============================================================
# 11) PRINTING
# ============================================================

# --- 11.1 Gross sensitivities --- #
cmdl_gross_sk_print = format_table(
    cmdl_gross_sk,
    columns=['trade_id', 'bucket', 'product', 'maturity', 'location', 'gross_s_k'],
    col_formats={'gross_s_k': ',.0f'},
    alignments=('left', 'left', 'left', 'left', 'left', 'right')
)

# --- 11.2 Net sensitivities (s_k) --- #
cmdl_net_sk_print = format_table(
    cmdl_net_sk,
    columns=['bucket', 'product', 'maturity', 'location', 's_k'],
    col_formats={'s_k': ',.0f'},
    alignments=('left', 'left', 'left', 'left', 'right')
)

# --- 11.3 Weighted sensitivities (WS_k) --- #
cmdl_wsk_print = format_table(
    cmdl_wsk,
    columns=['bucket', 'product', 'maturity', 'location', 's_k', 'rw', 'WS_k'],
    col_formats={'s_k': ',.0f', 'rw': '.2f', 'WS_k': ',.2f'},
    alignments=('left', 'left', 'left', 'left', 'right', 'right', 'right')
)

# --- 11.4 Intra-bucket correlation (rho_kl) --- #
# Mapping rho parameters to match WS Risk Factor columns
cmdl_rho_print = format_table(
    pd.DataFrame([
        {'bucket': '6', 'pair': '0M vs 3M', 'rho_product': cmdl_rho_commodity_b6, 'rho_maturity': cmdl_rho_tenor_b6, 'rho_location': cmdl_rho_basis_b6, 'rho_kl': cmdl_rho_kl_b6},
        {'bucket': '7', 'pair': '0M vs 3M', 'rho_product': cmdl_rho_commodity_b7, 'rho_maturity': cmdl_rho_tenor_b7, 'rho_location': cmdl_rho_basis_b7, 'rho_kl': cmdl_rho_kl_b7},
    ]),
    columns=['bucket', 'pair', 'rho_product', 'rho_maturity', 'rho_location', 'rho_kl'],
    col_formats={'rho_product': '.2f', 'rho_maturity': '.2f', 'rho_location': '.2f', 'rho_kl': '.4f'},
    alignments=('left', 'left', 'right', 'right', 'right', 'right')
)

# --- 11.5 Intra-bucket aggregation (K_b) --- #
# Bucket 6
cmdl_K_b6_print = format_table(
    pd.DataFrame([
        {'component': 'WS_k_1 (0M)', 'value': cmdl_WS_k_b6_0m},
        {'component': 'WS_k_2 (3M)', 'value': cmdl_WS_k_b6_3m},
        {'component': 'rho_kl', 'value': f"{cmdl_rho_kl_b6:.4f}"},
        {'component': 'K_b', 'value': cmdl_K_b6},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# Bucket 7
cmdl_K_b7_print = format_table(
    pd.DataFrame([
        {'component': 'WS_k_1 (0M)', 'value': cmdl_WS_k_b7_0m},
        {'component': 'WS_k_2 (3M)', 'value': cmdl_WS_k_b7_3m},
        {'component': 'rho_kl', 'value': f"{cmdl_rho_kl_b7:.4f}"},
        {'component': 'K_b', 'value': cmdl_K_b7},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# --- 11.6 Bucket sums (S_b) --- #
cmdl_S_b_print = format_table(
    pd.DataFrame([
        {'bucket': '6', 'WS_k_1': cmdl_WS_k_b6_0m, 'WS_k_2': cmdl_WS_k_b6_3m, 'S_b': cmdl_S_b6},
        {'bucket': '7', 'WS_k_1': cmdl_WS_k_b7_0m, 'WS_k_2': cmdl_WS_k_b7_3m, 'S_b': cmdl_S_b7},
    ]),
    columns=['bucket', 'WS_k_1', 'WS_k_2', 'S_b'],
    col_formats={'WS_k_1': ',.2f', 'WS_k_2': ',.2f', 'S_b': ',.2f'},
    alignments=('left', 'right', 'right', 'right')
)

# --- 11.7 Cross-bucket correlation (gamma_bc) --- #
cmdl_gamma_print = format_table(
    pd.DataFrame([
        {'buckets': '6 vs 7', 'gamma_bc': cmdl_gamma},
    ]),
    columns=['buckets', 'gamma_bc'],
    col_formats={'gamma_bc': '.2f'},
    alignments=('left', 'right')
)

# --- 11.8 Cross-bucket aggregation --- #
cmdl_cross_agg_print = format_table(
    pd.DataFrame([
        {'component': 'K_b6', 'value': cmdl_K_b6},
        {'component': 'K_b7', 'value': cmdl_K_b7},
        {'component': 'S_b6', 'value': cmdl_S_b6},
        {'component': 'S_b7', 'value': cmdl_S_b7},
        {'component': 'gamma_bc', 'value': f"{cmdl_gamma:.2f}"},
        {'component': 'capital', 'value': cmdl_K_medium},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# --- 11.9 Correlation scenarios --- #
cmdl_scenarios_print = format_table(
    pd.DataFrame([
        {'parameter': 'rho_kl_b6', 'medium': cmdl_rho_kl_b6, 'high': cmdl_rho_b6_high, 'low': cmdl_rho_b6_low},
        {'parameter': 'rho_kl_b7', 'medium': cmdl_rho_kl_b7, 'high': cmdl_rho_b7_high, 'low': cmdl_rho_b7_low},
        {'parameter': 'gamma_bc', 'medium': cmdl_gamma, 'high': cmdl_gamma_high, 'low': cmdl_gamma_low},
    ]),
    columns=['parameter', 'medium', 'high', 'low'],
    col_formats={'medium': '.4f', 'high': '.4f', 'low': '.4f'},
    alignments=('left', 'right', 'right', 'right')
)

# --- 11.10 Capital --- #
cmdl_capital_print = format_table(
    pd.DataFrame([
        {'scenario': 'medium', 'capital': cmdl_K_medium},
        {'scenario': 'high', 'capital': cmdl_K_high},
        {'scenario': 'low', 'capital': cmdl_K_low},
    ]),
    columns=['scenario', 'capital'],
    col_formats={'capital': ',.2f'},
    alignments=('left', 'right')
)


# ============================================================
# DISPLAY ALL OUTPUTS
# ============================================================

print("=" * 60)
print("CMDL")
print("=" * 60)

print("\n1) GROSS SENSITIVITIES")
print(cmdl_gross_sk_print)

print("\n2) NET SENSITIVITIES (s_k)")
print(cmdl_net_sk_print)

print("\n3) WEIGHTED SENSITIVITIES (WS_k)")
print(cmdl_wsk_print)

print("\n4) INTRA-BUCKET CORRELATION (rho_kl)")
print(cmdl_rho_print)

print("\n5) INTRA-BUCKET AGGREGATION (K_b)")
print("\n--- Bucket 6 ---")
print(cmdl_Kb6_formula_str)
print(cmdl_K_b6_print)
print("\n--- Bucket 7 ---")
print(cmdl_Kb7_formula_str)
print(cmdl_K_b7_print)

print("\n6) BUCKET SUMS (S_b)")
print(cmdl_S_b_print)

print("\n7) CROSS-BUCKET CORRELATION (gamma_bc)")
print(cmdl_gamma_print)

print("\n8) CROSS-BUCKET AGGREGATION")
print(cmdl_K_formula_str)
print(cmdl_cross_agg_print)

print("\n9) CORRELATION SCENARIOS")
print(cmdl_scenarios_print)

print("\n10) CAPITAL")
print(cmdl_capital_print)

CMDL

1) GROSS SENSITIVITIES
┌────────────┬──────────┬───────────┬────────────┬────────────┬─────────────┐
│ trade_id   │ bucket   │ product   │ maturity   │ location   │   gross_s_k │
├────────────┼──────────┼───────────┼────────────┼────────────┼─────────────┤
│ 1          │ 7        │ silver    │ 0M         │ none       │       1,700 │
│ 2          │ 7        │ silver    │ 0M         │ none       │      -1,900 │
│ 3          │ 7        │ silver    │ 3M         │ none       │         800 │
│ 4          │ 7        │ silver    │ 3M         │ none       │      -1,300 │
│ 5          │ 6        │ gas       │ 3M         │ UK         │       1,200 │
│ 6          │ 6        │ gas       │ 0M         │ UK         │       1,900 │
└────────────┴──────────┴───────────┴────────────┴────────────┴─────────────┘

2) NET SENSITIVITIES (s_k)
┌──────────┬───────────┬────────────┬────────────┬───────┐
│ bucket   │ product   │ maturity   │ location   │   s_k │
├──────────┼───────────┼────────────┼────────

### FXDL

In [10]:
# ============================================================
# FXDL
# ============================================================


# TABLE OF CONTENTS
#   1) Gross sensitivities
#   2) Net sensitivities (s_k)
#   3) Weighted sensitivities (WS_k)
#   4) Intra-bucket correlation (rho_kl)
#   5) Intra-bucket aggregation (K_b)
#   6) Bucket sums (S_b)
#   7) Cross-bucket correlation (gamma_bc)
#   8) Cross-bucket aggregation
#   9) Correlation scenarios (high/low)
#  10) Capital
#  11) Printing
# ============================================================


# ============================================================
# 1) GROSS SENSITIVITIES
# ============================================================
# Position-level delta sensitivities to FX rates.

data = [
    {'trade_id': 3, 'bucket': 'gbp_usd', 'gross_s_k': -1000},
    {'trade_id': 4, 'bucket': 'gbp_usd', 'gross_s_k': -940},
    {'trade_id': 5, 'bucket': 'jpy_usd', 'gross_s_k': 1700},
    {'trade_id': 6, 'bucket': 'jpy_usd', 'gross_s_k': 1400},
]

fxdl_gross_sk = pd.DataFrame(data)


# ============================================================
# 2) NET SENSITIVITIES (s_k)
# ============================================================
# Net sensitivity s_k = Σ gross sensitivities for each unique risk factor.
# Risk factor defined by: bucket (currency pair).

fxdl_risk_factor_cols = ['bucket']

fxdl_net_sk = (
    fxdl_gross_sk
    .groupby(fxdl_risk_factor_cols)
    .agg(s_k=('gross_s_k', 'sum'))
    .reset_index()
)


# ============================================================
# 3) WEIGHTED SENSITIVITIES (WS_k)
# ============================================================
# WS_k = s_k × RW_k

# Risk weight (RW_k) from Article 325av
# For liquid FX pairs: RW = 15% / √2
fxdl_risk_weight = 0.15 / np.sqrt(2)

fxdl_wsk = fxdl_net_sk.copy()
fxdl_wsk['rw'] = fxdl_risk_weight
fxdl_wsk['WS_k'] = fxdl_wsk['s_k'] * fxdl_wsk['rw']

# Extract weighted sensitivities per bucket for aggregation
fxdl_WS_k_gbp_usd = fxdl_wsk[fxdl_wsk['bucket'] == 'gbp_usd']['WS_k'].values[0]
fxdl_WS_k_jpy_usd = fxdl_wsk[fxdl_wsk['bucket'] == 'jpy_usd']['WS_k'].values[0]


# ============================================================
# 4) INTRA-BUCKET CORRELATION (rho_kl)
# ============================================================
# For FX Delta, each bucket contains only one risk factor (the currency pair).
# Therefore, no intra-bucket correlation is needed (effectively N/A).

fxdl_rho_kl_gbp_usd = None
fxdl_rho_kl_jpy_usd = None


# ============================================================
# 5) INTRA-BUCKET AGGREGATION (K_b)
# ============================================================
# K_b = |WS_k| for single risk factor buckets

fxdl_K_gbp_usd = abs(fxdl_WS_k_gbp_usd)
fxdl_K_jpy_usd = abs(fxdl_WS_k_jpy_usd)

# --- Explicit K_b formula strings for printing ---
fxdl_Kb_gbp_formula_str = f"K_b = abs({fxdl_WS_k_gbp_usd:.2f})"
fxdl_Kb_jpy_formula_str = f"K_b = abs({fxdl_WS_k_jpy_usd:.2f})"


# ============================================================
# 6) BUCKET SUMS (S_b)
# ============================================================
# S_b = Σ WS_k  (simple sum of weighted sensitivities in bucket)

fxdl_S_gbp_usd = fxdl_WS_k_gbp_usd
fxdl_S_jpy_usd = fxdl_WS_k_jpy_usd


# ============================================================
# 7) CROSS-BUCKET CORRELATION (gamma_bc)
# ============================================================
# γ = 0.60 for FX bucket pairs (Article 325aw)

fxdl_gamma = 0.60


# ============================================================
# 8) CROSS-BUCKET AGGREGATION
# ============================================================
# K = √( Σ K_b² + Σ Σ γ_bc × S_b × S_c )  for b ≠ c

fxdl_sum_K_sq = fxdl_K_gbp_usd**2 + fxdl_K_jpy_usd**2
fxdl_cross_bucket = 2 * fxdl_gamma * fxdl_S_gbp_usd * fxdl_S_jpy_usd

fxdl_k_squared = fxdl_sum_K_sq + fxdl_cross_bucket

# Apply the check at the overall level (Floor at 0)
fxdl_k_squared = max(fxdl_k_squared, 0)

fxdl_K_medium = np.sqrt(fxdl_k_squared)

# --- Explicit Cross-bucket formula string for printing ---
fxdl_K_formula_str = (
    f"Capital = sqrt( ({fxdl_K_gbp_usd:.2f}^2 + {fxdl_K_jpy_usd:.2f}^2) "
    f"+ (2 * {fxdl_gamma:.2f} * {fxdl_S_gbp_usd:.2f} * {fxdl_S_jpy_usd:.2f}) )"
)


# ============================================================
# 9) CORRELATION SCENARIOS (HIGH/LOW)
# ============================================================
# High: γ_high = min(1.25 × γ_medium, 1.0)
# Low:  γ_low  = max(2 × γ_medium - 1, 0.75 × γ_medium)

# --- High scenario correlations --- #
fxdl_gamma_high = min(fxdl_gamma * 1.25, 1.0)

# --- High scenario intra-bucket K_b --- #
fxdl_K_gbp_usd_high = abs(fxdl_WS_k_gbp_usd)  # Unchanged
fxdl_K_jpy_usd_high = abs(fxdl_WS_k_jpy_usd)  # Unchanged

# --- High scenario cross-bucket --- #
fxdl_k_sq_high = (
    fxdl_K_gbp_usd_high**2 + fxdl_K_jpy_usd_high**2
    + 2 * fxdl_gamma_high * fxdl_S_gbp_usd * fxdl_S_jpy_usd
)
fxdl_K_high = np.sqrt(max(fxdl_k_sq_high, 0))

# --- Low scenario correlations --- #
fxdl_gamma_low = max(2 * fxdl_gamma - 1.0, 0.75 * fxdl_gamma)

# --- Low scenario intra-bucket K_b --- #
fxdl_K_gbp_usd_low = abs(fxdl_WS_k_gbp_usd)  # Unchanged
fxdl_K_jpy_usd_low = abs(fxdl_WS_k_jpy_usd)  # Unchanged

# --- Low scenario cross-bucket --- #
fxdl_k_sq_low = (
    fxdl_K_gbp_usd_low**2 + fxdl_K_jpy_usd_low**2
    + 2 * fxdl_gamma_low * fxdl_S_gbp_usd * fxdl_S_jpy_usd
)
fxdl_K_low = np.sqrt(max(fxdl_k_sq_low, 0))


# ============================================================
# 10) CAPITAL
# ============================================================
# Final capital = max(K_medium, K_high, K_low)

fxdl_capital = max(fxdl_K_medium, fxdl_K_high, fxdl_K_low)


# ============================================================
# 11) PRINTING
# ============================================================

# --- 11.1 Gross sensitivities --- #
fxdl_gross_sk_print = format_table(
    fxdl_gross_sk,
    columns=['trade_id', 'bucket', 'gross_s_k'],
    col_formats={'gross_s_k': ',.0f'},
    alignments=('left', 'left', 'right')
)

# --- 11.2 Net sensitivities (s_k) --- #
fxdl_net_sk_print = format_table(
    fxdl_net_sk,
    columns=['bucket', 's_k'],
    col_formats={'s_k': ',.0f'},
    alignments=('left', 'right')
)

# --- 11.3 Weighted sensitivities (WS_k) --- #
fxdl_wsk_print = format_table(
    fxdl_wsk,
    columns=['bucket', 's_k', 'rw', 'WS_k'],
    col_formats={'s_k': ',.0f', 'rw': '.4f', 'WS_k': ',.0f'},
    alignments=('left', 'right', 'right', 'right')
)

# --- 11.4 Intra-bucket correlation (rho_kl) --- #
fxdl_rho_print = format_table(
    pd.DataFrame([
        {'bucket': 'gbp_usd', 'pair': 'n/a', 'rho_kl': 'n/a'},
        {'bucket': 'jpy_usd', 'pair': 'n/a', 'rho_kl': 'n/a'},
    ]),
    columns=['bucket', 'pair', 'rho_kl'],
    col_formats={'rho_kl': '.4f'},
    alignments=('left', 'left', 'right')
)

# --- 11.5 Intra-bucket aggregation (K_b) --- #
# GBP Bucket
fxdl_K_gbp_print = format_table(
    pd.DataFrame([
        {'component': 'WS_k_1', 'value': fxdl_WS_k_gbp_usd},
        {'component': 'K_b', 'value': fxdl_K_gbp_usd},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# JPY Bucket
fxdl_K_jpy_print = format_table(
    pd.DataFrame([
        {'component': 'WS_k_1', 'value': fxdl_WS_k_jpy_usd},
        {'component': 'K_b', 'value': fxdl_K_jpy_usd},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# --- 11.6 Bucket sums (S_b) --- #
fxdl_S_b_print = format_table(
    pd.DataFrame([
        {'bucket': 'gbp_usd', 'WS_k_1': fxdl_WS_k_gbp_usd, 'S_b': fxdl_S_gbp_usd},
        {'bucket': 'jpy_usd', 'WS_k_1': fxdl_WS_k_jpy_usd, 'S_b': fxdl_S_jpy_usd},
    ]),
    columns=['bucket', 'WS_k_1', 'S_b'],
    col_formats={'WS_k_1': ',.2f', 'S_b': ',.2f'},
    alignments=('left', 'right', 'right')
)

# --- 11.7 Cross-bucket correlation (gamma_bc) --- #
fxdl_gamma_print = format_table(
    pd.DataFrame([
        {'buckets': 'GBP vs JPY', 'gamma_bc': fxdl_gamma},
    ]),
    columns=['buckets', 'gamma_bc'],
    col_formats={'gamma_bc': '.2f'},
    alignments=('left', 'right')
)

# --- 11.8 Cross-bucket aggregation --- #
fxdl_cross_agg_print = format_table(
    pd.DataFrame([
        {'component': 'K_gbp_usd', 'value': fxdl_K_gbp_usd},
        {'component': 'K_jpy_usd', 'value': fxdl_K_jpy_usd},
        {'component': 'S_gbp_usd', 'value': fxdl_S_gbp_usd},
        {'component': 'S_jpy_usd', 'value': fxdl_S_jpy_usd},
        {'component': 'gamma_bc', 'value': f"{fxdl_gamma:.2f}"},
        {'component': 'capital', 'value': fxdl_K_medium},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# --- 11.9 Correlation scenarios --- #
fxdl_scenarios_print = format_table(
    pd.DataFrame([
        {'parameter': 'gamma_bc', 'medium': fxdl_gamma, 'high': fxdl_gamma_high, 'low': fxdl_gamma_low},
    ]),
    columns=['parameter', 'medium', 'high', 'low'],
    col_formats={'medium': '.2f', 'high': '.2f', 'low': '.2f'},
    alignments=('left', 'right', 'right', 'right')
)

# --- 11.10 Capital --- #
fxdl_capital_print = format_table(
    pd.DataFrame([
        {'scenario': 'medium', 'capital': fxdl_K_medium},
        {'scenario': 'high', 'capital': fxdl_K_high},
        {'scenario': 'low', 'capital': fxdl_K_low},
    ]),
    columns=['scenario', 'capital'],
    col_formats={'capital': ',.2f'},
    alignments=('left', 'right')
)


# ============================================================
# DISPLAY ALL OUTPUTS
# ============================================================

print("=" * 60)
print("FXDL")
print("=" * 60)

print("\n1) GROSS SENSITIVITIES")
print(fxdl_gross_sk_print)

print("\n2) NET SENSITIVITIES (s_k)")
print(fxdl_net_sk_print)

print("\n3) WEIGHTED SENSITIVITIES (WS_k)")
print(fxdl_wsk_print)

print("\n4) INTRA-BUCKET CORRELATION (rho_kl)")
print(fxdl_rho_print)

print("\n5) INTRA-BUCKET AGGREGATION (K_b)")
print("\n--- GBP Bucket ---")
print(fxdl_Kb_gbp_formula_str)
print(fxdl_K_gbp_print)
print("\n--- JPY Bucket ---")
print(fxdl_Kb_jpy_formula_str)
print(fxdl_K_jpy_print)

print("\n6) BUCKET SUMS (S_b)")
print(fxdl_S_b_print)

print("\n7) CROSS-BUCKET CORRELATION (gamma_bc)")
print(fxdl_gamma_print)

print("\n8) CROSS-BUCKET AGGREGATION")
print(fxdl_K_formula_str)
print(fxdl_cross_agg_print)

print("\n9) CORRELATION SCENARIOS")
print(fxdl_scenarios_print)

print("\n10) CAPITAL")
print(fxdl_capital_print)

FXDL

1) GROSS SENSITIVITIES
┌────────────┬──────────┬─────────────┐
│ trade_id   │ bucket   │   gross_s_k │
├────────────┼──────────┼─────────────┤
│ 3          │ gbp_usd  │      -1,000 │
│ 4          │ gbp_usd  │        -940 │
│ 5          │ jpy_usd  │       1,700 │
│ 6          │ jpy_usd  │       1,400 │
└────────────┴──────────┴─────────────┘

2) NET SENSITIVITIES (s_k)
┌──────────┬────────┐
│ bucket   │    s_k │
├──────────┼────────┤
│ gbp_usd  │ -1,940 │
│ jpy_usd  │  3,100 │
└──────────┴────────┘

3) WEIGHTED SENSITIVITIES (WS_k)
┌──────────┬────────┬────────┬────────┐
│ bucket   │    s_k │     rw │   WS_k │
├──────────┼────────┼────────┼────────┤
│ gbp_usd  │ -1,940 │ 0.1061 │   -206 │
│ jpy_usd  │  3,100 │ 0.1061 │    329 │
└──────────┴────────┴────────┴────────┘

4) INTRA-BUCKET CORRELATION (rho_kl)
┌──────────┬────────┬──────────┐
│ bucket   │ pair   │   rho_kl │
├──────────┼────────┼──────────┤
│ gbp_usd  │ n/a    │      n/a │
│ jpy_usd  │ n/a    │      n/a │
└──────────┴──

## Vega

### IRVG

In [11]:
# ============================================================
# IRVG
# ============================================================


# TABLE OF CONTENTS
#   1) Gross sensitivities
#   2) Net sensitivities (s_k)
#   3) Weighted sensitivities (WS_k)
#   4) Intra-bucket correlation (rho_kl)
#   5) Intra-bucket aggregation (K_b)
#   6) Bucket sums (S_b)
#   7) Cross-bucket correlation (gamma_bc)
#   8) Cross-bucket aggregation
#   9) Correlation scenarios (high/low)
#  10) Capital
#  11) Printing
# ============================================================


# ============================================================
# 1) GROSS SENSITIVITIES
# ============================================================
# Position-level vega sensitivities to interest rate volatility.

data = [
    {'trade_id': 1, 'bucket': 'eur', 'rfr': 'inflation',  'opt_maturity': 10.0, 'under_maturity': np.nan, 'gross_s_k': 2700},
    {'trade_id': 2, 'bucket': 'eur', 'rfr': 'inflation',  'opt_maturity': 10.0, 'under_maturity': np.nan, 'gross_s_k': -1000},
    {'trade_id': 3, 'bucket': 'eur', 'rfr': 'inflation',  'opt_maturity': 0.5,  'under_maturity': np.nan, 'gross_s_k': -900},
    {'trade_id': 4, 'bucket': 'usd', 'rfr': 'ois',        'opt_maturity': 0.5,  'under_maturity': 1.0,    'gross_s_k': -530},
    {'trade_id': 5, 'bucket': 'usd', 'rfr': 'ois',        'opt_maturity': 0.5,  'under_maturity': 3.0,    'gross_s_k': -640},
]

irvg_gross_sk = pd.DataFrame(data)


# ============================================================
# 2) NET SENSITIVITIES (s_k)
# ============================================================
# Net sensitivity s_k = Σ gross sensitivities for each unique risk factor.
# Risk factor defined by: bucket + rfr + option_maturity + underlying_maturity.

irvg_risk_factor_cols = ['bucket', 'rfr', 'opt_maturity', 'under_maturity']

irvg_net_sk = (
    irvg_gross_sk
    .groupby(irvg_risk_factor_cols, dropna=False)
    .agg(s_k=('gross_s_k', 'sum'))
    .reset_index()
)


# ============================================================
# 3) WEIGHTED SENSITIVITIES (WS_k)
# ============================================================
# WS_k = s_k × RW_k

# Risk weight (RW_k) from Article 325ax
# For GIRR Vega: RW = 100%
irvg_risk_weight = 1.00

irvg_wsk = irvg_net_sk.copy()
irvg_wsk['rw'] = irvg_risk_weight
irvg_wsk['WS_k'] = irvg_wsk['s_k'] * irvg_wsk['rw']

# Extract weighted sensitivities per bucket for aggregation
# EUR bucket: 2 risk factors (10Y and 0.5Y option maturities, Inflation type)
irvg_ws_eur = irvg_wsk[irvg_wsk['bucket'] == 'eur']['WS_k'].values
irvg_WS_k_eur_10y = irvg_ws_eur[0]  # 10Y option maturity (netted: 27949 - 10962)
irvg_WS_k_eur_05y = irvg_ws_eur[1]  # 0.5Y option maturity

# USD bucket: 2 risk factors (0.5Y option maturity with 1Y and 3Y underlying)
irvg_ws_usd = irvg_wsk[irvg_wsk['bucket'] == 'usd']['WS_k'].values
irvg_WS_k_usd_1y = irvg_ws_usd[0]   # 0.5Y option, 1Y underlying
irvg_WS_k_usd_3y = irvg_ws_usd[1]   # 0.5Y option, 3Y underlying


# ============================================================
# 4) INTRA-BUCKET CORRELATION (rho_kl)
# ============================================================
# ρ_kl = min( ρ_opt × ρ_under , 1 )
# ρ_opt = exp(-0.01 * |Tk - Tl| / min(Tk, Tl))
# ρ_under = exp(-0.01 * |Uk - Ul| / min(Uk, Ul))  [Default]

# EUR Bucket: 10Y vs 0.5Y Option Maturity (Same Underlying: Inflation)
# Note: For Inflation, underlying maturity is often treated as constant/irrelevant, so rho_under = 1.0
tk_eur, tl_eur = 10.0, 0.5
irvg_rho_opt_eur = np.exp(-0.01 * abs(tk_eur - tl_eur) / min(tk_eur, tl_eur))
irvg_rho_under_eur = 1.0
irvg_rho_kl_eur = min(irvg_rho_opt_eur * irvg_rho_under_eur, 1.0)

# USD Bucket: 1Y vs 3Y Underlying Maturity (Same Option Maturity: 0.5Y)
# Option maturities are identical (0.5), so rho_opt = 1.0
tk_usd, tl_usd = 0.5, 0.5
irvg_rho_opt_usd = 1.0

# Underlying maturities: 1.0 vs 3.0
uk_usd, ul_usd = 1.0, 3.0
irvg_rho_under_usd = np.exp(-0.01 * abs(uk_usd - ul_usd) / min(uk_usd, ul_usd))
irvg_rho_kl_usd = min(irvg_rho_opt_usd * irvg_rho_under_usd, 1.0)


# ============================================================
# 5) INTRA-BUCKET AGGREGATION (K_b)
# ============================================================
# K_b = √( Σ WS_k² + Σ Σ ρ_kl × WS_k × WS_l )  for k ≠ l

# EUR bucket: 2 risk factors
irvg_sum_sq_eur = irvg_WS_k_eur_10y**2 + irvg_WS_k_eur_05y**2
irvg_cross_eur = 2 * irvg_rho_kl_eur * irvg_WS_k_eur_10y * irvg_WS_k_eur_05y
irvg_K_eur = np.sqrt(max(0, irvg_sum_sq_eur + irvg_cross_eur))

# USD bucket: 2 risk factors
irvg_sum_sq_usd = irvg_WS_k_usd_1y**2 + irvg_WS_k_usd_3y**2
irvg_cross_usd = 2 * irvg_rho_kl_usd * irvg_WS_k_usd_1y * irvg_WS_k_usd_3y
irvg_K_usd = np.sqrt(max(0, irvg_sum_sq_usd + irvg_cross_usd))

# --- Explicit K_b formula strings for printing ---
irvg_Kb_eur_formula_str = (
    f"K_eur = sqrt( ({irvg_WS_k_eur_10y:.2f}^2 + {irvg_WS_k_eur_05y:.2f}^2) "
    f"+ (2 * {irvg_rho_kl_eur:.4f} * {irvg_WS_k_eur_10y:.2f} * {irvg_WS_k_eur_05y:.2f}) )"
)

irvg_Kb_usd_formula_str = (
    f"K_usd = sqrt( ({irvg_WS_k_usd_1y:.2f}^2 + {irvg_WS_k_usd_3y:.2f}^2) "
    f"+ (2 * {irvg_rho_kl_usd:.4f} * {irvg_WS_k_usd_1y:.2f} * {irvg_WS_k_usd_3y:.2f}) )"
)


# ============================================================
# 6) BUCKET SUMS (S_b)
# ============================================================
# S_b = Σ WS_k  (simple sum of weighted sensitivities in bucket)

irvg_S_eur = irvg_wsk[irvg_wsk['bucket'] == 'eur']['WS_k'].sum()
irvg_S_usd = irvg_wsk[irvg_wsk['bucket'] == 'usd']['WS_k'].sum()


# ============================================================
# 7) CROSS-BUCKET CORRELATION (gamma_bc)
# ============================================================
# γ = 0.50 for GIRR bucket pairs (Article 325ag)

irvg_gamma = 0.50


# ============================================================
# 8) CROSS-BUCKET AGGREGATION
# ============================================================
# K = √( Σ K_b² + Σ Σ γ_bc × S_b × S_c )  for b ≠ c

irvg_sum_K_sq = irvg_K_eur**2 + irvg_K_usd**2
irvg_cross_bucket = 2 * irvg_gamma * irvg_S_eur * irvg_S_usd

irvg_k_squared = irvg_sum_K_sq + irvg_cross_bucket

# Apply the check at the overall level (Floor at 0)
irvg_k_squared = max(irvg_k_squared, 0)

irvg_K_medium = np.sqrt(irvg_k_squared)

# --- Explicit Cross-bucket formula string for printing ---
irvg_K_formula_str = (
    f"Capital = sqrt( ({irvg_K_eur:.2f}^2 + {irvg_K_usd:.2f}^2) "
    f"+ (2 * {irvg_gamma:.2f} * {irvg_S_eur:.2f} * {irvg_S_usd:.2f}) )"
)


# ============================================================
# 9) CORRELATION SCENARIOS (HIGH/LOW)
# ============================================================
# High: ρ_high = min(1.25 × ρ_medium, 1.0)
# Low:  ρ_low  = max(2 × ρ_medium - 1, 0.75 × ρ_medium)

# --- High scenario correlations --- #
irvg_rho_eur_high = min(irvg_rho_kl_eur * 1.25, 1.0)
irvg_rho_usd_high = min(irvg_rho_kl_usd * 1.25, 1.0)
irvg_gamma_high = min(irvg_gamma * 1.25, 1.0)

# --- Low scenario correlations --- #
irvg_rho_eur_low = max(2 * irvg_rho_kl_eur - 1.0, 0.75 * irvg_rho_kl_eur)
irvg_rho_usd_low = max(2 * irvg_rho_kl_usd - 1.0, 0.75 * irvg_rho_kl_usd)
irvg_gamma_low = max(2 * irvg_gamma - 1.0, 0.75 * irvg_gamma)

# --- High scenario intra-bucket K_b --- #
irvg_cross_eur_high = 2 * irvg_rho_eur_high * irvg_WS_k_eur_10y * irvg_WS_k_eur_05y
irvg_K_eur_high = np.sqrt(max(0, irvg_sum_sq_eur + irvg_cross_eur_high))

irvg_cross_usd_high = 2 * irvg_rho_usd_high * irvg_WS_k_usd_1y * irvg_WS_k_usd_3y
irvg_K_usd_high = np.sqrt(max(0, irvg_sum_sq_usd + irvg_cross_usd_high))

# --- Low scenario intra-bucket K_b --- #
irvg_cross_eur_low = 2 * irvg_rho_eur_low * irvg_WS_k_eur_10y * irvg_WS_k_eur_05y
irvg_K_eur_low = np.sqrt(max(0, irvg_sum_sq_eur + irvg_cross_eur_low))

irvg_cross_usd_low = 2 * irvg_rho_usd_low * irvg_WS_k_usd_1y * irvg_WS_k_usd_3y
irvg_K_usd_low = np.sqrt(max(0, irvg_sum_sq_usd + irvg_cross_usd_low))

# --- High scenario cross-bucket --- #
irvg_k_sq_high = (
    irvg_K_eur_high**2 + irvg_K_usd_high**2
    + 2 * irvg_gamma_high * irvg_S_eur * irvg_S_usd
)
irvg_K_high = np.sqrt(max(irvg_k_sq_high, 0))

# --- Low scenario cross-bucket --- #
irvg_k_sq_low = (
    irvg_K_eur_low**2 + irvg_K_usd_low**2
    + 2 * irvg_gamma_low * irvg_S_eur * irvg_S_usd
)
irvg_K_low = np.sqrt(max(irvg_k_sq_low, 0))


# ============================================================
# 10) CAPITAL
# ============================================================
# Final capital = max(K_medium, K_high, K_low)

irvg_capital = max(irvg_K_medium, irvg_K_high, irvg_K_low)


# ============================================================
# 11) PRINTING
# ============================================================

# --- 11.1 Gross sensitivities --- #
irvg_gross_sk_print = format_table(
    irvg_gross_sk,
    columns=['trade_id', 'bucket', 'rfr', 'opt_maturity', 'under_maturity', 'gross_s_k'],
    col_formats={'gross_s_k': ',.0f'},
    alignments=('left', 'left', 'left', 'right', 'right', 'right')
)

# --- 11.2 Net sensitivities (s_k) --- #
irvg_net_sk_print = format_table(
    irvg_net_sk,
    columns=['bucket', 'opt_maturity', 'under_maturity', 's_k'],
    col_formats={'s_k': ',.0f'},
    alignments=('left', 'right', 'right', 'right')
)

# --- 11.3 Weighted sensitivities (WS_k) --- #
irvg_wsk_print = format_table(
    irvg_wsk,
    columns=['bucket', 's_k', 'rw', 'WS_k'],
    col_formats={'s_k': ',.0f', 'rw': '.2f', 'WS_k': ',.0f'},
    alignments=('left', 'right', 'right', 'right')
)

# --- 11.4 Intra-bucket correlation (rho_kl) --- #
# Columns renamed to match specific Vega risk drivers
irvg_rho_print = format_table(
    pd.DataFrame([
        {'bucket': 'eur', 'pair': '10y vs 0.5y', 'rho_opt_maturity': irvg_rho_opt_eur, 'rho_under_maturity': irvg_rho_under_eur, 'rho_kl': irvg_rho_kl_eur},
        {'bucket': 'usd', 'pair': '1y vs 3y', 'rho_opt_maturity': irvg_rho_opt_usd, 'rho_under_maturity': irvg_rho_under_usd, 'rho_kl': irvg_rho_kl_usd},
    ]),
    columns=['bucket', 'pair', 'rho_opt_maturity', 'rho_under_maturity', 'rho_kl'],
    col_formats={'rho_opt_maturity': '.3f', 'rho_under_maturity': '.3f', 'rho_kl': '.3f'},
    alignments=('left', 'left', 'right', 'right', 'right')
)

# --- 11.5 Intra-bucket aggregation (K_b) --- #
# EUR Bucket
irvg_K_eur_print = format_table(
    pd.DataFrame([
        {'component': 'WS_k_1 (10Y)', 'value': irvg_WS_k_eur_10y},
        {'component': 'WS_k_2 (0.5Y)', 'value': irvg_WS_k_eur_05y},
        {'component': 'rho_kl', 'value': f"{irvg_rho_kl_eur:.3f}"},
        {'component': 'K_b', 'value': irvg_K_eur},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# USD Bucket
irvg_K_usd_print = format_table(
    pd.DataFrame([
        {'component': 'WS_k_1 (1Y)', 'value': irvg_WS_k_usd_1y},
        {'component': 'WS_k_2 (3Y)', 'value': irvg_WS_k_usd_3y},
        {'component': 'rho_kl', 'value': f"{irvg_rho_kl_usd:.3f}"},
        {'component': 'K_b', 'value': irvg_K_usd},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# --- 11.6 Bucket sums (S_b) --- #
irvg_S_b_print = format_table(
    pd.DataFrame([
        {'bucket': 'eur', 'WS_k_1': irvg_WS_k_eur_10y, 'WS_k_2': irvg_WS_k_eur_05y, 'S_b': irvg_S_eur},
        {'bucket': 'usd', 'WS_k_1': irvg_WS_k_usd_1y, 'WS_k_2': irvg_WS_k_usd_3y, 'S_b': irvg_S_usd},
    ]),
    columns=['bucket', 'WS_k_1', 'WS_k_2', 'S_b'],
    col_formats={'WS_k_1': ',.2f', 'WS_k_2': ',.2f', 'S_b': ',.2f'},
    alignments=('left', 'right', 'right', 'right')
)

# --- 11.7 Cross-bucket correlation (gamma_bc) --- #
irvg_gamma_print = format_table(
    pd.DataFrame([
        {'buckets': 'EUR vs USD', 'gamma_bc': irvg_gamma},
    ]),
    columns=['buckets', 'gamma_bc'],
    col_formats={'gamma_bc': '.2f'},
    alignments=('left', 'right')
)

# --- 11.8 Cross-bucket aggregation --- #
irvg_cross_agg_print = format_table(
    pd.DataFrame([
        {'component': 'K_eur', 'value': irvg_K_eur},
        {'component': 'K_usd', 'value': irvg_K_usd},
        {'component': 'S_eur', 'value': irvg_S_eur},
        {'component': 'S_usd', 'value': irvg_S_usd},
        {'component': 'gamma_bc', 'value': f"{irvg_gamma:.2f}"},
        {'component': 'capital', 'value': irvg_K_medium},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# --- 11.9 Correlation scenarios --- #
irvg_scenarios_print = format_table(
    pd.DataFrame([
        {'parameter': 'rho_kl_eur', 'medium': irvg_rho_kl_eur, 'high': irvg_rho_eur_high, 'low': irvg_rho_eur_low},
        {'parameter': 'rho_kl_usd', 'medium': irvg_rho_kl_usd, 'high': irvg_rho_usd_high, 'low': irvg_rho_usd_low},
        {'parameter': 'gamma_bc', 'medium': irvg_gamma, 'high': irvg_gamma_high, 'low': irvg_gamma_low},
    ]),
    columns=['parameter', 'medium', 'high', 'low'],
    col_formats={'medium': '.3f', 'high': '.3f', 'low': '.3f'},
    alignments=('left', 'right', 'right', 'right')
)

# --- 11.10 Capital --- #
irvg_capital_print = format_table(
    pd.DataFrame([
        {'scenario': 'medium', 'capital': irvg_K_medium},
        {'scenario': 'high', 'capital': irvg_K_high},
        {'scenario': 'low', 'capital': irvg_K_low},
    ]),
    columns=['scenario', 'capital'],
    col_formats={'capital': ',.2f'},
    alignments=('left', 'right')
)


# ============================================================
# DISPLAY ALL OUTPUTS
# ============================================================

print("=" * 60)
print("IRVG")
print("=" * 60)

print("\n1) GROSS SENSITIVITIES")
print(irvg_gross_sk_print)

print("\n2) NET SENSITIVITIES (s_k)")
print(irvg_net_sk_print)

print("\n3) WEIGHTED SENSITIVITIES (WS_k)")
print(irvg_wsk_print)

print("\n4) INTRA-BUCKET CORRELATION (rho_kl)")
print(irvg_rho_print)

print("\n5) INTRA-BUCKET AGGREGATION (K_b)")
print("\n--- EUR bucket ---")
print(irvg_Kb_eur_formula_str)
print(irvg_K_eur_print)
print("\n--- USD bucket ---")
print(irvg_Kb_usd_formula_str)
print(irvg_K_usd_print)

print("\n6) BUCKET SUMS (S_b)")
print(irvg_S_b_print)

print("\n7) CROSS-BUCKET CORRELATION (gamma_bc)")
print(irvg_gamma_print)

print("\n8) CROSS-BUCKET AGGREGATION")
print(irvg_K_formula_str)
print(irvg_cross_agg_print)

print("\n9) CORRELATION SCENARIOS")
print(irvg_scenarios_print)

print("\n10) CAPITAL")
print(irvg_capital_print)

IRVG

1) GROSS SENSITIVITIES
┌────────────┬──────────┬───────────┬────────────────┬──────────────────┬─────────────┐
│ trade_id   │ bucket   │ rfr       │   opt_maturity │   under_maturity │   gross_s_k │
├────────────┼──────────┼───────────┼────────────────┼──────────────────┼─────────────┤
│ 1          │ eur      │ inflation │             10 │              nan │       2,700 │
│ 2          │ eur      │ inflation │             10 │              nan │      -1,000 │
│ 3          │ eur      │ inflation │            0.5 │              nan │        -900 │
│ 4          │ usd      │ ois       │            0.5 │                1 │        -530 │
│ 5          │ usd      │ ois       │            0.5 │                3 │        -640 │
└────────────┴──────────┴───────────┴────────────────┴──────────────────┴─────────────┘

2) NET SENSITIVITIES (s_k)
┌──────────┬────────────────┬──────────────────┬───────┐
│ bucket   │   opt_maturity │   under_maturity │   s_k │
├──────────┼────────────────┼────────

### CRVG Non Sec

In [12]:
# ============================================================
# CRVG NON-SEC
# ============================================================


# TABLE OF CONTENTS
#   1) Gross sensitivities
#   2) Net sensitivities (s_k)
#   3) Weighted sensitivities (WS_k)
#   4) Intra-bucket correlation (rho_kl)
#   5) Intra-bucket aggregation (K_b)
#   6) Bucket sums (S_b)
#   7) Cross-bucket correlation (gamma_bc)
#   8) Cross-bucket aggregation
#   9) Correlation scenarios (high/low)
#  10) Capital
#  11) Printing
# ============================================================


# ============================================================
# 1) GROSS SENSITIVITIES
# ============================================================
# Position-level vega sensitivities to credit spread volatility.

data = [
    {'trade_id': 1, 'bucket': '5',  'rating': 'ig', 'sector': 'cons', 'curve': 'bond', 'issuer': 'issuer_1', 'opt_maturity': '5y', 'gross_s_k': 200},
    {'trade_id': 2, 'bucket': '5',  'rating': 'ig', 'sector': 'cons', 'curve': 'bond', 'issuer': 'issuer_1', 'opt_maturity': '5y', 'gross_s_k': 300},
    {'trade_id': 3, 'bucket': '5',  'rating': 'ig', 'sector': 'cons', 'curve': 'bond', 'issuer': 'issuer_2', 'opt_maturity': '5y', 'gross_s_k': -500},
    {'trade_id': 4, 'bucket': '5',  'rating': 'ig', 'sector': 'cons', 'curve': 'bond', 'issuer': 'issuer_2', 'opt_maturity': '5y', 'gross_s_k': -140},
]

crvg_non_sec_gross_sk = pd.DataFrame(data)


# ============================================================
# 2) NET SENSITIVITIES (s_k)
# ============================================================
# Net sensitivity s_k = Σ gross sensitivities for each unique risk factor.
# Risk factor defined by: bucket + issuer + option maturity.

crvg_non_sec_risk_factor_cols = ['bucket', 'rating', 'sector', 'curve', 'issuer', 'opt_maturity']

crvg_non_sec_net_sk = (
    crvg_non_sec_gross_sk
    .groupby(crvg_non_sec_risk_factor_cols)
    .agg(s_k=('gross_s_k', 'sum'))
    .reset_index()
)

# Helper map for numerical tenor
TENOR_TO_YEARS = {'5y': 5.0}
crvg_non_sec_net_sk['tenor_years'] = crvg_non_sec_net_sk['opt_maturity'].map(TENOR_TO_YEARS)


# ============================================================
# 3) WEIGHTED SENSITIVITIES (WS_k)
# ============================================================
# WS_k = s_k × RW_k

# Risk weight (RW_k) from Article 325ax
# For Vega CSR Non-Sec: RW = 100%
crvg_non_sec_risk_weight = 1.00

crvg_non_sec_wsk = crvg_non_sec_net_sk.copy()
crvg_non_sec_wsk['rw'] = crvg_non_sec_risk_weight
crvg_non_sec_wsk['WS_k'] = crvg_non_sec_wsk['s_k'] * crvg_non_sec_wsk['rw']

# Extract weighted sensitivities per bucket for aggregation
# Bucket 5: 2 risk factors (Issuer 1 vs Issuer 2, both 5Y tenor)
crvg_non_sec_ws_b5 = crvg_non_sec_wsk[crvg_non_sec_wsk['bucket'] == '5']
crvg_non_sec_WS_k_1_5y = crvg_non_sec_ws_b5[crvg_non_sec_ws_b5['issuer'] == 'issuer_1']['WS_k'].values[0]
crvg_non_sec_WS_k_2_5y = crvg_non_sec_ws_b5[crvg_non_sec_ws_b5['issuer'] == 'issuer_2']['WS_k'].values[0]


# ============================================================
# 4) INTRA-BUCKET CORRELATION (rho_kl)
# ============================================================
# ρ_kl = ρ_delta × ρ_option_maturity
# ρ_delta = 1.0 (same issuer) else 0.35 (Article 325ai)
# ρ_option_maturity = exp(-alpha * |Tk - Tl| / min(Tk, Tl)) if different tenors, else 1.0

ALPHA = 0.01

# Issuer 1 (5Y) vs Issuer 2 (5Y)
# Delta correlation (Different Issuer)
crvg_rho_delta = 0.35

# Option Maturity correlation (Same Tenor: 5Y vs 5Y)
# Note: Same tenor implies rho = 1.0
crvg_rho_opt_maturity = 1.0

crvg_rho_kl_b5 = crvg_rho_delta * crvg_rho_opt_maturity


# ============================================================
# 5) INTRA-BUCKET AGGREGATION (K_b)
# ============================================================
# K_b = √( Σ WS_k² + Σ Σ ρ_kl × WS_k × WS_l )  for k ≠ l

# Bucket 5: 2 risk factors
crvg_sum_sq_b5 = crvg_non_sec_WS_k_1_5y**2 + crvg_non_sec_WS_k_2_5y**2
crvg_cross_b5 = 2 * crvg_rho_kl_b5 * crvg_non_sec_WS_k_1_5y * crvg_non_sec_WS_k_2_5y
crvg_K_b5 = np.sqrt(max(0, crvg_sum_sq_b5 + crvg_cross_b5))

# --- Explicit K_b formula string for printing ---
crvg_Kb_formula_str = (
    f"K_b5 = sqrt( ({crvg_non_sec_WS_k_1_5y:.2f}^2 + {crvg_non_sec_WS_k_2_5y:.2f}^2) "
    f"+ (2 * {crvg_rho_kl_b5:.4f} * {crvg_non_sec_WS_k_1_5y:.2f} * {crvg_non_sec_WS_k_2_5y:.2f}) )"
)


# ============================================================
# 6) BUCKET SUMS (S_b)
# ============================================================
# S_b = Σ WS_k  (simple sum of weighted sensitivities in bucket)

crvg_non_sec_S_b5 = crvg_non_sec_wsk[crvg_non_sec_wsk['bucket'] == '5']['WS_k'].sum()


# ============================================================
# 7) CROSS-BUCKET CORRELATION (gamma_bc)
# ============================================================
# For single-bucket portfolios, cross-bucket correlation is not applicable.

crvg_non_sec_gamma = 0.0  # Not applicable for single bucket


# ============================================================
# 8) CROSS-BUCKET AGGREGATION
# ============================================================
# K = √( Σ K_b² + Σ Σ γ_bc × S_b × S_c )  for b ≠ c
# With single bucket, no cross-bucket term: K = K_b5

# Calculate the variance term
# FIX: Use crvg_K_b5 defined in Section 5
crvg_non_sec_k_squared = crvg_K_b5**2

# Apply the check at the overall level (Floor at 0)
crvg_non_sec_k_squared = max(crvg_non_sec_k_squared, 0)

crvg_non_sec_K_medium = np.sqrt(crvg_non_sec_k_squared)

# --- Explicit Cross-bucket formula string for printing ---
crvg_K_formula_str = (
    f"Capital = sqrt( {crvg_K_b5:.2f}^2 )"
)


# ============================================================
# 9) CORRELATION SCENARIOS (HIGH/LOW)
# ============================================================
# High: ρ_high = min(1.25 × ρ_medium, 1.0)
# Low:  ρ_low  = max(2 × ρ_medium - 1, 0.75 × ρ_medium)

# --- High scenario correlations --- #
crvg_rho_kl_b5_high = min(crvg_rho_kl_b5 * 1.25, 1.0)
crvg_gamma_high = min(crvg_non_sec_gamma * 1.25, 1.0)

# --- Low scenario correlations --- #
crvg_rho_kl_b5_low = max(2 * crvg_rho_kl_b5 - 1.0, 0.75 * crvg_rho_kl_b5)
crvg_gamma_low = max(2 * crvg_non_sec_gamma - 1.0, 0.75 * crvg_non_sec_gamma)

# --- High scenario intra-bucket K_b --- #
crvg_cross_b5_high = 2 * crvg_rho_kl_b5_high * crvg_non_sec_WS_k_1_5y * crvg_non_sec_WS_k_2_5y
crvg_K_b5_high = np.sqrt(max(0, crvg_sum_sq_b5 + crvg_cross_b5_high))

# --- Low scenario intra-bucket K_b --- #
crvg_cross_b5_low = 2 * crvg_rho_kl_b5_low * crvg_non_sec_WS_k_1_5y * crvg_non_sec_WS_k_2_5y
crvg_K_b5_low = np.sqrt(max(0, crvg_sum_sq_b5 + crvg_cross_b5_low))

# --- High scenario cross-bucket --- #
crvg_k_sq_high = crvg_K_b5_high**2
crvg_K_high = np.sqrt(max(crvg_k_sq_high, 0))

# --- Low scenario cross-bucket --- #
crvg_k_sq_low = crvg_K_b5_low**2
crvg_K_low = np.sqrt(max(crvg_k_sq_low, 0))


# ============================================================
# 10) CAPITAL
# ============================================================
# Final capital = max(K_medium, K_high, K_low)

crvg_non_sec_capital = max(crvg_non_sec_K_medium, crvg_K_high, crvg_K_low)


# ============================================================
# 11) PRINTING
# ============================================================

# --- 11.1 Gross sensitivities --- #
crvg_non_sec_gross_sk_print = format_table(
    crvg_non_sec_gross_sk,
    columns=['bucket', 'curve', 'issuer', 'opt_maturity', 'gross_s_k'],
    col_formats={'gross_s_k': ',.0f'},
    alignments=('left', 'left', 'left', 'left', 'right')
)

# --- 11.2 Net sensitivities (s_k) --- #
crvg_non_sec_net_sk_print = format_table(
    crvg_non_sec_net_sk,
    columns=['bucket', 'curve', 'issuer', 'opt_maturity', 's_k'],
    col_formats={'s_k': ',.0f'},
    alignments=('left', 'left', 'left', 'left', 'right')
)

# --- 11.3 Weighted sensitivities (WS_k) --- #
crvg_non_sec_wsk_print = format_table(
    crvg_non_sec_wsk,
    columns=['bucket', 'issuer', 'opt_maturity', 's_k', 'rw', 'WS_k'],
    col_formats={'s_k': ',.0f', 'rw': '.4f', 'WS_k': ',.2f'},
    alignments=('left', 'left', 'left', 'right', 'right', 'right')
)

# --- 11.4 Intra-bucket correlation (rho_kl) --- #
# Columns renamed to match Vega-specific correlation components
crvg_non_sec_rho_print = format_table(
    pd.DataFrame([
        {'bucket': '5', 'pair': 'Iss1 vs Iss2', 'rho_delta': crvg_rho_delta, 'rho_opt_maturity': crvg_rho_opt_maturity, 'rho_kl': crvg_rho_kl_b5},
    ]),
    columns=['bucket', 'pair', 'rho_delta', 'rho_opt_maturity', 'rho_kl'],
    col_formats={'rho_delta': '.2f', 'rho_opt_maturity': '.4f', 'rho_kl': '.4f'},
    alignments=('left', 'left', 'right', 'right', 'right')
)

# --- 11.5 Intra-bucket aggregation (K_b) --- #
# Bucket 5
crvg_non_sec_K_b5_print = format_table(
    pd.DataFrame([
        {'component': 'WS_k_1 (Iss1 5Y)', 'value': crvg_non_sec_WS_k_1_5y},
        {'component': 'WS_k_2 (Iss2 5Y)', 'value': crvg_non_sec_WS_k_2_5y},
        {'component': 'rho_kl', 'value': f"{crvg_rho_kl_b5:.4f}"},
        {'component': 'K_b', 'value': crvg_K_b5},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# --- 11.6 Bucket sums (S_b) --- #
crvg_non_sec_bucket_sums_print = format_table(
    pd.DataFrame([
        {'bucket': '5', 'WS_k_1': crvg_non_sec_WS_k_1_5y, 'WS_k_2': crvg_non_sec_WS_k_2_5y, 'S_b': crvg_non_sec_S_b5},
    ]),
    columns=['bucket', 'WS_k_1', 'WS_k_2', 'S_b'],
    col_formats={'WS_k_1': ',.2f', 'WS_k_2': ',.2f', 'S_b': ',.2f'},
    alignments=('left', 'right', 'right', 'right')
)

# --- 11.7 Cross-bucket correlation (gamma_bc) --- #
crvg_non_sec_gamma_print = format_table(
    pd.DataFrame([
        {'buckets': 'n/a (single)', 'gamma_bc': crvg_non_sec_gamma},
    ]),
    columns=['buckets', 'gamma_bc'],
    col_formats={'gamma_bc': '.2f'},
    alignments=('left', 'right')
)

# --- 11.8 Cross-bucket aggregation --- #
crvg_non_sec_cross_agg_print = format_table(
    pd.DataFrame([
        {'component': 'K_b5', 'value': crvg_K_b5},
        {'component': 'S_b5', 'value': crvg_non_sec_S_b5},
        {'component': 'gamma_bc', 'value': f"{crvg_non_sec_gamma:.2f}"},
        {'component': 'capital', 'value': crvg_non_sec_K_medium},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# --- 11.9 Correlation scenarios --- #
crvg_non_sec_scenarios_print = format_table(
    pd.DataFrame([
        {'parameter': 'rho_kl_b5', 'medium': crvg_rho_kl_b5, 'high': crvg_rho_kl_b5_high, 'low': crvg_rho_kl_b5_low},
        {'parameter': 'gamma_bc', 'medium': crvg_non_sec_gamma, 'high': crvg_gamma_high, 'low': crvg_gamma_low},
    ]),
    columns=['parameter', 'medium', 'high', 'low'],
    col_formats={'medium': '.4f', 'high': '.4f', 'low': '.4f'},
    alignments=('left', 'right', 'right', 'right')
)

# --- 11.10 Capital --- #
crvg_non_sec_capital_print = format_table(
    pd.DataFrame([
        {'scenario': 'medium', 'capital': crvg_non_sec_K_medium},
        {'scenario': 'high', 'capital': crvg_K_high},
        {'scenario': 'low', 'capital': crvg_K_low},
    ]),
    columns=['scenario', 'capital'],
    col_formats={'capital': ',.2f'},
    alignments=('left', 'right')
)


# ============================================================
# DISPLAY ALL OUTPUTS
# ============================================================

print("=" * 60)
print("CRVG NON-SECURITISATION")
print("=" * 60)

print("\n1) GROSS SENSITIVITIES")
print(crvg_non_sec_gross_sk_print)

print("\n2) NET SENSITIVITIES (s_k)")
print(crvg_non_sec_net_sk_print)

print("\n3) WEIGHTED SENSITIVITIES (WS_k)")
print(crvg_non_sec_wsk_print)

print("\n4) INTRA-BUCKET CORRELATION (rho_kl)")
print(crvg_non_sec_rho_print)

print("\n5) INTRA-BUCKET AGGREGATION (K_b)")
print("\n--- Bucket 5 ---")
print(crvg_Kb_formula_str)
print(crvg_non_sec_K_b5_print)

print("\n6) BUCKET SUMS (S_b)")
print(crvg_non_sec_bucket_sums_print)

print("\n7) CROSS-BUCKET CORRELATION (gamma_bc)")
print(crvg_non_sec_gamma_print)

print("\n8) CROSS-BUCKET AGGREGATION")
print(crvg_K_formula_str)
print(crvg_non_sec_cross_agg_print)

print("\n9) CORRELATION SCENARIOS")
print(crvg_non_sec_scenarios_print)

print("\n10) CAPITAL")
print(crvg_non_sec_capital_print)

CRVG NON-SECURITISATION

1) GROSS SENSITIVITIES
┌──────────┬─────────┬──────────┬────────────────┬─────────────┐
│ bucket   │ curve   │ issuer   │ opt_maturity   │   gross_s_k │
├──────────┼─────────┼──────────┼────────────────┼─────────────┤
│ 5        │ bond    │ issuer_1 │ 5y             │         200 │
│ 5        │ bond    │ issuer_1 │ 5y             │         300 │
│ 5        │ bond    │ issuer_2 │ 5y             │        -500 │
│ 5        │ bond    │ issuer_2 │ 5y             │        -140 │
└──────────┴─────────┴──────────┴────────────────┴─────────────┘

2) NET SENSITIVITIES (s_k)
┌──────────┬─────────┬──────────┬────────────────┬───────┐
│ bucket   │ curve   │ issuer   │ opt_maturity   │   s_k │
├──────────┼─────────┼──────────┼────────────────┼───────┤
│ 5        │ bond    │ issuer_1 │ 5y             │   500 │
│ 5        │ bond    │ issuer_2 │ 5y             │  -640 │
└──────────┴─────────┴──────────┴────────────────┴───────┘

3) WEIGHTED SENSITIVITIES (WS_k)
┌──────────┬────

### CRVG Sec CTP

In [13]:
# ============================================================
# CRVG SEC CTP
# ============================================================


# TABLE OF CONTENTS
#   1) Gross sensitivities
#   2) Net sensitivities (s_k)
#   3) Weighted sensitivities (WS_k)
#   4) Intra-bucket correlation (rho_kl)
#   5) Intra-bucket aggregation (K_b)
#   6) Bucket sums (S_b)
#   7) Cross-bucket correlation (gamma_bc)
#   8) Cross-bucket aggregation
#   9) Correlation scenarios (high/low)
#  10) Capital
#  11) Printing
# ============================================================


# ============================================================
# 1) GROSS SENSITIVITIES
# ============================================================
# Position-level vega sensitivities to credit spread volatility.

data = [
    {'trade_id': 1, 'bucket': '3', 'rating': 'ig', 'sector': 'fin',  'curve': 'bond',  'opt_maturity': '1y', 'issuer_underlying': 'issuer_1',  'gross_s_k': 350},
    {'trade_id': 2, 'bucket': '3', 'rating': 'ig', 'sector': 'fin',  'curve': 'bond',  'opt_maturity': '3y', 'issuer_underlying': 'issuer_2',  'gross_s_k': -450},
    {'trade_id': 3, 'bucket': '5', 'rating': 'ig', 'sector': 'cons', 'curve': 'bond',  'opt_maturity': '1y', 'issuer_underlying': 'issuer_3',  'gross_s_k': 200},
    {'trade_id': 4, 'bucket': '5', 'rating': 'ig', 'sector': 'cons', 'curve': 'bond',  'opt_maturity': '5y', 'issuer_underlying': 'issuer_4',  'gross_s_k': 500},
]

crvg_sec_ctp_gross_sk = pd.DataFrame(data)


# ============================================================
# 2) NET SENSITIVITIES (s_k)
# ============================================================
# Net sensitivity s_k = Σ gross sensitivities for each unique risk factor.
# Risk factor defined by: bucket + issuer_underlying + opt_maturity.

crvg_sec_ctp_risk_factor_cols = ['bucket', 'rating', 'sector', 'curve', 'issuer_underlying', 'opt_maturity']

crvg_sec_ctp_net_sk = (
    crvg_sec_ctp_gross_sk
    .groupby(crvg_sec_ctp_risk_factor_cols)
    .agg(s_k=('gross_s_k', 'sum'))
    .reset_index()
)

# Helper map for numerical tenor
TENOR_TO_YEARS = {'1y': 1.0, '3y': 3.0, '5y': 5.0}
crvg_sec_ctp_net_sk['tenor_years'] = crvg_sec_ctp_net_sk['opt_maturity'].map(TENOR_TO_YEARS)


# ============================================================
# 3) WEIGHTED SENSITIVITIES (WS_k)
# ============================================================
# WS_k = s_k × RW_k

# Risk weight (RW_k) from Article 325ax
# For Vega CSR Sec CTP: RW = 100%
crvg_sec_ctp_risk_weight = 1.00

crvg_sec_ctp_wsk = crvg_sec_ctp_net_sk.copy()
crvg_sec_ctp_wsk['rw'] = crvg_sec_ctp_risk_weight
crvg_sec_ctp_wsk['WS_k'] = crvg_sec_ctp_wsk['s_k'] * crvg_sec_ctp_wsk['rw']

# Extract weighted sensitivities per bucket for aggregation
# Bucket 3: Issuer 1 (1Y) vs Issuer 2 (3Y)
crvg_sec_ctp_ws_b3 = crvg_sec_ctp_wsk[crvg_sec_ctp_wsk['bucket'] == '3']
crvg_sec_ctp_WS_k_1_1y = crvg_sec_ctp_ws_b3[crvg_sec_ctp_ws_b3['issuer_underlying'] == 'issuer_1']['WS_k'].values[0]
crvg_sec_ctp_WS_k_2_3y = crvg_sec_ctp_ws_b3[crvg_sec_ctp_ws_b3['issuer_underlying'] == 'issuer_2']['WS_k'].values[0]

# Bucket 5: Issuer 3 (1Y) vs Issuer 4 (5Y)
crvg_sec_ctp_ws_b5 = crvg_sec_ctp_wsk[crvg_sec_ctp_wsk['bucket'] == '5']
crvg_sec_ctp_WS_k_3_1y = crvg_sec_ctp_ws_b5[crvg_sec_ctp_ws_b5['issuer_underlying'] == 'issuer_3']['WS_k'].values[0]
crvg_sec_ctp_WS_k_4_5y = crvg_sec_ctp_ws_b5[crvg_sec_ctp_ws_b5['issuer_underlying'] == 'issuer_4']['WS_k'].values[0]


# ============================================================
# 4) INTRA-BUCKET CORRELATION (rho_kl)
# ============================================================
# ρ_kl = ρ_delta × ρ_option_maturity (Article 325ax)
# ρ_delta = 1.0 (same issuer) else 0.35 (Article 325ai)
# ρ_option_maturity = exp(-alpha * |Tk - Tl| / min(Tk, Tl)) if different tenors, else 1.0

ALPHA = 0.01
RHO_DELTA_DIFFERENT_ISSUER = 0.35

# --- Bucket 3: Issuer 1 (1Y) vs Issuer 2 (3Y) ---
# Different issuers -> rho_delta = 0.35
crvg_rho_delta_b3 = RHO_DELTA_DIFFERENT_ISSUER

# Different tenors -> rho_opt
t_1y, t_3y = 1.0, 3.0
crvg_rho_opt_maturity_b3 = np.exp(-ALPHA * abs(t_1y - t_3y) / min(t_1y, t_3y))

crvg_rho_kl_b3 = min(crvg_rho_delta_b3 * crvg_rho_opt_maturity_b3, 1.0)

# --- Bucket 5: Issuer 3 (1Y) vs Issuer 4 (5Y) ---
# Different issuers -> rho_delta = 0.35
crvg_rho_delta_b5 = RHO_DELTA_DIFFERENT_ISSUER

# Different tenors -> rho_opt
t_1y, t_5y = 1.0, 5.0
crvg_rho_opt_maturity_b5 = np.exp(-ALPHA * abs(t_1y - t_5y) / min(t_1y, t_5y))

crvg_rho_kl_b5 = min(crvg_rho_delta_b5 * crvg_rho_opt_maturity_b5, 1.0)


# ============================================================
# 5) INTRA-BUCKET AGGREGATION (K_b)
# ============================================================
# K_b = √( Σ WS_k² + Σ Σ ρ_kl × WS_k × WS_l )  for k ≠ l

# Bucket 3: 2 risk factors
crvg_sum_sq_b3 = crvg_sec_ctp_WS_k_1_1y**2 + crvg_sec_ctp_WS_k_2_3y**2
crvg_cross_b3 = 2 * crvg_rho_kl_b3 * crvg_sec_ctp_WS_k_1_1y * crvg_sec_ctp_WS_k_2_3y
crvg_sec_ctp_K_b3 = np.sqrt(max(0, crvg_sum_sq_b3 + crvg_cross_b3))

# Bucket 5: 2 risk factors
crvg_sum_sq_b5 = crvg_sec_ctp_WS_k_3_1y**2 + crvg_sec_ctp_WS_k_4_5y**2
crvg_cross_b5 = 2 * crvg_rho_kl_b5 * crvg_sec_ctp_WS_k_3_1y * crvg_sec_ctp_WS_k_4_5y
crvg_sec_ctp_K_b5 = np.sqrt(max(0, crvg_sum_sq_b5 + crvg_cross_b5))

# --- Explicit K_b formula strings for printing ---
crvg_sec_ctp_Kb3_formula_str = (
    f"K_b3 = sqrt( ({crvg_sec_ctp_WS_k_1_1y:.2f}^2 + {crvg_sec_ctp_WS_k_2_3y:.2f}^2) "
    f"+ (2 * {crvg_rho_kl_b3:.4f} * {crvg_sec_ctp_WS_k_1_1y:.2f} * {crvg_sec_ctp_WS_k_2_3y:.2f}) )"
)

crvg_sec_ctp_Kb5_formula_str = (
    f"K_b5 = sqrt( ({crvg_sec_ctp_WS_k_3_1y:.2f}^2 + {crvg_sec_ctp_WS_k_4_5y:.2f}^2) "
    f"+ (2 * {crvg_rho_kl_b5:.4f} * {crvg_sec_ctp_WS_k_3_1y:.2f} * {crvg_sec_ctp_WS_k_4_5y:.2f}) )"
)


# ============================================================
# 6) BUCKET SUMS (S_b)
# ============================================================
# S_b = Σ WS_k  (simple sum of weighted sensitivities in bucket)

crvg_sec_ctp_S_b3 = crvg_sec_ctp_wsk[crvg_sec_ctp_wsk['bucket'] == '3']['WS_k'].sum()
crvg_sec_ctp_S_b5 = crvg_sec_ctp_wsk[crvg_sec_ctp_wsk['bucket'] == '5']['WS_k'].sum()


# ============================================================
# 7) CROSS-BUCKET CORRELATION (gamma_bc)
# ============================================================
# γ = 0.15 for buckets 3 (Financials) vs 5 (Consumer)

crvg_sec_ctp_gamma = 0.15


# ============================================================
# 8) CROSS-BUCKET AGGREGATION
# ============================================================
# K = √( Σ K_b² + Σ Σ γ_bc × S_b × S_c )  for b ≠ c

crvg_sum_K_sq = crvg_sec_ctp_K_b3**2 + crvg_sec_ctp_K_b5**2
crvg_cross_bucket = 2 * crvg_sec_ctp_gamma * crvg_sec_ctp_S_b3 * crvg_sec_ctp_S_b5

crvg_sec_ctp_k_squared = crvg_sum_K_sq + crvg_cross_bucket

# Apply the check at the overall level (Floor at 0)
crvg_sec_ctp_k_squared = max(crvg_sec_ctp_k_squared, 0)

crvg_sec_ctp_K_medium = np.sqrt(crvg_sec_ctp_k_squared)

# --- Explicit Cross-bucket formula string for printing ---
crvg_sec_ctp_K_formula_str = (
    f"Capital = sqrt( ({crvg_sec_ctp_K_b3:.2f}^2 + {crvg_sec_ctp_K_b5:.2f}^2) "
    f"+ (2 * {crvg_sec_ctp_gamma:.2f} * {crvg_sec_ctp_S_b3:.2f} * {crvg_sec_ctp_S_b5:.2f}) )"
)


# ============================================================
# 9) CORRELATION SCENARIOS (HIGH/LOW)
# ============================================================
# High: ρ_high = min(1.25 × ρ_medium, 1.0)
# Low:  ρ_low  = max(2 × ρ_medium - 1, 0.75 × ρ_medium)

# --- High scenario correlations --- #
crvg_rho_kl_b3_high = min(crvg_rho_kl_b3 * 1.25, 1.0)
crvg_rho_kl_b5_high = min(crvg_rho_kl_b5 * 1.25, 1.0)
crvg_gamma_high = min(crvg_sec_ctp_gamma * 1.25, 1.0)

# --- Low scenario correlations --- #
crvg_rho_kl_b3_low = max(2 * crvg_rho_kl_b3 - 1.0, 0.75 * crvg_rho_kl_b3)
crvg_rho_kl_b5_low = max(2 * crvg_rho_kl_b5 - 1.0, 0.75 * crvg_rho_kl_b5)
crvg_gamma_low = max(2 * crvg_sec_ctp_gamma - 1.0, 0.75 * crvg_sec_ctp_gamma)

# --- High scenario intra-bucket K_b --- #
crvg_cross_b3_high = 2 * crvg_rho_kl_b3_high * crvg_sec_ctp_WS_k_1_1y * crvg_sec_ctp_WS_k_2_3y
crvg_sec_ctp_K_b3_high = np.sqrt(max(0, crvg_sum_sq_b3 + crvg_cross_b3_high))

crvg_cross_b5_high = 2 * crvg_rho_kl_b5_high * crvg_sec_ctp_WS_k_3_1y * crvg_sec_ctp_WS_k_4_5y
crvg_sec_ctp_K_b5_high = np.sqrt(max(0, crvg_sum_sq_b5 + crvg_cross_b5_high))

# --- Low scenario intra-bucket K_b --- #
crvg_cross_b3_low = 2 * crvg_rho_kl_b3_low * crvg_sec_ctp_WS_k_1_1y * crvg_sec_ctp_WS_k_2_3y
crvg_sec_ctp_K_b3_low = np.sqrt(max(0, crvg_sum_sq_b3 + crvg_cross_b3_low))

crvg_cross_b5_low = 2 * crvg_rho_kl_b5_low * crvg_sec_ctp_WS_k_3_1y * crvg_sec_ctp_WS_k_4_5y
crvg_sec_ctp_K_b5_low = np.sqrt(max(0, crvg_sum_sq_b5 + crvg_cross_b5_low))

# --- High scenario cross-bucket --- #
crvg_k_sq_high = (
    crvg_sec_ctp_K_b3_high**2 + crvg_sec_ctp_K_b5_high**2
    + 2 * crvg_gamma_high * crvg_sec_ctp_S_b3 * crvg_sec_ctp_S_b5
)
crvg_sec_ctp_K_high = np.sqrt(max(crvg_k_sq_high, 0))

# --- Low scenario cross-bucket --- #
crvg_k_sq_low = (
    crvg_sec_ctp_K_b3_low**2 + crvg_sec_ctp_K_b5_low**2
    + 2 * crvg_gamma_low * crvg_sec_ctp_S_b3 * crvg_sec_ctp_S_b5
)
crvg_sec_ctp_K_low = np.sqrt(max(crvg_k_sq_low, 0))


# ============================================================
# 10) CAPITAL
# ============================================================
# Final capital = max(K_medium, K_high, K_low)

crvg_sec_ctp_capital = max(crvg_sec_ctp_K_medium, crvg_sec_ctp_K_high, crvg_sec_ctp_K_low)


# ============================================================
# 11) PRINTING
# ============================================================

# --- 11.1 Gross sensitivities --- #
crvg_sec_ctp_gross_sk_print = format_table(
    crvg_sec_ctp_gross_sk,
    columns=['bucket', 'curve', 'issuer_underlying', 'opt_maturity', 'gross_s_k'],
    col_formats={'gross_s_k': ',.0f'},
    alignments=('left', 'left', 'left', 'left', 'right')
)

# --- 11.2 Net sensitivities (s_k) --- #
crvg_sec_ctp_net_sk_print = format_table(
    crvg_sec_ctp_net_sk,
    columns=['bucket', 'curve', 'issuer_underlying', 'opt_maturity', 's_k'],
    col_formats={'s_k': ',.0f'},
    alignments=('left', 'left', 'left', 'left', 'right')
)

# --- 11.3 Weighted sensitivities (WS_k) --- #
crvg_sec_ctp_wsk_print = format_table(
    crvg_sec_ctp_wsk,
    columns=['bucket', 'issuer_underlying', 'opt_maturity', 's_k', 'rw', 'WS_k'],
    col_formats={'s_k': ',.0f', 'rw': '.4f', 'WS_k': ',.2f'},
    alignments=('left', 'left', 'left', 'right', 'right', 'right')
)

# --- 11.4 Intra-bucket correlation (rho_kl) --- #
# Columns renamed to match Vega-specific correlation components
crvg_sec_ctp_rho_print = format_table(
    pd.DataFrame([
        {'bucket': '3', 'pair': 'Iss1 1Y vs Iss2 3Y', 'rho_delta': crvg_rho_delta_b3, 'rho_opt_maturity': crvg_rho_opt_maturity_b3, 'rho_kl': crvg_rho_kl_b3},
        {'bucket': '5', 'pair': 'Iss3 1Y vs Iss4 5Y', 'rho_delta': crvg_rho_delta_b5, 'rho_opt_maturity': crvg_rho_opt_maturity_b5, 'rho_kl': crvg_rho_kl_b5},
    ]),
    columns=['bucket', 'pair', 'rho_delta', 'rho_opt_maturity', 'rho_kl'],
    col_formats={'rho_delta': '.2f', 'rho_opt_maturity': '.4f', 'rho_kl': '.4f'},
    alignments=('left', 'left', 'right', 'right', 'right')
)

# --- 11.5 Intra-bucket aggregation (K_b) --- #
# Bucket 3
crvg_sec_ctp_K_b3_print = format_table(
    pd.DataFrame([
        {'component': 'WS_k_1 (Iss1 1Y)', 'value': crvg_sec_ctp_WS_k_1_1y},
        {'component': 'WS_k_2 (Iss2 3Y)', 'value': crvg_sec_ctp_WS_k_2_3y},
        {'component': 'rho_kl', 'value': f"{crvg_rho_kl_b3:.4f}"},
        {'component': 'K_b', 'value': crvg_sec_ctp_K_b3},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# Bucket 5
crvg_sec_ctp_K_b5_print = format_table(
    pd.DataFrame([
        {'component': 'WS_k_1 (Iss3 1Y)', 'value': crvg_sec_ctp_WS_k_3_1y},
        {'component': 'WS_k_2 (Iss4 5Y)', 'value': crvg_sec_ctp_WS_k_4_5y},
        {'component': 'rho_kl', 'value': f"{crvg_rho_kl_b5:.4f}"},
        {'component': 'K_b', 'value': crvg_sec_ctp_K_b5},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# --- 11.6 Bucket sums (S_b) --- #
crvg_sec_ctp_bucket_sums_print = format_table(
    pd.DataFrame([
        {'bucket': '3', 'WS_k_1': crvg_sec_ctp_WS_k_1_1y, 'WS_k_2': crvg_sec_ctp_WS_k_2_3y, 'S_b': crvg_sec_ctp_S_b3},
        {'bucket': '5', 'WS_k_1': crvg_sec_ctp_WS_k_3_1y, 'WS_k_2': crvg_sec_ctp_WS_k_4_5y, 'S_b': crvg_sec_ctp_S_b5},
    ]),
    columns=['bucket', 'WS_k_1', 'WS_k_2', 'S_b'],
    col_formats={'WS_k_1': ',.2f', 'WS_k_2': ',.2f', 'S_b': ',.2f'},
    alignments=('left', 'right', 'right', 'right')
)

# --- 11.7 Cross-bucket correlation (gamma_bc) --- #
crvg_sec_ctp_gamma_print = format_table(
    pd.DataFrame([
        {'buckets': '3 vs 5', 'gamma_bc': crvg_sec_ctp_gamma},
    ]),
    columns=['buckets', 'gamma_bc'],
    col_formats={'gamma_bc': '.2f'},
    alignments=('left', 'right')
)

# --- 11.8 Cross-bucket aggregation --- #
crvg_sec_ctp_cross_agg_print = format_table(
    pd.DataFrame([
        {'component': 'K_b3', 'value': crvg_sec_ctp_K_b3},
        {'component': 'K_b5', 'value': crvg_sec_ctp_K_b5},
        {'component': 'S_b3', 'value': crvg_sec_ctp_S_b3},
        {'component': 'S_b5', 'value': crvg_sec_ctp_S_b5},
        {'component': 'gamma_bc', 'value': f"{crvg_sec_ctp_gamma:.2f}"},
        {'component': 'capital', 'value': crvg_sec_ctp_K_medium},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# --- 11.9 Correlation scenarios --- #
crvg_sec_ctp_scenarios_print = format_table(
    pd.DataFrame([
        {'parameter': 'rho_kl_b3', 'medium': crvg_rho_kl_b3, 'high': crvg_rho_kl_b3_high, 'low': crvg_rho_kl_b3_low},
        {'parameter': 'rho_kl_b5', 'medium': crvg_rho_kl_b5, 'high': crvg_rho_kl_b5_high, 'low': crvg_rho_kl_b5_low},
        {'parameter': 'gamma_bc', 'medium': crvg_sec_ctp_gamma, 'high': crvg_gamma_high, 'low': crvg_gamma_low},
    ]),
    columns=['parameter', 'medium', 'high', 'low'],
    col_formats={'medium': '.4f', 'high': '.4f', 'low': '.4f'},
    alignments=('left', 'right', 'right', 'right')
)

# --- 11.10 Capital --- #
crvg_sec_ctp_capital_print = format_table(
    pd.DataFrame([
        {'scenario': 'medium', 'capital': crvg_sec_ctp_K_medium},
        {'scenario': 'high', 'capital': crvg_sec_ctp_K_high},
        {'scenario': 'low', 'capital': crvg_sec_ctp_K_low},
    ]),
    columns=['scenario', 'capital'],
    col_formats={'capital': ',.2f'},
    alignments=('left', 'right')
)


# ============================================================
# DISPLAY ALL OUTPUTS
# ============================================================

print("=" * 60)
print("CRVG SECURITISATION CTP")
print("=" * 60)

print("\n1) GROSS SENSITIVITIES")
print(crvg_sec_ctp_gross_sk_print)

print("\n2) NET SENSITIVITIES (s_k)")
print(crvg_sec_ctp_net_sk_print)

print("\n3) WEIGHTED SENSITIVITIES (WS_k)")
print(crvg_sec_ctp_wsk_print)

print("\n4) INTRA-BUCKET CORRELATION (rho_kl)")
print(crvg_sec_ctp_rho_print)

print("\n5) INTRA-BUCKET AGGREGATION (K_b)")
print("\n--- Bucket 3 ---")
print(crvg_sec_ctp_Kb3_formula_str)
print(crvg_sec_ctp_K_b3_print)
print("\n--- Bucket 5 ---")
print(crvg_sec_ctp_Kb5_formula_str)
print(crvg_sec_ctp_K_b5_print)

print("\n6) BUCKET SUMS (S_b)")
print(crvg_sec_ctp_bucket_sums_print)

print("\n7) CROSS-BUCKET CORRELATION (gamma_bc)")
print(crvg_sec_ctp_gamma_print)

print("\n8) CROSS-BUCKET AGGREGATION")
print(crvg_sec_ctp_K_formula_str)
print(crvg_sec_ctp_cross_agg_print)

print("\n9) CORRELATION SCENARIOS")
print(crvg_sec_ctp_scenarios_print)

print("\n10) CAPITAL")
print(crvg_sec_ctp_capital_print)

CRVG SECURITISATION CTP

1) GROSS SENSITIVITIES
┌──────────┬─────────┬─────────────────────┬────────────────┬─────────────┐
│ bucket   │ curve   │ issuer_underlying   │ opt_maturity   │   gross_s_k │
├──────────┼─────────┼─────────────────────┼────────────────┼─────────────┤
│ 3        │ bond    │ issuer_1            │ 1y             │         350 │
│ 3        │ bond    │ issuer_2            │ 3y             │        -450 │
│ 5        │ bond    │ issuer_3            │ 1y             │         200 │
│ 5        │ bond    │ issuer_4            │ 5y             │         500 │
└──────────┴─────────┴─────────────────────┴────────────────┴─────────────┘

2) NET SENSITIVITIES (s_k)
┌──────────┬─────────┬─────────────────────┬────────────────┬───────┐
│ bucket   │ curve   │ issuer_underlying   │ opt_maturity   │   s_k │
├──────────┼─────────┼─────────────────────┼────────────────┼───────┤
│ 3        │ bond    │ issuer_1            │ 1y             │   350 │
│ 3        │ bond    │ issuer_2     

### CRVG Sec

In [14]:
# ============================================================
# CRVG SEC
# ============================================================


# TABLE OF CONTENTS
#   1) Gross sensitivities
#   2) Net sensitivities (s_k)
#   3) Weighted sensitivities (WS_k)
#   4) Intra-bucket correlation (rho_kl)
#   5) Intra-bucket aggregation (K_b)
#   6) Bucket sums (S_b)
#   7) Cross-bucket correlation (gamma_bc)
#   8) Cross-bucket aggregation
#   9) Correlation scenarios (high/low)
#  10) Capital
#  11) Printing
# ============================================================


# ============================================================
# 1) GROSS SENSITIVITIES
# ============================================================
# Position-level vega sensitivities to credit spread volatility.

data = [
    {'trade_id': 1, 'bucket': '1',  'rating': 'IG',  'sector': 'rmbs_prime',  'tranche': 'tranche_a', 'opt_maturity': '1y', 'gross_s_k': 800},
    {'trade_id': 2, 'bucket': '1',  'rating': 'IG',  'sector': 'rmbs_prime',  'tranche': 'tranche_b', 'opt_maturity': '3y', 'gross_s_k': -600},
    {'trade_id': 3, 'bucket': '12', 'rating': 'NIG', 'sector': 'cmbs',        'tranche': 'tranche_c', 'opt_maturity': '1y', 'gross_s_k': 900},
    {'trade_id': 4, 'bucket': '12', 'rating': 'NIG', 'sector': 'cmbs',        'tranche': 'tranche_c', 'opt_maturity': '5y', 'gross_s_k': -700},
]

crvg_sec_gross_sk = pd.DataFrame(data)


# ============================================================
# 2) NET SENSITIVITIES (s_k)
# ============================================================
# Net sensitivity s_k = Σ gross sensitivities for each unique risk factor.
# Risk factor defined by: bucket + tranche + opt_maturity.

crvg_sec_risk_factor_cols = ['bucket', 'rating', 'sector', 'tranche', 'opt_maturity']

crvg_sec_net_sk = (
    crvg_sec_gross_sk
    .groupby(crvg_sec_risk_factor_cols)
    .agg(s_k=('gross_s_k', 'sum'))
    .reset_index()
)

# Helper map for numerical tenor
TENOR_TO_YEARS = {'1y': 1.0, '3y': 3.0, '5y': 5.0}
crvg_sec_net_sk['tenor_years'] = crvg_sec_net_sk['opt_maturity'].map(TENOR_TO_YEARS)


# ============================================================
# 3) WEIGHTED SENSITIVITIES (WS_k)
# ============================================================
# WS_k = s_k × RW_k

# Risk weight (RW_k) from Article 325ax
# For Vega CSR Sec Non-CTP: RW = 100%
crvg_sec_risk_weight = 1.00

crvg_sec_wsk = crvg_sec_net_sk.copy()
crvg_sec_wsk['rw'] = crvg_sec_risk_weight
crvg_sec_wsk['WS_k'] = crvg_sec_wsk['s_k'] * crvg_sec_wsk['rw']

# Extract weighted sensitivities per bucket for aggregation
# Bucket 1: Tranche A (1Y) vs Tranche B (3Y)
crvg_sec_ws_b1 = crvg_sec_wsk[crvg_sec_wsk['bucket'] == '1']
crvg_sec_WS_k_a_1y = crvg_sec_ws_b1[crvg_sec_ws_b1['tranche'] == 'tranche_a']['WS_k'].values[0]
crvg_sec_WS_k_b_3y = crvg_sec_ws_b1[crvg_sec_ws_b1['tranche'] == 'tranche_b']['WS_k'].values[0]

# Bucket 12: Tranche C (1Y) vs Tranche C (5Y)
crvg_sec_ws_b12 = crvg_sec_wsk[crvg_sec_wsk['bucket'] == '12']
crvg_sec_WS_k_c_1y = crvg_sec_ws_b12[crvg_sec_ws_b12['opt_maturity'] == '1y']['WS_k'].values[0]
crvg_sec_WS_k_c_5y = crvg_sec_ws_b12[crvg_sec_ws_b12['opt_maturity'] == '5y']['WS_k'].values[0]


# ============================================================
# 4) INTRA-BUCKET CORRELATION (rho_kl)
# ============================================================
# ρ_kl = ρ_delta × ρ_option_maturity
# ρ_delta = 1.0 (same tranche) else 0.40 (Article 325ao)
# ρ_option_maturity = exp(-alpha * |Tk - Tl| / min(Tk, Tl)) if different tenors, else 1.0

ALPHA = 0.01
RHO_DELTA_TRANCHE_SAME = 1.00
RHO_DELTA_TRANCHE_DIFF = 0.40

# --- Bucket 1: Tranche A (1Y) vs Tranche B (3Y) ---
# Different tranche -> rho_delta = 0.40
crvg_rho_delta_b1 = RHO_DELTA_TRANCHE_DIFF

# Different tenors -> rho_opt
t_1y, t_3y = 1.0, 3.0
crvg_rho_opt_maturity_b1 = np.exp(-ALPHA * abs(t_1y - t_3y) / min(t_1y, t_3y))

crvg_rho_kl_b1 = min(crvg_rho_delta_b1 * crvg_rho_opt_maturity_b1, 1.0)

# --- Bucket 12: Tranche C (1Y) vs Tranche C (5Y) ---
# Same tranche -> rho_delta = 1.00
crvg_rho_delta_b12 = RHO_DELTA_TRANCHE_SAME

# Different tenors -> rho_opt
t_1y, t_5y = 1.0, 5.0
crvg_rho_opt_maturity_b12 = np.exp(-ALPHA * abs(t_1y - t_5y) / min(t_1y, t_5y))

crvg_rho_kl_b12 = min(crvg_rho_delta_b12 * crvg_rho_opt_maturity_b12, 1.0)


# ============================================================
# 5) INTRA-BUCKET AGGREGATION (K_b)
# ============================================================
# K_b = √( Σ WS_k² + Σ Σ ρ_kl × WS_k × WS_l )  for k ≠ l

# Bucket 1: 2 risk factors
crvg_sum_sq_b1 = crvg_sec_WS_k_a_1y**2 + crvg_sec_WS_k_b_3y**2
crvg_cross_b1 = 2 * crvg_rho_kl_b1 * crvg_sec_WS_k_a_1y * crvg_sec_WS_k_b_3y
crvg_sec_K_b1 = np.sqrt(max(0, crvg_sum_sq_b1 + crvg_cross_b1))

# Bucket 12: 2 risk factors
crvg_sum_sq_b12 = crvg_sec_WS_k_c_1y**2 + crvg_sec_WS_k_c_5y**2
crvg_cross_b12 = 2 * crvg_rho_kl_b12 * crvg_sec_WS_k_c_1y * crvg_sec_WS_k_c_5y
crvg_sec_K_b12 = np.sqrt(max(0, crvg_sum_sq_b12 + crvg_cross_b12))

# --- Explicit K_b formula strings for printing ---
crvg_sec_Kb1_formula_str = (
    f"K_b1 = sqrt( ({crvg_sec_WS_k_a_1y:.2f}^2 + {crvg_sec_WS_k_b_3y:.2f}^2) "
    f"+ (2 * {crvg_rho_kl_b1:.4f} * {crvg_sec_WS_k_a_1y:.2f} * {crvg_sec_WS_k_b_3y:.2f}) )"
)

crvg_sec_Kb12_formula_str = (
    f"K_b12 = sqrt( ({crvg_sec_WS_k_c_1y:.2f}^2 + {crvg_sec_WS_k_c_5y:.2f}^2) "
    f"+ (2 * {crvg_rho_kl_b12:.4f} * {crvg_sec_WS_k_c_1y:.2f} * {crvg_sec_WS_k_c_5y:.2f}) )"
)


# ============================================================
# 6) BUCKET SUMS (S_b)
# ============================================================
# S_b = Σ WS_k  (simple sum of weighted sensitivities in bucket)

crvg_sec_S_b1 = crvg_sec_wsk[crvg_sec_wsk['bucket'] == '1']['WS_k'].sum()
crvg_sec_S_b12 = crvg_sec_wsk[crvg_sec_wsk['bucket'] == '12']['WS_k'].sum()


# ============================================================
# 7) CROSS-BUCKET CORRELATION (gamma_bc)
# ============================================================
# γ_bc from Article 325ao
# For CSR Sec Non-CTP: γ = 0%

crvg_sec_gamma = 0.0


# ============================================================
# 8) CROSS-BUCKET AGGREGATION
# ============================================================
# K = √( Σ K_b² + Σ Σ γ_bc × S_b × S_c )  for b ≠ c

crvg_sum_K_sq = crvg_sec_K_b1**2 + crvg_sec_K_b12**2
crvg_cross_bucket = 2 * crvg_sec_gamma * crvg_sec_S_b1 * crvg_sec_S_b12

crvg_sec_k_squared = crvg_sum_K_sq + crvg_cross_bucket

# Apply the check at the overall level (Floor at 0)
crvg_sec_k_squared = max(crvg_sec_k_squared, 0)

crvg_sec_K_medium = np.sqrt(crvg_sec_k_squared)

# --- Explicit Cross-bucket formula string for printing ---
crvg_sec_K_formula_str = (
    f"Capital = sqrt( ({crvg_sec_K_b1:.2f}^2 + {crvg_sec_K_b12:.2f}^2) "
    f"+ (2 * {crvg_sec_gamma:.2f} * {crvg_sec_S_b1:.2f} * {crvg_sec_S_b12:.2f}) )"
)


# ============================================================
# 9) CORRELATION SCENARIOS (HIGH/LOW)
# ============================================================
# High: ρ_high = min(1.25 × ρ_medium, 1.0)
# Low:  ρ_low  = max(2 × ρ_medium - 1, 0.75 × ρ_medium)

# --- High scenario correlations --- #
crvg_rho_kl_b1_high = min(crvg_rho_kl_b1 * 1.25, 1.0)
crvg_rho_kl_b12_high = min(crvg_rho_kl_b12 * 1.25, 1.0)
crvg_gamma_high = min(crvg_sec_gamma * 1.25, 1.0)

# --- Low scenario correlations --- #
crvg_rho_kl_b1_low = max(2 * crvg_rho_kl_b1 - 1.0, 0.75 * crvg_rho_kl_b1)
crvg_rho_kl_b12_low = max(2 * crvg_rho_kl_b12 - 1.0, 0.75 * crvg_rho_kl_b12)
crvg_gamma_low = max(2 * crvg_sec_gamma - 1.0, 0.75 * crvg_sec_gamma)

# --- High scenario intra-bucket K_b --- #
crvg_cross_b1_high = 2 * crvg_rho_kl_b1_high * crvg_sec_WS_k_a_1y * crvg_sec_WS_k_b_3y
crvg_sec_K_b1_high = np.sqrt(max(0, crvg_sum_sq_b1 + crvg_cross_b1_high))

crvg_cross_b12_high = 2 * crvg_rho_kl_b12_high * crvg_sec_WS_k_c_1y * crvg_sec_WS_k_c_5y
crvg_sec_K_b12_high = np.sqrt(max(0, crvg_sum_sq_b12 + crvg_cross_b12_high))

# --- Low scenario intra-bucket K_b --- #
crvg_cross_b1_low = 2 * crvg_rho_kl_b1_low * crvg_sec_WS_k_a_1y * crvg_sec_WS_k_b_3y
crvg_sec_K_b1_low = np.sqrt(max(0, crvg_sum_sq_b1 + crvg_cross_b1_low))

crvg_cross_b12_low = 2 * crvg_rho_kl_b12_low * crvg_sec_WS_k_c_1y * crvg_sec_WS_k_c_5y
crvg_sec_K_b12_low = np.sqrt(max(0, crvg_sum_sq_b12 + crvg_cross_b12_low))

# --- High scenario cross-bucket --- #
crvg_k_sq_high = (
    crvg_sec_K_b1_high**2 + crvg_sec_K_b12_high**2
    + 2 * crvg_gamma_high * crvg_sec_S_b1 * crvg_sec_S_b12
)
crvg_sec_K_high = np.sqrt(max(crvg_k_sq_high, 0))

# --- Low scenario cross-bucket --- #
crvg_k_sq_low = (
    crvg_sec_K_b1_low**2 + crvg_sec_K_b12_low**2
    + 2 * crvg_gamma_low * crvg_sec_S_b1 * crvg_sec_S_b12
)
crvg_sec_K_low = np.sqrt(max(crvg_k_sq_low, 0))


# ============================================================
# 10) CAPITAL
# ============================================================
# Final capital = max(K_medium, K_high, K_low)

crvg_sec_capital = max(crvg_sec_K_medium, crvg_sec_K_high, crvg_sec_K_low)


# ============================================================
# 11) PRINTING
# ============================================================

# --- 11.1 Gross sensitivities --- #
crvg_sec_gross_sk_print = format_table(
    crvg_sec_gross_sk,
    columns=['trade_id', 'bucket', 'rating', 'sector', 'tranche', 'opt_maturity', 'gross_s_k'],
    col_formats={'gross_s_k': ',.0f'},
    alignments=('left', 'left', 'left', 'left', 'left', 'left', 'right')
)

# --- 11.2 Net sensitivities (s_k) --- #
crvg_sec_net_sk_print = format_table(
    crvg_sec_net_sk,
    columns=['bucket', 'rating', 'sector', 'tranche', 'opt_maturity', 's_k'],
    col_formats={'s_k': ',.0f'},
    alignments=('left', 'left', 'left', 'left', 'left', 'right')
)

# --- 11.3 Weighted sensitivities (WS_k) --- #
crvg_sec_wsk_print = format_table(
    crvg_sec_wsk,
    columns=['bucket', 'rating', 'sector', 'tranche', 'opt_maturity', 's_k', 'rw', 'WS_k'],
    col_formats={'s_k': ',.0f', 'rw': '.4f', 'WS_k': ',.2f'},
    alignments=('left', 'left', 'left', 'left', 'left', 'right', 'right', 'right')
)

# --- 11.4 Intra-bucket correlation (rho_kl) --- #
# Columns renamed to match Vega-specific correlation components
crvg_sec_rho_print = format_table(
    pd.DataFrame([
        {'bucket': '1', 'pair': 'Tr A 1Y vs Tr B 3Y', 'rho_delta': crvg_rho_delta_b1, 'rho_opt_maturity': crvg_rho_opt_maturity_b1, 'rho_kl': crvg_rho_kl_b1},
        {'bucket': '12', 'pair': 'Tr C 1Y vs Tr C 5Y', 'rho_delta': crvg_rho_delta_b12, 'rho_opt_maturity': crvg_rho_opt_maturity_b12, 'rho_kl': crvg_rho_kl_b12},
    ]),
    columns=['bucket', 'pair', 'rho_delta', 'rho_opt_maturity', 'rho_kl'],
    col_formats={'rho_delta': '.2f', 'rho_opt_maturity': '.4f', 'rho_kl': '.4f'},
    alignments=('left', 'left', 'right', 'right', 'right')
)

# --- 11.5 Intra-bucket aggregation (K_b) --- #
# Bucket 1
crvg_sec_K_b1_print = format_table(
    pd.DataFrame([
        {'component': 'WS_k_1 (Tr A 1Y)', 'value': crvg_sec_WS_k_a_1y},
        {'component': 'WS_k_2 (Tr B 3Y)', 'value': crvg_sec_WS_k_b_3y},
        {'component': 'rho_kl', 'value': f"{crvg_rho_kl_b1:.4f}"},
        {'component': 'K_b', 'value': crvg_sec_K_b1},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# Bucket 12
crvg_sec_K_b12_print = format_table(
    pd.DataFrame([
        {'component': 'WS_k_1 (Tr C 1Y)', 'value': crvg_sec_WS_k_c_1y},
        {'component': 'WS_k_2 (Tr C 5Y)', 'value': crvg_sec_WS_k_c_5y},
        {'component': 'rho_kl', 'value': f"{crvg_rho_kl_b12:.4f}"},
        {'component': 'K_b', 'value': crvg_sec_K_b12},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# --- 11.6 Bucket sums (S_b) --- #
crvg_sec_bucket_sums_print = format_table(
    pd.DataFrame([
        {'bucket': '1', 'WS_k_1': crvg_sec_WS_k_a_1y, 'WS_k_2': crvg_sec_WS_k_b_3y, 'S_b': crvg_sec_S_b1},
        {'bucket': '12', 'WS_k_1': crvg_sec_WS_k_c_1y, 'WS_k_2': crvg_sec_WS_k_c_5y, 'S_b': crvg_sec_S_b12},
    ]),
    columns=['bucket', 'WS_k_1', 'WS_k_2', 'S_b'],
    col_formats={'WS_k_1': ',.2f', 'WS_k_2': ',.2f', 'S_b': ',.2f'},
    alignments=('left', 'right', 'right', 'right')
)

# --- 11.7 Cross-bucket correlation (gamma_bc) --- #
crvg_sec_gamma_print = format_table(
    pd.DataFrame([
        {'buckets': '1 vs 12', 'gamma_bc': crvg_sec_gamma},
    ]),
    columns=['buckets', 'gamma_bc'],
    col_formats={'gamma_bc': '.2f'},
    alignments=('left', 'right')
)

# --- 11.8 Cross-bucket aggregation --- #
crvg_sec_cross_agg_print = format_table(
    pd.DataFrame([
        {'component': 'K_b1', 'value': crvg_sec_K_b1},
        {'component': 'K_b12', 'value': crvg_sec_K_b12},
        {'component': 'S_b1', 'value': crvg_sec_S_b1},
        {'component': 'S_b12', 'value': crvg_sec_S_b12},
        {'component': 'gamma_bc', 'value': f"{crvg_sec_gamma:.2f}"},
        {'component': 'capital', 'value': crvg_sec_K_medium},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# --- 11.9 Correlation scenarios --- #
crvg_sec_scenarios_print = format_table(
    pd.DataFrame([
        {'parameter': 'rho_kl_b1', 'medium': crvg_rho_kl_b1, 'high': crvg_rho_kl_b1_high, 'low': crvg_rho_kl_b1_low},
        {'parameter': 'rho_kl_b12', 'medium': crvg_rho_kl_b12, 'high': crvg_rho_kl_b12_high, 'low': crvg_rho_kl_b12_low},
        {'parameter': 'gamma_bc', 'medium': crvg_sec_gamma, 'high': crvg_gamma_high, 'low': crvg_gamma_low},
    ]),
    columns=['parameter', 'medium', 'high', 'low'],
    col_formats={'medium': '.4f', 'high': '.4f', 'low': '.4f'},
    alignments=('left', 'right', 'right', 'right')
)

# --- 11.10 Capital --- #
crvg_sec_capital_print = format_table(
    pd.DataFrame([
        {'scenario': 'medium', 'capital': crvg_sec_K_medium},
        {'scenario': 'high', 'capital': crvg_sec_K_high},
        {'scenario': 'low', 'capital': crvg_sec_K_low},
    ]),
    columns=['scenario', 'capital'],
    col_formats={'capital': ',.2f'},
    alignments=('left', 'right')
)


# ============================================================
# DISPLAY ALL OUTPUTS
# ============================================================

print("=" * 60)
print("CRVG SECURITISATION")
print("=" * 60)

print("\n1) GROSS SENSITIVITIES")
print(crvg_sec_gross_sk_print)

print("\n2) NET SENSITIVITIES (s_k)")
print(crvg_sec_net_sk_print)

print("\n3) WEIGHTED SENSITIVITIES (WS_k)")
print(crvg_sec_wsk_print)

print("\n4) INTRA-BUCKET CORRELATION (rho_kl)")
print(crvg_sec_rho_print)

print("\n5) INTRA-BUCKET AGGREGATION (K_b)")
print("\n--- Bucket 1 ---")
print(crvg_sec_Kb1_formula_str)
print(crvg_sec_K_b1_print)
print("\n--- Bucket 12 ---")
print(crvg_sec_Kb12_formula_str)
print(crvg_sec_K_b12_print)

print("\n6) BUCKET SUMS (S_b)")
print(crvg_sec_bucket_sums_print)

print("\n7) CROSS-BUCKET CORRELATION (gamma_bc)")
print(crvg_sec_gamma_print)

print("\n8) CROSS-BUCKET AGGREGATION")
print(crvg_sec_K_formula_str)
print(crvg_sec_cross_agg_print)

print("\n9) CORRELATION SCENARIOS")
print(crvg_sec_scenarios_print)

print("\n10) CAPITAL")
print(crvg_sec_capital_print)

CRVG SECURITISATION

1) GROSS SENSITIVITIES
┌────────────┬──────────┬──────────┬────────────┬───────────┬────────────────┬─────────────┐
│ trade_id   │ bucket   │ rating   │ sector     │ tranche   │ opt_maturity   │   gross_s_k │
├────────────┼──────────┼──────────┼────────────┼───────────┼────────────────┼─────────────┤
│ 1          │ 1        │ IG       │ rmbs_prime │ tranche_a │ 1y             │         800 │
│ 2          │ 1        │ IG       │ rmbs_prime │ tranche_b │ 3y             │        -600 │
│ 3          │ 12       │ NIG      │ cmbs       │ tranche_c │ 1y             │         900 │
│ 4          │ 12       │ NIG      │ cmbs       │ tranche_c │ 5y             │        -700 │
└────────────┴──────────┴──────────┴────────────┴───────────┴────────────────┴─────────────┘

2) NET SENSITIVITIES (s_k)
┌──────────┬──────────┬────────────┬───────────┬────────────────┬───────┐
│ bucket   │ rating   │ sector     │ tranche   │ opt_maturity   │   s_k │
├──────────┼──────────┼────────────┼

### EQVG

In [15]:
# ============================================================
# EQVG
# ============================================================


# TABLE OF CONTENTS
#   1) Gross sensitivities
#   2) Net sensitivities (s_k)
#   3) Weighted sensitivities (WS_k)
#   4) Intra-bucket correlation (rho_kl)
#   5) Intra-bucket aggregation (K_b)
#   6) Bucket sums (S_b)
#   7) Cross-bucket correlation (gamma_bc)
#   8) Cross-bucket aggregation
#   9) Correlation scenarios (high/low)
#  10) Capital
#  11) Printing
# ============================================================


# ============================================================
# 1) GROSS SENSITIVITIES
# ============================================================
# Position-level vega sensitivities to equity volatility.

data = [
    {'trade_id': 1, 'bucket': '12', 'spot': 'issuer_a', 'opt_maturity': '10y', 'gross_s_k': -1800},
    {'trade_id': 2, 'bucket': '12', 'spot': 'issuer_a', 'opt_maturity': '10y', 'gross_s_k': -3600},
    {'trade_id': 3, 'bucket': '12', 'spot': 'issuer_a', 'opt_maturity': '6m',  'gross_s_k': -3900},
    {'trade_id': 4, 'bucket': '12', 'spot': 'issuer_a', 'opt_maturity': '6m',  'gross_s_k': 1400},
    {'trade_id': 5, 'bucket': '12', 'spot': 'issuer_b', 'opt_maturity': '10y', 'gross_s_k': -1200},
    {'trade_id': 6, 'bucket': '12', 'spot': 'issuer_b', 'opt_maturity': '6m',  'gross_s_k': 1700},
]

eqvg_gross_sk = pd.DataFrame(data)


# ============================================================
# 2) NET SENSITIVITIES (s_k)
# ============================================================
# Net sensitivity s_k = Σ gross sensitivities for each unique risk factor.
# Risk factor defined by: bucket + spot + opt_maturity.

eqvg_risk_factor_cols = ['bucket', 'spot', 'opt_maturity']

eqvg_net_sk = (
    eqvg_gross_sk
    .groupby(eqvg_risk_factor_cols)
    .agg(s_k=('gross_s_k', 'sum'))
    .reset_index()
)

# Helper map for numerical tenor
TENOR_TO_YEARS = {'6m': 0.5, '10y': 10.0}
eqvg_net_sk['tenor_years'] = eqvg_net_sk['opt_maturity'].map(TENOR_TO_YEARS)


# ============================================================
# 3) WEIGHTED SENSITIVITIES (WS_k)
# ============================================================
# WS_k = s_k × RW_k

# Risk weight (RW_k) from Article 325ay
# For Equity Vega Bucket 12: RW = 77.78%
eqvg_risk_weight = 0.7778

eqvg_wsk = eqvg_net_sk.copy()
eqvg_wsk['rw'] = eqvg_risk_weight
eqvg_wsk['WS_k'] = eqvg_wsk['s_k'] * eqvg_wsk['rw']

# Extract weighted sensitivities per bucket for aggregation
# Bucket 12: 4 risk factors (Issuer A 10Y/6M, Issuer B 10Y/6M)
eqvg_ws_b12 = eqvg_wsk[eqvg_wsk['bucket'] == '12']
eqvg_WS_k_a_10y = eqvg_ws_b12[(eqvg_ws_b12['spot'] == 'issuer_a') & (eqvg_ws_b12['opt_maturity'] == '10y')]['WS_k'].values[0]
eqvg_WS_k_a_6m  = eqvg_ws_b12[(eqvg_ws_b12['spot'] == 'issuer_a') & (eqvg_ws_b12['opt_maturity'] == '6m')]['WS_k'].values[0]
eqvg_WS_k_b_10y = eqvg_ws_b12[(eqvg_ws_b12['spot'] == 'issuer_b') & (eqvg_ws_b12['opt_maturity'] == '10y')]['WS_k'].values[0]
eqvg_WS_k_b_6m  = eqvg_ws_b12[(eqvg_ws_b12['spot'] == 'issuer_b') & (eqvg_ws_b12['opt_maturity'] == '6m')]['WS_k'].values[0]


# ============================================================
# 4) INTRA-BUCKET CORRELATION (rho_kl)
# ============================================================
# ρ_kl = ρ_delta × ρ_option_maturity (Article 325ay)
# ρ_delta = 1.0 (same issuer) else 0.80 (Article 325ay)
# ρ_option_maturity = exp(-alpha * |Tk - Tl| / min(Tk, Tl))

ALPHA = 0.01
RHO_DELTA_DIFFERENT_ISSUER = 0.80

t_6m, t_10y = 0.5, 10.0
rho_maturity_diff = np.exp(-ALPHA * abs(t_6m - t_10y) / min(t_6m, t_10y))

# --- Issuer A 10Y vs Issuer A 6M (same issuer, different tenor) ---
eqvg_rho_delta_a10y_a6m = 1.0
eqvg_rho_maturity_a10y_a6m = rho_maturity_diff
eqvg_rho_kl_a10y_a6m = eqvg_rho_delta_a10y_a6m * eqvg_rho_maturity_a10y_a6m

# --- Issuer A 10Y vs Issuer B 10Y (different issuer, same tenor) ---
eqvg_rho_delta_a10y_b10y = RHO_DELTA_DIFFERENT_ISSUER
eqvg_rho_maturity_a10y_b10y = 1.0
eqvg_rho_kl_a10y_b10y = eqvg_rho_delta_a10y_b10y * eqvg_rho_maturity_a10y_b10y

# --- Issuer A 10Y vs Issuer B 6M (different issuer, different tenor) ---
eqvg_rho_delta_a10y_b6m = RHO_DELTA_DIFFERENT_ISSUER
eqvg_rho_maturity_a10y_b6m = rho_maturity_diff
eqvg_rho_kl_a10y_b6m = eqvg_rho_delta_a10y_b6m * eqvg_rho_maturity_a10y_b6m

# --- Issuer A 6M vs Issuer B 10Y (different issuer, different tenor) ---
eqvg_rho_delta_a6m_b10y = RHO_DELTA_DIFFERENT_ISSUER
eqvg_rho_maturity_a6m_b10y = rho_maturity_diff
eqvg_rho_kl_a6m_b10y = eqvg_rho_delta_a6m_b10y * eqvg_rho_maturity_a6m_b10y

# --- Issuer A 6M vs Issuer B 6M (different issuer, same tenor) ---
eqvg_rho_delta_a6m_b6m = RHO_DELTA_DIFFERENT_ISSUER
eqvg_rho_maturity_a6m_b6m = 1.0
eqvg_rho_kl_a6m_b6m = eqvg_rho_delta_a6m_b6m * eqvg_rho_maturity_a6m_b6m

# --- Issuer B 10Y vs Issuer B 6M (same issuer, different tenor) ---
eqvg_rho_delta_b10y_b6m = 1.0
eqvg_rho_maturity_b10y_b6m = rho_maturity_diff
eqvg_rho_kl_b10y_b6m = eqvg_rho_delta_b10y_b6m * eqvg_rho_maturity_b10y_b6m


# ============================================================
# 5) INTRA-BUCKET AGGREGATION (K_b)
# ============================================================
# K_b = √( Σ WS_k² + Σ Σ ρ_kl × WS_k × WS_l )  for k ≠ l

# Sum of squares
eqvg_sum_sq_b12 = (
    eqvg_WS_k_a_10y**2 + eqvg_WS_k_a_6m**2 + eqvg_WS_k_b_10y**2 + eqvg_WS_k_b_6m**2
)

# Cross-product terms (6 pairs)
eqvg_cross_b12 = (
    2 * eqvg_rho_kl_a10y_a6m * eqvg_WS_k_a_10y * eqvg_WS_k_a_6m
    + 2 * eqvg_rho_kl_a10y_b10y * eqvg_WS_k_a_10y * eqvg_WS_k_b_10y
    + 2 * eqvg_rho_kl_a10y_b6m * eqvg_WS_k_a_10y * eqvg_WS_k_b_6m
    + 2 * eqvg_rho_kl_a6m_b10y * eqvg_WS_k_a_6m * eqvg_WS_k_b_10y
    + 2 * eqvg_rho_kl_a6m_b6m * eqvg_WS_k_a_6m * eqvg_WS_k_b_6m
    + 2 * eqvg_rho_kl_b10y_b6m * eqvg_WS_k_b_10y * eqvg_WS_k_b_6m
)

eqvg_K_b12 = np.sqrt(max(0, eqvg_sum_sq_b12 + eqvg_cross_b12))

# --- Explicit K_b formula string for printing ---
# Displaying simplified form with first few terms to fit
eqvg_Kb12_formula_str = (
    f"K_b12 = sqrt( ({eqvg_WS_k_a_10y:.2f}^2 + {eqvg_WS_k_a_6m:.2f}^2 + ...) "
    f"+ (2 * {eqvg_rho_kl_a10y_a6m:.4f} * ...) + ... )"
)


# ============================================================
# 6) BUCKET SUMS (S_b)
# ============================================================
# S_b = Σ WS_k

eqvg_S_b12 = eqvg_wsk[eqvg_wsk['bucket'] == '12']['WS_k'].sum()


# ============================================================
# 7) CROSS-BUCKET CORRELATION (gamma_bc)
# ============================================================
# For single-bucket portfolios, cross-bucket correlation is not applicable.

eqvg_gamma = 0.0  # Not applicable for single bucket


# ============================================================
# 8) CROSS-BUCKET AGGREGATION
# ============================================================
# K = √( Σ K_b² + Σ Σ γ_bc × S_b × S_c )  for b ≠ c
# With single bucket, no cross-bucket term: K = K_b12

# Calculate the variance term
eqvg_k_squared = eqvg_K_b12**2

# Apply the check at the overall level (Floor at 0)
eqvg_k_squared = max(eqvg_k_squared, 0)

eqvg_K_medium = np.sqrt(eqvg_k_squared)

# --- Explicit Cross-bucket formula string for printing ---
eqvg_K_formula_str = (
    f"Capital = sqrt( {eqvg_K_b12:.2f}^2 )"
)


# ============================================================
# 9) CORRELATION SCENARIOS (HIGH/LOW)
# ============================================================
# High: ρ_high = min(1.25 × ρ_medium, 1.0)
# Low:  ρ_low  = max(2 × ρ_medium - 1, 0.75 × ρ_medium)

# --- High scenario correlations --- #
eqvg_rho_a10y_a6m_high = min(eqvg_rho_kl_a10y_a6m * 1.25, 1.0)
eqvg_rho_a10y_b10y_high = min(eqvg_rho_kl_a10y_b10y * 1.25, 1.0)
eqvg_rho_a10y_b6m_high = min(eqvg_rho_kl_a10y_b6m * 1.25, 1.0)
eqvg_rho_a6m_b10y_high = min(eqvg_rho_kl_a6m_b10y * 1.25, 1.0)
eqvg_rho_a6m_b6m_high = min(eqvg_rho_kl_a6m_b6m * 1.25, 1.0)
eqvg_rho_b10y_b6m_high = min(eqvg_rho_kl_b10y_b6m * 1.25, 1.0)
eqvg_gamma_high = min(eqvg_gamma * 1.25, 1.0)

# --- Low scenario correlations --- #
eqvg_rho_a10y_a6m_low = max(2 * eqvg_rho_kl_a10y_a6m - 1.0, 0.75 * eqvg_rho_kl_a10y_a6m)
eqvg_rho_a10y_b10y_low = max(2 * eqvg_rho_kl_a10y_b10y - 1.0, 0.75 * eqvg_rho_kl_a10y_b10y)
eqvg_rho_a10y_b6m_low = max(2 * eqvg_rho_kl_a10y_b6m - 1.0, 0.75 * eqvg_rho_kl_a10y_b6m)
eqvg_rho_a6m_b10y_low = max(2 * eqvg_rho_kl_a6m_b10y - 1.0, 0.75 * eqvg_rho_kl_a6m_b10y)
eqvg_rho_a6m_b6m_low = max(2 * eqvg_rho_kl_a6m_b6m - 1.0, 0.75 * eqvg_rho_kl_a6m_b6m)
eqvg_rho_b10y_b6m_low = max(2 * eqvg_rho_kl_b10y_b6m - 1.0, 0.75 * eqvg_rho_kl_b10y_b6m)
eqvg_gamma_low = max(2 * eqvg_gamma - 1.0, 0.75 * eqvg_gamma)

# --- High scenario intra-bucket K_b --- #
eqvg_cross_b12_high = (
    2 * eqvg_rho_a10y_a6m_high * eqvg_WS_k_a_10y * eqvg_WS_k_a_6m
    + 2 * eqvg_rho_a10y_b10y_high * eqvg_WS_k_a_10y * eqvg_WS_k_b_10y
    + 2 * eqvg_rho_a10y_b6m_high * eqvg_WS_k_a_10y * eqvg_WS_k_b_6m
    + 2 * eqvg_rho_a6m_b10y_high * eqvg_WS_k_a_6m * eqvg_WS_k_b_10y
    + 2 * eqvg_rho_a6m_b6m_high * eqvg_WS_k_a_6m * eqvg_WS_k_b_6m
    + 2 * eqvg_rho_b10y_b6m_high * eqvg_WS_k_b_10y * eqvg_WS_k_b_6m
)
eqvg_K_b12_high = np.sqrt(max(0, eqvg_sum_sq_b12 + eqvg_cross_b12_high))

# --- Low scenario intra-bucket K_b --- #
eqvg_cross_b12_low = (
    2 * eqvg_rho_a10y_a6m_low * eqvg_WS_k_a_10y * eqvg_WS_k_a_6m
    + 2 * eqvg_rho_a10y_b10y_low * eqvg_WS_k_a_10y * eqvg_WS_k_b_10y
    + 2 * eqvg_rho_a10y_b6m_low * eqvg_WS_k_a_10y * eqvg_WS_k_b_6m
    + 2 * eqvg_rho_a6m_b10y_low * eqvg_WS_k_a_6m * eqvg_WS_k_b_10y
    + 2 * eqvg_rho_a6m_b6m_low * eqvg_WS_k_a_6m * eqvg_WS_k_b_6m
    + 2 * eqvg_rho_b10y_b6m_low * eqvg_WS_k_b_10y * eqvg_WS_k_b_6m
)
eqvg_K_b12_low = np.sqrt(max(0, eqvg_sum_sq_b12 + eqvg_cross_b12_low))

# --- High scenario cross-bucket --- #
eqvg_k_sq_high = eqvg_K_b12_high**2
eqvg_K_high = np.sqrt(max(eqvg_k_sq_high, 0))

# --- Low scenario cross-bucket --- #
eqvg_k_sq_low = eqvg_K_b12_low**2
eqvg_K_low = np.sqrt(max(eqvg_k_sq_low, 0))


# ============================================================
# 10) CAPITAL
# ============================================================
# Final capital = max(K_medium, K_high, K_low)

eqvg_capital = max(eqvg_K_medium, eqvg_K_high, eqvg_K_low)


# ============================================================
# 11) PRINTING
# ============================================================

# --- 11.1 Gross sensitivities --- #
eqvg_gross_sk_print = format_table(
    eqvg_gross_sk,
    columns=['trade_id', 'bucket', 'spot', 'opt_maturity', 'gross_s_k'],
    col_formats={'gross_s_k': ',.0f'},
    alignments=('left', 'left', 'left', 'left', 'right')
)

# --- 11.2 Net sensitivities (s_k) --- #
eqvg_net_sk_print = format_table(
    eqvg_net_sk,
    columns=['bucket', 'spot', 'opt_maturity', 's_k'],
    col_formats={'s_k': ',.0f'},
    alignments=('left', 'left', 'left', 'right')
)

# --- 11.3 Weighted sensitivities (WS_k) --- #
eqvg_wsk_print = format_table(
    eqvg_wsk,
    columns=['bucket', 'spot', 'opt_maturity', 's_k', 'rw', 'WS_k'],
    col_formats={'s_k': ',.0f', 'rw': '.4f', 'WS_k': ',.0f'},
    alignments=('left', 'left', 'left', 'right', 'right', 'right')
)

# --- 11.4 Intra-bucket correlation (rho_kl) --- #
# Columns renamed to match Vega-specific correlation components
eqvg_rho_print = format_table(
    pd.DataFrame([
        {'bucket': '12', 'pair': 'A 10Y vs A 6M', 'rho_delta': eqvg_rho_delta_a10y_a6m, 'rho_opt_maturity': eqvg_rho_maturity_a10y_a6m, 'rho_kl': eqvg_rho_kl_a10y_a6m},
        {'bucket': '12', 'pair': 'A 10Y vs B 10Y', 'rho_delta': eqvg_rho_delta_a10y_b10y, 'rho_opt_maturity': eqvg_rho_maturity_a10y_b10y, 'rho_kl': eqvg_rho_kl_a10y_b10y},
        {'bucket': '12', 'pair': 'A 10Y vs B 6M', 'rho_delta': eqvg_rho_delta_a10y_b6m, 'rho_opt_maturity': eqvg_rho_maturity_a10y_b6m, 'rho_kl': eqvg_rho_kl_a10y_b6m},
        {'bucket': '12', 'pair': 'A 6M vs B 10Y', 'rho_delta': eqvg_rho_delta_a6m_b10y, 'rho_opt_maturity': eqvg_rho_maturity_a6m_b10y, 'rho_kl': eqvg_rho_kl_a6m_b10y},
        {'bucket': '12', 'pair': 'A 6M vs B 6M', 'rho_delta': eqvg_rho_delta_a6m_b6m, 'rho_opt_maturity': eqvg_rho_maturity_a6m_b6m, 'rho_kl': eqvg_rho_kl_a6m_b6m},
        {'bucket': '12', 'pair': 'B 10Y vs B 6M', 'rho_delta': eqvg_rho_delta_b10y_b6m, 'rho_opt_maturity': eqvg_rho_maturity_b10y_b6m, 'rho_kl': eqvg_rho_kl_b10y_b6m},
    ]),
    columns=['bucket', 'pair', 'rho_delta', 'rho_opt_maturity', 'rho_kl'],
    col_formats={'rho_delta': '.2f', 'rho_opt_maturity': '.4f', 'rho_kl': '.4f'},
    alignments=('left', 'left', 'right', 'right', 'right')
)

# --- 11.5 Intra-bucket aggregation (K_b) --- #
# Bucket 12
eqvg_K_b12_print = format_table(
    pd.DataFrame([
        {'component': 'WS_k_1 (A 10Y)', 'value': eqvg_WS_k_a_10y},
        {'component': 'WS_k_2 (A 6M)', 'value': eqvg_WS_k_a_6m},
        {'component': 'WS_k_3 (B 10Y)', 'value': eqvg_WS_k_b_10y},
        {'component': 'WS_k_4 (B 6M)', 'value': eqvg_WS_k_b_6m},
        {'component': 'K_b', 'value': eqvg_K_b12},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# --- 11.6 Bucket sums (S_b) --- #
eqvg_bucket_sums_print = format_table(
    pd.DataFrame([
        {'bucket': '12', 'WS_k_1': eqvg_WS_k_a_10y, 'WS_k_2': eqvg_WS_k_a_6m, 'WS_k_3': eqvg_WS_k_b_10y, 'WS_k_4': eqvg_WS_k_b_6m, 'S_b': eqvg_S_b12},
    ]),
    columns=['bucket', 'WS_k_1', 'WS_k_2', 'WS_k_3', 'WS_k_4', 'S_b'],
    col_formats={'WS_k_1': ',.0f', 'WS_k_2': ',.0f', 'WS_k_3': ',.0f', 'WS_k_4': ',.0f', 'S_b': ',.0f'},
    alignments=('left', 'right', 'right', 'right', 'right', 'right')
)

# --- 11.7 Cross-bucket correlation (gamma_bc) --- #
eqvg_gamma_print = format_table(
    pd.DataFrame([
        {'buckets': 'n/a (single)', 'gamma_bc': eqvg_gamma},
    ]),
    columns=['buckets', 'gamma_bc'],
    col_formats={'gamma_bc': '.2f'},
    alignments=('left', 'right')
)

# --- 11.8 Cross-bucket aggregation --- #
eqvg_cross_agg_print = format_table(
    pd.DataFrame([
        {'component': 'K_b12', 'value': eqvg_K_b12},
        {'component': 'S_b12', 'value': eqvg_S_b12},
        {'component': 'gamma_bc', 'value': f"{eqvg_gamma:.2f}"},
        {'component': 'capital', 'value': eqvg_K_medium},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# --- 11.9 Correlation scenarios --- #
eqvg_scenarios_print = format_table(
    pd.DataFrame([
        {'parameter': 'rho_kl_a10y_b10y', 'medium': eqvg_rho_kl_a10y_b10y, 'high': eqvg_rho_a10y_b10y_high, 'low': eqvg_rho_a10y_b10y_low},
        {'parameter': 'gamma_bc', 'medium': eqvg_gamma, 'high': eqvg_gamma_high, 'low': eqvg_gamma_low},
    ]),
    columns=['parameter', 'medium', 'high', 'low'],
    col_formats={'medium': '.4f', 'high': '.4f', 'low': '.4f'},
    alignments=('left', 'right', 'right', 'right')
)

# --- 11.10 Capital --- #
eqvg_capital_print = format_table(
    pd.DataFrame([
        {'scenario': 'medium', 'capital': eqvg_K_medium},
        {'scenario': 'high', 'capital': eqvg_K_high},
        {'scenario': 'low', 'capital': eqvg_K_low},
    ]),
    columns=['scenario', 'capital'],
    col_formats={'capital': ',.2f'},
    alignments=('left', 'right')
)


# ============================================================
# DISPLAY ALL OUTPUTS
# ============================================================

print("=" * 60)
print("EQVG")
print("=" * 60)

print("\n1) GROSS SENSITIVITIES")
print(eqvg_gross_sk_print)

print("\n2) NET SENSITIVITIES (s_k)")
print(eqvg_net_sk_print)

print("\n3) WEIGHTED SENSITIVITIES (WS_k)")
print(eqvg_wsk_print)

print("\n4) INTRA-BUCKET CORRELATION (rho_kl)")
print(eqvg_rho_print)

print("\n5) INTRA-BUCKET AGGREGATION (K_b)")
print("\n--- Bucket 12 ---")
print(eqvg_Kb12_formula_str)
print(eqvg_K_b12_print)

print("\n6) BUCKET SUMS (S_b)")
print(eqvg_bucket_sums_print)

print("\n7) CROSS-BUCKET CORRELATION (gamma_bc)")
print(eqvg_gamma_print)

print("\n8) CROSS-BUCKET AGGREGATION")
print(eqvg_K_formula_str)
print(eqvg_cross_agg_print)

print("\n9) CORRELATION SCENARIOS")
print(eqvg_scenarios_print)

print("\n10) CAPITAL")
print(eqvg_capital_print)

EQVG

1) GROSS SENSITIVITIES
┌────────────┬──────────┬──────────┬────────────────┬─────────────┐
│ trade_id   │ bucket   │ spot     │ opt_maturity   │   gross_s_k │
├────────────┼──────────┼──────────┼────────────────┼─────────────┤
│ 1          │ 12       │ issuer_a │ 10y            │      -1,800 │
│ 2          │ 12       │ issuer_a │ 10y            │      -3,600 │
│ 3          │ 12       │ issuer_a │ 6m             │      -3,900 │
│ 4          │ 12       │ issuer_a │ 6m             │       1,400 │
│ 5          │ 12       │ issuer_b │ 10y            │      -1,200 │
│ 6          │ 12       │ issuer_b │ 6m             │       1,700 │
└────────────┴──────────┴──────────┴────────────────┴─────────────┘

2) NET SENSITIVITIES (s_k)
┌──────────┬──────────┬────────────────┬────────┐
│ bucket   │ spot     │ opt_maturity   │    s_k │
├──────────┼──────────┼────────────────┼────────┤
│ 12       │ issuer_a │ 10y            │ -5,400 │
│ 12       │ issuer_a │ 6m             │ -2,500 │
│ 12       │ 

### CMVG

In [16]:
# ============================================================
# CMVG
# ============================================================


# TABLE OF CONTENTS
#   1) Gross sensitivities
#   2) Net sensitivities (s_k)
#   3) Weighted sensitivities (WS_k)
#   4) Intra-bucket correlation (rho_kl)
#   5) Intra-bucket aggregation (K_b)
#   6) Bucket sums (S_b)
#   7) Cross-bucket correlation (gamma_bc)
#   8) Cross-bucket aggregation
#   9) Correlation scenarios (high/low)
#  10) Capital
#  11) Printing
# ============================================================


# ============================================================
# 1) GROSS SENSITIVITIES
# ============================================================
# Position-level vega sensitivities to commodity volatility.

data = [
    {'trade_id': 1, 'bucket': 6, 'commodity': 'bnp',   'opt_maturity': '1y', 'gross_s_k': 750},
    {'trade_id': 2, 'bucket': 6, 'commodity': 'ttf',   'opt_maturity': '6m', 'gross_s_k': -800},
    {'trade_id': 3, 'bucket': 2, 'commodity': 'brent', 'opt_maturity': '1y', 'gross_s_k': 3100},
    {'trade_id': 4, 'bucket': 2, 'commodity': 'brent', 'opt_maturity': '6m', 'gross_s_k': 1600},
    {'trade_id': 5, 'bucket': 2, 'commodity': 'brent', 'opt_maturity': '6m', 'gross_s_k': -1200},
]

cmvg_gross_sk = pd.DataFrame(data)


# ============================================================
# 2) NET SENSITIVITIES (s_k)
# ============================================================
# Net sensitivity s_k = Σ gross sensitivities for each unique risk factor.
# Risk factor defined by: bucket + commodity + option_maturity.

cmvg_risk_factor_cols = ['bucket', 'commodity', 'opt_maturity']

cmvg_net_sk = (
    cmvg_gross_sk
    .groupby(cmvg_risk_factor_cols)
    .agg(s_k=('gross_s_k', 'sum'))
    .reset_index()
)

# Add option maturity in years for correlation calculation
OPTION_MATURITY_TO_YEARS = {'6m': 0.5, '1y': 1.0}
cmvg_net_sk['maturity_years'] = cmvg_net_sk['opt_maturity'].map(OPTION_MATURITY_TO_YEARS)


# ============================================================
# 3) WEIGHTED SENSITIVITIES (WS_k)
# ============================================================
# WS_k = s_k × RW_k

# Risk weight (RW_k) from Article 325ay
# For Commodity Vega: RW = 100%
cmvg_risk_weight = 1.00

cmvg_wsk = cmvg_net_sk.copy()
cmvg_wsk['rw'] = cmvg_risk_weight
cmvg_wsk['WS_k'] = cmvg_wsk['s_k'] * cmvg_wsk['rw']

# Extract weighted sensitivities per bucket for aggregation
# Bucket 2: Brent 1Y vs Brent 6M
cmvg_ws_b2 = cmvg_wsk[cmvg_wsk['bucket'] == 2]['WS_k'].values
cmvg_WS_k_b2_1y = cmvg_ws_b2[0]  # Brent 1Y
cmvg_WS_k_b2_6m = cmvg_ws_b2[1]  # Brent 6M (netted)

# Bucket 6: NLTTF 6M vs UKNBP 1Y
cmvg_ws_b6 = cmvg_wsk[cmvg_wsk['bucket'] == 6]['WS_k'].values
cmvg_WS_k_b6_nlttf = cmvg_ws_b6[0]  # NLTTF 6M (Assuming 'ttf' maps here)
cmvg_WS_k_b6_uknbp = cmvg_ws_b6[1]  # UKNBP 1Y (Assuming 'bnp' maps here)


# ============================================================
# 4) INTRA-BUCKET CORRELATION (rho_kl)
# ============================================================
# ρ_kl = ρ_delta_cty × ρ_option_maturity (Article 325ay)

ALPHA = 0.01

t_1y, t_6m = 1.0, 0.5
cmvg_rho_option_maturity = np.exp(-ALPHA * abs(t_1y - t_6m) / min(t_1y, t_6m))

# Bucket 2: Brent 1Y vs Brent 6M (same commodity)
cmvg_rho_delta_cty_b2 = 1.0   # identical commodities
cmvg_rho_kl_b2 = cmvg_rho_delta_cty_b2 * cmvg_rho_option_maturity

# Bucket 6: NLTTF 6M vs UKNBP 1Y (different commodity)
cmvg_rho_delta_cty_b6 = 0.65  # different commodities (Table 10)
cmvg_rho_kl_b6 = cmvg_rho_delta_cty_b6 * cmvg_rho_option_maturity


# ============================================================
# 5) INTRA-BUCKET AGGREGATION (K_b)
# ============================================================
# K_b = √( Σ WS_k² + Σ Σ ρ_kl × WS_k × WS_l )  for k ≠ l

# Bucket 2: 2 risk factors
cmvg_sum_sq_b2 = cmvg_WS_k_b2_1y**2 + cmvg_WS_k_b2_6m**2
cmvg_cross_b2 = 2 * cmvg_rho_kl_b2 * cmvg_WS_k_b2_1y * cmvg_WS_k_b2_6m
cmvg_K_b2 = np.sqrt(max(0, cmvg_sum_sq_b2 + cmvg_cross_b2))

# Bucket 6: 2 risk factors
cmvg_sum_sq_b6 = cmvg_WS_k_b6_nlttf**2 + cmvg_WS_k_b6_uknbp**2
cmvg_cross_b6 = 2 * cmvg_rho_kl_b6 * cmvg_WS_k_b6_nlttf * cmvg_WS_k_b6_uknbp
cmvg_K_b6 = np.sqrt(max(0, cmvg_sum_sq_b6 + cmvg_cross_b6))

# --- Explicit K_b formula strings for printing ---
cmvg_Kb2_formula_str = (
    f"K_b2 = sqrt( ({cmvg_WS_k_b2_1y:.2f}^2 + {cmvg_WS_k_b2_6m:.2f}^2) "
    f"+ (2 * {cmvg_rho_kl_b2:.4f} * {cmvg_WS_k_b2_1y:.2f} * {cmvg_WS_k_b2_6m:.2f}) )"
)

cmvg_Kb6_formula_str = (
    f"K_b6 = sqrt( ({cmvg_WS_k_b6_nlttf:.2f}^2 + {cmvg_WS_k_b6_uknbp:.2f}^2) "
    f"+ (2 * {cmvg_rho_kl_b6:.4f} * {cmvg_WS_k_b6_nlttf:.2f} * {cmvg_WS_k_b6_uknbp:.2f}) )"
)


# ============================================================
# 6) BUCKET SUMS (S_b)
# ============================================================
# S_b = Σ WS_k

cmvg_S_b2 = cmvg_wsk[cmvg_wsk['bucket'] == 2]['WS_k'].sum()
cmvg_S_b6 = cmvg_wsk[cmvg_wsk['bucket'] == 6]['WS_k'].sum()


# ============================================================
# 7) CROSS-BUCKET CORRELATION (gamma_bc)
# ============================================================
# γ = 0.20 for Commodity bucket pairs (Article 325au)

cmvg_gamma = 0.20


# ============================================================
# 8) CROSS-BUCKET AGGREGATION
# ============================================================
# K = √( Σ K_b² + Σ Σ γ_bc × S_b × S_c )  for b ≠ c

cmvg_sum_K_sq = cmvg_K_b2**2 + cmvg_K_b6**2
cmvg_cross_bucket = 2 * cmvg_gamma * cmvg_S_b2 * cmvg_S_b6

cmvg_k_squared = cmvg_sum_K_sq + cmvg_cross_bucket

# Apply the check at the overall level (Floor at 0)
cmvg_k_squared = max(cmvg_k_squared, 0)

cmvg_K_medium = np.sqrt(cmvg_k_squared)

# --- Explicit Cross-bucket formula string for printing ---
cmvg_K_formula_str = (
    f"Capital = sqrt( ({cmvg_K_b2:.2f}^2 + {cmvg_K_b6:.2f}^2) "
    f"+ (2 * {cmvg_gamma:.2f} * {cmvg_S_b2:.2f} * {cmvg_S_b6:.2f}) )"
)


# ============================================================
# 9) CORRELATION SCENARIOS (HIGH/LOW)
# ============================================================
# High: ρ_high = min(1.25 × ρ_medium, 1.0)
# Low:  ρ_low  = max(2 × ρ_medium - 1, 0.75 × ρ_medium)

# --- High scenario correlations --- #
cmvg_rho_b2_high = min(cmvg_rho_kl_b2 * 1.25, 1.0)
cmvg_rho_b6_high = min(cmvg_rho_kl_b6 * 1.25, 1.0)
cmvg_gamma_high = min(cmvg_gamma * 1.25, 1.0)

# --- Low scenario correlations --- #
cmvg_rho_b2_low = max(2 * cmvg_rho_kl_b2 - 1.0, 0.75 * cmvg_rho_kl_b2)
cmvg_rho_b6_low = max(2 * cmvg_rho_kl_b6 - 1.0, 0.75 * cmvg_rho_kl_b6)
cmvg_gamma_low = max(2 * cmvg_gamma - 1.0, 0.75 * cmvg_gamma)

# --- High scenario intra-bucket K_b --- #
cmvg_cross_b2_high = 2 * cmvg_rho_b2_high * cmvg_WS_k_b2_1y * cmvg_WS_k_b2_6m
cmvg_K_b2_high = np.sqrt(max(0, cmvg_sum_sq_b2 + cmvg_cross_b2_high))

cmvg_cross_b6_high = 2 * cmvg_rho_b6_high * cmvg_WS_k_b6_nlttf * cmvg_WS_k_b6_uknbp
cmvg_K_b6_high = np.sqrt(max(0, cmvg_sum_sq_b6 + cmvg_cross_b6_high))

# --- Low scenario intra-bucket K_b --- #
cmvg_cross_b2_low = 2 * cmvg_rho_b2_low * cmvg_WS_k_b2_1y * cmvg_WS_k_b2_6m
cmvg_K_b2_low = np.sqrt(max(0, cmvg_sum_sq_b2 + cmvg_cross_b2_low))

cmvg_cross_b6_low = 2 * cmvg_rho_b6_low * cmvg_WS_k_b6_nlttf * cmvg_WS_k_b6_uknbp
cmvg_K_b6_low = np.sqrt(max(0, cmvg_sum_sq_b6 + cmvg_cross_b6_low))

# --- High scenario cross-bucket --- #
cmvg_k_sq_high = (
    cmvg_K_b2_high**2 + cmvg_K_b6_high**2
    + 2 * cmvg_gamma_high * cmvg_S_b2 * cmvg_S_b6
)
cmvg_K_high = np.sqrt(max(cmvg_k_sq_high, 0))

# --- Low scenario cross-bucket --- #
cmvg_k_sq_low = (
    cmvg_K_b2_low**2 + cmvg_K_b6_low**2
    + 2 * cmvg_gamma_low * cmvg_S_b2 * cmvg_S_b6
)
cmvg_K_low = np.sqrt(max(cmvg_k_sq_low, 0))


# ============================================================
# 10) CAPITAL
# ============================================================
# Final capital = max(K_medium, K_high, K_low)

cmvg_capital = max(cmvg_K_medium, cmvg_K_high, cmvg_K_low)


# ============================================================
# 11) PRINTING
# ============================================================

# --- 11.1 Gross sensitivities --- #
cmvg_gross_sk_print = format_table(
    cmvg_gross_sk,
    columns=['trade_id', 'bucket', 'commodity', 'opt_maturity', 'gross_s_k'],
    col_formats={'gross_s_k': ',.0f'},
    alignments=('left', 'left', 'left', 'left', 'right')
)

# --- 11.2 Net sensitivities (s_k) --- #
cmvg_net_sk_print = format_table(
    cmvg_net_sk,
    columns=['bucket', 'commodity', 'opt_maturity', 's_k'],
    col_formats={'s_k': ',.0f'},
    alignments=('left', 'left', 'left', 'right')
)

# --- 11.3 Weighted sensitivities (WS_k) --- #
cmvg_wsk_print = format_table(
    cmvg_wsk,
    columns=['bucket', 'commodity', 'opt_maturity', 's_k', 'rw', 'WS_k'],
    col_formats={'s_k': ',.0f', 'rw': '.2f', 'WS_k': ',.0f'},
    alignments=('left', 'left', 'left', 'right', 'right', 'right')
)

# --- 11.4 Intra-bucket correlation (rho_kl) --- #
# Columns renamed to match Vega-specific correlation components
cmvg_rho_print = format_table(
    pd.DataFrame([
        {'bucket': 2, 'pair': 'Brent 1Y vs 6M', 'rho_delta_cty': cmvg_rho_delta_cty_b2, 'rho_opt_maturity': cmvg_rho_option_maturity, 'rho_kl': cmvg_rho_kl_b2},
        {'bucket': 6, 'pair': 'NLTTF 6M vs UKNBP 1Y', 'rho_delta_cty': cmvg_rho_delta_cty_b6, 'rho_opt_maturity': cmvg_rho_option_maturity, 'rho_kl': cmvg_rho_kl_b6},
    ]),
    columns=['bucket', 'pair', 'rho_delta_cty', 'rho_opt_maturity', 'rho_kl'],
    col_formats={'rho_delta_cty': '.2f', 'rho_opt_maturity': '.4f', 'rho_kl': '.4f'},
    alignments=('left', 'left', 'right', 'right', 'right')
)

# --- 11.5 Intra-bucket aggregation (K_b) --- #
# Bucket 2
cmvg_K_b2_print = format_table(
    pd.DataFrame([
        {'component': 'WS_k_1 (Brent 1Y)', 'value': cmvg_WS_k_b2_1y},
        {'component': 'WS_k_2 (Brent 6M)', 'value': cmvg_WS_k_b2_6m},
        {'component': 'rho_kl', 'value': f"{cmvg_rho_kl_b2:.4f}"},
        {'component': 'K_b', 'value': cmvg_K_b2},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# Bucket 6
cmvg_K_b6_print = format_table(
    pd.DataFrame([
        {'component': 'WS_k_1 (NLTTF 6M)', 'value': cmvg_WS_k_b6_nlttf},
        {'component': 'WS_k_2 (UKNBP 1Y)', 'value': cmvg_WS_k_b6_uknbp},
        {'component': 'rho_kl', 'value': f"{cmvg_rho_kl_b6:.4f}"},
        {'component': 'K_b', 'value': cmvg_K_b6},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# --- 11.6 Bucket sums (S_b) --- #
cmvg_bucket_sums_print = format_table(
    pd.DataFrame([
        {'bucket': 2, 'WS_k_1': cmvg_WS_k_b2_1y, 'WS_k_2': cmvg_WS_k_b2_6m, 'S_b': cmvg_S_b2},
        {'bucket': 6, 'WS_k_1': cmvg_WS_k_b6_nlttf, 'WS_k_2': cmvg_WS_k_b6_uknbp, 'S_b': cmvg_S_b6},
    ]),
    columns=['bucket', 'WS_k_1', 'WS_k_2', 'S_b'],
    col_formats={'WS_k_1': ',.2f', 'WS_k_2': ',.2f', 'S_b': ',.2f'},
    alignments=('left', 'right', 'right', 'right')
)

# --- 11.7 Cross-bucket correlation (gamma_bc) --- #
cmvg_gamma_print = format_table(
    pd.DataFrame([
        {'buckets': '2 vs 6', 'gamma_bc': cmvg_gamma},
    ]),
    columns=['buckets', 'gamma_bc'],
    col_formats={'gamma_bc': '.2f'},
    alignments=('left', 'right')
)

# --- 11.8 Cross-bucket aggregation --- #
cmvg_cross_agg_print = format_table(
    pd.DataFrame([
        {'component': 'K_b2', 'value': cmvg_K_b2},
        {'component': 'K_b6', 'value': cmvg_K_b6},
        {'component': 'S_b2', 'value': cmvg_S_b2},
        {'component': 'S_b6', 'value': cmvg_S_b6},
        {'component': 'gamma_bc', 'value': f"{cmvg_gamma:.2f}"},
        {'component': 'capital', 'value': cmvg_K_medium},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# --- 11.9 Correlation scenarios --- #
cmvg_scenarios_print = format_table(
    pd.DataFrame([
        {'parameter': 'rho_kl_b2', 'medium': cmvg_rho_kl_b2, 'high': cmvg_rho_b2_high, 'low': cmvg_rho_b2_low},
        {'parameter': 'rho_kl_b6', 'medium': cmvg_rho_kl_b6, 'high': cmvg_rho_b6_high, 'low': cmvg_rho_b6_low},
        {'parameter': 'gamma_bc', 'medium': cmvg_gamma, 'high': cmvg_gamma_high, 'low': cmvg_gamma_low},
    ]),
    columns=['parameter', 'medium', 'high', 'low'],
    col_formats={'medium': '.4f', 'high': '.4f', 'low': '.4f'},
    alignments=('left', 'right', 'right', 'right')
)

# --- 11.10 Capital --- #
cmvg_capital_print = format_table(
    pd.DataFrame([
        {'scenario': 'medium', 'capital': cmvg_K_medium},
        {'scenario': 'high', 'capital': cmvg_K_high},
        {'scenario': 'low', 'capital': cmvg_K_low},
    ]),
    columns=['scenario', 'capital'],
    col_formats={'capital': ',.2f'},
    alignments=('left', 'right')
)


# ============================================================
# DISPLAY ALL OUTPUTS
# ============================================================

print("=" * 60)
print("CMVG")
print("=" * 60)

print("\n1) GROSS SENSITIVITIES")
print(cmvg_gross_sk_print)

print("\n2) NET SENSITIVITIES (s_k)")
print(cmvg_net_sk_print)

print("\n3) WEIGHTED SENSITIVITIES (WS_k)")
print(cmvg_wsk_print)

print("\n4) INTRA-BUCKET CORRELATION (rho_kl)")
print(cmvg_rho_print)

print("\n5) INTRA-BUCKET AGGREGATION (K_b)")
print("\n--- Bucket 2 ---")
print(cmvg_Kb2_formula_str)
print(cmvg_K_b2_print)
print("\n--- Bucket 6 ---")
print(cmvg_Kb6_formula_str)
print(cmvg_K_b6_print)

print("\n6) BUCKET SUMS (S_b)")
print(cmvg_bucket_sums_print)

print("\n7) CROSS-BUCKET CORRELATION (gamma_bc)")
print(cmvg_gamma_print)

print("\n8) CROSS-BUCKET AGGREGATION")
print(cmvg_K_formula_str)
print(cmvg_cross_agg_print)

print("\n9) CORRELATION SCENARIOS")
print(cmvg_scenarios_print)

print("\n10) CAPITAL")
print(cmvg_capital_print)

CMVG

1) GROSS SENSITIVITIES
┌────────────┬──────────┬─────────────┬────────────────┬─────────────┐
│ trade_id   │ bucket   │ commodity   │ opt_maturity   │   gross_s_k │
├────────────┼──────────┼─────────────┼────────────────┼─────────────┤
│ 1          │ 6        │ bnp         │ 1y             │         750 │
│ 2          │ 6        │ ttf         │ 6m             │        -800 │
│ 3          │ 2        │ brent       │ 1y             │       3,100 │
│ 4          │ 2        │ brent       │ 6m             │       1,600 │
│ 5          │ 2        │ brent       │ 6m             │      -1,200 │
└────────────┴──────────┴─────────────┴────────────────┴─────────────┘

2) NET SENSITIVITIES (s_k)
┌──────────┬─────────────┬────────────────┬───────┐
│ bucket   │ commodity   │ opt_maturity   │   s_k │
├──────────┼─────────────┼────────────────┼───────┤
│ 2        │ brent       │ 1y             │ 3,100 │
│ 2        │ brent       │ 6m             │   400 │
│ 6        │ bnp         │ 1y             │ 

### FXVG

In [17]:
# ============================================================
# FXVG
# ============================================================


# TABLE OF CONTENTS
#   1) Gross sensitivities
#   2) Net sensitivities (s_k)
#   3) Weighted sensitivities (WS_k)
#   4) Intra-bucket correlation (rho_kl)
#   5) Intra-bucket aggregation (K_b)
#   6) Bucket sums (S_b)
#   7) Cross-bucket correlation (gamma_bc)
#   8) Cross-bucket aggregation
#   9) Correlation scenarios (high/low)
#  10) Capital
#  11) Printing
# ============================================================


# ============================================================
# 1) GROSS SENSITIVITIES
# ============================================================
# Position-level vega sensitivities to FX volatility.

data = [
    {'trade_id': 1, 'bucket': 'gbp_usd', 'opt_maturity': '6m', 'gross_s_k': -1600},
    {'trade_id': 2, 'bucket': 'gbp_usd', 'opt_maturity': '6m', 'gross_s_k':  2200},
    {'trade_id': 3, 'bucket': 'gbp_usd', 'opt_maturity': '1y', 'gross_s_k':   830},
    {'trade_id': 4, 'bucket': 'jpy_usd', 'opt_maturity': '1y', 'gross_s_k': -1200},
]

fxvg_gross_sk = pd.DataFrame(data)


# ============================================================
# 2) NET SENSITIVITIES (s_k)
# ============================================================
# Net sensitivity s_k = Σ gross sensitivities for each unique risk factor.
# Risk factor defined by: bucket (currency pair) + opt_maturity.

fxvg_risk_factor_cols = ['bucket', 'opt_maturity']

fxvg_net_sk = (
    fxvg_gross_sk
    .groupby(fxvg_risk_factor_cols)
    .agg(s_k=('gross_s_k', 'sum'))
    .reset_index()
)

# Add tenor in years for correlation calculation
TENOR_TO_YEARS = {'6m': 0.5, '1y': 1.0}
fxvg_net_sk['tenor_years'] = fxvg_net_sk['opt_maturity'].map(TENOR_TO_YEARS)


# ============================================================
# 3) WEIGHTED SENSITIVITIES (WS_k)
# ============================================================
# WS_k = s_k × RW_k

# Risk weight (RW_k) from Article 325ay
# For FX Vega: RW = 100%
fxvg_risk_weight = 1.00

fxvg_wsk = fxvg_net_sk.copy()
fxvg_wsk['rw'] = fxvg_risk_weight
fxvg_wsk['WS_k'] = fxvg_wsk['s_k'] * fxvg_wsk['rw']

# Extract weighted sensitivities per bucket for aggregation
# GBP^USD bucket: 2 risk factors (6M and 1Y tenors)
fxvg_ws_gbp_usd = fxvg_wsk[fxvg_wsk['bucket'] == 'gbp_usd']['WS_k'].values
fxvg_WS_k_gbp_usd_6m = fxvg_ws_gbp_usd[0]  # 6M tenor
fxvg_WS_k_gbp_usd_1y = fxvg_ws_gbp_usd[1]  # 1Y tenor

# JPY^USD bucket: 1 risk factor (1Y tenor)
fxvg_ws_jpy_usd = fxvg_wsk[fxvg_wsk['bucket'] == 'jpy_usd']['WS_k'].values
fxvg_WS_k_jpy_usd_1y = fxvg_ws_jpy_usd[0]  # 1Y tenor


# ============================================================
# 4) INTRA-BUCKET CORRELATION (rho_kl)
# ============================================================
# ρ_kl = ρ_delta × ρ_option_maturity (Article 325ay)

ALPHA = 0.01

# GBP^USD bucket: correlation between 6M and 1Y tenors
# Same currency pair -> rho_delta = 1.0
fxvg_rho_delta_gbp_usd = 1.0

# Different tenors -> rho_opt
t_6m, t_1y = 0.5, 1.0
fxvg_rho_opt_maturity_gbp_usd = np.exp(-ALPHA * abs(t_6m - t_1y) / min(t_6m, t_1y))

fxvg_rho_kl_gbp_usd = fxvg_rho_delta_gbp_usd * fxvg_rho_opt_maturity_gbp_usd

# JPY^USD bucket: single risk factor, no correlation needed
fxvg_rho_kl_jpy_usd = None


# ============================================================
# 5) INTRA-BUCKET AGGREGATION (K_b)
# ============================================================
# K_b = √( Σ WS_k² + Σ Σ ρ_kl × WS_k × WS_l )  for k ≠ l

# GBP^USD bucket: 2 risk factors
fxvg_sum_sq_gbp = fxvg_WS_k_gbp_usd_6m**2 + fxvg_WS_k_gbp_usd_1y**2
fxvg_cross_gbp = 2 * fxvg_rho_kl_gbp_usd * fxvg_WS_k_gbp_usd_6m * fxvg_WS_k_gbp_usd_1y
fxvg_K_gbp_usd = np.sqrt(max(0, fxvg_sum_sq_gbp + fxvg_cross_gbp))

# JPY^USD bucket: 1 risk factor → K_b = |WS_k|
fxvg_K_jpy_usd = abs(fxvg_WS_k_jpy_usd_1y)

# --- Explicit K_b formula strings for printing ---
fxvg_Kb_gbp_formula_str = (
    f"K_gbp = sqrt( ({fxvg_WS_k_gbp_usd_6m:.2f}^2 + {fxvg_WS_k_gbp_usd_1y:.2f}^2) "
    f"+ (2 * {fxvg_rho_kl_gbp_usd:.4f} * {fxvg_WS_k_gbp_usd_6m:.2f} * {fxvg_WS_k_gbp_usd_1y:.2f}) )"
)

fxvg_Kb_jpy_formula_str = f"K_jpy = abs({fxvg_WS_k_jpy_usd_1y:.2f})"


# ============================================================
# 6) BUCKET SUMS (S_b)
# ============================================================
# S_b = Σ WS_k

fxvg_S_gbp_usd = fxvg_wsk[fxvg_wsk['bucket'] == 'gbp_usd']['WS_k'].sum()
fxvg_S_jpy_usd = fxvg_wsk[fxvg_wsk['bucket'] == 'jpy_usd']['WS_k'].sum()


# ============================================================
# 7) CROSS-BUCKET CORRELATION (gamma_bc)
# ============================================================
# γ = 0.60 for FX bucket pairs (Article 325aw)

fxvg_gamma = 0.60


# ============================================================
# 8) CROSS-BUCKET AGGREGATION
# ============================================================
# K = √( Σ K_b² + Σ Σ γ_bc × S_b × S_c )  for b ≠ c

fxvg_sum_K_sq = fxvg_K_gbp_usd**2 + fxvg_K_jpy_usd**2
fxvg_cross_bucket = 2 * fxvg_gamma * fxvg_S_gbp_usd * fxvg_S_jpy_usd

fxvg_k_squared = fxvg_sum_K_sq + fxvg_cross_bucket

# Apply the check at the overall level (Floor at 0)
fxvg_k_squared = max(fxvg_k_squared, 0)

fxvg_K_medium = np.sqrt(fxvg_k_squared)

# --- Explicit Cross-bucket formula string for printing ---
fxvg_K_formula_str = (
    f"Capital = sqrt( ({fxvg_K_gbp_usd:.2f}^2 + {fxvg_K_jpy_usd:.2f}^2) "
    f"+ (2 * {fxvg_gamma:.2f} * {fxvg_S_gbp_usd:.2f} * {fxvg_S_jpy_usd:.2f}) )"
)


# ============================================================
# 9) CORRELATION SCENARIOS (HIGH/LOW)
# ============================================================
# High: ρ_high = min(1.25 × ρ_medium, 1.0)
# Low:  ρ_low  = max(2 × ρ_medium - 1, 0.75 × ρ_medium)

# --- High scenario correlations --- #
fxvg_rho_gbp_usd_high = min(fxvg_rho_kl_gbp_usd * 1.25, 1.0)
fxvg_gamma_high = min(fxvg_gamma * 1.25, 1.0)

# --- Low scenario correlations --- #
fxvg_rho_gbp_usd_low = max(2 * fxvg_rho_kl_gbp_usd - 1.0, 0.75 * fxvg_rho_kl_gbp_usd)
fxvg_gamma_low = max(2 * fxvg_gamma - 1.0, 0.75 * fxvg_gamma)

# --- High scenario intra-bucket K_b --- #
fxvg_cross_gbp_high = 2 * fxvg_rho_gbp_usd_high * fxvg_WS_k_gbp_usd_6m * fxvg_WS_k_gbp_usd_1y
fxvg_K_gbp_usd_high = np.sqrt(max(0, fxvg_sum_sq_gbp + fxvg_cross_gbp_high))

fxvg_K_jpy_usd_high = abs(fxvg_WS_k_jpy_usd_1y)  # Single position, unchanged

# --- Low scenario intra-bucket K_b --- #
fxvg_cross_gbp_low = 2 * fxvg_rho_gbp_usd_low * fxvg_WS_k_gbp_usd_6m * fxvg_WS_k_gbp_usd_1y
fxvg_K_gbp_usd_low = np.sqrt(max(0, fxvg_sum_sq_gbp + fxvg_cross_gbp_low))

fxvg_K_jpy_usd_low = abs(fxvg_WS_k_jpy_usd_1y)  # Single position, unchanged

# --- High scenario cross-bucket --- #
fxvg_k_sq_high = (
    fxvg_K_gbp_usd_high**2 + fxvg_K_jpy_usd_high**2
    + 2 * fxvg_gamma_high * fxvg_S_gbp_usd * fxvg_S_jpy_usd
)
fxvg_K_high = np.sqrt(max(fxvg_k_sq_high, 0))

# --- Low scenario cross-bucket --- #
fxvg_k_sq_low = (
    fxvg_K_gbp_usd_low**2 + fxvg_K_jpy_usd_low**2
    + 2 * fxvg_gamma_low * fxvg_S_gbp_usd * fxvg_S_jpy_usd
)
fxvg_K_low = np.sqrt(max(fxvg_k_sq_low, 0))


# ============================================================
# 10) CAPITAL
# ============================================================
# Final capital = max(K_medium, K_high, K_low)

fxvg_capital = max(fxvg_K_medium, fxvg_K_high, fxvg_K_low)


# ============================================================
# 11) PRINTING
# ============================================================

# --- 11.1 Gross sensitivities --- #
fxvg_gross_sk_print = format_table(
    fxvg_gross_sk,
    columns=['trade_id', 'bucket', 'opt_maturity', 'gross_s_k'],
    col_formats={'gross_s_k': ',.0f'},
    alignments=('left', 'left', 'left', 'right')
)

# --- 11.2 Net sensitivities (s_k) --- #
fxvg_net_sk_print = format_table(
    fxvg_net_sk,
    columns=['bucket', 'opt_maturity', 's_k'],
    col_formats={'s_k': ',.0f'},
    alignments=('left', 'left', 'right')
)

# --- 11.3 Weighted sensitivities (WS_k) --- #
fxvg_wsk_print = format_table(
    fxvg_wsk,
    columns=['bucket', 'opt_maturity', 's_k', 'rw', 'WS_k'],
    col_formats={'s_k': ',.0f', 'rw': '.2f', 'WS_k': ',.0f'},
    alignments=('left', 'left', 'right', 'right', 'right')
)

# --- 11.4 Intra-bucket correlation (rho_kl) --- #
# Columns renamed to match Vega-specific correlation components
fxvg_rho_print = format_table(
    pd.DataFrame([
        {'bucket': 'gbp_usd', 'pair': '6M vs 1Y', 'rho_delta': fxvg_rho_delta_gbp_usd, 'rho_opt_maturity': fxvg_rho_opt_maturity_gbp_usd, 'rho_kl': fxvg_rho_kl_gbp_usd},
        {'bucket': 'jpy_usd', 'pair': 'n/a', 'rho_delta': 'n/a', 'rho_opt_maturity': 'n/a', 'rho_kl': 'n/a'},
    ]),
    columns=['bucket', 'pair', 'rho_delta', 'rho_opt_maturity', 'rho_kl'],
    col_formats={'rho_delta': '.2f', 'rho_opt_maturity': '.4f', 'rho_kl': '.4f'},
    alignments=('left', 'left', 'right', 'right', 'right')
)

# --- 11.5 Intra-bucket aggregation (K_b) --- #
# GBP Bucket
fxvg_K_gbp_print = format_table(
    pd.DataFrame([
        {'component': 'WS_k_1 (6m)', 'value': fxvg_WS_k_gbp_usd_6m},
        {'component': 'WS_k_2 (1y)', 'value': fxvg_WS_k_gbp_usd_1y},
        {'component': 'rho_kl', 'value': f"{fxvg_rho_kl_gbp_usd:.4f}"},
        {'component': 'K_b', 'value': fxvg_K_gbp_usd},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# JPY Bucket
fxvg_K_jpy_print = format_table(
    pd.DataFrame([
        {'component': 'WS_k_1 (1y)', 'value': fxvg_WS_k_jpy_usd_1y},
        {'component': 'K_b', 'value': fxvg_K_jpy_usd},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# --- 11.6 Bucket sums (S_b) --- #
fxvg_bucket_sums_print = format_table(
    pd.DataFrame([
        {'bucket': 'gbp_usd', 'WS_k_1': fxvg_WS_k_gbp_usd_6m, 'WS_k_2': fxvg_WS_k_gbp_usd_1y, 'S_b': fxvg_S_gbp_usd},
        {'bucket': 'jpy_usd', 'WS_k_1': fxvg_WS_k_jpy_usd_1y, 'WS_k_2': 'n/a', 'S_b': fxvg_S_jpy_usd},
    ]),
    columns=['bucket', 'WS_k_1', 'WS_k_2', 'S_b'],
    col_formats={'WS_k_1': ',.2f', 'WS_k_2': ',.2f', 'S_b': ',.2f'},
    alignments=('left', 'right', 'right', 'right')
)

# --- 11.7 Cross-bucket correlation (gamma_bc) --- #
fxvg_gamma_print = format_table(
    pd.DataFrame([
        {'buckets': 'GBP vs JPY', 'gamma_bc': fxvg_gamma},
    ]),
    columns=['buckets', 'gamma_bc'],
    col_formats={'gamma_bc': '.2f'},
    alignments=('left', 'right')
)

# --- 11.8 Cross-bucket aggregation --- #
fxvg_cross_agg_print = format_table(
    pd.DataFrame([
        {'component': 'K_gbp_usd', 'value': fxvg_K_gbp_usd},
        {'component': 'K_jpy_usd', 'value': fxvg_K_jpy_usd},
        {'component': 'S_gbp_usd', 'value': fxvg_S_gbp_usd},
        {'component': 'S_jpy_usd', 'value': fxvg_S_jpy_usd},
        {'component': 'gamma_bc', 'value': f"{fxvg_gamma:.2f}"},
        {'component': 'capital', 'value': fxvg_K_medium},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# --- 11.9 Correlation scenarios --- #
fxvg_scenarios_print = format_table(
    pd.DataFrame([
        {'parameter': 'rho_kl_gbp_usd', 'medium': fxvg_rho_kl_gbp_usd, 'high': fxvg_rho_gbp_usd_high, 'low': fxvg_rho_gbp_usd_low},
        {'parameter': 'gamma_bc', 'medium': fxvg_gamma, 'high': fxvg_gamma_high, 'low': fxvg_gamma_low},
    ]),
    columns=['parameter', 'medium', 'high', 'low'],
    col_formats={'medium': '.4f', 'high': '.4f', 'low': '.4f'},
    alignments=('left', 'right', 'right', 'right')
)

# --- 11.10 Capital --- #
fxvg_capital_print = format_table(
    pd.DataFrame([
        {'scenario': 'medium', 'capital': fxvg_K_medium},
        {'scenario': 'high', 'capital': fxvg_K_high},
        {'scenario': 'low', 'capital': fxvg_K_low},
    ]),
    columns=['scenario', 'capital'],
    col_formats={'capital': ',.2f'},
    alignments=('left', 'right')
)


# ============================================================
# DISPLAY ALL OUTPUTS
# ============================================================

print("=" * 60)
print("FXVG")
print("=" * 60)

print("\n1) GROSS SENSITIVITIES")
print(fxvg_gross_sk_print)

print("\n2) NET SENSITIVITIES (s_k)")
print(fxvg_net_sk_print)

print("\n3) WEIGHTED SENSITIVITIES (WS_k)")
print(fxvg_wsk_print)

print("\n4) INTRA-BUCKET CORRELATION (rho_kl)")
print(fxvg_rho_print)

print("\n5) INTRA-BUCKET AGGREGATION (K_b)")
print("\n--- GBP Bucket ---")
print(fxvg_Kb_gbp_formula_str)
print(fxvg_K_gbp_print)
print("\n--- JPY Bucket ---")
print(fxvg_Kb_jpy_formula_str)
print(fxvg_K_jpy_print)

print("\n6) BUCKET SUMS (S_b)")
print(fxvg_bucket_sums_print)

print("\n7) CROSS-BUCKET CORRELATION (gamma_bc)")
print(fxvg_gamma_print)

print("\n8) CROSS-BUCKET AGGREGATION")
print(fxvg_K_formula_str)
print(fxvg_cross_agg_print)

print("\n9) CORRELATION SCENARIOS")
print(fxvg_scenarios_print)

print("\n10) CAPITAL")
print(fxvg_capital_print)

FXVG

1) GROSS SENSITIVITIES
┌────────────┬──────────┬────────────────┬─────────────┐
│ trade_id   │ bucket   │ opt_maturity   │   gross_s_k │
├────────────┼──────────┼────────────────┼─────────────┤
│ 1          │ gbp_usd  │ 6m             │      -1,600 │
│ 2          │ gbp_usd  │ 6m             │       2,200 │
│ 3          │ gbp_usd  │ 1y             │         830 │
│ 4          │ jpy_usd  │ 1y             │      -1,200 │
└────────────┴──────────┴────────────────┴─────────────┘

2) NET SENSITIVITIES (s_k)
┌──────────┬────────────────┬────────┐
│ bucket   │ opt_maturity   │    s_k │
├──────────┼────────────────┼────────┤
│ gbp_usd  │ 1y             │    830 │
│ gbp_usd  │ 6m             │    600 │
│ jpy_usd  │ 1y             │ -1,200 │
└──────────┴────────────────┴────────┘

3) WEIGHTED SENSITIVITIES (WS_k)
┌──────────┬────────────────┬────────┬──────┬────────┐
│ bucket   │ opt_maturity   │    s_k │   rw │   WS_k │
├──────────┼────────────────┼────────┼──────┼────────┤
│ gbp_usd  │ 1y

## Curvature

### IRCV

In [18]:
# ============================================================
# IRCV
# ============================================================


# TABLE OF CONTENTS
#   1) Gross curvature positions
#   2) Net curvature positions (CVR+/CVR-)
#   3) Intra-bucket correlation (rho_kl)
#   4) Intra-bucket safeguard function (ψ)
#   5) Upward scenario (K_b+)
#   6) Downward scenario (K_b-)
#   7) Intra-bucket aggregation (K_b)
#   8) Bucket sums (S_b)
#   9) Cross-bucket correlation (gamma_bc)
#  10) Cross-bucket safeguard function (ψ)
#  11) Cross-bucket aggregation
#  12) Correlation scenarios (high/low)
#  13) Capital
#  14) Printing
# ============================================================


# ============================================================
# 1) GROSS CURVATURE POSITIONS
# ============================================================
# Position-level curvature values from upward (+) and downward (-) shocks.

data = [
    {'trade_id': 1, 'bucket': 'eur', 'curve': 'ois',     'cvr_plus':  2400, 'cvr_minus':  2600},
    {'trade_id': 2, 'bucket': 'eur', 'curve': 'ois',     'cvr_plus':  1300, 'cvr_minus':  1400},
    {'trade_id': 3, 'bucket': 'eur', 'curve': 'implied', 'cvr_plus':  1100, 'cvr_minus':  1600},
    {'trade_id': 4, 'bucket': 'eur', 'curve': 'implied', 'cvr_plus': -1700, 'cvr_minus': -2200},
    {'trade_id': 5, 'bucket': 'usd', 'curve': 'ois',     'cvr_plus':  3400, 'cvr_minus': -2700},
    {'trade_id': 6, 'bucket': 'usd', 'curve': 'ois',     'cvr_plus':  1400, 'cvr_minus':  1500},
    {'trade_id': 7, 'bucket': 'usd', 'curve': 'implied', 'cvr_plus': -3800, 'cvr_minus':  4500},
    {'trade_id': 8, 'bucket': 'usd', 'curve': 'implied', 'cvr_plus': -2600, 'cvr_minus': -3700},
]

ircv_gross_cvr = pd.DataFrame(data)


# ============================================================
# 2) NET CURVATURE POSITIONS (CVR+/CVR-)
# ============================================================
# Net CVR = Σ gross CVR for each unique risk factor.
# Per Article 325l, all curves within a currency = single risk factor.

ircv_risk_factor_cols = ['bucket']

ircv_net_cvr = (
    ircv_gross_cvr
    .groupby(ircv_risk_factor_cols)
    .agg(cvr_plus=('cvr_plus', 'sum'), cvr_minus=('cvr_minus', 'sum'))
    .reset_index()
)

# Extract CVR vectors per bucket
ircv_cvr_eur = ircv_net_cvr[ircv_net_cvr['bucket'] == 'eur']
ircv_cvr_plus_eur = ircv_cvr_eur['cvr_plus'].values[0]
ircv_cvr_minus_eur = ircv_cvr_eur['cvr_minus'].values[0]

ircv_cvr_usd = ircv_net_cvr[ircv_net_cvr['bucket'] == 'usd']
ircv_cvr_plus_usd = ircv_cvr_usd['cvr_plus'].values[0]
ircv_cvr_minus_usd = ircv_cvr_usd['cvr_minus'].values[0]


# ============================================================
# 3) INTRA-BUCKET CORRELATION (rho_kl)
# ============================================================
# GIRR consists of a single risk factor per bucket.
# Intra-bucket correlation is not applicable (effectively 0 or 1, but no cross terms).

ircv_rho_eur = None
ircv_rho_usd = None


# ============================================================
# 4) INTRA-BUCKET SAFEGUARD FUNCTION (ψ)
# ============================================================
# ψ(x, y) = 0 if both x < 0 and y < 0, else 1

def ircv_psi(x, y):
    """Safeguard function: returns 0 if both x and y are negative, else 1"""
    return 0 if x < 0 and y < 0 else 1

# Single risk factor per bucket -> No intra-bucket psi needed
ircv_psi_eur_plus = None
ircv_psi_eur_minus = None
ircv_psi_usd_plus = None
ircv_psi_usd_minus = None


# ============================================================
# 5) UPWARD SCENARIO (K_b+)
# ============================================================
# K_b+ = √( Σ max(CVR_k+, 0)² )  -- (No cross terms for single RF)

# --- EUR Bucket --- #
ircv_sum_sq_plus_eur = np.sum(np.maximum(ircv_cvr_plus_eur, 0)**2)
ircv_cross_plus_eur = 0
ircv_K_eur_plus = np.sqrt(max(0, ircv_sum_sq_plus_eur + ircv_cross_plus_eur))

# --- USD Bucket --- #
ircv_sum_sq_plus_usd = np.sum(np.maximum(ircv_cvr_plus_usd, 0)**2)
ircv_cross_plus_usd = 0
ircv_K_usd_plus = np.sqrt(max(0, ircv_sum_sq_plus_usd + ircv_cross_plus_usd))


# ============================================================
# 6) DOWNWARD SCENARIO (K_b-)
# ============================================================
# K_b- = √( Σ max(CVR_k-, 0)² ) -- (No cross terms for single RF)

# --- EUR Bucket --- #
ircv_sum_sq_minus_eur = np.sum(np.maximum(ircv_cvr_minus_eur, 0)**2)
ircv_cross_minus_eur = 0
ircv_K_eur_minus = np.sqrt(max(0, ircv_sum_sq_minus_eur + ircv_cross_minus_eur))

# --- USD Bucket --- #
ircv_sum_sq_minus_usd = np.sum(np.maximum(ircv_cvr_minus_usd, 0)**2)
ircv_cross_minus_usd = 0
ircv_K_usd_minus = np.sqrt(max(0, ircv_sum_sq_minus_usd + ircv_cross_minus_usd))


# ============================================================
# 7) INTRA-BUCKET AGGREGATION (K_b)
# ============================================================
# K_b = max(K_b+, K_b-)

# --- EUR Bucket --- #
if ircv_K_eur_plus > ircv_K_eur_minus:
    ircv_K_eur = ircv_K_eur_plus
    ircv_scenario_eur = "upward"
elif ircv_K_eur_minus > ircv_K_eur_plus:
    ircv_K_eur = ircv_K_eur_minus
    ircv_scenario_eur = "downward"
else:
    ircv_K_eur = ircv_K_eur_plus
    ircv_scenario_eur = "upward" if np.sum(ircv_cvr_plus_eur) >= np.sum(ircv_cvr_minus_eur) else "downward"

# --- USD Bucket --- #
if ircv_K_usd_plus > ircv_K_usd_minus:
    ircv_K_usd = ircv_K_usd_plus
    ircv_scenario_usd = "upward"
elif ircv_K_usd_minus > ircv_K_usd_plus:
    ircv_K_usd = ircv_K_usd_minus
    ircv_scenario_usd = "downward"
else:
    ircv_K_usd = ircv_K_usd_plus
    ircv_scenario_usd = "upward" if np.sum(ircv_cvr_plus_usd) >= np.sum(ircv_cvr_minus_usd) else "downward"


# ============================================================
# 8) BUCKET SUMS (S_b)
# ============================================================
# S_b = Σ CVR+ if upward scenario, Σ CVR- if downward scenario

ircv_S_eur = (
    ircv_cvr_plus_eur if ircv_scenario_eur == "upward"
    else ircv_cvr_minus_eur
)

ircv_S_usd = (
    ircv_cvr_plus_usd if ircv_scenario_usd == "upward"
    else ircv_cvr_minus_usd
)


# ============================================================
# 9) CROSS-BUCKET CORRELATION (gamma_bc)
# ============================================================
# γ_bc = (delta_γ_bc)²
# Delta correlation between currencies = 50%

IRCV_DELTA_GAMMA = 0.50
ircv_gamma = IRCV_DELTA_GAMMA ** 2


# ============================================================
# 10) CROSS-BUCKET SAFEGUARD FUNCTION (ψ)
# ============================================================
# ψ(S_b, S_c) = 0 if both S_b < 0 and S_c < 0, else 1

ircv_psi_cross = ircv_psi(ircv_S_eur, ircv_S_usd)


# ============================================================
# 11) CROSS-BUCKET AGGREGATION
# ============================================================
# K = √( Σ K_b² + Σ Σ γ_bc × S_b × S_c × ψ(S_b, S_c) )

ircv_sum_K_sq = ircv_K_eur**2 + ircv_K_usd**2
ircv_cross_bucket = (
    2 * ircv_gamma
    * ircv_S_eur
    * ircv_S_usd
    * ircv_psi_cross
)
ircv_K_medium = np.sqrt(max(0, ircv_sum_K_sq + ircv_cross_bucket))


# ============================================================
# 12) CORRELATION SCENARIOS (HIGH/LOW)
# ============================================================
# High: γ_high = min(1.25 × γ_medium, 1.0)
# Low:  γ_low  = max(2 × γ_medium - 1, 0.75 × γ_medium)
# Note: Intra-bucket K_b does not change because there are no intra-correlations to shock.

# --- High scenario correlations --- #
ircv_gamma_high = min(ircv_gamma * 1.25, 1.0)

# --- Low scenario correlations --- #
ircv_gamma_low = max(2 * ircv_gamma - 1.0, 0.75 * ircv_gamma)

# --- High scenario K (Scenarios/Sums identical to medium for single RF) --- #
ircv_S_eur_high = ircv_S_eur
ircv_S_usd_high = ircv_S_usd
ircv_psi_cross_high = ircv_psi(ircv_S_eur_high, ircv_S_usd_high)

ircv_sum_K_sq_high = ircv_K_eur**2 + ircv_K_usd**2
ircv_cross_bucket_high = 2 * ircv_gamma_high * ircv_S_eur_high * ircv_S_usd_high * ircv_psi_cross_high
ircv_K_high = np.sqrt(max(0, ircv_sum_K_sq_high + ircv_cross_bucket_high))

# --- Low scenario K --- #
ircv_S_eur_low = ircv_S_eur
ircv_S_usd_low = ircv_S_usd
ircv_psi_cross_low = ircv_psi(ircv_S_eur_low, ircv_S_usd_low)

ircv_sum_K_sq_low = ircv_K_eur**2 + ircv_K_usd**2
ircv_cross_bucket_low = 2 * ircv_gamma_low * ircv_S_eur_low * ircv_S_usd_low * ircv_psi_cross_low
ircv_K_low = np.sqrt(max(0, ircv_sum_K_sq_low + ircv_cross_bucket_low))


# ============================================================
# 13) CAPITAL
# ============================================================

ircv_capital_df = pd.DataFrame([
    {'scenario': 'medium', 'capital': ircv_K_medium},
    {'scenario': 'high', 'capital': ircv_K_high},
    {'scenario': 'low', 'capital': ircv_K_low},
])


# ============================================================
# 14) PRINTING
# ============================================================

# --- 14.1 Gross curvature positions --- #
ircv_gross_cvr_print = format_table(
    ircv_gross_cvr,
    columns=['trade_id', 'bucket', 'curve', 'cvr_plus', 'cvr_minus'],
    col_formats={'cvr_plus': ',.0f', 'cvr_minus': ',.0f'},
    alignments=('left', 'left', 'left', 'right', 'right')
)

# --- 14.2 Net curvature positions --- #
ircv_net_cvr_print = format_table(
    ircv_net_cvr,
    columns=['bucket', 'cvr_plus', 'cvr_minus'],
    col_formats={'cvr_plus': ',.0f', 'cvr_minus': ',.0f'},
    alignments=('left', 'right', 'right')
)

# --- 14.3 Intra-bucket correlation --- #
ircv_intra_corr_print = format_table(
    pd.DataFrame([
        {'bucket': 'eur', 'rho': 'n/a (single RF)'},
        {'bucket': 'usd', 'rho': 'n/a (single RF)'},
    ]),
    columns=['bucket', 'rho'],
    col_formats={},
    alignments=('left', 'right')
)

# --- 14.4 Intra-bucket safeguard function (ψ) --- #
ircv_psi_intra_print = format_table(
    pd.DataFrame([
        {'bucket': 'eur', 'cvr_k+': ircv_cvr_plus_eur, 'ψ_up': 'n/a', 'cvr_k-': ircv_cvr_minus_eur, 'ψ_down': 'n/a'},
        {'bucket': 'usd', 'cvr_k+': ircv_cvr_plus_usd, 'ψ_up': 'n/a', 'cvr_k-': ircv_cvr_minus_usd, 'ψ_down': 'n/a'},
    ]),
    columns=['bucket', 'cvr_k+', 'ψ_up', 'cvr_k-', 'ψ_down'],
    col_formats={'cvr_k+': ',.0f', 'cvr_k-': ',.0f'},
    alignments=('left', 'right', 'center', 'right', 'center')
)

# --- 14.5 Upward scenario (K_b+) --- #
# Calculation comments
ircv_Kb_eur_plus_str = f"K_b+ = max({ircv_cvr_plus_eur:,.2f}, 0)"
ircv_Kb_usd_plus_str = f"K_b+ = max({ircv_cvr_plus_usd:,.2f}, 0)"

ircv_upward_print = format_table(
    pd.DataFrame([
        {'bucket': 'eur', 'cvr_k+': ircv_cvr_plus_eur, 'rho': 'n/a', 'ψ': 'n/a', 'K_b+': ircv_K_eur_plus},
        {'bucket': 'usd', 'cvr_k+': ircv_cvr_plus_usd, 'rho': 'n/a', 'ψ': 'n/a', 'K_b+': ircv_K_usd_plus},
    ]),
    columns=['bucket', 'cvr_k+', 'rho', 'ψ', 'K_b+'],
    col_formats={'cvr_k+': ',.0f', 'K_b+': ',.2f'},
    alignments=('left', 'right', 'right', 'center', 'right')
)

# --- 14.6 Downward scenario (K_b-) --- #
# Calculation comments
ircv_Kb_eur_minus_str = f"K_b- = max({ircv_cvr_minus_eur:,.2f}, 0)"
ircv_Kb_usd_minus_str = f"K_b- = max({ircv_cvr_minus_usd:,.2f}, 0)"

ircv_downward_print = format_table(
    pd.DataFrame([
        {'bucket': 'eur', 'cvr_k-': ircv_cvr_minus_eur, 'rho': 'n/a', 'ψ': 'n/a', 'K_b-': ircv_K_eur_minus},
        {'bucket': 'usd', 'cvr_k-': ircv_cvr_minus_usd, 'rho': 'n/a', 'ψ': 'n/a', 'K_b-': ircv_K_usd_minus},
    ]),
    columns=['bucket', 'cvr_k-', 'rho', 'ψ', 'K_b-'],
    col_formats={'cvr_k-': ',.0f', 'K_b-': ',.2f'},
    alignments=('left', 'right', 'right', 'center', 'right')
)

# --- 14.7 Intra-bucket aggregation (K_b) --- #
ircv_intra_agg_print = format_table(
    pd.DataFrame([
        {'bucket': 'eur', 'K_b+': ircv_K_eur_plus, 'K_b-': ircv_K_eur_minus, 'K_b': ircv_K_eur, 'selected': ircv_scenario_eur},
        {'bucket': 'usd', 'K_b+': ircv_K_usd_plus, 'K_b-': ircv_K_usd_minus, 'K_b': ircv_K_usd, 'selected': ircv_scenario_usd},
    ]),
    columns=['bucket', 'K_b+', 'K_b-', 'K_b', 'selected'],
    col_formats={'K_b+': ',.2f', 'K_b-': ',.2f', 'K_b': ',.2f'},
    alignments=('left', 'right', 'right', 'right', 'left')
)

# --- 14.8 Bucket sums (S_b) --- #
ircv_bucket_sums_print = format_table(
    pd.DataFrame([
        {'bucket': 'eur', 'selected': ircv_scenario_eur, 'cvr_k': ircv_cvr_minus_eur if ircv_scenario_eur == "downward" else ircv_cvr_plus_eur, 'S_b': ircv_S_eur},
        {'bucket': 'usd', 'selected': ircv_scenario_usd, 'cvr_k': ircv_cvr_minus_usd if ircv_scenario_usd == "downward" else ircv_cvr_plus_usd, 'S_b': ircv_S_usd},
    ]),
    columns=['bucket', 'selected', 'cvr_k', 'S_b'],
    col_formats={'cvr_k': ',.0f', 'S_b': ',.2f'},
    alignments=('left', 'left', 'right', 'right')
)

# --- 14.9 Cross-bucket correlation --- #
ircv_cross_corr_print = format_table(
    pd.DataFrame([
        {'buckets': 'eur vs usd', 'delta_gamma': IRCV_DELTA_GAMMA, 'gamma': ircv_gamma},
    ]),
    columns=['buckets', 'delta_gamma', 'gamma'],
    col_formats={'delta_gamma': '.4f', 'gamma': '.4f'},
    alignments=('left', 'right', 'right')
)

# --- 14.10 Cross-bucket safeguard function (ψ) --- #
ircv_psi_cross_print = format_table(
    pd.DataFrame([
        {'S_eur': ircv_S_eur, 'S_usd': ircv_S_usd, 'ψ': ircv_psi_cross},
    ]),
    columns=['S_eur', 'S_usd', 'ψ'],
    col_formats={'S_eur': ',.2f', 'S_usd': ',.2f'},
    alignments=('right', 'right', 'center')
)

# --- 14.11 Cross-bucket aggregation --- #
# Calculation comment
ircv_cross_agg_formula_str = (
    f"Capital = sqrt( ({ircv_K_eur:,.2f}^2 + {ircv_K_usd:,.2f}^2) + "
    f"(2 * {ircv_gamma:.2f} * {ircv_S_eur:,.2f} * {ircv_S_usd:,.2f} * {ircv_psi_cross}) )"
)

ircv_cross_agg_print = format_table(
    pd.DataFrame([
        {'component': 'K_eur', 'value': ircv_K_eur},
        {'component': 'K_usd', 'value': ircv_K_usd},
        {'component': 'S_eur', 'value': ircv_S_eur},
        {'component': 'S_usd', 'value': ircv_S_usd},
        {'component': 'gamma', 'value': f"{ircv_gamma:.4f}"},
        {'component': 'ψ(S_eur, S_usd)', 'value': ircv_psi_cross},
        {'component': 'capital', 'value': ircv_K_medium},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# --- 14.12 Correlation scenarios --- #
ircv_corr_scenarios_print = format_table(
    pd.DataFrame([
        {'parameter': 'rho', 'medium': 'n/a', 'high': 'n/a', 'low': 'n/a'},
        {'parameter': 'gamma', 'medium': ircv_gamma, 'high': ircv_gamma_high, 'low': ircv_gamma_low},
    ]),
    columns=['parameter', 'medium', 'high', 'low'],
    col_formats={'medium': '.4f', 'high': '.4f', 'low': '.4f'},
    alignments=('left', 'right', 'right', 'right')
)

# --- 14.13 Capital --- #
ircv_capital_print = format_table(
    ircv_capital_df,
    columns=['scenario', 'capital'],
    col_formats={'capital': ',.2f'},
    alignments=('left', 'right')
)


# ============================================================
# DISPLAY ALL OUTPUTS
# ============================================================

print("=" * 60)
print("IRCV")
print("=" * 60)

print("\n1) GROSS CURVATURE POSITIONS")
print(ircv_gross_cvr_print)

print("\n2) NET CURVATURE POSITIONS")
print(ircv_net_cvr_print)

print("\n3) INTRA-BUCKET CORRELATION (rho_kl)")
print(ircv_intra_corr_print)

print("\n4) INTRA-BUCKET SAFEGUARD FUNCTION (ψ)")
print(ircv_psi_intra_print)

print("\n5) UPWARD SCENARIO (K_b+)")
print("\n--- EUR Bucket ---")
print(ircv_Kb_eur_plus_str)
print("\n--- USD Bucket ---")
print(ircv_Kb_usd_plus_str)
print(ircv_upward_print)

print("\n6) DOWNWARD SCENARIO (K_b-)")
print("\n--- EUR Bucket ---")
print(ircv_Kb_eur_minus_str)
print("\n--- USD Bucket ---")
print(ircv_Kb_usd_minus_str)
print(ircv_downward_print)

print("\n7) INTRA-BUCKET AGGREGATION (K_b)")
print(ircv_intra_agg_print)

print("\n8) BUCKET SUMS (S_b)")
print(ircv_bucket_sums_print)

print("\n9) CROSS-BUCKET CORRELATION (gamma_bc)")
print(ircv_cross_corr_print)

print("\n10) CROSS-BUCKET SAFEGUARD FUNCTION (ψ)")
print(ircv_psi_cross_print)

print("\n11) CROSS-BUCKET AGGREGATION")
print(ircv_cross_agg_formula_str)
print(ircv_cross_agg_print)

print("\n12) CORRELATION SCENARIOS")
print(ircv_corr_scenarios_print)

print("\n13) CAPITAL")
print(ircv_capital_print)

IRCV

1) GROSS CURVATURE POSITIONS
┌────────────┬──────────┬─────────┬────────────┬─────────────┐
│ trade_id   │ bucket   │ curve   │   cvr_plus │   cvr_minus │
├────────────┼──────────┼─────────┼────────────┼─────────────┤
│ 1          │ eur      │ ois     │      2,400 │       2,600 │
│ 2          │ eur      │ ois     │      1,300 │       1,400 │
│ 3          │ eur      │ implied │      1,100 │       1,600 │
│ 4          │ eur      │ implied │     -1,700 │      -2,200 │
│ 5          │ usd      │ ois     │      3,400 │      -2,700 │
│ 6          │ usd      │ ois     │      1,400 │       1,500 │
│ 7          │ usd      │ implied │     -3,800 │       4,500 │
│ 8          │ usd      │ implied │     -2,600 │      -3,700 │
└────────────┴──────────┴─────────┴────────────┴─────────────┘

2) NET CURVATURE POSITIONS
┌──────────┬────────────┬─────────────┐
│ bucket   │   cvr_plus │   cvr_minus │
├──────────┼────────────┼─────────────┤
│ eur      │      3,100 │       3,400 │
│ usd      │     -1,6

### CRCV Non Sec

In [19]:
# ============================================================
# CRCV NON-SEC
# ============================================================


# TABLE OF CONTENTS
#   1) Gross curvature positions
#   2) Net curvature positions (CVR+/CVR-)
#   3) Intra-bucket correlation (rho_kl)
#   4) Intra-bucket safeguard function (ψ)
#   5) Upward scenario (K_b+)
#   6) Downward scenario (K_b-)
#   7) Intra-bucket aggregation (K_b)
#   8) Bucket sums (S_b)
#   9) Cross-bucket correlation (gamma_bc)
#  10) Cross-bucket safeguard function (ψ)
#  11) Cross-bucket aggregation
#  12) Correlation scenarios (high/low)
#  13) Capital
#  14) Printing
# ============================================================


# ============================================================
# 1) GROSS CURVATURE POSITIONS
# ============================================================
# Position-level curvature values from upward (+) and downward (-) shocks.

data = [
    {'trade_id': 1, 'bucket': '5', 'issuer': 'issuer_a', 'cvr_plus': -800, 'cvr_minus': 1600},
    {'trade_id': 2, 'bucket': '5', 'issuer': 'issuer_b', 'cvr_plus': -4500, 'cvr_minus': 4000},
    {'trade_id': 3, 'bucket': '6', 'issuer': 'issuer_f', 'cvr_plus': -380, 'cvr_minus': 1090},
    {'trade_id': 4, 'bucket': '6', 'issuer': 'issuer_i', 'cvr_plus': -2240, 'cvr_minus': 2850},
]

crcv_sec_ctp_gross_cvr = pd.DataFrame(data)


# ============================================================
# 2) NET CURVATURE POSITIONS (CVR+/CVR-)
# ============================================================
# Net CVR = Σ gross CVR for each unique risk factor.
# Risk factor defined by: bucket + issuer.

crcv_sec_ctp_risk_factor_cols = ['bucket', 'issuer']

crcv_sec_ctp_net_cvr = (
    crcv_sec_ctp_gross_cvr
    .groupby(crcv_sec_ctp_risk_factor_cols)
    .agg(cvr_plus=('cvr_plus', 'sum'), cvr_minus=('cvr_minus', 'sum'))
    .reset_index()
)

# Extract CVR vectors per bucket
# Bucket 5
crcv_sec_ctp_cvr_b5 = crcv_sec_ctp_net_cvr[crcv_sec_ctp_net_cvr['bucket'] == '5']
crcv_sec_ctp_cvr_plus_b5 = crcv_sec_ctp_cvr_b5['cvr_plus'].values
crcv_sec_ctp_cvr_minus_b5 = crcv_sec_ctp_cvr_b5['cvr_minus'].values

# Bucket 6
crcv_sec_ctp_cvr_b6 = crcv_sec_ctp_net_cvr[crcv_sec_ctp_net_cvr['bucket'] == '6']
crcv_sec_ctp_cvr_plus_b6 = crcv_sec_ctp_cvr_b6['cvr_plus'].values
crcv_sec_ctp_cvr_minus_b6 = crcv_sec_ctp_cvr_b6['cvr_minus'].values


# ============================================================
# 3) INTRA-BUCKET CORRELATION (rho_kl)
# ============================================================
# For curvature: ρ_kl = (delta_ρ_kl)²

CRCV_SEC_CTP_DELTA_RHO = 0.35
crcv_sec_ctp_rho = CRCV_SEC_CTP_DELTA_RHO ** 2

# Both buckets have multiple risk factors -> correlation applies
crcv_sec_ctp_rho_b5 = crcv_sec_ctp_rho
crcv_sec_ctp_rho_b6 = crcv_sec_ctp_rho


# ============================================================
# 4) INTRA-BUCKET SAFEGUARD FUNCTION (ψ)
# ============================================================
# ψ(x, y) = 0 if both x < 0 and y < 0, else 1

def crcv_sec_ctp_psi(x, y):
    """Safeguard function: returns 0 if both x and y are negative, else 1"""
    return 0 if x < 0 and y < 0 else 1

# Bucket 5 (two risk factors)
crcv_sec_ctp_psi_b5_plus = crcv_sec_ctp_psi(crcv_sec_ctp_cvr_plus_b5[0], crcv_sec_ctp_cvr_plus_b5[1])
crcv_sec_ctp_psi_b5_minus = crcv_sec_ctp_psi(crcv_sec_ctp_cvr_minus_b5[0], crcv_sec_ctp_cvr_minus_b5[1])

# Bucket 6 (two risk factors)
crcv_sec_ctp_psi_b6_plus = crcv_sec_ctp_psi(crcv_sec_ctp_cvr_plus_b6[0], crcv_sec_ctp_cvr_plus_b6[1])
crcv_sec_ctp_psi_b6_minus = crcv_sec_ctp_psi(crcv_sec_ctp_cvr_minus_b6[0], crcv_sec_ctp_cvr_minus_b6[1])


# ============================================================
# 5) UPWARD SCENARIO (K_b+)
# ============================================================
# K_b+ = √( Σ max(CVR_k+, 0)² + Σ Σ ρ_kl × CVR_k+ × CVR_l+ × ψ(CVR_k+, CVR_l+) )

# --- Bucket 5 --- #
crcv_sec_ctp_sum_sq_plus_b5 = np.sum(np.maximum(crcv_sec_ctp_cvr_plus_b5, 0)**2)
crcv_sec_ctp_cross_plus_b5 = (
    2 * crcv_sec_ctp_rho_b5
    * crcv_sec_ctp_cvr_plus_b5[0]
    * crcv_sec_ctp_cvr_plus_b5[1]
    * crcv_sec_ctp_psi_b5_plus
)
crcv_sec_ctp_K_b5_plus = np.sqrt(max(0, crcv_sec_ctp_sum_sq_plus_b5 + crcv_sec_ctp_cross_plus_b5))


# --- Bucket 6 --- #
crcv_sec_ctp_sum_sq_plus_b6 = np.sum(np.maximum(crcv_sec_ctp_cvr_plus_b6, 0)**2)
crcv_sec_ctp_cross_plus_b6 = (
    2 * crcv_sec_ctp_rho_b6
    * crcv_sec_ctp_cvr_plus_b6[0]
    * crcv_sec_ctp_cvr_plus_b6[1]
    * crcv_sec_ctp_psi_b6_plus
)
crcv_sec_ctp_K_b6_plus = np.sqrt(max(0, crcv_sec_ctp_sum_sq_plus_b6 + crcv_sec_ctp_cross_plus_b6))


# ============================================================
# 6) DOWNWARD SCENARIO (K_b-)
# ============================================================
# K_b- = √( Σ max(CVR_k-, 0)² + Σ Σ ρ_kl × CVR_k- × CVR_l- × ψ(CVR_k-, CVR_l-) )

# --- Bucket 5 --- #
crcv_sec_ctp_sum_sq_minus_b5 = np.sum(np.maximum(crcv_sec_ctp_cvr_minus_b5, 0)**2)
crcv_sec_ctp_cross_minus_b5 = (
    2 * crcv_sec_ctp_rho_b5
    * crcv_sec_ctp_cvr_minus_b5[0]
    * crcv_sec_ctp_cvr_minus_b5[1]
    * crcv_sec_ctp_psi_b5_minus
)
crcv_sec_ctp_K_b5_minus = np.sqrt(max(0, crcv_sec_ctp_sum_sq_minus_b5 + crcv_sec_ctp_cross_minus_b5))


# --- Bucket 6 --- #
crcv_sec_ctp_sum_sq_minus_b6 = np.sum(np.maximum(crcv_sec_ctp_cvr_minus_b6, 0)**2)
crcv_sec_ctp_cross_minus_b6 = (
    2 * crcv_sec_ctp_rho_b6
    * crcv_sec_ctp_cvr_minus_b6[0]
    * crcv_sec_ctp_cvr_minus_b6[1]
    * crcv_sec_ctp_psi_b6_minus
)
crcv_sec_ctp_K_b6_minus = np.sqrt(max(0, crcv_sec_ctp_sum_sq_minus_b6 + crcv_sec_ctp_cross_minus_b6))


# ============================================================
# 7) INTRA-BUCKET AGGREGATION (K_b)
# ============================================================
# K_b = max(K_b+, K_b-)

# --- Bucket 5 --- #
if crcv_sec_ctp_K_b5_plus > crcv_sec_ctp_K_b5_minus:
    crcv_sec_ctp_K_b5 = crcv_sec_ctp_K_b5_plus
    crcv_sec_ctp_scenario_b5 = "upward"
elif crcv_sec_ctp_K_b5_minus > crcv_sec_ctp_K_b5_plus:
    crcv_sec_ctp_K_b5 = crcv_sec_ctp_K_b5_minus
    crcv_sec_ctp_scenario_b5 = "downward"
else:
    crcv_sec_ctp_K_b5 = crcv_sec_ctp_K_b5_plus
    crcv_sec_ctp_scenario_b5 = "upward" if np.sum(crcv_sec_ctp_cvr_plus_b5) >= np.sum(crcv_sec_ctp_cvr_minus_b5) else "downward"

# --- Bucket 6 --- #
if crcv_sec_ctp_K_b6_plus > crcv_sec_ctp_K_b6_minus:
    crcv_sec_ctp_K_b6 = crcv_sec_ctp_K_b6_plus
    crcv_sec_ctp_scenario_b6 = "upward"
elif crcv_sec_ctp_K_b6_minus > crcv_sec_ctp_K_b6_plus:
    crcv_sec_ctp_K_b6 = crcv_sec_ctp_K_b6_minus
    crcv_sec_ctp_scenario_b6 = "downward"
else:
    crcv_sec_ctp_K_b6 = crcv_sec_ctp_K_b6_plus
    crcv_sec_ctp_scenario_b6 = "upward" if np.sum(crcv_sec_ctp_cvr_plus_b6) >= np.sum(crcv_sec_ctp_cvr_minus_b6) else "downward"


# ============================================================
# 8) BUCKET SUMS (S_b)
# ============================================================
# S_b = Σ CVR+ if upward scenario, Σ CVR- if downward scenario

crcv_sec_ctp_S_b5 = (
    crcv_sec_ctp_cvr_plus_b5.sum() if crcv_sec_ctp_scenario_b5 == "upward"
    else crcv_sec_ctp_cvr_minus_b5.sum()
)

crcv_sec_ctp_S_b6 = (
    crcv_sec_ctp_cvr_plus_b6.sum() if crcv_sec_ctp_scenario_b6 == "upward"
    else crcv_sec_ctp_cvr_minus_b6.sum()
)


# ============================================================
# 9) CROSS-BUCKET CORRELATION (gamma_bc)
# ============================================================
# γ_bc = (delta_γ_bc)²

CRCV_SEC_CTP_DELTA_GAMMA = 0.25
crcv_sec_ctp_gamma = CRCV_SEC_CTP_DELTA_GAMMA ** 2


# ============================================================
# 10) CROSS-BUCKET SAFEGUARD FUNCTION (ψ)
# ============================================================
# ψ(S_b, S_c) = 0 if both S_b < 0 and S_c < 0, else 1

crcv_sec_ctp_psi_cross = crcv_sec_ctp_psi(crcv_sec_ctp_S_b5, crcv_sec_ctp_S_b6)


# ============================================================
# 11) CROSS-BUCKET AGGREGATION
# ============================================================
# K = √( Σ K_b² + Σ Σ γ_bc × S_b × S_c × ψ(S_b, S_c) )

crcv_sec_ctp_sum_K_sq = crcv_sec_ctp_K_b5**2 + crcv_sec_ctp_K_b6**2
crcv_sec_ctp_cross_bucket = (
    2 * crcv_sec_ctp_gamma
    * crcv_sec_ctp_S_b5
    * crcv_sec_ctp_S_b6
    * crcv_sec_ctp_psi_cross
)
crcv_sec_ctp_K_medium = np.sqrt(max(0, crcv_sec_ctp_sum_K_sq + crcv_sec_ctp_cross_bucket))


# ============================================================
# 12) CORRELATION SCENARIOS (HIGH/LOW)
# ============================================================
# High: ρ_high = min(1.25 × ρ_medium, 1.0)
# Low:  ρ_low  = max(2 × ρ_medium - 1, 0.75 × ρ_medium)

# --- High scenario correlations --- #
crcv_sec_ctp_rho_high = min(crcv_sec_ctp_rho * 1.25, 1.0)
crcv_sec_ctp_gamma_high = min(crcv_sec_ctp_gamma * 1.25, 1.0)

# --- Low scenario correlations --- #
crcv_sec_ctp_rho_low = max(2 * crcv_sec_ctp_rho - 1.0, 0.75 * crcv_sec_ctp_rho)
crcv_sec_ctp_gamma_low = max(2 * crcv_sec_ctp_gamma - 1.0, 0.75 * crcv_sec_ctp_gamma)

# --- High scenario intra-bucket K_b --- #

# Bucket 5 High
crcv_sec_ctp_cross_plus_b5_high = 2 * crcv_sec_ctp_rho_high * crcv_sec_ctp_cvr_plus_b5[0] * crcv_sec_ctp_cvr_plus_b5[1] * crcv_sec_ctp_psi_b5_plus
crcv_sec_ctp_K_b5_plus_high = np.sqrt(max(0, crcv_sec_ctp_sum_sq_plus_b5 + crcv_sec_ctp_cross_plus_b5_high))

crcv_sec_ctp_cross_minus_b5_high = 2 * crcv_sec_ctp_rho_high * crcv_sec_ctp_cvr_minus_b5[0] * crcv_sec_ctp_cvr_minus_b5[1] * crcv_sec_ctp_psi_b5_minus
crcv_sec_ctp_K_b5_minus_high = np.sqrt(max(0, crcv_sec_ctp_sum_sq_minus_b5 + crcv_sec_ctp_cross_minus_b5_high))

if crcv_sec_ctp_K_b5_plus_high > crcv_sec_ctp_K_b5_minus_high:
    crcv_sec_ctp_K_b5_high = crcv_sec_ctp_K_b5_plus_high
    crcv_sec_ctp_scenario_b5_high = "upward"
elif crcv_sec_ctp_K_b5_minus_high > crcv_sec_ctp_K_b5_plus_high:
    crcv_sec_ctp_K_b5_high = crcv_sec_ctp_K_b5_minus_high
    crcv_sec_ctp_scenario_b5_high = "downward"
else:
    crcv_sec_ctp_K_b5_high = crcv_sec_ctp_K_b5_plus_high
    crcv_sec_ctp_scenario_b5_high = "upward" if np.sum(crcv_sec_ctp_cvr_plus_b5) >= np.sum(crcv_sec_ctp_cvr_minus_b5) else "downward"

# Bucket 6 High
crcv_sec_ctp_cross_plus_b6_high = 2 * crcv_sec_ctp_rho_high * crcv_sec_ctp_cvr_plus_b6[0] * crcv_sec_ctp_cvr_plus_b6[1] * crcv_sec_ctp_psi_b6_plus
crcv_sec_ctp_K_b6_plus_high = np.sqrt(max(0, crcv_sec_ctp_sum_sq_plus_b6 + crcv_sec_ctp_cross_plus_b6_high))

crcv_sec_ctp_cross_minus_b6_high = 2 * crcv_sec_ctp_rho_high * crcv_sec_ctp_cvr_minus_b6[0] * crcv_sec_ctp_cvr_minus_b6[1] * crcv_sec_ctp_psi_b6_minus
crcv_sec_ctp_K_b6_minus_high = np.sqrt(max(0, crcv_sec_ctp_sum_sq_minus_b6 + crcv_sec_ctp_cross_minus_b6_high))

if crcv_sec_ctp_K_b6_plus_high > crcv_sec_ctp_K_b6_minus_high:
    crcv_sec_ctp_K_b6_high = crcv_sec_ctp_K_b6_plus_high
    crcv_sec_ctp_scenario_b6_high = "upward"
elif crcv_sec_ctp_K_b6_minus_high > crcv_sec_ctp_K_b6_plus_high:
    crcv_sec_ctp_K_b6_high = crcv_sec_ctp_K_b6_minus_high
    crcv_sec_ctp_scenario_b6_high = "downward"
else:
    crcv_sec_ctp_K_b6_high = crcv_sec_ctp_K_b6_plus_high
    crcv_sec_ctp_scenario_b6_high = "upward" if np.sum(crcv_sec_ctp_cvr_plus_b6) >= np.sum(crcv_sec_ctp_cvr_minus_b6) else "downward"

# --- Low scenario intra-bucket K_b --- #

# Bucket 5 Low
crcv_sec_ctp_cross_plus_b5_low = 2 * crcv_sec_ctp_rho_low * crcv_sec_ctp_cvr_plus_b5[0] * crcv_sec_ctp_cvr_plus_b5[1] * crcv_sec_ctp_psi_b5_plus
crcv_sec_ctp_K_b5_plus_low = np.sqrt(max(0, crcv_sec_ctp_sum_sq_plus_b5 + crcv_sec_ctp_cross_plus_b5_low))

crcv_sec_ctp_cross_minus_b5_low = 2 * crcv_sec_ctp_rho_low * crcv_sec_ctp_cvr_minus_b5[0] * crcv_sec_ctp_cvr_minus_b5[1] * crcv_sec_ctp_psi_b5_minus
crcv_sec_ctp_K_b5_minus_low = np.sqrt(max(0, crcv_sec_ctp_sum_sq_minus_b5 + crcv_sec_ctp_cross_minus_b5_low))

if crcv_sec_ctp_K_b5_plus_low > crcv_sec_ctp_K_b5_minus_low:
    crcv_sec_ctp_K_b5_low = crcv_sec_ctp_K_b5_plus_low
    crcv_sec_ctp_scenario_b5_low = "upward"
elif crcv_sec_ctp_K_b5_minus_low > crcv_sec_ctp_K_b5_plus_low:
    crcv_sec_ctp_K_b5_low = crcv_sec_ctp_K_b5_minus_low
    crcv_sec_ctp_scenario_b5_low = "downward"
else:
    crcv_sec_ctp_K_b5_low = crcv_sec_ctp_K_b5_plus_low
    crcv_sec_ctp_scenario_b5_low = "upward" if np.sum(crcv_sec_ctp_cvr_plus_b5) >= np.sum(crcv_sec_ctp_cvr_minus_b5) else "downward"

# Bucket 6 Low
crcv_sec_ctp_cross_plus_b6_low = 2 * crcv_sec_ctp_rho_low * crcv_sec_ctp_cvr_plus_b6[0] * crcv_sec_ctp_cvr_plus_b6[1] * crcv_sec_ctp_psi_b6_plus
crcv_sec_ctp_K_b6_plus_low = np.sqrt(max(0, crcv_sec_ctp_sum_sq_plus_b6 + crcv_sec_ctp_cross_plus_b6_low))

crcv_sec_ctp_cross_minus_b6_low = 2 * crcv_sec_ctp_rho_low * crcv_sec_ctp_cvr_minus_b6[0] * crcv_sec_ctp_cvr_minus_b6[1] * crcv_sec_ctp_psi_b6_minus
crcv_sec_ctp_K_b6_minus_low = np.sqrt(max(0, crcv_sec_ctp_sum_sq_minus_b6 + crcv_sec_ctp_cross_minus_b6_low))

if crcv_sec_ctp_K_b6_plus_low > crcv_sec_ctp_K_b6_minus_low:
    crcv_sec_ctp_K_b6_low = crcv_sec_ctp_K_b6_plus_low
    crcv_sec_ctp_scenario_b6_low = "upward"
elif crcv_sec_ctp_K_b6_minus_low > crcv_sec_ctp_K_b6_plus_low:
    crcv_sec_ctp_K_b6_low = crcv_sec_ctp_K_b6_minus_low
    crcv_sec_ctp_scenario_b6_low = "downward"
else:
    crcv_sec_ctp_K_b6_low = crcv_sec_ctp_K_b6_plus_low
    crcv_sec_ctp_scenario_b6_low = "upward" if np.sum(crcv_sec_ctp_cvr_plus_b6) >= np.sum(crcv_sec_ctp_cvr_minus_b6) else "downward"

# --- High scenario bucket sums and cross-bucket --- #
crcv_sec_ctp_S_b5_high = crcv_sec_ctp_cvr_plus_b5.sum() if crcv_sec_ctp_scenario_b5_high == "upward" else crcv_sec_ctp_cvr_minus_b5.sum()
crcv_sec_ctp_S_b6_high = crcv_sec_ctp_cvr_plus_b6.sum() if crcv_sec_ctp_scenario_b6_high == "upward" else crcv_sec_ctp_cvr_minus_b6.sum()
crcv_sec_ctp_psi_cross_high = crcv_sec_ctp_psi(crcv_sec_ctp_S_b5_high, crcv_sec_ctp_S_b6_high)

crcv_sec_ctp_sum_K_sq_high = crcv_sec_ctp_K_b5_high**2 + crcv_sec_ctp_K_b6_high**2
crcv_sec_ctp_cross_bucket_high = 2 * crcv_sec_ctp_gamma_high * crcv_sec_ctp_S_b5_high * crcv_sec_ctp_S_b6_high * crcv_sec_ctp_psi_cross_high
crcv_sec_ctp_K_high = np.sqrt(max(0, crcv_sec_ctp_sum_K_sq_high + crcv_sec_ctp_cross_bucket_high))

# --- Low scenario bucket sums and cross-bucket --- #
crcv_sec_ctp_S_b5_low = crcv_sec_ctp_cvr_plus_b5.sum() if crcv_sec_ctp_scenario_b5_low == "upward" else crcv_sec_ctp_cvr_minus_b5.sum()
crcv_sec_ctp_S_b6_low = crcv_sec_ctp_cvr_plus_b6.sum() if crcv_sec_ctp_scenario_b6_low == "upward" else crcv_sec_ctp_cvr_minus_b6.sum()
crcv_sec_ctp_psi_cross_low = crcv_sec_ctp_psi(crcv_sec_ctp_S_b5_low, crcv_sec_ctp_S_b6_low)

crcv_sec_ctp_sum_K_sq_low = crcv_sec_ctp_K_b5_low**2 + crcv_sec_ctp_K_b6_low**2
crcv_sec_ctp_cross_bucket_low = 2 * crcv_sec_ctp_gamma_low * crcv_sec_ctp_S_b5_low * crcv_sec_ctp_S_b6_low * crcv_sec_ctp_psi_cross_low
crcv_sec_ctp_K_low = np.sqrt(max(0, crcv_sec_ctp_sum_K_sq_low + crcv_sec_ctp_cross_bucket_low))


# ============================================================
# 13) CAPITAL
# ============================================================

crcv_sec_ctp_capital_df = pd.DataFrame([
    {'scenario': 'medium', 'capital': crcv_sec_ctp_K_medium},
    {'scenario': 'high', 'capital': crcv_sec_ctp_K_high},
    {'scenario': 'low', 'capital': crcv_sec_ctp_K_low},
])


# ============================================================
# 14) PRINTING
# ============================================================

# --- 14.1 Gross curvature positions --- #
crcv_sec_ctp_gross_cvr_print = format_table(
    crcv_sec_ctp_gross_cvr,
    columns=['trade_id', 'bucket', 'issuer', 'cvr_plus', 'cvr_minus'],
    col_formats={'cvr_plus': ',.0f', 'cvr_minus': ',.0f'},
    alignments=('left', 'left', 'left', 'right', 'right')
)

# --- 14.2 Net curvature positions --- #
crcv_sec_ctp_net_cvr_print = format_table(
    crcv_sec_ctp_net_cvr,
    columns=['bucket', 'issuer', 'cvr_plus', 'cvr_minus'],
    col_formats={'cvr_plus': ',.0f', 'cvr_minus': ',.0f'},
    alignments=('left', 'left', 'right', 'right')
)

# --- 14.3 Intra-bucket correlation --- #
crcv_sec_ctp_intra_corr_print = format_table(
    pd.DataFrame([
        {'bucket': '5', 'delta_rho': CRCV_SEC_CTP_DELTA_RHO, 'rho': crcv_sec_ctp_rho_b5},
        {'bucket': '6', 'delta_rho': CRCV_SEC_CTP_DELTA_RHO, 'rho': crcv_sec_ctp_rho_b6},
    ]),
    columns=['bucket', 'delta_rho', 'rho'],
    col_formats={'delta_rho': '.4f', 'rho': '.4f'},
    alignments=('left', 'right', 'right')
)

# --- 14.4 Intra-bucket safeguard function (ψ) --- #
crcv_sec_ctp_psi_intra_print = format_table(
    pd.DataFrame([
        {'bucket': '5', 'cvr_k+': crcv_sec_ctp_cvr_plus_b5[0], 'cvr_l+': crcv_sec_ctp_cvr_plus_b5[1], 'ψ_up': crcv_sec_ctp_psi_b5_plus, 'cvr_k-': crcv_sec_ctp_cvr_minus_b5[0], 'cvr_l-': crcv_sec_ctp_cvr_minus_b5[1], 'ψ_down': crcv_sec_ctp_psi_b5_minus},
        {'bucket': '6', 'cvr_k+': crcv_sec_ctp_cvr_plus_b6[0], 'cvr_l+': crcv_sec_ctp_cvr_plus_b6[1], 'ψ_up': crcv_sec_ctp_psi_b6_plus, 'cvr_k-': crcv_sec_ctp_cvr_minus_b6[0], 'cvr_l-': crcv_sec_ctp_cvr_minus_b6[1], 'ψ_down': crcv_sec_ctp_psi_b6_minus},
    ]),
    columns=['bucket', 'cvr_k+', 'cvr_l+', 'ψ_up', 'cvr_k-', 'cvr_l-', 'ψ_down'],
    col_formats={'cvr_k+': ',.0f', 'cvr_l+': ',.0f', 'cvr_k-': ',.0f', 'cvr_l-': ',.0f'},
    alignments=('left', 'right', 'right', 'center', 'right', 'right', 'center')
)

# --- 14.5 Upward scenario (K_b+) --- #
crcv_sec_ctp_upward_print = format_table(
    pd.DataFrame([
        {'bucket': '5', 'cvr_k+': crcv_sec_ctp_cvr_plus_b5[0], 'cvr_l+': crcv_sec_ctp_cvr_plus_b5[1], 'rho': crcv_sec_ctp_rho_b5, 'ψ': crcv_sec_ctp_psi_b5_plus, 'K_b+': crcv_sec_ctp_K_b5_plus},
        {'bucket': '6', 'cvr_k+': crcv_sec_ctp_cvr_plus_b6[0], 'cvr_l+': crcv_sec_ctp_cvr_plus_b6[1], 'rho': crcv_sec_ctp_rho_b6, 'ψ': crcv_sec_ctp_psi_b6_plus, 'K_b+': crcv_sec_ctp_K_b6_plus},
    ]),
    columns=['bucket', 'cvr_k+', 'cvr_l+', 'rho', 'ψ', 'K_b+'],
    col_formats={'cvr_k+': ',.0f', 'cvr_l+': ',.0f', 'rho': '.4f', 'K_b+': ',.2f'},
    alignments=('left', 'right', 'right', 'right', 'center', 'right')
)

# Formula String for Printing
crcv_sec_ctp_Kb5_plus_formula_print = (
    f"K_b6+ = sqrt( (max({crcv_sec_ctp_cvr_plus_b5[0]},0)^2 + max({crcv_sec_ctp_cvr_plus_b5[1]},0)^2) + "
    f"(2 * {crcv_sec_ctp_rho_b5:.4f} * {crcv_sec_ctp_cvr_plus_b5[0]} * {crcv_sec_ctp_cvr_plus_b5[1]} * {crcv_sec_ctp_psi_b5_plus}) )"
)

# Formula String for Printing
crcv_sec_ctp_Kb6_plus_formula_print = (
    f"K_b6+ = sqrt( (max({crcv_sec_ctp_cvr_plus_b6[0]},0)^2 + max({crcv_sec_ctp_cvr_plus_b6[1]},0)^2) + "
    f"(2 * {crcv_sec_ctp_rho_b6:.4f} * {crcv_sec_ctp_cvr_plus_b6[0]} * {crcv_sec_ctp_cvr_plus_b6[1]} * {crcv_sec_ctp_psi_b6_plus}) )"
)


# --- 14.6 Downward scenario (K_b-) --- #
crcv_sec_ctp_downward_print = format_table(
    pd.DataFrame([
        {'bucket': '5', 'cvr_k-': crcv_sec_ctp_cvr_minus_b5[0], 'cvr_l-': crcv_sec_ctp_cvr_minus_b5[1], 'rho': crcv_sec_ctp_rho_b5, 'ψ': crcv_sec_ctp_psi_b5_minus, 'K_b-': crcv_sec_ctp_K_b5_minus},
        {'bucket': '6', 'cvr_k-': crcv_sec_ctp_cvr_minus_b6[0], 'cvr_l-': crcv_sec_ctp_cvr_minus_b6[1], 'rho': crcv_sec_ctp_rho_b6, 'ψ': crcv_sec_ctp_psi_b6_minus, 'K_b-': crcv_sec_ctp_K_b6_minus},
    ]),
    columns=['bucket', 'cvr_k-', 'cvr_l-', 'rho', 'ψ', 'K_b-'],
    col_formats={'cvr_k-': ',.0f', 'cvr_l-': ',.0f', 'rho': '.4f', 'K_b-': ',.2f'},
    alignments=('left', 'right', 'right', 'right', 'center', 'right')
)


# Formula String for Printing
crcv_sec_ctp_Kb5_minus_formula_print = (
    f"K_b5- = sqrt( (max({crcv_sec_ctp_cvr_minus_b5[0]},0)^2 + max({crcv_sec_ctp_cvr_minus_b5[1]},0)^2) + "
    f"(2 * {crcv_sec_ctp_rho_b5:.4f} * {crcv_sec_ctp_cvr_minus_b5[0]} * {crcv_sec_ctp_cvr_minus_b5[1]} * {crcv_sec_ctp_psi_b5_minus}) )"
)


# Formula String for Printing
crcv_sec_ctp_Kb6_minus_formula_print = (
    f"K_b6- = sqrt( (max({crcv_sec_ctp_cvr_minus_b6[0]},0)^2 + max({crcv_sec_ctp_cvr_minus_b6[1]},0)^2) + "
    f"(2 * {crcv_sec_ctp_rho_b6:.4f} * {crcv_sec_ctp_cvr_minus_b6[0]} * {crcv_sec_ctp_cvr_minus_b6[1]} * {crcv_sec_ctp_psi_b6_minus}) )"
)


# --- 14.7 Intra-bucket aggregation (K_b) --- #
crcv_sec_ctp_intra_agg_print = format_table(
    pd.DataFrame([
        {'bucket': '5', 'K_b+': crcv_sec_ctp_K_b5_plus, 'K_b-': crcv_sec_ctp_K_b5_minus, 'K_b': crcv_sec_ctp_K_b5, 'selected': crcv_sec_ctp_scenario_b5},
        {'bucket': '6', 'K_b+': crcv_sec_ctp_K_b6_plus, 'K_b-': crcv_sec_ctp_K_b6_minus, 'K_b': crcv_sec_ctp_K_b6, 'selected': crcv_sec_ctp_scenario_b6},
    ]),
    columns=['bucket', 'K_b+', 'K_b-', 'K_b', 'selected'],
    col_formats={'K_b+': ',.2f', 'K_b-': ',.2f', 'K_b': ',.2f'},
    alignments=('left', 'right', 'right', 'right', 'left')
)

# --- 14.8 Bucket sums (S_b) --- #
crcv_sec_ctp_bucket_sums_print = format_table(
    pd.DataFrame([
        {'bucket': '5', 'selected': crcv_sec_ctp_scenario_b5, 'cvr_k': crcv_sec_ctp_cvr_minus_b5[0] if crcv_sec_ctp_scenario_b5 == "downward" else crcv_sec_ctp_cvr_plus_b5[0], 'cvr_l': crcv_sec_ctp_cvr_minus_b5[1] if crcv_sec_ctp_scenario_b5 == "downward" else crcv_sec_ctp_cvr_plus_b5[1], 'S_b': crcv_sec_ctp_S_b5},
        {'bucket': '6', 'selected': crcv_sec_ctp_scenario_b6, 'cvr_k': crcv_sec_ctp_cvr_minus_b6[0] if crcv_sec_ctp_scenario_b6 == "downward" else crcv_sec_ctp_cvr_plus_b6[0], 'cvr_l': crcv_sec_ctp_cvr_minus_b6[1] if crcv_sec_ctp_scenario_b6 == "downward" else crcv_sec_ctp_cvr_plus_b6[1], 'S_b': crcv_sec_ctp_S_b6},
    ]),
    columns=['bucket', 'selected', 'cvr_k', 'cvr_l', 'S_b'],
    col_formats={'cvr_k': ',.0f', 'cvr_l': ',.0f', 'S_b': ',.2f'},
    alignments=('left', 'left', 'right', 'right', 'right')
)

# --- 14.9 Cross-bucket correlation --- #
crcv_sec_ctp_cross_corr_print = format_table(
    pd.DataFrame([
        {'buckets': '5 vs 6', 'delta_gamma': CRCV_SEC_CTP_DELTA_GAMMA, 'gamma': crcv_sec_ctp_gamma},
    ]),
    columns=['buckets', 'delta_gamma', 'gamma'],
    col_formats={'delta_gamma': '.4f', 'gamma': '.4f'},
    alignments=('left', 'right', 'right')
)


# --- 14.10 Cross-bucket safeguard function (ψ) --- #
crcv_sec_ctp_psi_cross_print = format_table(
    pd.DataFrame([
        {'S_b5': crcv_sec_ctp_S_b5, 'S_b6': crcv_sec_ctp_S_b6, 'ψ': crcv_sec_ctp_psi_cross},
    ]),
    columns=['S_b5', 'S_b6', 'ψ'],
    col_formats={'S_b5': ',.2f', 'S_b6': ',.2f'},
    alignments=('right', 'right', 'center')
)

# --- 14.11 Cross-bucket aggregation --- #
crcv_sec_ctp_cross_agg_print = format_table(
    pd.DataFrame([
        {'component': 'K_b5', 'value': crcv_sec_ctp_K_b5},
        {'component': 'K_b6', 'value': crcv_sec_ctp_K_b6},
        {'component': 'S_b5', 'value': crcv_sec_ctp_S_b5},
        {'component': 'S_b6', 'value': crcv_sec_ctp_S_b6},
        {'component': 'gamma', 'value': f"{crcv_sec_ctp_gamma:.4f}"},
        {'component': 'ψ(S_b5, S_b6)', 'value': crcv_sec_ctp_psi_cross},
        {'component': 'capital', 'value': crcv_sec_ctp_K_medium},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# Formula String for Printing
crcv_sec_ctp_cross_agg_formula_print = (
    f"Capital = sqrt( ({crcv_sec_ctp_K_b5:,.2f}^2 + {crcv_sec_ctp_K_b6:,.2f}^2) + "
    f"(2 * {crcv_sec_ctp_gamma:.4f} * {crcv_sec_ctp_S_b5:,.2f} * {crcv_sec_ctp_S_b6:,.2f} * {crcv_sec_ctp_psi_cross}) )"
)


# --- 14.12 Correlation scenarios --- #
crcv_sec_ctp_corr_scenarios_print = format_table(
    pd.DataFrame([
        {'parameter': 'rho', 'medium': crcv_sec_ctp_rho, 'high': crcv_sec_ctp_rho_high, 'low': crcv_sec_ctp_rho_low},
        {'parameter': 'gamma', 'medium': crcv_sec_ctp_gamma, 'high': crcv_sec_ctp_gamma_high, 'low': crcv_sec_ctp_gamma_low},
    ]),
    columns=['parameter', 'medium', 'high', 'low'],
    col_formats={'medium': '.4f', 'high': '.4f', 'low': '.4f'},
    alignments=('left', 'right', 'right', 'right')
)

# --- 14.13 Capital --- #
crcv_sec_ctp_capital_print = format_table(
    crcv_sec_ctp_capital_df,
    columns=['scenario', 'capital'],
    col_formats={'capital': ',.2f'},
    alignments=('left', 'right')
)


# ============================================================
# DISPLAY ALL OUTPUTS
# ============================================================

print("=" * 60)
print("FRTB — CSR Sec CTP Curvature Capital Calculation (Buckets 5 & 6)")
print("=" * 60)

print("\n1) GROSS CURVATURE POSITIONS")
print(crcv_sec_ctp_gross_cvr_print)

print("\n2) NET CURVATURE POSITIONS")
print(crcv_sec_ctp_net_cvr_print)

print("\n3) INTRA-BUCKET CORRELATION (rho_kl)")
print(crcv_sec_ctp_intra_corr_print)

print("\n4) INTRA-BUCKET SAFEGUARD FUNCTION (ψ)")
print(crcv_sec_ctp_psi_intra_print)

print("\n5) UPWARD SCENARIO (K_b+)")
print("\n--- Bucket 5 ---")
print(crcv_sec_ctp_Kb5_plus_formula_print)
print("\n--- Bucket 6 ---")
print(crcv_sec_ctp_Kb6_plus_formula_print)
print(crcv_sec_ctp_upward_print)

print("\n6) DOWNWARD SCENARIO (K_b-)")
print("\n--- Bucket 5 ---")
print(crcv_sec_ctp_Kb5_minus_formula_print)
print("\n--- Bucket 6 ---")
print(crcv_sec_ctp_Kb6_minus_formula_print)
print(crcv_sec_ctp_downward_print)

print("\n7) INTRA-BUCKET AGGREGATION (K_b)")
print(crcv_sec_ctp_intra_agg_print)

print("\n8) BUCKET SUMS (S_b)")
print(crcv_sec_ctp_bucket_sums_print)

print("\n9) CROSS-BUCKET CORRELATION (gamma_bc)")
print(crcv_sec_ctp_cross_corr_print)

print("\n10) CROSS-BUCKET SAFEGUARD FUNCTION (ψ)")
print(crcv_sec_ctp_psi_cross_print)

print("\n11) CROSS-BUCKET AGGREGATION")
print(crcv_sec_ctp_cross_agg_formula_print)
print(crcv_sec_ctp_cross_agg_print)

print("\n12) CORRELATION SCENARIOS")
print(crcv_sec_ctp_corr_scenarios_print)

print("\n13) CAPITAL")
print(crcv_sec_ctp_capital_print)

FRTB — CSR Sec CTP Curvature Capital Calculation (Buckets 5 & 6)

1) GROSS CURVATURE POSITIONS
┌────────────┬──────────┬──────────┬────────────┬─────────────┐
│ trade_id   │ bucket   │ issuer   │   cvr_plus │   cvr_minus │
├────────────┼──────────┼──────────┼────────────┼─────────────┤
│ 1          │ 5        │ issuer_a │       -800 │       1,600 │
│ 2          │ 5        │ issuer_b │     -4,500 │       4,000 │
│ 3          │ 6        │ issuer_f │       -380 │       1,090 │
│ 4          │ 6        │ issuer_i │     -2,240 │       2,850 │
└────────────┴──────────┴──────────┴────────────┴─────────────┘

2) NET CURVATURE POSITIONS
┌──────────┬──────────┬────────────┬─────────────┐
│ bucket   │ issuer   │   cvr_plus │   cvr_minus │
├──────────┼──────────┼────────────┼─────────────┤
│ 5        │ issuer_a │       -800 │       1,600 │
│ 5        │ issuer_b │     -4,500 │       4,000 │
│ 6        │ issuer_f │       -380 │       1,090 │
│ 6        │ issuer_i │     -2,240 │       2,850 │
└───────

### CRCV Sec CTP

In [20]:
# ============================================================
# CRCV SEC CTP
# ============================================================


# TABLE OF CONTENTS
#   1) Gross curvature positions
#   2) Net curvature positions (CVR+/CVR-)
#   3) Intra-bucket correlation (rho_kl)
#   4) Intra-bucket safeguard function (ψ)
#   5) Upward scenario (K_b+)
#   6) Downward scenario (K_b-)
#   7) Intra-bucket aggregation (K_b)
#   8) Bucket sums (S_b)
#   9) Cross-bucket correlation (gamma_bc)
#  10) Cross-bucket safeguard function (ψ)
#  11) Cross-bucket aggregation
#  12) Correlation scenarios (high/low)
#  13) Capital
#  14) Printing
# ============================================================


# ============================================================
# 1) GROSS CURVATURE POSITIONS
# ============================================================
# Position-level curvature values from upward (+) and downward (-) shocks.

data = [
    {'trade_id': 1, 'bucket': '5', 'issuer': 'issuer_a', 'cvr_plus': -800, 'cvr_minus': 1600},
    {'trade_id': 2, 'bucket': '5', 'issuer': 'issuer_b', 'cvr_plus': -4500, 'cvr_minus': 4000},
    {'trade_id': 3, 'bucket': '6', 'issuer': 'issuer_f', 'cvr_plus': -380, 'cvr_minus': 1090},
    {'trade_id': 4, 'bucket': '6', 'issuer': 'issuer_i', 'cvr_plus': -2240, 'cvr_minus': 2850},
]

crcv_sec_ctp_gross_cvr = pd.DataFrame(data)


# ============================================================
# 2) NET CURVATURE POSITIONS (CVR+/CVR-)
# ============================================================
# Net CVR = Σ gross CVR for each unique risk factor.
# Risk factor defined by: bucket + issuer.

crcv_sec_ctp_risk_factor_cols = ['bucket', 'issuer']

crcv_sec_ctp_net_cvr = (
    crcv_sec_ctp_gross_cvr
    .groupby(crcv_sec_ctp_risk_factor_cols)
    .agg(cvr_plus=('cvr_plus', 'sum'), cvr_minus=('cvr_minus', 'sum'))
    .reset_index()
)

# Extract CVR vectors per bucket
# Bucket 5
crcv_sec_ctp_cvr_b5 = crcv_sec_ctp_net_cvr[crcv_sec_ctp_net_cvr['bucket'] == '5']
crcv_sec_ctp_cvr_plus_b5 = crcv_sec_ctp_cvr_b5['cvr_plus'].values
crcv_sec_ctp_cvr_minus_b5 = crcv_sec_ctp_cvr_b5['cvr_minus'].values

# Bucket 6
crcv_sec_ctp_cvr_b6 = crcv_sec_ctp_net_cvr[crcv_sec_ctp_net_cvr['bucket'] == '6']
crcv_sec_ctp_cvr_plus_b6 = crcv_sec_ctp_cvr_b6['cvr_plus'].values
crcv_sec_ctp_cvr_minus_b6 = crcv_sec_ctp_cvr_b6['cvr_minus'].values


# ============================================================
# 3) INTRA-BUCKET CORRELATION (rho_kl)
# ============================================================
# For curvature: ρ_kl = (delta_ρ_kl)²

CRCV_SEC_CTP_DELTA_RHO = 0.35
crcv_sec_ctp_rho = CRCV_SEC_CTP_DELTA_RHO ** 2

# Both buckets have multiple risk factors -> correlation applies
crcv_sec_ctp_rho_b5 = crcv_sec_ctp_rho
crcv_sec_ctp_rho_b6 = crcv_sec_ctp_rho


# ============================================================
# 4) INTRA-BUCKET SAFEGUARD FUNCTION (ψ)
# ============================================================
# ψ(x, y) = 0 if both x < 0 and y < 0, else 1

def crcv_sec_ctp_psi(x, y):
    """Safeguard function: returns 0 if both x and y are negative, else 1"""
    return 0 if x < 0 and y < 0 else 1

# Bucket 5 (two risk factors)
crcv_sec_ctp_psi_b5_plus = crcv_sec_ctp_psi(crcv_sec_ctp_cvr_plus_b5[0], crcv_sec_ctp_cvr_plus_b5[1])
crcv_sec_ctp_psi_b5_minus = crcv_sec_ctp_psi(crcv_sec_ctp_cvr_minus_b5[0], crcv_sec_ctp_cvr_minus_b5[1])

# Bucket 6 (two risk factors)
crcv_sec_ctp_psi_b6_plus = crcv_sec_ctp_psi(crcv_sec_ctp_cvr_plus_b6[0], crcv_sec_ctp_cvr_plus_b6[1])
crcv_sec_ctp_psi_b6_minus = crcv_sec_ctp_psi(crcv_sec_ctp_cvr_minus_b6[0], crcv_sec_ctp_cvr_minus_b6[1])


# ============================================================
# 5) UPWARD SCENARIO (K_b+)
# ============================================================
# K_b+ = √( Σ max(CVR_k+, 0)² + Σ Σ ρ_kl × CVR_k+ × CVR_l+ × ψ(CVR_k+, CVR_l+) )

# --- Bucket 5 --- #
crcv_sec_ctp_sum_sq_plus_b5 = np.sum(np.maximum(crcv_sec_ctp_cvr_plus_b5, 0)**2)
crcv_sec_ctp_cross_plus_b5 = (
    2 * crcv_sec_ctp_rho_b5
    * crcv_sec_ctp_cvr_plus_b5[0]
    * crcv_sec_ctp_cvr_plus_b5[1]
    * crcv_sec_ctp_psi_b5_plus
)
crcv_sec_ctp_K_b5_plus = np.sqrt(max(0, crcv_sec_ctp_sum_sq_plus_b5 + crcv_sec_ctp_cross_plus_b5))


# --- Bucket 6 --- #
crcv_sec_ctp_sum_sq_plus_b6 = np.sum(np.maximum(crcv_sec_ctp_cvr_plus_b6, 0)**2)
crcv_sec_ctp_cross_plus_b6 = (
    2 * crcv_sec_ctp_rho_b6
    * crcv_sec_ctp_cvr_plus_b6[0]
    * crcv_sec_ctp_cvr_plus_b6[1]
    * crcv_sec_ctp_psi_b6_plus
)
crcv_sec_ctp_K_b6_plus = np.sqrt(max(0, crcv_sec_ctp_sum_sq_plus_b6 + crcv_sec_ctp_cross_plus_b6))


# ============================================================
# 6) DOWNWARD SCENARIO (K_b-)
# ============================================================
# K_b- = √( Σ max(CVR_k-, 0)² + Σ Σ ρ_kl × CVR_k- × CVR_l- × ψ(CVR_k-, CVR_l-) )

# --- Bucket 5 --- #
crcv_sec_ctp_sum_sq_minus_b5 = np.sum(np.maximum(crcv_sec_ctp_cvr_minus_b5, 0)**2)
crcv_sec_ctp_cross_minus_b5 = (
    2 * crcv_sec_ctp_rho_b5
    * crcv_sec_ctp_cvr_minus_b5[0]
    * crcv_sec_ctp_cvr_minus_b5[1]
    * crcv_sec_ctp_psi_b5_minus
)
crcv_sec_ctp_K_b5_minus = np.sqrt(max(0, crcv_sec_ctp_sum_sq_minus_b5 + crcv_sec_ctp_cross_minus_b5))

# --- Bucket 6 --- #
crcv_sec_ctp_sum_sq_minus_b6 = np.sum(np.maximum(crcv_sec_ctp_cvr_minus_b6, 0)**2)
crcv_sec_ctp_cross_minus_b6 = (
    2 * crcv_sec_ctp_rho_b6
    * crcv_sec_ctp_cvr_minus_b6[0]
    * crcv_sec_ctp_cvr_minus_b6[1]
    * crcv_sec_ctp_psi_b6_minus
)
crcv_sec_ctp_K_b6_minus = np.sqrt(max(0, crcv_sec_ctp_sum_sq_minus_b6 + crcv_sec_ctp_cross_minus_b6))

# ============================================================
# 7) INTRA-BUCKET AGGREGATION (K_b)
# ============================================================
# K_b = max(K_b+, K_b-)

# --- Bucket 5 --- #
if crcv_sec_ctp_K_b5_plus > crcv_sec_ctp_K_b5_minus:
    crcv_sec_ctp_K_b5 = crcv_sec_ctp_K_b5_plus
    crcv_sec_ctp_scenario_b5 = "upward"
elif crcv_sec_ctp_K_b5_minus > crcv_sec_ctp_K_b5_plus:
    crcv_sec_ctp_K_b5 = crcv_sec_ctp_K_b5_minus
    crcv_sec_ctp_scenario_b5 = "downward"
else:
    crcv_sec_ctp_K_b5 = crcv_sec_ctp_K_b5_plus
    crcv_sec_ctp_scenario_b5 = "upward" if np.sum(crcv_sec_ctp_cvr_plus_b5) >= np.sum(crcv_sec_ctp_cvr_minus_b5) else "downward"

# --- Bucket 6 --- #
if crcv_sec_ctp_K_b6_plus > crcv_sec_ctp_K_b6_minus:
    crcv_sec_ctp_K_b6 = crcv_sec_ctp_K_b6_plus
    crcv_sec_ctp_scenario_b6 = "upward"
elif crcv_sec_ctp_K_b6_minus > crcv_sec_ctp_K_b6_plus:
    crcv_sec_ctp_K_b6 = crcv_sec_ctp_K_b6_minus
    crcv_sec_ctp_scenario_b6 = "downward"
else:
    crcv_sec_ctp_K_b6 = crcv_sec_ctp_K_b6_plus
    crcv_sec_ctp_scenario_b6 = "upward" if np.sum(crcv_sec_ctp_cvr_plus_b6) >= np.sum(crcv_sec_ctp_cvr_minus_b6) else "downward"


# ============================================================
# 8) BUCKET SUMS (S_b)
# ============================================================
# S_b = Σ CVR+ if upward scenario, Σ CVR- if downward scenario

crcv_sec_ctp_S_b5 = (
    crcv_sec_ctp_cvr_plus_b5.sum() if crcv_sec_ctp_scenario_b5 == "upward"
    else crcv_sec_ctp_cvr_minus_b5.sum()
)

crcv_sec_ctp_S_b6 = (
    crcv_sec_ctp_cvr_plus_b6.sum() if crcv_sec_ctp_scenario_b6 == "upward"
    else crcv_sec_ctp_cvr_minus_b6.sum()
)


# ============================================================
# 9) CROSS-BUCKET CORRELATION (gamma_bc)
# ============================================================
# γ_bc = (delta_γ_bc)²

CRCV_SEC_CTP_DELTA_GAMMA = 0.25
crcv_sec_ctp_gamma = CRCV_SEC_CTP_DELTA_GAMMA ** 2


# ============================================================
# 10) CROSS-BUCKET SAFEGUARD FUNCTION (ψ)
# ============================================================
# ψ(S_b, S_c) = 0 if both S_b < 0 and S_c < 0, else 1

crcv_sec_ctp_psi_cross = crcv_sec_ctp_psi(crcv_sec_ctp_S_b5, crcv_sec_ctp_S_b6)


# ============================================================
# 11) CROSS-BUCKET AGGREGATION
# ============================================================
# K = √( Σ K_b² + Σ Σ γ_bc × S_b × S_c × ψ(S_b, S_c) )

crcv_sec_ctp_sum_K_sq = crcv_sec_ctp_K_b5**2 + crcv_sec_ctp_K_b6**2
crcv_sec_ctp_cross_bucket = (
    2 * crcv_sec_ctp_gamma
    * crcv_sec_ctp_S_b5
    * crcv_sec_ctp_S_b6
    * crcv_sec_ctp_psi_cross
)
crcv_sec_ctp_K_medium = np.sqrt(max(0, crcv_sec_ctp_sum_K_sq + crcv_sec_ctp_cross_bucket))


# ============================================================
# 12) CORRELATION SCENARIOS (HIGH/LOW)
# ============================================================
# High: ρ_high = min(1.25 × ρ_medium, 1.0)
# Low:  ρ_low  = max(2 × ρ_medium - 1, 0.75 × ρ_medium)

# --- High scenario correlations --- #
crcv_sec_ctp_rho_high = min(crcv_sec_ctp_rho * 1.25, 1.0)
crcv_sec_ctp_gamma_high = min(crcv_sec_ctp_gamma * 1.25, 1.0)

# --- Low scenario correlations --- #
crcv_sec_ctp_rho_low = max(2 * crcv_sec_ctp_rho - 1.0, 0.75 * crcv_sec_ctp_rho)
crcv_sec_ctp_gamma_low = max(2 * crcv_sec_ctp_gamma - 1.0, 0.75 * crcv_sec_ctp_gamma)

# --- High scenario intra-bucket K_b --- #

# Bucket 5 High
crcv_sec_ctp_cross_plus_b5_high = 2 * crcv_sec_ctp_rho_high * crcv_sec_ctp_cvr_plus_b5[0] * crcv_sec_ctp_cvr_plus_b5[1] * crcv_sec_ctp_psi_b5_plus
crcv_sec_ctp_K_b5_plus_high = np.sqrt(max(0, crcv_sec_ctp_sum_sq_plus_b5 + crcv_sec_ctp_cross_plus_b5_high))

crcv_sec_ctp_cross_minus_b5_high = 2 * crcv_sec_ctp_rho_high * crcv_sec_ctp_cvr_minus_b5[0] * crcv_sec_ctp_cvr_minus_b5[1] * crcv_sec_ctp_psi_b5_minus
crcv_sec_ctp_K_b5_minus_high = np.sqrt(max(0, crcv_sec_ctp_sum_sq_minus_b5 + crcv_sec_ctp_cross_minus_b5_high))

if crcv_sec_ctp_K_b5_plus_high > crcv_sec_ctp_K_b5_minus_high:
    crcv_sec_ctp_K_b5_high = crcv_sec_ctp_K_b5_plus_high
    crcv_sec_ctp_scenario_b5_high = "upward"
elif crcv_sec_ctp_K_b5_minus_high > crcv_sec_ctp_K_b5_plus_high:
    crcv_sec_ctp_K_b5_high = crcv_sec_ctp_K_b5_minus_high
    crcv_sec_ctp_scenario_b5_high = "downward"
else:
    crcv_sec_ctp_K_b5_high = crcv_sec_ctp_K_b5_plus_high
    crcv_sec_ctp_scenario_b5_high = "upward" if np.sum(crcv_sec_ctp_cvr_plus_b5) >= np.sum(crcv_sec_ctp_cvr_minus_b5) else "downward"

# Bucket 6 High
crcv_sec_ctp_cross_plus_b6_high = 2 * crcv_sec_ctp_rho_high * crcv_sec_ctp_cvr_plus_b6[0] * crcv_sec_ctp_cvr_plus_b6[1] * crcv_sec_ctp_psi_b6_plus
crcv_sec_ctp_K_b6_plus_high = np.sqrt(max(0, crcv_sec_ctp_sum_sq_plus_b6 + crcv_sec_ctp_cross_plus_b6_high))

crcv_sec_ctp_cross_minus_b6_high = 2 * crcv_sec_ctp_rho_high * crcv_sec_ctp_cvr_minus_b6[0] * crcv_sec_ctp_cvr_minus_b6[1] * crcv_sec_ctp_psi_b6_minus
crcv_sec_ctp_K_b6_minus_high = np.sqrt(max(0, crcv_sec_ctp_sum_sq_minus_b6 + crcv_sec_ctp_cross_minus_b6_high))

if crcv_sec_ctp_K_b6_plus_high > crcv_sec_ctp_K_b6_minus_high:
    crcv_sec_ctp_K_b6_high = crcv_sec_ctp_K_b6_plus_high
    crcv_sec_ctp_scenario_b6_high = "upward"
elif crcv_sec_ctp_K_b6_minus_high > crcv_sec_ctp_K_b6_plus_high:
    crcv_sec_ctp_K_b6_high = crcv_sec_ctp_K_b6_minus_high
    crcv_sec_ctp_scenario_b6_high = "downward"
else:
    crcv_sec_ctp_K_b6_high = crcv_sec_ctp_K_b6_plus_high
    crcv_sec_ctp_scenario_b6_high = "upward" if np.sum(crcv_sec_ctp_cvr_plus_b6) >= np.sum(crcv_sec_ctp_cvr_minus_b6) else "downward"

# --- Low scenario intra-bucket K_b --- #

# Bucket 5 Low
crcv_sec_ctp_cross_plus_b5_low = 2 * crcv_sec_ctp_rho_low * crcv_sec_ctp_cvr_plus_b5[0] * crcv_sec_ctp_cvr_plus_b5[1] * crcv_sec_ctp_psi_b5_plus
crcv_sec_ctp_K_b5_plus_low = np.sqrt(max(0, crcv_sec_ctp_sum_sq_plus_b5 + crcv_sec_ctp_cross_plus_b5_low))

crcv_sec_ctp_cross_minus_b5_low = 2 * crcv_sec_ctp_rho_low * crcv_sec_ctp_cvr_minus_b5[0] * crcv_sec_ctp_cvr_minus_b5[1] * crcv_sec_ctp_psi_b5_minus
crcv_sec_ctp_K_b5_minus_low = np.sqrt(max(0, crcv_sec_ctp_sum_sq_minus_b5 + crcv_sec_ctp_cross_minus_b5_low))

if crcv_sec_ctp_K_b5_plus_low > crcv_sec_ctp_K_b5_minus_low:
    crcv_sec_ctp_K_b5_low = crcv_sec_ctp_K_b5_plus_low
    crcv_sec_ctp_scenario_b5_low = "upward"
elif crcv_sec_ctp_K_b5_minus_low > crcv_sec_ctp_K_b5_plus_low:
    crcv_sec_ctp_K_b5_low = crcv_sec_ctp_K_b5_minus_low
    crcv_sec_ctp_scenario_b5_low = "downward"
else:
    crcv_sec_ctp_K_b5_low = crcv_sec_ctp_K_b5_plus_low
    crcv_sec_ctp_scenario_b5_low = "upward" if np.sum(crcv_sec_ctp_cvr_plus_b5) >= np.sum(crcv_sec_ctp_cvr_minus_b5) else "downward"

# Bucket 6 Low
crcv_sec_ctp_cross_plus_b6_low = 2 * crcv_sec_ctp_rho_low * crcv_sec_ctp_cvr_plus_b6[0] * crcv_sec_ctp_cvr_plus_b6[1] * crcv_sec_ctp_psi_b6_plus
crcv_sec_ctp_K_b6_plus_low = np.sqrt(max(0, crcv_sec_ctp_sum_sq_plus_b6 + crcv_sec_ctp_cross_plus_b6_low))

crcv_sec_ctp_cross_minus_b6_low = 2 * crcv_sec_ctp_rho_low * crcv_sec_ctp_cvr_minus_b6[0] * crcv_sec_ctp_cvr_minus_b6[1] * crcv_sec_ctp_psi_b6_minus
crcv_sec_ctp_K_b6_minus_low = np.sqrt(max(0, crcv_sec_ctp_sum_sq_minus_b6 + crcv_sec_ctp_cross_minus_b6_low))

if crcv_sec_ctp_K_b6_plus_low > crcv_sec_ctp_K_b6_minus_low:
    crcv_sec_ctp_K_b6_low = crcv_sec_ctp_K_b6_minus_low
    crcv_sec_ctp_scenario_b6_low = "upward"
elif crcv_sec_ctp_K_b6_minus_low > crcv_sec_ctp_K_b6_plus_low:
    crcv_sec_ctp_K_b6_low = crcv_sec_ctp_K_b6_minus_low
    crcv_sec_ctp_scenario_b6_low = "downward"
else:
    crcv_sec_ctp_K_b6_low = crcv_sec_ctp_K_b6_plus_low
    crcv_sec_ctp_scenario_b6_low = "upward" if np.sum(crcv_sec_ctp_cvr_plus_b6) >= np.sum(crcv_sec_ctp_cvr_minus_b6) else "downward"

# --- High scenario bucket sums and cross-bucket --- #
crcv_sec_ctp_S_b5_high = crcv_sec_ctp_cvr_plus_b5.sum() if crcv_sec_ctp_scenario_b5_high == "upward" else crcv_sec_ctp_cvr_minus_b5.sum()
crcv_sec_ctp_S_b6_high = crcv_sec_ctp_cvr_plus_b6.sum() if crcv_sec_ctp_scenario_b6_high == "upward" else crcv_sec_ctp_cvr_minus_b6.sum()
crcv_sec_ctp_psi_cross_high = crcv_sec_ctp_psi(crcv_sec_ctp_S_b5_high, crcv_sec_ctp_S_b6_high)

crcv_sec_ctp_sum_K_sq_high = crcv_sec_ctp_K_b5_high**2 + crcv_sec_ctp_K_b6_high**2
crcv_sec_ctp_cross_bucket_high = 2 * crcv_sec_ctp_gamma_high * crcv_sec_ctp_S_b5_high * crcv_sec_ctp_S_b6_high * crcv_sec_ctp_psi_cross_high
crcv_sec_ctp_K_high = np.sqrt(max(0, crcv_sec_ctp_sum_K_sq_high + crcv_sec_ctp_cross_bucket_high))

# --- Low scenario bucket sums and cross-bucket --- #
crcv_sec_ctp_S_b5_low = crcv_sec_ctp_cvr_plus_b5.sum() if crcv_sec_ctp_scenario_b5_low == "upward" else crcv_sec_ctp_cvr_minus_b5.sum()
crcv_sec_ctp_S_b6_low = crcv_sec_ctp_cvr_plus_b6.sum() if crcv_sec_ctp_scenario_b6_low == "upward" else crcv_sec_ctp_cvr_minus_b6.sum()
crcv_sec_ctp_psi_cross_low = crcv_sec_ctp_psi(crcv_sec_ctp_S_b5_low, crcv_sec_ctp_S_b6_low)

crcv_sec_ctp_sum_K_sq_low = crcv_sec_ctp_K_b5_low**2 + crcv_sec_ctp_K_b6_low**2
crcv_sec_ctp_cross_bucket_low = 2 * crcv_sec_ctp_gamma_low * crcv_sec_ctp_S_b5_low * crcv_sec_ctp_S_b6_low * crcv_sec_ctp_psi_cross_low
crcv_sec_ctp_K_low = np.sqrt(max(0, crcv_sec_ctp_sum_K_sq_low + crcv_sec_ctp_cross_bucket_low))


# ============================================================
# 13) CAPITAL
# ============================================================

crcv_sec_ctp_capital_df = pd.DataFrame([
    {'scenario': 'medium', 'capital': crcv_sec_ctp_K_medium},
    {'scenario': 'high', 'capital': crcv_sec_ctp_K_high},
    {'scenario': 'low', 'capital': crcv_sec_ctp_K_low},
])


# ============================================================
# 14) PRINTING
# ============================================================

# --- 14.1 Gross curvature positions --- #
crcv_sec_ctp_gross_cvr_print = format_table(
    crcv_sec_ctp_gross_cvr,
    columns=['trade_id', 'bucket', 'issuer', 'cvr_plus', 'cvr_minus'],
    col_formats={'cvr_plus': ',.0f', 'cvr_minus': ',.0f'},
    alignments=('left', 'left', 'left', 'right', 'right')
)

# --- 14.2 Net curvature positions --- #
crcv_sec_ctp_net_cvr_print = format_table(
    crcv_sec_ctp_net_cvr,
    columns=['bucket', 'issuer', 'cvr_plus', 'cvr_minus'],
    col_formats={'cvr_plus': ',.0f', 'cvr_minus': ',.0f'},
    alignments=('left', 'left', 'right', 'right')
)

# --- 14.3 Intra-bucket correlation --- #
crcv_sec_ctp_intra_corr_print = format_table(
    pd.DataFrame([
        {'bucket': '5', 'delta_rho': CRCV_SEC_CTP_DELTA_RHO, 'rho': crcv_sec_ctp_rho_b5},
        {'bucket': '6', 'delta_rho': CRCV_SEC_CTP_DELTA_RHO, 'rho': crcv_sec_ctp_rho_b6},
    ]),
    columns=['bucket', 'delta_rho', 'rho'],
    col_formats={'delta_rho': '.4f', 'rho': '.4f'},
    alignments=('left', 'right', 'right')
)

# --- 14.4 Intra-bucket safeguard function (ψ) --- #
# Exactly matching Gold Standard column structure
crcv_sec_ctp_psi_intra_print = format_table(
    pd.DataFrame([
        {'bucket': '5', 'cvr_k+': crcv_sec_ctp_cvr_plus_b5[0], 'cvr_l+': crcv_sec_ctp_cvr_plus_b5[1], 'ψ_up': crcv_sec_ctp_psi_b5_plus, 'cvr_k-': crcv_sec_ctp_cvr_minus_b5[0], 'cvr_l-': crcv_sec_ctp_cvr_minus_b5[1], 'ψ_down': crcv_sec_ctp_psi_b5_minus},
        {'bucket': '6', 'cvr_k+': crcv_sec_ctp_cvr_plus_b6[0], 'cvr_l+': crcv_sec_ctp_cvr_plus_b6[1], 'ψ_up': crcv_sec_ctp_psi_b6_plus, 'cvr_k-': crcv_sec_ctp_cvr_minus_b6[0], 'cvr_l-': crcv_sec_ctp_cvr_minus_b6[1], 'ψ_down': crcv_sec_ctp_psi_b6_minus},
    ]),
    columns=['bucket', 'cvr_k+', 'cvr_l+', 'ψ_up', 'cvr_k-', 'cvr_l-', 'ψ_down'],
    col_formats={'cvr_k+': ',.0f', 'cvr_l+': ',.0f', 'cvr_k-': ',.0f', 'cvr_l-': ',.0f'},
    alignments=('left', 'right', 'right', 'center', 'right', 'right', 'center')
)

# --- 14.5 Upward scenario (K_b+) --- #
# Exactly matching Gold Standard column structure
crcv_sec_ctp_upward_print = format_table(
    pd.DataFrame([
        {'bucket': '5', 'cvr_k+': crcv_sec_ctp_cvr_plus_b5[0], 'cvr_l+': crcv_sec_ctp_cvr_plus_b5[1], 'rho': crcv_sec_ctp_rho_b5, 'ψ': crcv_sec_ctp_psi_b5_plus, 'K_b+': crcv_sec_ctp_K_b5_plus},
        {'bucket': '6', 'cvr_k+': crcv_sec_ctp_cvr_plus_b6[0], 'cvr_l+': crcv_sec_ctp_cvr_plus_b6[1], 'rho': crcv_sec_ctp_rho_b6, 'ψ': crcv_sec_ctp_psi_b6_plus, 'K_b+': crcv_sec_ctp_K_b6_plus},
    ]),
    columns=['bucket', 'cvr_k+', 'cvr_l+', 'rho', 'ψ', 'K_b+'],
    col_formats={'cvr_k+': ',.0f', 'cvr_l+': ',.0f', 'rho': '.4f', 'K_b+': ',.2f'},
    alignments=('left', 'right', 'right', 'right', 'center', 'right')
)

# Formula String for Printing
crcv_sec_ctp_Kb5_plus_formula_print = (
    f"K_b5+ = sqrt( (max({crcv_sec_ctp_cvr_plus_b5[0]},0)^2 + max({crcv_sec_ctp_cvr_plus_b5[1]},0)^2) + "
    f"(2 * {crcv_sec_ctp_rho_b5:.4f} * {crcv_sec_ctp_cvr_plus_b5[0]} * {crcv_sec_ctp_cvr_plus_b5[1]} * {crcv_sec_ctp_psi_b5_plus}) )"
)

# Formula String for Printing
crcv_sec_ctp_Kb6_plus_formula_print = (
    f"K_b6+ = sqrt( (max({crcv_sec_ctp_cvr_plus_b6[0]},0)^2 + max({crcv_sec_ctp_cvr_plus_b6[1]},0)^2) + "
    f"(2 * {crcv_sec_ctp_rho_b6:.4f} * {crcv_sec_ctp_cvr_plus_b6[0]} * {crcv_sec_ctp_cvr_plus_b6[1]} * {crcv_sec_ctp_psi_b6_plus}) )"
)

# --- 14.6 Downward scenario (K_b-) --- #
# Exactly matching Gold Standard column structure
crcv_sec_ctp_downward_print = format_table(
    pd.DataFrame([
        {'bucket': '5', 'cvr_k-': crcv_sec_ctp_cvr_minus_b5[0], 'cvr_l-': crcv_sec_ctp_cvr_minus_b5[1], 'rho': crcv_sec_ctp_rho_b5, 'ψ': crcv_sec_ctp_psi_b5_minus, 'K_b-': crcv_sec_ctp_K_b5_minus},
        {'bucket': '6', 'cvr_k-': crcv_sec_ctp_cvr_minus_b6[0], 'cvr_l-': crcv_sec_ctp_cvr_minus_b6[1], 'rho': crcv_sec_ctp_rho_b6, 'ψ': crcv_sec_ctp_psi_b6_minus, 'K_b-': crcv_sec_ctp_K_b6_minus},
    ]),
    columns=['bucket', 'cvr_k-', 'cvr_l-', 'rho', 'ψ', 'K_b-'],
    col_formats={'cvr_k-': ',.0f', 'cvr_l-': ',.0f', 'rho': '.4f', 'K_b-': ',.2f'},
    alignments=('left', 'right', 'right', 'right', 'center', 'right')
)

# Formula String for Printing
crcv_sec_ctp_Kb5_minus_formula_print = (
    f"K_b5- = sqrt( (max({crcv_sec_ctp_cvr_minus_b5[0]},0)^2 + max({crcv_sec_ctp_cvr_minus_b5[1]},0)^2) + "
    f"(2 * {crcv_sec_ctp_rho_b5:.4f} * {crcv_sec_ctp_cvr_minus_b5[0]} * {crcv_sec_ctp_cvr_minus_b5[1]} * {crcv_sec_ctp_psi_b5_minus}) )"
)

# Formula String for Printing
crcv_sec_ctp_Kb6_minus_formula_print = (
    f"K_b6- = sqrt( (max({crcv_sec_ctp_cvr_minus_b6[0]},0)^2 + max({crcv_sec_ctp_cvr_minus_b6[1]},0)^2) + "
    f"(2 * {crcv_sec_ctp_rho_b6:.4f} * {crcv_sec_ctp_cvr_minus_b6[0]} * {crcv_sec_ctp_cvr_minus_b6[1]} * {crcv_sec_ctp_psi_b6_minus}) )"
)


# --- 14.7 Intra-bucket aggregation (K_b) --- #
crcv_sec_ctp_intra_agg_print = format_table(
    pd.DataFrame([
        {'bucket': '5', 'K_b+': crcv_sec_ctp_K_b5_plus, 'K_b-': crcv_sec_ctp_K_b5_minus, 'K_b': crcv_sec_ctp_K_b5, 'selected': crcv_sec_ctp_scenario_b5},
        {'bucket': '6', 'K_b+': crcv_sec_ctp_K_b6_plus, 'K_b-': crcv_sec_ctp_K_b6_minus, 'K_b': crcv_sec_ctp_K_b6, 'selected': crcv_sec_ctp_scenario_b6},
    ]),
    columns=['bucket', 'K_b+', 'K_b-', 'K_b', 'selected'],
    col_formats={'K_b+': ',.2f', 'K_b-': ',.2f', 'K_b': ',.2f'},
    alignments=('left', 'right', 'right', 'right', 'left')
)

# --- 14.8 Bucket sums (S_b) --- #
crcv_sec_ctp_bucket_sums_print = format_table(
    pd.DataFrame([
        {'bucket': '5', 'selected': crcv_sec_ctp_scenario_b5, 'cvr_k': crcv_sec_ctp_cvr_minus_b5[0] if crcv_sec_ctp_scenario_b5 == "downward" else crcv_sec_ctp_cvr_plus_b5[0], 'cvr_l': crcv_sec_ctp_cvr_minus_b5[1] if crcv_sec_ctp_scenario_b5 == "downward" else crcv_sec_ctp_cvr_plus_b5[1], 'S_b': crcv_sec_ctp_S_b5},
        {'bucket': '6', 'selected': crcv_sec_ctp_scenario_b6, 'cvr_k': crcv_sec_ctp_cvr_minus_b6[0] if crcv_sec_ctp_scenario_b6 == "downward" else crcv_sec_ctp_cvr_plus_b6[0], 'cvr_l': crcv_sec_ctp_cvr_minus_b6[1] if crcv_sec_ctp_scenario_b6 == "downward" else crcv_sec_ctp_cvr_plus_b6[1], 'S_b': crcv_sec_ctp_S_b6},
    ]),
    columns=['bucket', 'selected', 'cvr_k', 'cvr_l', 'S_b'],
    col_formats={'cvr_k': ',.0f', 'cvr_l': ',.0f', 'S_b': ',.2f'},
    alignments=('left', 'left', 'right', 'right', 'right')
)

# --- 14.9 Cross-bucket correlation --- #
crcv_sec_ctp_cross_corr_print = format_table(
    pd.DataFrame([
        {'buckets': '5 vs 6', 'delta_gamma': CRCV_SEC_CTP_DELTA_GAMMA, 'gamma': crcv_sec_ctp_gamma},
    ]),
    columns=['buckets', 'delta_gamma', 'gamma'],
    col_formats={'delta_gamma': '.4f', 'gamma': '.4f'},
    alignments=('left', 'right', 'right')
)

# --- 14.10 Cross-bucket safeguard function (ψ) --- #
crcv_sec_ctp_psi_cross_print = format_table(
    pd.DataFrame([
        {'S_b5': crcv_sec_ctp_S_b5, 'S_b6': crcv_sec_ctp_S_b6, 'ψ': crcv_sec_ctp_psi_cross},
    ]),
    columns=['S_b5', 'S_b6', 'ψ'],
    col_formats={'S_b5': ',.2f', 'S_b6': ',.2f'},
    alignments=('right', 'right', 'center')
)

# --- 14.11 Cross-bucket aggregation --- #
crcv_sec_ctp_cross_agg_print = format_table(
    pd.DataFrame([
        {'component': 'K_b5', 'value': crcv_sec_ctp_K_b5},
        {'component': 'K_b6', 'value': crcv_sec_ctp_K_b6},
        {'component': 'S_b5', 'value': crcv_sec_ctp_S_b5},
        {'component': 'S_b6', 'value': crcv_sec_ctp_S_b6},
        {'component': 'gamma', 'value': f"{crcv_sec_ctp_gamma:.4f}"},
        {'component': 'ψ(S_b5, S_b6)', 'value': crcv_sec_ctp_psi_cross},
        {'component': 'capital', 'value': crcv_sec_ctp_K_medium},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# Formula String for Printing
crcv_sec_ctp_cross_agg_formula_print = (
    f"Capital = sqrt( ({crcv_sec_ctp_K_b5:,.2f}^2 + {crcv_sec_ctp_K_b6:,.2f}^2) + "
    f"(2 * {crcv_sec_ctp_gamma:.4f} * {crcv_sec_ctp_S_b5:,.2f} * {crcv_sec_ctp_S_b6:,.2f} * {crcv_sec_ctp_psi_cross}) )"
)


# --- 14.12 Correlation scenarios --- #
crcv_sec_ctp_corr_scenarios_print = format_table(
    pd.DataFrame([
        {'parameter': 'rho', 'medium': crcv_sec_ctp_rho, 'high': crcv_sec_ctp_rho_high, 'low': crcv_sec_ctp_rho_low},
        {'parameter': 'gamma', 'medium': crcv_sec_ctp_gamma, 'high': crcv_sec_ctp_gamma_high, 'low': crcv_sec_ctp_gamma_low},
    ]),
    columns=['parameter', 'medium', 'high', 'low'],
    col_formats={'medium': '.4f', 'high': '.4f', 'low': '.4f'},
    alignments=('left', 'right', 'right', 'right')
)

# --- 14.13 Capital --- #
crcv_sec_ctp_capital_print = format_table(
    crcv_sec_ctp_capital_df,
    columns=['scenario', 'capital'],
    col_formats={'capital': ',.2f'},
    alignments=('left', 'right')
)


# ============================================================
# DISPLAY ALL OUTPUTS
# ============================================================

print("=" * 60)
print("CRCV Securitisation CTP")
print("=" * 60)

print("\n1) GROSS CURVATURE POSITIONS")
print(crcv_sec_ctp_gross_cvr_print)

print("\n2) NET CURVATURE POSITIONS")
print(crcv_sec_ctp_net_cvr_print)

print("\n3) INTRA-BUCKET CORRELATION (rho_kl)")
print(crcv_sec_ctp_intra_corr_print)

print("\n4) INTRA-BUCKET SAFEGUARD FUNCTION (ψ)")
print(crcv_sec_ctp_psi_intra_print)

print("\n5) UPWARD SCENARIO (K_b+)")
print("\n--- Bucket 5 ---")
print(crcv_sec_ctp_Kb5_plus_formula_print)
print("\n--- Bucket 6 ---")
print(crcv_sec_ctp_Kb6_plus_formula_print)
print(crcv_sec_ctp_upward_print)

print("\n6) DOWNWARD SCENARIO (K_b-)")
print("\n--- Bucket 5 ---")
print(crcv_sec_ctp_Kb5_minus_formula_print)
print("\n--- Bucket 6 ---")
print(crcv_sec_ctp_Kb6_minus_formula_print)
print(crcv_sec_ctp_downward_print)

print("\n7) INTRA-BUCKET AGGREGATION (K_b)")
print(crcv_sec_ctp_intra_agg_print)

print("\n8) BUCKET SUMS (S_b)")
print(crcv_sec_ctp_bucket_sums_print)

print("\n9) CROSS-BUCKET CORRELATION (gamma_bc)")
print(crcv_sec_ctp_cross_corr_print)

print("\n10) CROSS-BUCKET SAFEGUARD FUNCTION (ψ)")
print(crcv_sec_ctp_psi_cross_print)

print("\n11) CROSS-BUCKET AGGREGATION")
print(crcv_sec_ctp_cross_agg_formula_print)
print(crcv_sec_ctp_cross_agg_print)

print("\n12) CORRELATION SCENARIOS")
print(crcv_sec_ctp_corr_scenarios_print)

print("\n13) CAPITAL")
print(crcv_sec_ctp_capital_print)

CRCV Securitisation CTP

1) GROSS CURVATURE POSITIONS
┌────────────┬──────────┬──────────┬────────────┬─────────────┐
│ trade_id   │ bucket   │ issuer   │   cvr_plus │   cvr_minus │
├────────────┼──────────┼──────────┼────────────┼─────────────┤
│ 1          │ 5        │ issuer_a │       -800 │       1,600 │
│ 2          │ 5        │ issuer_b │     -4,500 │       4,000 │
│ 3          │ 6        │ issuer_f │       -380 │       1,090 │
│ 4          │ 6        │ issuer_i │     -2,240 │       2,850 │
└────────────┴──────────┴──────────┴────────────┴─────────────┘

2) NET CURVATURE POSITIONS
┌──────────┬──────────┬────────────┬─────────────┐
│ bucket   │ issuer   │   cvr_plus │   cvr_minus │
├──────────┼──────────┼────────────┼─────────────┤
│ 5        │ issuer_a │       -800 │       1,600 │
│ 5        │ issuer_b │     -4,500 │       4,000 │
│ 6        │ issuer_f │       -380 │       1,090 │
│ 6        │ issuer_i │     -2,240 │       2,850 │
└──────────┴──────────┴────────────┴─────────────

### CRCV Sec

In [21]:
# ============================================================
# CRCV SEC
# ============================================================


# TABLE OF CONTENTS
#   1) Gross curvature positions
#   2) Net curvature positions (CVR+/CVR-)
#   3) Intra-bucket correlation (rho_kl)
#   4) Intra-bucket safeguard function (ψ)
#   5) Upward scenario (K_b+)
#   6) Downward scenario (K_b-)
#   7) Intra-bucket aggregation (K_b)
#   8) Bucket sums (S_b)
#   9) Cross-bucket correlation (gamma_bc)
#  10) Cross-bucket safeguard function (ψ)
#  11) Cross-bucket aggregation
#  12) Correlation scenarios (high/low)
#  13) Capital
#  14) Printing
# ============================================================


# ============================================================
# 1) GROSS CURVATURE POSITIONS
# ============================================================
# Position-level curvature values from upward (+) and downward (-) shocks.

data = [
    {'trade_id': 1, 'bucket': '3', 'tranche': 'rmbs_tranche_a', 'cvr_plus': 750, 'cvr_minus': -600},
    {'trade_id': 2, 'bucket': '3', 'tranche': 'rmbs_tranche_b', 'cvr_plus': -400, 'cvr_minus': 550},
    {'trade_id': 3, 'bucket': '14', 'tranche': 'abs_tranche_1', 'cvr_plus': -800, 'cvr_minus': 950},
    {'trade_id': 4, 'bucket': '14', 'tranche': 'abs_tranche_2', 'cvr_plus': 300, 'cvr_minus': -200}
]

crcv_sec_gross_cvr = pd.DataFrame(data)


# ============================================================
# 2) NET CURVATURE POSITIONS (CVR+/CVR-)
# ============================================================
# Net CVR = Σ gross CVR for each unique risk factor.
# Risk factor defined by: bucket + tranche.

crcv_sec_risk_factor_cols = ['bucket', 'tranche']

crcv_sec_net_cvr = (
    crcv_sec_gross_cvr
    .groupby(crcv_sec_risk_factor_cols)
    .agg(cvr_plus=('cvr_plus', 'sum'), cvr_minus=('cvr_minus', 'sum'))
    .reset_index()
)

# Extract CVR vectors per bucket
crcv_sec_cvr_b3 = crcv_sec_net_cvr[crcv_sec_net_cvr['bucket'] == '3']
crcv_sec_cvr_plus_b3 = crcv_sec_cvr_b3['cvr_plus'].values
crcv_sec_cvr_minus_b3 = crcv_sec_cvr_b3['cvr_minus'].values

crcv_sec_cvr_b14 = crcv_sec_net_cvr[crcv_sec_net_cvr['bucket'] == '14']
crcv_sec_cvr_plus_b14 = crcv_sec_cvr_b14['cvr_plus'].values
crcv_sec_cvr_minus_b14 = crcv_sec_cvr_b14['cvr_minus'].values


# ============================================================
# 3) INTRA-BUCKET CORRELATION (rho_kl)
# ============================================================
# For curvature: ρ_kl = (delta_ρ_kl)²

CRCV_SEC_DELTA_RHO = 0.40
crcv_sec_rho = CRCV_SEC_DELTA_RHO ** 2

# Buckets 3 & 14 have multiple risk factors -> correlation applies
crcv_sec_rho_b3 = crcv_sec_rho
crcv_sec_rho_b14 = crcv_sec_rho


# ============================================================
# 4) INTRA-BUCKET SAFEGUARD FUNCTION (ψ)
# ============================================================
# ψ(x, y) = 0 if both x < 0 and y < 0, else 1

def crcv_sec_psi(x, y):
    """Safeguard function: returns 0 if both x and y are negative, else 1"""
    return 0 if x < 0 and y < 0 else 1

# Bucket 3 (two risk factors)
crcv_sec_psi_b3_plus = crcv_sec_psi(crcv_sec_cvr_plus_b3[0], crcv_sec_cvr_plus_b3[1])
crcv_sec_psi_b3_minus = crcv_sec_psi(crcv_sec_cvr_minus_b3[0], crcv_sec_cvr_minus_b3[1])

# Bucket 14 (two risk factors)
crcv_sec_psi_b14_plus = crcv_sec_psi(crcv_sec_cvr_plus_b14[0], crcv_sec_cvr_plus_b14[1])
crcv_sec_psi_b14_minus = crcv_sec_psi(crcv_sec_cvr_minus_b14[0], crcv_sec_cvr_minus_b14[1])


# ============================================================
# 5) UPWARD SCENARIO (K_b+)
# ============================================================
# K_b+ = √( Σ max(CVR_k+, 0)² + Σ Σ ρ_kl × CVR_k+ × CVR_l+ × ψ(CVR_k+, CVR_l+) )

# --- Bucket 3 --- #
crcv_sec_sum_sq_plus_b3 = np.sum(np.maximum(crcv_sec_cvr_plus_b3, 0)**2)
crcv_sec_cross_plus_b3 = (
    2 * crcv_sec_rho_b3
    * crcv_sec_cvr_plus_b3[0]
    * crcv_sec_cvr_plus_b3[1]
    * crcv_sec_psi_b3_plus
)
crcv_sec_K_b3_plus = np.sqrt(max(0, crcv_sec_sum_sq_plus_b3 + crcv_sec_cross_plus_b3))

# --- Bucket 14 --- #
crcv_sec_sum_sq_plus_b14 = np.sum(np.maximum(crcv_sec_cvr_plus_b14, 0)**2)
crcv_sec_cross_plus_b14 = (
    2 * crcv_sec_rho_b14
    * crcv_sec_cvr_plus_b14[0]
    * crcv_sec_cvr_plus_b14[1]
    * crcv_sec_psi_b14_plus
)
crcv_sec_K_b14_plus = np.sqrt(max(0, crcv_sec_sum_sq_plus_b14 + crcv_sec_cross_plus_b14))


# ============================================================
# 6) DOWNWARD SCENARIO (K_b-)
# ============================================================
# K_b- = √( Σ max(CVR_k-, 0)² + Σ Σ ρ_kl × CVR_k- × CVR_l- × ψ(CVR_k-, CVR_l-) )

# --- Bucket 3 --- #
crcv_sec_sum_sq_minus_b3 = np.sum(np.maximum(crcv_sec_cvr_minus_b3, 0)**2)
crcv_sec_cross_minus_b3 = (
    2 * crcv_sec_rho_b3
    * crcv_sec_cvr_minus_b3[0]
    * crcv_sec_cvr_minus_b3[1]
    * crcv_sec_psi_b3_minus
)
crcv_sec_K_b3_minus = np.sqrt(max(0, crcv_sec_sum_sq_minus_b3 + crcv_sec_cross_minus_b3))


# --- Bucket 14 --- #
crcv_sec_sum_sq_minus_b14 = np.sum(np.maximum(crcv_sec_cvr_minus_b14, 0)**2)
crcv_sec_cross_minus_b14 = (
    2 * crcv_sec_rho_b14
    * crcv_sec_cvr_minus_b14[0]
    * crcv_sec_cvr_minus_b14[1]
    * crcv_sec_psi_b14_minus
)
crcv_sec_K_b14_minus = np.sqrt(max(0, crcv_sec_sum_sq_minus_b14 + crcv_sec_cross_minus_b14))


# ============================================================
# 7) INTRA-BUCKET AGGREGATION (K_b)
# ============================================================
# K_b = max(K_b+, K_b-)

# --- Bucket 3 --- #
if crcv_sec_K_b3_plus > crcv_sec_K_b3_minus:
    crcv_sec_K_b3 = crcv_sec_K_b3_plus
    crcv_sec_scenario_b3 = "upward"
elif crcv_sec_K_b3_minus > crcv_sec_K_b3_plus:
    crcv_sec_K_b3 = crcv_sec_K_b3_minus
    crcv_sec_scenario_b3 = "downward"
else:
    crcv_sec_K_b3 = crcv_sec_K_b3_plus
    crcv_sec_scenario_b3 = "upward" if np.sum(crcv_sec_cvr_plus_b3) >= np.sum(crcv_sec_cvr_minus_b3) else "downward"

# --- Bucket 14 --- #
if crcv_sec_K_b14_plus > crcv_sec_K_b14_minus:
    crcv_sec_K_b14 = crcv_sec_K_b14_plus
    crcv_sec_scenario_b14 = "upward"
elif crcv_sec_K_b14_minus > crcv_sec_K_b14_plus:
    crcv_sec_K_b14 = crcv_sec_K_b14_minus
    crcv_sec_scenario_b14 = "downward"
else:
    crcv_sec_K_b14 = crcv_sec_K_b14_plus
    crcv_sec_scenario_b14 = "upward" if np.sum(crcv_sec_cvr_plus_b14) >= np.sum(crcv_sec_cvr_minus_b14) else "downward"


# ============================================================
# 8) BUCKET SUMS (S_b)
# ============================================================
# S_b = Σ CVR+ if upward scenario, Σ CVR- if downward scenario

crcv_sec_S_b3 = (
    crcv_sec_cvr_plus_b3.sum() if crcv_sec_scenario_b3 == "upward"
    else crcv_sec_cvr_minus_b3.sum()
)

crcv_sec_S_b14 = (
    crcv_sec_cvr_plus_b14.sum() if crcv_sec_scenario_b14 == "upward"
    else crcv_sec_cvr_minus_b14.sum()
)


# ============================================================
# 9) CROSS-BUCKET CORRELATION (gamma_bc)
# ============================================================
# γ_bc = (delta_γ_bc)²

CRCV_SEC_DELTA_GAMMA = 0.00
crcv_sec_gamma = CRCV_SEC_DELTA_GAMMA ** 2


# ============================================================
# 10) CROSS-BUCKET SAFEGUARD FUNCTION (ψ)
# ============================================================
# ψ(S_b, S_c) = 0 if both S_b < 0 and S_c < 0, else 1

crcv_sec_psi_cross = crcv_sec_psi(crcv_sec_S_b3, crcv_sec_S_b14)


# ============================================================
# 11) CROSS-BUCKET AGGREGATION
# ============================================================
# K = √( Σ K_b² + Σ Σ γ_bc × S_b × S_c × ψ(S_b, S_c) )

crcv_sec_sum_K_sq = crcv_sec_K_b3**2 + crcv_sec_K_b14**2
crcv_sec_cross_bucket = (
    2 * crcv_sec_gamma
    * crcv_sec_S_b3
    * crcv_sec_S_b14
    * crcv_sec_psi_cross
)
crcv_sec_K_medium = np.sqrt(max(0, crcv_sec_sum_K_sq + crcv_sec_cross_bucket))


# ============================================================
# 12) CORRELATION SCENARIOS (HIGH/LOW)
# ============================================================
# High: ρ_high = min(1.25 × ρ_medium, 1.0)
# Low:  ρ_low  = max(2 × ρ_medium - 1, 0.75 × ρ_medium)

# --- High scenario correlations --- #
crcv_sec_rho_high = min(crcv_sec_rho * 1.25, 1.0)
crcv_sec_gamma_high = min(crcv_sec_gamma * 1.25, 1.0) # Still 0

# --- Low scenario correlations --- #
crcv_sec_rho_low = max(2 * crcv_sec_rho - 1.0, 0.75 * crcv_sec_rho)
crcv_sec_gamma_low = max(2 * crcv_sec_gamma - 1.0, 0.75 * crcv_sec_gamma) # Still 0

# --- High scenario intra-bucket K_b --- #

# Bucket 3 High
crcv_sec_cross_plus_b3_high = 2 * crcv_sec_rho_high * crcv_sec_cvr_plus_b3[0] * crcv_sec_cvr_plus_b3[1] * crcv_sec_psi_b3_plus
crcv_sec_K_b3_plus_high = np.sqrt(max(0, crcv_sec_sum_sq_plus_b3 + crcv_sec_cross_plus_b3_high))

crcv_sec_cross_minus_b3_high = 2 * crcv_sec_rho_high * crcv_sec_cvr_minus_b3[0] * crcv_sec_cvr_minus_b3[1] * crcv_sec_psi_b3_minus
crcv_sec_K_b3_minus_high = np.sqrt(max(0, crcv_sec_sum_sq_minus_b3 + crcv_sec_cross_minus_b3_high))

if crcv_sec_K_b3_plus_high > crcv_sec_K_b3_minus_high:
    crcv_sec_K_b3_high = crcv_sec_K_b3_plus_high
    crcv_sec_scenario_b3_high = "upward"
elif crcv_sec_K_b3_minus_high > crcv_sec_K_b3_plus_high:
    crcv_sec_K_b3_high = crcv_sec_K_b3_minus_high
    crcv_sec_scenario_b3_high = "downward"
else:
    crcv_sec_K_b3_high = crcv_sec_K_b3_plus_high
    crcv_sec_scenario_b3_high = "upward" if np.sum(crcv_sec_cvr_plus_b3) >= np.sum(crcv_sec_cvr_minus_b3) else "downward"

# Bucket 14 High
crcv_sec_cross_plus_b14_high = 2 * crcv_sec_rho_high * crcv_sec_cvr_plus_b14[0] * crcv_sec_cvr_plus_b14[1] * crcv_sec_psi_b14_plus
crcv_sec_K_b14_plus_high = np.sqrt(max(0, crcv_sec_sum_sq_plus_b14 + crcv_sec_cross_plus_b14_high))

crcv_sec_cross_minus_b14_high = 2 * crcv_sec_rho_high * crcv_sec_cvr_minus_b14[0] * crcv_sec_cvr_minus_b14[1] * crcv_sec_psi_b14_minus
crcv_sec_K_b14_minus_high = np.sqrt(max(0, crcv_sec_sum_sq_minus_b14 + crcv_sec_cross_minus_b14_high))

if crcv_sec_K_b14_plus_high > crcv_sec_K_b14_minus_high:
    crcv_sec_K_b14_high = crcv_sec_K_b14_plus_high
    crcv_sec_scenario_b14_high = "upward"
elif crcv_sec_K_b14_minus_high > crcv_sec_K_b14_plus_high:
    crcv_sec_K_b14_high = crcv_sec_K_b14_minus_high
    crcv_sec_scenario_b14_high = "downward"
else:
    crcv_sec_K_b14_high = crcv_sec_K_b14_plus_high
    crcv_sec_scenario_b14_high = "upward" if np.sum(crcv_sec_cvr_plus_b14) >= np.sum(crcv_sec_cvr_minus_b14) else "downward"

# --- Low scenario intra-bucket K_b --- #

# Bucket 3 Low
crcv_sec_cross_plus_b3_low = 2 * crcv_sec_rho_low * crcv_sec_cvr_plus_b3[0] * crcv_sec_cvr_plus_b3[1] * crcv_sec_psi_b3_plus
crcv_sec_K_b3_plus_low = np.sqrt(max(0, crcv_sec_sum_sq_plus_b3 + crcv_sec_cross_plus_b3_low))

crcv_sec_cross_minus_b3_low = 2 * crcv_sec_rho_low * crcv_sec_cvr_minus_b3[0] * crcv_sec_cvr_minus_b3[1] * crcv_sec_psi_b3_minus
crcv_sec_K_b3_minus_low = np.sqrt(max(0, crcv_sec_sum_sq_minus_b3 + crcv_sec_cross_minus_b3_low))

if crcv_sec_K_b3_plus_low > crcv_sec_K_b3_minus_low:
    crcv_sec_K_b3_low = crcv_sec_K_b3_plus_low
    crcv_sec_scenario_b3_low = "upward"
elif crcv_sec_K_b3_minus_low > crcv_sec_K_b3_plus_low:
    crcv_sec_K_b3_low = crcv_sec_K_b3_minus_low
    crcv_sec_scenario_b3_low = "downward"
else:
    crcv_sec_K_b3_low = crcv_sec_K_b3_plus_low
    crcv_sec_scenario_b3_low = "upward" if np.sum(crcv_sec_cvr_plus_b3) >= np.sum(crcv_sec_cvr_minus_b3) else "downward"

# Bucket 14 Low
crcv_sec_cross_plus_b14_low = 2 * crcv_sec_rho_low * crcv_sec_cvr_plus_b14[0] * crcv_sec_cvr_plus_b14[1] * crcv_sec_psi_b14_plus
crcv_sec_K_b14_plus_low = np.sqrt(max(0, crcv_sec_sum_sq_plus_b14 + crcv_sec_cross_plus_b14_low))

crcv_sec_cross_minus_b14_low = 2 * crcv_sec_rho_low * crcv_sec_cvr_minus_b14[0] * crcv_sec_cvr_minus_b14[1] * crcv_sec_psi_b14_minus
crcv_sec_K_b14_minus_low = np.sqrt(max(0, crcv_sec_sum_sq_minus_b14 + crcv_sec_cross_minus_b14_low))

if crcv_sec_K_b14_plus_low > crcv_sec_K_b14_minus_low:
    crcv_sec_K_b14_low = crcv_sec_K_b14_minus_low
    crcv_sec_scenario_b14_low = "upward"
elif crcv_sec_K_b14_minus_low > crcv_sec_K_b14_plus_low:
    crcv_sec_K_b14_low = crcv_sec_K_b14_minus_low
    crcv_sec_scenario_b14_low = "downward"
else:
    crcv_sec_K_b14_low = crcv_sec_K_b14_plus_low
    crcv_sec_scenario_b14_low = "upward" if np.sum(crcv_sec_cvr_plus_b14) >= np.sum(crcv_sec_cvr_minus_b14) else "downward"

# --- High scenario bucket sums and cross-bucket --- #
crcv_sec_S_b3_high = crcv_sec_cvr_plus_b3.sum() if crcv_sec_scenario_b3_high == "upward" else crcv_sec_cvr_minus_b3.sum()
crcv_sec_S_b14_high = crcv_sec_cvr_plus_b14.sum() if crcv_sec_scenario_b14_high == "upward" else crcv_sec_cvr_minus_b14.sum()
crcv_sec_psi_cross_high = crcv_sec_psi(crcv_sec_S_b3_high, crcv_sec_S_b14_high)

crcv_sec_sum_K_sq_high = crcv_sec_K_b3_high**2 + crcv_sec_K_b14_high**2
crcv_sec_cross_bucket_high = 2 * crcv_sec_gamma_high * crcv_sec_S_b3_high * crcv_sec_S_b14_high * crcv_sec_psi_cross_high
crcv_sec_K_high = np.sqrt(max(0, crcv_sec_sum_K_sq_high + crcv_sec_cross_bucket_high))

# --- Low scenario bucket sums and cross-bucket --- #
crcv_sec_S_b3_low = crcv_sec_cvr_plus_b3.sum() if crcv_sec_scenario_b3_low == "upward" else crcv_sec_cvr_minus_b3.sum()
crcv_sec_S_b14_low = crcv_sec_cvr_plus_b14.sum() if crcv_sec_scenario_b14_low == "upward" else crcv_sec_cvr_minus_b14.sum()
crcv_sec_psi_cross_low = crcv_sec_psi(crcv_sec_S_b3_low, crcv_sec_S_b14_low)

crcv_sec_sum_K_sq_low = crcv_sec_K_b3_low**2 + crcv_sec_K_b14_low**2
crcv_sec_cross_bucket_low = 2 * crcv_sec_gamma_low * crcv_sec_S_b3_low * crcv_sec_S_b14_low * crcv_sec_psi_cross_low
crcv_sec_K_low = np.sqrt(max(0, crcv_sec_sum_K_sq_low + crcv_sec_cross_bucket_low))


# ============================================================
# 13) CAPITAL
# ============================================================

crcv_sec_capital_df = pd.DataFrame([
    {'scenario': 'medium', 'capital': crcv_sec_K_medium},
    {'scenario': 'high', 'capital': crcv_sec_K_high},
    {'scenario': 'low', 'capital': crcv_sec_K_low},
])


# ============================================================
# 14) PRINTING
# ============================================================

# --- 14.1 Gross curvature positions --- #
crcv_sec_gross_cvr_print = format_table(
    crcv_sec_gross_cvr,
    columns=['trade_id', 'bucket', 'tranche', 'cvr_plus', 'cvr_minus'],
    col_formats={'cvr_plus': ',.0f', 'cvr_minus': ',.0f'},
    alignments=('left', 'left', 'left', 'right', 'right')
)

# --- 14.2 Net curvature positions --- #
crcv_sec_net_cvr_print = format_table(
    crcv_sec_net_cvr,
    columns=['bucket', 'tranche', 'cvr_plus', 'cvr_minus'],
    col_formats={'cvr_plus': ',.0f', 'cvr_minus': ',.0f'},
    alignments=('left', 'left', 'right', 'right')
)

# --- 14.3 Intra-bucket correlation --- #
crcv_sec_intra_corr_print = format_table(
    pd.DataFrame([
        {'bucket': '3', 'delta_rho': CRCV_SEC_DELTA_RHO, 'rho': crcv_sec_rho_b3},
        {'bucket': '14', 'delta_rho': CRCV_SEC_DELTA_RHO, 'rho': crcv_sec_rho_b14},
    ]),
    columns=['bucket', 'delta_rho', 'rho'],
    col_formats={'delta_rho': '.4f', 'rho': '.4f'},
    alignments=('left', 'right', 'right')
)

# --- 14.4 Intra-bucket safeguard function (ψ) --- #
crcv_sec_psi_intra_print = format_table(
    pd.DataFrame([
        {'bucket': '3', 'cvr_k+': crcv_sec_cvr_plus_b3[0], 'cvr_l+': crcv_sec_cvr_plus_b3[1], 'ψ_up': crcv_sec_psi_b3_plus, 'cvr_k-': crcv_sec_cvr_minus_b3[0], 'cvr_l-': crcv_sec_cvr_minus_b3[1], 'ψ_down': crcv_sec_psi_b3_minus},
        {'bucket': '14', 'cvr_k+': crcv_sec_cvr_plus_b14[0], 'cvr_l+': crcv_sec_cvr_plus_b14[1], 'ψ_up': crcv_sec_psi_b14_plus, 'cvr_k-': crcv_sec_cvr_minus_b14[0], 'cvr_l-': crcv_sec_cvr_minus_b14[1], 'ψ_down': crcv_sec_psi_b14_minus},
    ]),
    columns=['bucket', 'cvr_k+', 'cvr_l+', 'ψ_up', 'cvr_k-', 'cvr_l-', 'ψ_down'],
    col_formats={'cvr_k+': ',.0f', 'cvr_l+': ',.0f', 'cvr_k-': ',.0f', 'cvr_l-': ',.0f'},
    alignments=('left', 'right', 'right', 'center', 'right', 'right', 'center')
)

# --- 14.5 Upward scenario (K_b+) --- #
crcv_sec_upward_print = format_table(
    pd.DataFrame([
        {'bucket': '3', 'cvr_k+': crcv_sec_cvr_plus_b3[0], 'cvr_l+': crcv_sec_cvr_plus_b3[1], 'rho': crcv_sec_rho_b3, 'ψ': crcv_sec_psi_b3_plus, 'K_b+': crcv_sec_K_b3_plus},
        {'bucket': '14', 'cvr_k+': crcv_sec_cvr_plus_b14[0], 'cvr_l+': crcv_sec_cvr_plus_b14[1], 'rho': crcv_sec_rho_b14, 'ψ': crcv_sec_psi_b14_plus, 'K_b+': crcv_sec_K_b14_plus},
    ]),
    columns=['bucket', 'cvr_k+', 'cvr_l+', 'rho', 'ψ', 'K_b+'],
    col_formats={'cvr_k+': ',.0f', 'cvr_l+': ',.0f', 'rho': '.4f', 'K_b+': ',.2f'},
    alignments=('left', 'right', 'right', 'right', 'center', 'right')
)

# Formula String
crcv_sec_Kb3_plus_formula_print = (
    f"K_b3+ = sqrt( (max({crcv_sec_cvr_plus_b3[0]},0)^2 + max({crcv_sec_cvr_plus_b3[1]},0)^2) + "
    f"(2 * {crcv_sec_rho_b3:.4f} * {crcv_sec_cvr_plus_b3[0]} * {crcv_sec_cvr_plus_b3[1]} * {crcv_sec_psi_b3_plus}) )"
)

# Formula String
crcv_sec_Kb14_plus_formula_print = (
    f"K_b14+ = sqrt( (max({crcv_sec_cvr_plus_b14[0]},0)^2 + max({crcv_sec_cvr_plus_b14[1]},0)^2) + "
    f"(2 * {crcv_sec_rho_b14:.4f} * {crcv_sec_cvr_plus_b14[0]} * {crcv_sec_cvr_plus_b14[1]} * {crcv_sec_psi_b14_plus}) )"
)

# --- 14.6 Downward scenario (K_b-) --- #
crcv_sec_downward_print = format_table(
    pd.DataFrame([
        {'bucket': '3', 'cvr_k-': crcv_sec_cvr_minus_b3[0], 'cvr_l-': crcv_sec_cvr_minus_b3[1], 'rho': crcv_sec_rho_b3, 'ψ': crcv_sec_psi_b3_minus, 'K_b-': crcv_sec_K_b3_minus},
        {'bucket': '14', 'cvr_k-': crcv_sec_cvr_minus_b14[0], 'cvr_l-': crcv_sec_cvr_minus_b14[1], 'rho': crcv_sec_rho_b14, 'ψ': crcv_sec_psi_b14_minus, 'K_b-': crcv_sec_K_b14_minus},
    ]),
    columns=['bucket', 'cvr_k-', 'cvr_l-', 'rho', 'ψ', 'K_b-'],
    col_formats={'cvr_k-': ',.0f', 'cvr_l-': ',.0f', 'rho': '.4f', 'K_b-': ',.2f'},
    alignments=('left', 'right', 'right', 'right', 'center', 'right')
)

# Formula String
crcv_sec_Kb3_minus_formula_print = (
    f"K_b3- = sqrt( (max({crcv_sec_cvr_minus_b3[0]},0)^2 + max({crcv_sec_cvr_minus_b3[1]},0)^2) + "
    f"(2 * {crcv_sec_rho_b3:.4f} * {crcv_sec_cvr_minus_b3[0]} * {crcv_sec_cvr_minus_b3[1]} * {crcv_sec_psi_b3_minus}) )"
)

# Formula String
crcv_sec_Kb14_minus_formula_print = (
    f"K_b14- = sqrt( (max({crcv_sec_cvr_minus_b14[0]},0)^2 + max({crcv_sec_cvr_minus_b14[1]},0)^2) + "
    f"(2 * {crcv_sec_rho_b14:.4f} * {crcv_sec_cvr_minus_b14[0]} * {crcv_sec_cvr_minus_b14[1]} * {crcv_sec_psi_b14_minus}) )"
)

# --- 14.7 Intra-bucket aggregation (K_b) --- #
crcv_sec_intra_agg_print = format_table(
    pd.DataFrame([
        {'bucket': '3', 'K_b+': crcv_sec_K_b3_plus, 'K_b-': crcv_sec_K_b3_minus, 'K_b': crcv_sec_K_b3, 'selected': crcv_sec_scenario_b3},
        {'bucket': '14', 'K_b+': crcv_sec_K_b14_plus, 'K_b-': crcv_sec_K_b14_minus, 'K_b': crcv_sec_K_b14, 'selected': crcv_sec_scenario_b14},
    ]),
    columns=['bucket', 'K_b+', 'K_b-', 'K_b', 'selected'],
    col_formats={'K_b+': ',.2f', 'K_b-': ',.2f', 'K_b': ',.2f'},
    alignments=('left', 'right', 'right', 'right', 'left')
)

# --- 14.8 Bucket sums (S_b) --- #
crcv_sec_bucket_sums_print = format_table(
    pd.DataFrame([
        {'bucket': '3', 'selected': crcv_sec_scenario_b3, 'cvr_k': crcv_sec_cvr_minus_b3[0] if crcv_sec_scenario_b3 == "downward" else crcv_sec_cvr_plus_b3[0], 'cvr_l': crcv_sec_cvr_minus_b3[1] if crcv_sec_scenario_b3 == "downward" else crcv_sec_cvr_plus_b3[1], 'S_b': crcv_sec_S_b3},
        {'bucket': '14', 'selected': crcv_sec_scenario_b14, 'cvr_k': crcv_sec_cvr_minus_b14[0] if crcv_sec_scenario_b14 == "downward" else crcv_sec_cvr_plus_b14[0], 'cvr_l': crcv_sec_cvr_minus_b14[1] if crcv_sec_scenario_b14 == "downward" else crcv_sec_cvr_plus_b14[1], 'S_b': crcv_sec_S_b14},
    ]),
    columns=['bucket', 'selected', 'cvr_k', 'cvr_l', 'S_b'],
    col_formats={'cvr_k': ',.0f', 'cvr_l': ',.0f', 'S_b': ',.2f'},
    alignments=('left', 'left', 'right', 'right', 'right')
)

# --- 14.9 Cross-bucket correlation --- #
crcv_sec_cross_corr_print = format_table(
    pd.DataFrame([
        {'buckets': '3 vs 14', 'delta_gamma': CRCV_SEC_DELTA_GAMMA, 'gamma': crcv_sec_gamma},
    ]),
    columns=['buckets', 'delta_gamma', 'gamma'],
    col_formats={'delta_gamma': '.4f', 'gamma': '.4f'},
    alignments=('left', 'right', 'right')
)

# --- 14.10 Cross-bucket safeguard function (ψ) --- #
crcv_sec_psi_cross_print = format_table(
    pd.DataFrame([
        {'S_b3': crcv_sec_S_b3, 'S_b14': crcv_sec_S_b14, 'ψ': crcv_sec_psi_cross},
    ]),
    columns=['S_b3', 'S_b14', 'ψ'],
    col_formats={'S_b3': ',.2f', 'S_b14': ',.2f'},
    alignments=('right', 'right', 'center')
)

# --- 14.11 Cross-bucket aggregation --- #
crcv_sec_cross_agg_print = format_table(
    pd.DataFrame([
        {'component': 'K_b3', 'value': crcv_sec_K_b3},
        {'component': 'K_b14', 'value': crcv_sec_K_b14},
        {'component': 'S_b3', 'value': crcv_sec_S_b3},
        {'component': 'S_b14', 'value': crcv_sec_S_b14},
        {'component': 'gamma', 'value': f"{crcv_sec_gamma:.4f}"},
        {'component': 'ψ(S_b3, S_b14)', 'value': crcv_sec_psi_cross},
        {'component': 'capital', 'value': crcv_sec_K_medium},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# Formula String
crcv_sec_cross_agg_formula_print = (
    f"Capital = sqrt( ({crcv_sec_K_b3:,.2f}^2 + {crcv_sec_K_b14:,.2f}^2) + "
    f"(2 * {crcv_sec_gamma:.4f} * {crcv_sec_S_b3:,.2f} * {crcv_sec_S_b14:,.2f} * {crcv_sec_psi_cross}) )"
)

# --- 14.12 Correlation scenarios --- #
crcv_sec_corr_scenarios_print = format_table(
    pd.DataFrame([
        {'parameter': 'rho', 'medium': crcv_sec_rho, 'high': crcv_sec_rho_high, 'low': crcv_sec_rho_low},
        {'parameter': 'gamma', 'medium': crcv_sec_gamma, 'high': crcv_sec_gamma_high, 'low': crcv_sec_gamma_low},
    ]),
    columns=['parameter', 'medium', 'high', 'low'],
    col_formats={'medium': '.4f', 'high': '.4f', 'low': '.4f'},
    alignments=('left', 'right', 'right', 'right')
)

# --- 14.13 Capital --- #
crcv_sec_capital_print = format_table(
    crcv_sec_ctp_capital_df,
    columns=['scenario', 'capital'],
    col_formats={'capital': ',.2f'},
    alignments=('left', 'right')
)


# ============================================================
# DISPLAY ALL OUTPUTS
# ============================================================

print("=" * 60)
print("CRCV Securitisation")
print("=" * 60)

print("\n1) GROSS CURVATURE POSITIONS")
print(crcv_sec_gross_cvr_print)

print("\n2) NET CURVATURE POSITIONS")
print(crcv_sec_net_cvr_print)

print("\n3) INTRA-BUCKET CORRELATION (rho_kl)")
print(crcv_sec_intra_corr_print)

print("\n4) INTRA-BUCKET SAFEGUARD FUNCTION (ψ)")
print(crcv_sec_psi_intra_print)

print("\n5) UPWARD SCENARIO (K_b+)")
print("\n--- Bucket 3 ---")
print(crcv_sec_Kb3_plus_formula_print)
print("\n--- Bucket 14 ---")
print(crcv_sec_Kb14_plus_formula_print)
print(crcv_sec_upward_print)

print("\n6) DOWNWARD SCENARIO (K_b-)")
print("\n--- Bucket 3 ---")
print(crcv_sec_Kb3_minus_formula_print)
print("\n--- Bucket 14 ---")
print(crcv_sec_Kb14_minus_formula_print)
print(crcv_sec_downward_print)

print("\n7) INTRA-BUCKET AGGREGATION (K_b)")
print(crcv_sec_intra_agg_print)

print("\n8) BUCKET SUMS (S_b)")
print(crcv_sec_bucket_sums_print)

print("\n9) CROSS-BUCKET CORRELATION (gamma_bc)")
print(crcv_sec_cross_corr_print)

print("\n10) CROSS-BUCKET SAFEGUARD FUNCTION (ψ)")
print(crcv_sec_psi_cross_print)

print("\n11) CROSS-BUCKET AGGREGATION")
print(crcv_sec_cross_agg_formula_print)
print(crcv_sec_cross_agg_print)

print("\n12) CORRELATION SCENARIOS")
print(crcv_sec_corr_scenarios_print)

print("\n13) CAPITAL")
print(crcv_sec_capital_print)

CRCV Securitisation

1) GROSS CURVATURE POSITIONS
┌────────────┬──────────┬────────────────┬────────────┬─────────────┐
│ trade_id   │ bucket   │ tranche        │   cvr_plus │   cvr_minus │
├────────────┼──────────┼────────────────┼────────────┼─────────────┤
│ 1          │ 3        │ rmbs_tranche_a │        750 │        -600 │
│ 2          │ 3        │ rmbs_tranche_b │       -400 │         550 │
│ 3          │ 14       │ abs_tranche_1  │       -800 │         950 │
│ 4          │ 14       │ abs_tranche_2  │        300 │        -200 │
└────────────┴──────────┴────────────────┴────────────┴─────────────┘

2) NET CURVATURE POSITIONS
┌──────────┬────────────────┬────────────┬─────────────┐
│ bucket   │ tranche        │   cvr_plus │   cvr_minus │
├──────────┼────────────────┼────────────┼─────────────┤
│ 14       │ abs_tranche_1  │       -800 │         950 │
│ 14       │ abs_tranche_2  │        300 │        -200 │
│ 3        │ rmbs_tranche_a │        750 │        -600 │
│ 3        │ rmbs_tr

### EQCV

In [22]:
# ============================================================
# EQCV
# ============================================================


# TABLE OF CONTENTS
#   1) Gross curvature positions
#   2) Net curvature positions (CVR+/CVR-)
#   3) Intra-bucket correlation (rho_kl)
#   4) Intra-bucket safeguard function (ψ)
#   5) Upward scenario (K_b+)
#   6) Downward scenario (K_b-)
#   7) Intra-bucket aggregation (K_b)
#   8) Bucket sums (S_b)
#   9) Cross-bucket correlation (gamma_bc)
#  10) Cross-bucket safeguard function (ψ)
#  11) Cross-bucket aggregation
#  12) Correlation scenarios (high/low)
#  13) Capital
#  14) Printing
# ============================================================


# ============================================================
# 1) GROSS CURVATURE POSITIONS
# ============================================================
# Position-level curvature values from upward (+) and downward (-) shocks.

data = [
    {'trade_id': 1, 'bucket': '6', 'spot': 'issuer_a', 'cvr_plus': -6400, 'cvr_minus': 8800},
    {'trade_id': 2, 'bucket': '6', 'spot': 'issuer_b', 'cvr_plus': -4900, 'cvr_minus': 4800},
    {'trade_id': 3, 'bucket': '6', 'spot': 'issuer_b', 'cvr_plus': 6200, 'cvr_minus': -4800},
    {'trade_id': 4, 'bucket': '5', 'spot': 'issuer_c', 'cvr_plus': 8900, 'cvr_minus': -7200},
    {'trade_id': 5, 'bucket': '5', 'spot': 'issuer_d', 'cvr_plus': 3600, 'cvr_minus': -2800},
]

eqcv_gross_cvr = pd.DataFrame(data)


# ============================================================
# 2) NET CURVATURE POSITIONS (CVR+/CVR-)
# ============================================================
# Net CVR = Σ gross CVR for each unique risk factor.
# Risk factor defined by: bucket + issuer.

eqcv_risk_factor_cols = ['bucket', 'spot']

eqcv_net_cvr = (
    eqcv_gross_cvr
    .groupby(eqcv_risk_factor_cols)
    .agg(cvr_plus=('cvr_plus', 'sum'), cvr_minus=('cvr_minus', 'sum'))
    .reset_index()
)

# Extract CVR vectors per bucket
eqcv_cvr_b6 = eqcv_net_cvr[eqcv_net_cvr['bucket'] == '6']
eqcv_cvr_plus_b6 = eqcv_cvr_b6['cvr_plus'].values
eqcv_cvr_minus_b6 = eqcv_cvr_b6['cvr_minus'].values

eqcv_cvr_b5 = eqcv_net_cvr[eqcv_net_cvr['bucket'] == '5']
eqcv_cvr_plus_b5 = eqcv_cvr_b5['cvr_plus'].values
eqcv_cvr_minus_b5 = eqcv_cvr_b5['cvr_minus'].values


# ============================================================
# 3) INTRA-BUCKET CORRELATION (rho_kl)
# ============================================================
# For curvature: ρ_kl = (delta_ρ_kl)²

EQCV_DELTA_RHO = 0.25
eqcv_rho = EQCV_DELTA_RHO ** 2

# Both buckets have multiple risk factors -> correlation applies
eqcv_rho_b6 = eqcv_rho
eqcv_rho_b5 = eqcv_rho


# ============================================================
# 4) INTRA-BUCKET SAFEGUARD FUNCTION (ψ)
# ============================================================
# ψ(x, y) = 0 if both x < 0 and y < 0, else 1

def eqcv_psi(x, y):
    """Safeguard function: returns 0 if both x and y are negative, else 1"""
    return 0 if x < 0 and y < 0 else 1

# Bucket 6 (two risk factors)
eqcv_psi_b6_plus = eqcv_psi(eqcv_cvr_plus_b6[0], eqcv_cvr_plus_b6[1])
eqcv_psi_b6_minus = eqcv_psi(eqcv_cvr_minus_b6[0], eqcv_cvr_minus_b6[1])

# Bucket 5 (two risk factors)
eqcv_psi_b5_plus = eqcv_psi(eqcv_cvr_plus_b5[0], eqcv_cvr_plus_b5[1])
eqcv_psi_b5_minus = eqcv_psi(eqcv_cvr_minus_b5[0], eqcv_cvr_minus_b5[1])


# ============================================================
# 5) UPWARD SCENARIO (K_b+)
# ============================================================
# K_b+ = √( Σ max(CVR_k+, 0)² + Σ Σ ρ_kl × CVR_k+ × CVR_l+ × ψ(CVR_k+, CVR_l+) )

# --- Bucket 6 --- #
eqcv_sum_sq_plus_b6 = np.sum(np.maximum(eqcv_cvr_plus_b6, 0)**2)
eqcv_cross_plus_b6 = (
    2 * eqcv_rho_b6
    * eqcv_cvr_plus_b6[0]
    * eqcv_cvr_plus_b6[1]
    * eqcv_psi_b6_plus
)
eqcv_K_b6_plus = np.sqrt(max(0, eqcv_sum_sq_plus_b6 + eqcv_cross_plus_b6))


# --- Bucket 5 --- #
eqcv_sum_sq_plus_b5 = np.sum(np.maximum(eqcv_cvr_plus_b5, 0)**2)
eqcv_cross_plus_b5 = (
    2 * eqcv_rho_b5
    * eqcv_cvr_plus_b5[0]
    * eqcv_cvr_plus_b5[1]
    * eqcv_psi_b5_plus
)
eqcv_K_b5_plus = np.sqrt(max(0, eqcv_sum_sq_plus_b5 + eqcv_cross_plus_b5))


# ============================================================
# 6) DOWNWARD SCENARIO (K_b-)
# ============================================================
# K_b- = √( Σ max(CVR_k-, 0)² + Σ Σ ρ_kl × CVR_k- × CVR_l- × ψ(CVR_k-, CVR_l-) )

# --- Bucket 6 --- #
eqcv_sum_sq_minus_b6 = np.sum(np.maximum(eqcv_cvr_minus_b6, 0)**2)
eqcv_cross_minus_b6 = (
    2 * eqcv_rho_b6
    * eqcv_cvr_minus_b6[0]
    * eqcv_cvr_minus_b6[1]
    * eqcv_psi_b6_minus
)
eqcv_K_b6_minus = np.sqrt(max(0, eqcv_sum_sq_minus_b6 + eqcv_cross_minus_b6))


# --- Bucket 5 --- #
eqcv_sum_sq_minus_b5 = np.sum(np.maximum(eqcv_cvr_minus_b5, 0)**2)
eqcv_cross_minus_b5 = (
    2 * eqcv_rho_b5
    * eqcv_cvr_minus_b5[0]
    * eqcv_cvr_minus_b5[1]
    * eqcv_psi_b5_minus
)
eqcv_K_b5_minus = np.sqrt(max(0, eqcv_sum_sq_minus_b5 + eqcv_cross_minus_b5))


# ============================================================
# 7) INTRA-BUCKET AGGREGATION (K_b)
# ============================================================
# K_b = max(K_b+, K_b-)

# --- Bucket 6 --- #
if eqcv_K_b6_plus > eqcv_K_b6_minus:
    eqcv_K_b6 = eqcv_K_b6_plus
    eqcv_scenario_b6 = "upward"
elif eqcv_K_b6_minus > eqcv_K_b6_plus:
    eqcv_K_b6 = eqcv_K_b6_minus
    eqcv_scenario_b6 = "downward"
else:
    eqcv_K_b6 = eqcv_K_b6_plus
    eqcv_scenario_b6 = "upward" if np.sum(eqcv_cvr_plus_b6) >= np.sum(eqcv_cvr_minus_b6) else "downward"

# --- Bucket 5 --- #
if eqcv_K_b5_plus > eqcv_K_b5_minus:
    eqcv_K_b5 = eqcv_K_b5_plus
    eqcv_scenario_b5 = "upward"
elif eqcv_K_b5_minus > eqcv_K_b5_plus:
    eqcv_K_b5 = eqcv_K_b5_minus
    eqcv_scenario_b5 = "downward"
else:
    eqcv_K_b5 = eqcv_K_b5_plus
    eqcv_scenario_b5 = "upward" if np.sum(eqcv_cvr_plus_b5) >= np.sum(eqcv_cvr_minus_b5) else "downward"


# ============================================================
# 8) BUCKET SUMS (S_b)
# ============================================================
# S_b = Σ CVR+ if upward scenario, Σ CVR- if downward scenario

eqcv_S_b6 = (
    eqcv_cvr_plus_b6.sum() if eqcv_scenario_b6 == "upward"
    else eqcv_cvr_minus_b6.sum()
)

eqcv_S_b5 = (
    eqcv_cvr_plus_b5.sum() if eqcv_scenario_b5 == "upward"
    else eqcv_cvr_minus_b5.sum()
)


# ============================================================
# 9) CROSS-BUCKET CORRELATION (gamma_bc)
# ============================================================
# γ_bc = (delta_γ_bc)²

EQCV_DELTA_GAMMA = 0.15
eqcv_gamma = EQCV_DELTA_GAMMA ** 2


# ============================================================
# 10) CROSS-BUCKET SAFEGUARD FUNCTION (ψ)
# ============================================================
# ψ(S_b, S_c) = 0 if both S_b < 0 and S_c < 0, else 1

eqcv_psi_cross = eqcv_psi(eqcv_S_b5, eqcv_S_b6)


# ============================================================
# 11) CROSS-BUCKET AGGREGATION
# ============================================================
# K = √( Σ K_b² + Σ Σ γ_bc × S_b × S_c × ψ(S_b, S_c) )

eqcv_sum_K_sq = eqcv_K_b5**2 + eqcv_K_b6**2
eqcv_cross_bucket = (
    2 * eqcv_gamma
    * eqcv_S_b5
    * eqcv_S_b6
    * eqcv_psi_cross
)
eqcv_K_medium = np.sqrt(max(0, eqcv_sum_K_sq + eqcv_cross_bucket))


# ============================================================
# 12) CORRELATION SCENARIOS (HIGH/LOW)
# ============================================================
# High: ρ_high = min(1.25 × ρ_medium, 1.0)
# Low:  ρ_low  = max(2 × ρ_medium - 1, 0.75 × ρ_medium)

# --- High scenario correlations --- #
eqcv_rho_high = min(eqcv_rho * 1.25, 1.0)
eqcv_gamma_high = min(eqcv_gamma * 1.25, 1.0)

# --- Low scenario correlations --- #
eqcv_rho_low = max(2 * eqcv_rho - 1.0, 0.75 * eqcv_rho)
eqcv_gamma_low = max(2 * eqcv_gamma - 1.0, 0.75 * eqcv_gamma)

# --- High scenario intra-bucket K_b --- #

# Bucket 6 High
eqcv_cross_plus_b6_high = 2 * eqcv_rho_high * eqcv_cvr_plus_b6[0] * eqcv_cvr_plus_b6[1] * eqcv_psi_b6_plus
eqcv_K_b6_plus_high = np.sqrt(max(0, eqcv_sum_sq_plus_b6 + eqcv_cross_plus_b6_high))

eqcv_cross_minus_b6_high = 2 * eqcv_rho_high * eqcv_cvr_minus_b6[0] * eqcv_cvr_minus_b6[1] * eqcv_psi_b6_minus
eqcv_K_b6_minus_high = np.sqrt(max(0, eqcv_sum_sq_minus_b6 + eqcv_cross_minus_b6_high))

if eqcv_K_b6_plus_high > eqcv_K_b6_minus_high:
    eqcv_K_b6_high = eqcv_K_b6_plus_high
    eqcv_scenario_b6_high = "upward"
elif eqcv_K_b6_minus_high > eqcv_K_b6_plus_high:
    eqcv_K_b6_high = eqcv_K_b6_minus_high
    eqcv_scenario_b6_high = "downward"
else:
    eqcv_K_b6_high = eqcv_K_b6_plus_high
    eqcv_scenario_b6_high = "upward" if np.sum(eqcv_cvr_plus_b6) >= np.sum(eqcv_cvr_minus_b6) else "downward"

# Bucket 5 High
eqcv_cross_plus_b5_high = 2 * eqcv_rho_high * eqcv_cvr_plus_b5[0] * eqcv_cvr_plus_b5[1] * eqcv_psi_b5_plus
eqcv_K_b5_plus_high = np.sqrt(max(0, eqcv_sum_sq_plus_b5 + eqcv_cross_plus_b5_high))

eqcv_cross_minus_b5_high = 2 * eqcv_rho_high * eqcv_cvr_minus_b5[0] * eqcv_cvr_minus_b5[1] * eqcv_psi_b5_minus
eqcv_K_b5_minus_high = np.sqrt(max(0, eqcv_sum_sq_minus_b5 + eqcv_cross_minus_b5_high))

if eqcv_K_b5_plus_high > eqcv_K_b5_minus_high:
    eqcv_K_b5_high = eqcv_K_b5_plus_high
    eqcv_scenario_b5_high = "upward"
elif eqcv_K_b5_minus_high > eqcv_K_b5_plus_high:
    eqcv_K_b5_high = eqcv_K_b5_minus_high
    eqcv_scenario_b5_high = "downward"
else:
    eqcv_K_b5_high = eqcv_K_b5_plus_high
    eqcv_scenario_b5_high = "upward" if np.sum(eqcv_cvr_plus_b5) >= np.sum(eqcv_cvr_minus_b5) else "downward"

# --- Low scenario intra-bucket K_b --- #

# Bucket 6 Low
eqcv_cross_plus_b6_low = 2 * eqcv_rho_low * eqcv_cvr_plus_b6[0] * eqcv_cvr_plus_b6[1] * eqcv_psi_b6_plus
eqcv_K_b6_plus_low = np.sqrt(max(0, eqcv_sum_sq_plus_b6 + eqcv_cross_plus_b6_low))

eqcv_cross_minus_b6_low = 2 * eqcv_rho_low * eqcv_cvr_minus_b6[0] * eqcv_cvr_minus_b6[1] * eqcv_psi_b6_minus
eqcv_K_b6_minus_low = np.sqrt(max(0, eqcv_sum_sq_minus_b6 + eqcv_cross_minus_b6_low))

if eqcv_K_b6_plus_low > eqcv_K_b6_minus_low:
    eqcv_K_b6_low = eqcv_K_b6_plus_low
    eqcv_scenario_b6_low = "upward"
elif eqcv_K_b6_minus_low > eqcv_K_b6_plus_low:
    eqcv_K_b6_low = eqcv_K_b6_minus_low
    eqcv_scenario_b6_low = "downward"
else:
    eqcv_K_b6_low = eqcv_K_b6_plus_low
    eqcv_scenario_b6_low = "upward" if np.sum(eqcv_cvr_plus_b6) >= np.sum(eqcv_cvr_minus_b6) else "downward"

# Bucket 5 Low
eqcv_cross_plus_b5_low = 2 * eqcv_rho_low * eqcv_cvr_plus_b5[0] * eqcv_cvr_plus_b5[1] * eqcv_psi_b5_plus
eqcv_K_b5_plus_low = np.sqrt(max(0, eqcv_sum_sq_plus_b5 + eqcv_cross_plus_b5_low))

eqcv_cross_minus_b5_low = 2 * eqcv_rho_low * eqcv_cvr_minus_b5[0] * eqcv_cvr_minus_b5[1] * eqcv_psi_b5_minus
eqcv_K_b5_minus_low = np.sqrt(max(0, eqcv_sum_sq_minus_b5 + eqcv_cross_minus_b5_low))

if eqcv_K_b5_plus_low > eqcv_K_b5_minus_low:
    eqcv_K_b5_low = eqcv_K_b5_plus_low
    eqcv_scenario_b5_low = "upward"
elif eqcv_K_b5_minus_low > eqcv_K_b5_plus_low:
    eqcv_K_b5_low = eqcv_K_b5_minus_low
    eqcv_scenario_b5_low = "downward"
else:
    eqcv_K_b5_low = eqcv_K_b5_plus_low
    eqcv_scenario_b5_low = "upward" if np.sum(eqcv_cvr_plus_b5) >= np.sum(eqcv_cvr_minus_b5) else "downward"

# --- High scenario bucket sums and cross-bucket --- #
eqcv_S_b6_high = eqcv_cvr_plus_b6.sum() if eqcv_scenario_b6_high == "upward" else eqcv_cvr_minus_b6.sum()
eqcv_S_b5_high = eqcv_cvr_plus_b5.sum() if eqcv_scenario_b5_high == "upward" else eqcv_cvr_minus_b5.sum()
eqcv_psi_cross_high = eqcv_psi(eqcv_S_b5_high, eqcv_S_b6_high)

eqcv_sum_K_sq_high = eqcv_K_b5_high**2 + eqcv_K_b6_high**2
eqcv_cross_bucket_high = 2 * eqcv_gamma_high * eqcv_S_b5_high * eqcv_S_b6_high * eqcv_psi_cross_high
eqcv_K_high = np.sqrt(max(0, eqcv_sum_K_sq_high + eqcv_cross_bucket_high))

# --- Low scenario bucket sums and cross-bucket --- #
eqcv_S_b6_low = eqcv_cvr_plus_b6.sum() if eqcv_scenario_b6_low == "upward" else eqcv_cvr_minus_b6.sum()
eqcv_S_b5_low = eqcv_cvr_plus_b5.sum() if eqcv_scenario_b5_low == "upward" else eqcv_cvr_minus_b5.sum()
eqcv_psi_cross_low = eqcv_psi(eqcv_S_b5_low, eqcv_S_b6_low)

eqcv_sum_K_sq_low = eqcv_K_b5_low**2 + eqcv_K_b6_low**2
eqcv_cross_bucket_low = 2 * eqcv_gamma_low * eqcv_S_b5_low * eqcv_S_b6_low * eqcv_psi_cross_low
eqcv_K_low = np.sqrt(max(0, eqcv_sum_K_sq_low + eqcv_cross_bucket_low))


# ============================================================
# 13) CAPITAL
# ============================================================

eqcv_capital_df = pd.DataFrame([
    {'scenario': 'medium', 'capital': eqcv_K_medium},
    {'scenario': 'high', 'capital': eqcv_K_high},
    {'scenario': 'low', 'capital': eqcv_K_low},
])


# ============================================================
# 14) PRINTING
# ============================================================

# --- 14.1 Gross curvature positions --- #
eqcv_gross_cvr_print = format_table(
    eqcv_gross_cvr,
    columns=['trade_id', 'bucket', 'spot', 'cvr_plus', 'cvr_minus'],
    col_formats={'cvr_plus': ',.0f', 'cvr_minus': ',.0f'},
    alignments=('left', 'left', 'left', 'right', 'right')
)

# --- 14.2 Net curvature positions --- #
eqcv_net_cvr_print = format_table(
    eqcv_net_cvr,
    columns=['bucket', 'spot', 'cvr_plus', 'cvr_minus'],
    col_formats={'cvr_plus': ',.0f', 'cvr_minus': ',.0f'},
    alignments=('left', 'left', 'right', 'right')
)

# --- 14.3 Intra-bucket correlation --- #
eqcv_intra_corr_print = format_table(
    pd.DataFrame([
        {'bucket': '5', 'delta_rho': EQCV_DELTA_RHO, 'rho': eqcv_rho_b5},
        {'bucket': '6', 'delta_rho': EQCV_DELTA_RHO, 'rho': eqcv_rho_b6},
    ]),
    columns=['bucket', 'delta_rho', 'rho'],
    col_formats={'delta_rho': '.4f', 'rho': '.4f'},
    alignments=('left', 'right', 'right')
)

# --- 14.4 Intra-bucket safeguard function (ψ) --- #
eqcv_psi_intra_print = format_table(
    pd.DataFrame([
        {'bucket': '5', 'cvr_k+': eqcv_cvr_plus_b5[0], 'cvr_l+': eqcv_cvr_plus_b5[1], 'ψ_up': eqcv_psi_b5_plus, 'cvr_k-': eqcv_cvr_minus_b5[0], 'cvr_l-': eqcv_cvr_minus_b5[1], 'ψ_down': eqcv_psi_b5_minus},
        {'bucket': '6', 'cvr_k+': eqcv_cvr_plus_b6[0], 'cvr_l+': eqcv_cvr_plus_b6[1], 'ψ_up': eqcv_psi_b6_plus, 'cvr_k-': eqcv_cvr_minus_b6[0], 'cvr_l-': eqcv_cvr_minus_b6[1], 'ψ_down': eqcv_psi_b6_minus},
    ]),
    columns=['bucket', 'cvr_k+', 'cvr_l+', 'ψ_up', 'cvr_k-', 'cvr_l-', 'ψ_down'],
    col_formats={'cvr_k+': ',.0f', 'cvr_l+': ',.0f', 'cvr_k-': ',.0f', 'cvr_l-': ',.0f'},
    alignments=('left', 'right', 'right', 'center', 'right', 'right', 'center')
)

# --- 14.5 Upward scenario (K_b+) --- #
eqcv_upward_print = format_table(
    pd.DataFrame([
        {'bucket': '5', 'cvr_k+': eqcv_cvr_plus_b5[0], 'cvr_l+': eqcv_cvr_plus_b5[1], 'rho': eqcv_rho_b5, 'ψ': eqcv_psi_b5_plus, 'K_b+': eqcv_K_b5_plus},
        {'bucket': '6', 'cvr_k+': eqcv_cvr_plus_b6[0], 'cvr_l+': eqcv_cvr_plus_b6[1], 'rho': eqcv_rho_b6, 'ψ': eqcv_psi_b6_plus, 'K_b+': eqcv_K_b6_plus},
    ]),
    columns=['bucket', 'cvr_k+', 'cvr_l+', 'rho', 'ψ', 'K_b+'],
    col_formats={'cvr_k+': ',.0f', 'cvr_l+': ',.0f', 'rho': '.4f', 'K_b+': ',.2f'},
    alignments=('left', 'right', 'right', 'right', 'center', 'right')
)

# Formula String
eqcv_Kb6_plus_formula_print = (
    f"K_b6+ = sqrt( (max({eqcv_cvr_plus_b6[0]},0)^2 + max({eqcv_cvr_plus_b6[1]},0)^2) + "
    f"(2 * {eqcv_rho_b6:.4f} * {eqcv_cvr_plus_b6[0]} * {eqcv_cvr_plus_b6[1]} * {eqcv_psi_b6_plus}) )"
)

# Formula String
eqcv_Kb5_plus_formula_print = (
    f"K_b5+ = sqrt( (max({eqcv_cvr_plus_b5[0]},0)^2 + max({eqcv_cvr_plus_b5[1]},0)^2) + "
    f"(2 * {eqcv_rho_b5:.4f} * {eqcv_cvr_plus_b5[0]} * {eqcv_cvr_plus_b5[1]} * {eqcv_psi_b5_plus}) )"
)

# --- 14.6 Downward scenario (K_b-) --- #
eqcv_downward_print = format_table(
    pd.DataFrame([
        {'bucket': '5', 'cvr_k-': eqcv_cvr_minus_b5[0], 'cvr_l-': eqcv_cvr_minus_b5[1], 'rho': eqcv_rho_b5, 'ψ': eqcv_psi_b5_minus, 'K_b-': eqcv_K_b5_minus},
        {'bucket': '6', 'cvr_k-': eqcv_cvr_minus_b6[0], 'cvr_l-': eqcv_cvr_minus_b6[1], 'rho': eqcv_rho_b6, 'ψ': eqcv_psi_b6_minus, 'K_b-': eqcv_K_b6_minus},
    ]),
    columns=['bucket', 'cvr_k-', 'cvr_l-', 'rho', 'ψ', 'K_b-'],
    col_formats={'cvr_k-': ',.0f', 'cvr_l-': ',.0f', 'rho': '.4f', 'K_b-': ',.2f'},
    alignments=('left', 'right', 'right', 'right', 'center', 'right')
)

# Formula String
eqcv_Kb6_minus_formula_print = (
    f"K_b6- = sqrt( (max({eqcv_cvr_minus_b6[0]},0)^2 + max({eqcv_cvr_minus_b6[1]},0)^2) + "
    f"(2 * {eqcv_rho_b6:.4f} * {eqcv_cvr_minus_b6[0]} * {eqcv_cvr_minus_b6[1]} * {eqcv_psi_b6_minus}) )"
)

# Formula String
eqcv_Kb5_minus_formula_print = (
    f"K_b5- = sqrt( (max({eqcv_cvr_minus_b5[0]},0)^2 + max({eqcv_cvr_minus_b5[1]},0)^2) + "
    f"(2 * {eqcv_rho_b5:.4f} * {eqcv_cvr_minus_b5[0]} * {eqcv_cvr_minus_b5[1]} * {eqcv_psi_b5_minus}) )"
)

# --- 14.7 Intra-bucket aggregation (K_b) --- #
eqcv_intra_agg_print = format_table(
    pd.DataFrame([
        {'bucket': '5', 'K_b+': eqcv_K_b5_plus, 'K_b-': eqcv_K_b5_minus, 'K_b': eqcv_K_b5, 'selected': eqcv_scenario_b5},
        {'bucket': '6', 'K_b+': eqcv_K_b6_plus, 'K_b-': eqcv_K_b6_minus, 'K_b': eqcv_K_b6, 'selected': eqcv_scenario_b6},
    ]),
    columns=['bucket', 'K_b+', 'K_b-', 'K_b', 'selected'],
    col_formats={'K_b+': ',.2f', 'K_b-': ',.2f', 'K_b': ',.2f'},
    alignments=('left', 'right', 'right', 'right', 'left')
)

# --- 14.8 Bucket sums (S_b) --- #
eqcv_bucket_sums_print = format_table(
    pd.DataFrame([
        {'bucket': '5', 'selected': eqcv_scenario_b5, 'cvr_k': eqcv_cvr_minus_b5[0] if eqcv_scenario_b5 == "downward" else eqcv_cvr_plus_b5[0], 'cvr_l': eqcv_cvr_minus_b5[1] if eqcv_scenario_b5 == "downward" else eqcv_cvr_plus_b5[1], 'S_b': eqcv_S_b5},
        {'bucket': '6', 'selected': eqcv_scenario_b6, 'cvr_k': eqcv_cvr_minus_b6[0] if eqcv_scenario_b6 == "downward" else eqcv_cvr_plus_b6[0], 'cvr_l': eqcv_cvr_minus_b6[1] if eqcv_scenario_b6 == "downward" else eqcv_cvr_plus_b6[1], 'S_b': eqcv_S_b6},
    ]),
    columns=['bucket', 'selected', 'cvr_k', 'cvr_l', 'S_b'],
    col_formats={'cvr_k': ',.0f', 'cvr_l': ',.0f', 'S_b': ',.2f'},
    alignments=('left', 'left', 'right', 'right', 'right')
)

# --- 14.9 Cross-bucket correlation --- #
eqcv_cross_corr_print = format_table(
    pd.DataFrame([
        {'buckets': '5 vs 6', 'delta_gamma': EQCV_DELTA_GAMMA, 'gamma': eqcv_gamma},
    ]),
    columns=['buckets', 'delta_gamma', 'gamma'],
    col_formats={'delta_gamma': '.4f', 'gamma': '.4f'},
    alignments=('left', 'right', 'right')
)

# --- 14.10 Cross-bucket safeguard function (ψ) --- #
eqcv_psi_cross_print = format_table(
    pd.DataFrame([
        {'S_b5': eqcv_S_b5, 'S_b6': eqcv_S_b6, 'ψ': eqcv_psi_cross},
    ]),
    columns=['S_b5', 'S_b6', 'ψ'],
    col_formats={'S_b5': ',.2f', 'S_b6': ',.2f'},
    alignments=('right', 'right', 'center')
)

# --- 14.11 Cross-bucket aggregation --- #
eqcv_cross_agg_print = format_table(
    pd.DataFrame([
        {'component': 'K_b5', 'value': eqcv_K_b5},
        {'component': 'K_b6', 'value': eqcv_K_b6},
        {'component': 'S_b5', 'value': eqcv_S_b5},
        {'component': 'S_b6', 'value': eqcv_S_b6},
        {'component': 'gamma', 'value': f"{eqcv_gamma:.4f}"},
        {'component': 'ψ(S_b5, S_b6)', 'value': eqcv_psi_cross},
        {'component': 'capital', 'value': eqcv_K_medium},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# Formula String
eqcv_cross_agg_formula_print = (
    f"Capital = sqrt( ({eqcv_K_b5:,.2f}^2 + {eqcv_K_b6:,.2f}^2) + "
    f"(2 * {eqcv_gamma:.4f} * {eqcv_S_b5:,.2f} * {eqcv_S_b6:,.2f} * {eqcv_psi_cross}) )"
)

# --- 14.12 Correlation scenarios --- #
eqcv_corr_scenarios_print = format_table(
    pd.DataFrame([
        {'parameter': 'rho', 'medium': eqcv_rho, 'high': eqcv_rho_high, 'low': eqcv_rho_low},
        {'parameter': 'gamma', 'medium': eqcv_gamma, 'high': eqcv_gamma_high, 'low': eqcv_gamma_low},
    ]),
    columns=['parameter', 'medium', 'high', 'low'],
    col_formats={'medium': '.4f', 'high': '.4f', 'low': '.4f'},
    alignments=('left', 'right', 'right', 'right')
)

# --- 14.13 Capital --- #
eqcv_capital_print = format_table(
    eqcv_capital_df,
    columns=['scenario', 'capital'],
    col_formats={'capital': ',.2f'},
    alignments=('left', 'right')
)


# ============================================================
# DISPLAY ALL OUTPUTS
# ============================================================

print("=" * 60)
print("EQCV")
print("=" * 60)

print("\n1) GROSS CURVATURE POSITIONS")
print(eqcv_gross_cvr_print)

print("\n2) NET CURVATURE POSITIONS")
print(eqcv_net_cvr_print)

print("\n3) INTRA-BUCKET CORRELATION")
print(eqcv_intra_corr_print)

print("\n4) INTRA-BUCKET SAFEGUARD FUNCTION (ψ)")
print(eqcv_psi_intra_print)

print("\n5) UPWARD SCENARIO (K_b+)")
print("\n--- Bucket 6 ---")
print(eqcv_Kb6_plus_formula_print)
print("\n--- Bucket 5 ---")
print(eqcv_Kb5_plus_formula_print)

print(eqcv_upward_print)

print("\n6) DOWNWARD SCENARIO (K_b-)")
print("\n--- Bucket 6 ---")
print(eqcv_Kb6_minus_formula_print)
print("\n--- Bucket 5 ---")
print(eqcv_Kb5_minus_formula_print)
print(eqcv_downward_print)

print("\n7) INTRA-BUCKET AGGREGATION (K_b)")
print(eqcv_intra_agg_print)

print("\n8) BUCKET SUMS (S_b)")
print(eqcv_bucket_sums_print)

print("\n9) CROSS-BUCKET CORRELATION")
print(eqcv_cross_corr_print)

print("\n10) CROSS-BUCKET SAFEGUARD FUNCTION (ψ)")
print(eqcv_psi_cross_print)

print("\n11) CROSS-BUCKET AGGREGATION")
print(eqcv_cross_agg_formula_print)
print(eqcv_cross_agg_print)

print("\n12) CORRELATION SCENARIOS")
print(eqcv_corr_scenarios_print)

print("\n13) CAPITAL")
print(eqcv_capital_print)

EQCV

1) GROSS CURVATURE POSITIONS
┌────────────┬──────────┬──────────┬────────────┬─────────────┐
│ trade_id   │ bucket   │ spot     │   cvr_plus │   cvr_minus │
├────────────┼──────────┼──────────┼────────────┼─────────────┤
│ 1          │ 6        │ issuer_a │     -6,400 │       8,800 │
│ 2          │ 6        │ issuer_b │     -4,900 │       4,800 │
│ 3          │ 6        │ issuer_b │      6,200 │      -4,800 │
│ 4          │ 5        │ issuer_c │      8,900 │      -7,200 │
│ 5          │ 5        │ issuer_d │      3,600 │      -2,800 │
└────────────┴──────────┴──────────┴────────────┴─────────────┘

2) NET CURVATURE POSITIONS
┌──────────┬──────────┬────────────┬─────────────┐
│ bucket   │ spot     │   cvr_plus │   cvr_minus │
├──────────┼──────────┼────────────┼─────────────┤
│ 5        │ issuer_c │      8,900 │      -7,200 │
│ 5        │ issuer_d │      3,600 │      -2,800 │
│ 6        │ issuer_a │     -6,400 │       8,800 │
│ 6        │ issuer_b │      1,300 │           0 │
└───

### CMCV

In [23]:
# ============================================================
# CMCV
# ============================================================


# TABLE OF CONTENTS
#   1) Gross curvature positions
#   2) Net curvature positions (CVR+/CVR-)
#   3) Intra-bucket correlation (rho_kl)
#   4) Intra-bucket safeguard function (ψ)
#   5) Upward scenario (K_b+)
#   6) Downward scenario (K_b-)
#   7) Intra-bucket aggregation (K_b)
#   8) Bucket sums (S_b)
#   9) Cross-bucket correlation (gamma_bc)
#  10) Cross-bucket safeguard function (ψ)
#  11) Cross-bucket aggregation
#  12) Correlation scenarios (high/low)
#  13) Capital
#  14) Printing
# ============================================================


# ============================================================
# 1) GROSS CURVATURE POSITIONS
# ============================================================
# Position-level curvature values from upward (+) and downward (-) shocks.

data = [
    {'trade_id': 1, 'bucket': '10', 'commodity': 'sugar', 'cvr_plus':  2600, 'cvr_minus':  1400},
    {'trade_id': 2, 'bucket': '10', 'commodity': 'sugar', 'cvr_plus':  9100, 'cvr_minus':  4900},
    {'trade_id': 3, 'bucket': '2',  'commodity': 'brent', 'cvr_plus':  1200, 'cvr_minus': -1520},
    {'trade_id': 4, 'bucket': '2',  'commodity': 'wti',   'cvr_plus': -3500, 'cvr_minus':  6300},
]

cmcv_gross_cvr = pd.DataFrame(data)


# ============================================================
# 2) NET CURVATURE POSITIONS (CVR+/CVR-)
# ============================================================
# Net CVR = Σ gross CVR for each unique risk factor.
# Risk factor defined by: bucket + commodity.

cmcv_risk_factor_cols = ['bucket', 'commodity']

cmcv_net_cvr = (
    cmcv_gross_cvr
    .groupby(cmcv_risk_factor_cols)
    .agg(cvr_plus=('cvr_plus', 'sum'), cvr_minus=('cvr_minus', 'sum'))
    .reset_index()
)

# Extract CVR vectors per bucket
cmcv_cvr_b10 = cmcv_net_cvr[cmcv_net_cvr['bucket'] == '10']
cmcv_cvr_plus_b10 = cmcv_cvr_b10['cvr_plus'].values
cmcv_cvr_minus_b10 = cmcv_cvr_b10['cvr_minus'].values

cmcv_cvr_b2 = cmcv_net_cvr[cmcv_net_cvr['bucket'] == '2']
cmcv_cvr_plus_b2 = cmcv_cvr_b2['cvr_plus'].values
cmcv_cvr_minus_b2 = cmcv_cvr_b2['cvr_minus'].values


# ============================================================
# 3) INTRA-BUCKET CORRELATION (rho_kl)
# ============================================================
# For curvature: ρ_kl = (delta_ρ_kl)²

CMCV_DELTA_RHO = 0.95 * 1.0 * 1.0 # 0.95
cmcv_rho = CMCV_DELTA_RHO ** 2

# Bucket 10: single risk factor -> no intra-bucket correlation needed
# Bucket 2: two risk factors -> correlation applies
cmcv_rho_b10 = None
cmcv_rho_b2 = cmcv_rho


# ============================================================
# 4) INTRA-BUCKET SAFEGUARD FUNCTION (ψ)
# ============================================================
# ψ(x, y) = 0 if both x < 0 and y < 0, else 1

def cmcv_psi(x, y):
    """Safeguard function: returns 0 if both x and y are negative, else 1"""
    return 0 if x < 0 and y < 0 else 1

# Bucket 10 (single risk factor - no psi needed)
cmcv_psi_b10_plus = None
cmcv_psi_b10_minus = None

# Bucket 2 (two risk factors)
cmcv_psi_b2_plus = cmcv_psi(cmcv_cvr_plus_b2[0], cmcv_cvr_plus_b2[1])
cmcv_psi_b2_minus = cmcv_psi(cmcv_cvr_minus_b2[0], cmcv_cvr_minus_b2[1])


# ============================================================
# 5) UPWARD SCENARIO (K_b+)
# ============================================================
# K_b+ = √( Σ max(CVR_k+, 0)² + Σ Σ ρ_kl × CVR_k+ × CVR_l+ × ψ(CVR_k+, CVR_l+) )

# --- Bucket 10 (single RF - no cross term) --- #
cmcv_sum_sq_plus_b10 = np.sum(np.maximum(cmcv_cvr_plus_b10, 0)**2)
cmcv_cross_plus_b10 = 0
cmcv_K_b10_plus = np.sqrt(max(0, cmcv_sum_sq_plus_b10 + cmcv_cross_plus_b10))


# --- Bucket 2 --- #
cmcv_sum_sq_plus_b2 = np.sum(np.maximum(cmcv_cvr_plus_b2, 0)**2)
cmcv_cross_plus_b2 = (
    2 * cmcv_rho_b2
    * cmcv_cvr_plus_b2[0]
    * cmcv_cvr_plus_b2[1]
    * cmcv_psi_b2_plus
)
cmcv_K_b2_plus = np.sqrt(max(0, cmcv_sum_sq_plus_b2 + cmcv_cross_plus_b2))


# ============================================================
# 6) DOWNWARD SCENARIO (K_b-)
# ============================================================
# K_b- = √( Σ max(CVR_k-, 0)² + Σ Σ ρ_kl × CVR_k- × CVR_l- × ψ(CVR_k-, CVR_l-) )

# --- Bucket 10 (single RF - no cross term) --- #
cmcv_sum_sq_minus_b10 = np.sum(np.maximum(cmcv_cvr_minus_b10, 0)**2)
cmcv_cross_minus_b10 = 0
cmcv_K_b10_minus = np.sqrt(max(0, cmcv_sum_sq_minus_b10 + cmcv_cross_minus_b10))


# --- Bucket 2 --- #
cmcv_sum_sq_minus_b2 = np.sum(np.maximum(cmcv_cvr_minus_b2, 0)**2)
cmcv_cross_minus_b2 = (
    2 * cmcv_rho_b2
    * cmcv_cvr_minus_b2[0]
    * cmcv_cvr_minus_b2[1]
    * cmcv_psi_b2_minus
)
cmcv_K_b2_minus = np.sqrt(max(0, cmcv_sum_sq_minus_b2 + cmcv_cross_minus_b2))


# ============================================================
# 7) INTRA-BUCKET AGGREGATION (K_b)
# ============================================================
# K_b = max(K_b+, K_b-)

# --- Bucket 10 --- #
if cmcv_K_b10_plus > cmcv_K_b10_minus:
    cmcv_K_b10 = cmcv_K_b10_plus
    cmcv_scenario_b10 = "upward"
elif cmcv_K_b10_minus > cmcv_K_b10_plus:
    cmcv_K_b10 = cmcv_K_b10_minus
    cmcv_scenario_b10 = "downward"
else:
    cmcv_K_b10 = cmcv_K_b10_plus
    cmcv_scenario_b10 = "upward" if np.sum(cmcv_cvr_plus_b10) >= np.sum(cmcv_cvr_minus_b10) else "downward"

# --- Bucket 2 --- #
if cmcv_K_b2_plus > cmcv_K_b2_minus:
    cmcv_K_b2 = cmcv_K_b2_plus
    cmcv_scenario_b2 = "upward"
elif cmcv_K_b2_minus > cmcv_K_b2_plus:
    cmcv_K_b2 = cmcv_K_b2_minus
    cmcv_scenario_b2 = "downward"
else:
    cmcv_K_b2 = cmcv_K_b2_plus
    cmcv_scenario_b2 = "upward" if np.sum(cmcv_cvr_plus_b2) >= np.sum(cmcv_cvr_minus_b2) else "downward"


# ============================================================
# 8) BUCKET SUMS (S_b)
# ============================================================
# S_b = Σ CVR+ if upward scenario, Σ CVR- if downward scenario

cmcv_S_b10 = (
    cmcv_cvr_plus_b10.sum() if cmcv_scenario_b10 == "upward"
    else cmcv_cvr_minus_b10.sum()
)

cmcv_S_b2 = (
    cmcv_cvr_plus_b2.sum() if cmcv_scenario_b2 == "upward"
    else cmcv_cvr_minus_b2.sum()
)


# ============================================================
# 9) CROSS-BUCKET CORRELATION (gamma_bc)
# ============================================================
# γ_bc = (delta_γ_bc)²

CMCV_DELTA_GAMMA = 0.20
cmcv_gamma = CMCV_DELTA_GAMMA ** 2


# ============================================================
# 10) CROSS-BUCKET SAFEGUARD FUNCTION (ψ)
# ============================================================
# ψ(S_b, S_c) = 0 if both S_b < 0 and S_c < 0, else 1

cmcv_psi_cross = cmcv_psi(cmcv_S_b10, cmcv_S_b2)


# ============================================================
# 11) CROSS-BUCKET AGGREGATION
# ============================================================
# K = √( Σ K_b² + Σ Σ γ_bc × S_b × S_c × ψ(S_b, S_c) )

cmcv_sum_K_sq = cmcv_K_b10**2 + cmcv_K_b2**2
cmcv_cross_bucket = (
    2 * cmcv_gamma
    * cmcv_S_b10
    * cmcv_S_b2
    * cmcv_psi_cross
)
cmcv_K_medium = np.sqrt(max(0, cmcv_sum_K_sq + cmcv_cross_bucket))


# ============================================================
# 12) CORRELATION SCENARIOS (HIGH/LOW)
# ============================================================
# High: ρ_high = min(1.25 × ρ_medium, 1.0)
# Low:  ρ_low  = max(2 × ρ_medium - 1, 0.75 × ρ_medium)

# --- High scenario correlations --- #
cmcv_rho_high = min(cmcv_rho * 1.25, 1.0)
cmcv_gamma_high = min(cmcv_gamma * 1.25, 1.0)

# --- Low scenario correlations --- #
cmcv_rho_low = max(2 * cmcv_rho - 1.0, 0.75 * cmcv_rho)
cmcv_gamma_low = max(2 * cmcv_gamma - 1.0, 0.75 * cmcv_gamma)

# --- High scenario intra-bucket K_b --- #

# Bucket 10 high (single RF - unchanged)
cmcv_K_b10_high = cmcv_K_b10
cmcv_scenario_b10_high = cmcv_scenario_b10

# Bucket 2 high
cmcv_cross_plus_b2_high = 2 * cmcv_rho_high * cmcv_cvr_plus_b2[0] * cmcv_cvr_plus_b2[1] * cmcv_psi_b2_plus
cmcv_K_b2_plus_high = np.sqrt(max(0, cmcv_sum_sq_plus_b2 + cmcv_cross_plus_b2_high))

cmcv_cross_minus_b2_high = 2 * cmcv_rho_high * cmcv_cvr_minus_b2[0] * cmcv_cvr_minus_b2[1] * cmcv_psi_b2_minus
cmcv_K_b2_minus_high = np.sqrt(max(0, cmcv_sum_sq_minus_b2 + cmcv_cross_minus_b2_high))

if cmcv_K_b2_plus_high > cmcv_K_b2_minus_high:
    cmcv_K_b2_high = cmcv_K_b2_plus_high
    cmcv_scenario_b2_high = "upward"
elif cmcv_K_b2_minus_high > cmcv_K_b2_plus_high:
    cmcv_K_b2_high = cmcv_K_b2_minus_high
    cmcv_scenario_b2_high = "downward"
else:
    cmcv_K_b2_high = cmcv_K_b2_plus_high
    cmcv_scenario_b2_high = "upward" if np.sum(cmcv_cvr_plus_b2) >= np.sum(cmcv_cvr_minus_b2) else "downward"

# --- Low scenario intra-bucket K_b --- #

# Bucket 10 low (single RF - unchanged)
cmcv_K_b10_low = cmcv_K_b10
cmcv_scenario_b10_low = cmcv_scenario_b10

# Bucket 2 low
cmcv_cross_plus_b2_low = 2 * cmcv_rho_low * cmcv_cvr_plus_b2[0] * cmcv_cvr_plus_b2[1] * cmcv_psi_b2_plus
cmcv_K_b2_plus_low = np.sqrt(max(0, cmcv_sum_sq_plus_b2 + cmcv_cross_plus_b2_low))

cmcv_cross_minus_b2_low = 2 * cmcv_rho_low * cmcv_cvr_minus_b2[0] * cmcv_cvr_minus_b2[1] * cmcv_psi_b2_minus
cmcv_K_b2_minus_low = np.sqrt(max(0, cmcv_sum_sq_minus_b2 + cmcv_cross_minus_b2_low))

if cmcv_K_b2_plus_low > cmcv_K_b2_minus_low:
    cmcv_K_b2_low = cmcv_K_b2_plus_low
    cmcv_scenario_b2_low = "upward"
elif cmcv_K_b2_minus_low > cmcv_K_b2_plus_low:
    cmcv_K_b2_low = cmcv_K_b2_minus_low
    cmcv_scenario_b2_low = "downward"
else:
    cmcv_K_b2_low = cmcv_K_b2_plus_low
    cmcv_scenario_b2_low = "upward" if np.sum(cmcv_cvr_plus_b2) >= np.sum(cmcv_cvr_minus_b2) else "downward"

# --- High scenario bucket sums and cross-bucket --- #
cmcv_S_b10_high = cmcv_cvr_plus_b10.sum() if cmcv_scenario_b10_high == "upward" else cmcv_cvr_minus_b10.sum()
cmcv_S_b2_high = cmcv_cvr_plus_b2.sum() if cmcv_scenario_b2_high == "upward" else cmcv_cvr_minus_b2.sum()
cmcv_psi_cross_high = cmcv_psi(cmcv_S_b10_high, cmcv_S_b2_high)

cmcv_sum_K_sq_high = cmcv_K_b10_high**2 + cmcv_K_b2_high**2
cmcv_cross_bucket_high = 2 * cmcv_gamma_high * cmcv_S_b10_high * cmcv_S_b2_high * cmcv_psi_cross_high
cmcv_K_high = np.sqrt(max(0, cmcv_sum_K_sq_high + cmcv_cross_bucket_high))

# --- Low scenario bucket sums and cross-bucket --- #
cmcv_S_b10_low = cmcv_cvr_plus_b10.sum() if cmcv_scenario_b10_low == "upward" else cmcv_cvr_minus_b10.sum()
cmcv_S_b2_low = cmcv_cvr_plus_b2.sum() if cmcv_scenario_b2_low == "upward" else cmcv_cvr_minus_b2.sum()
cmcv_psi_cross_low = cmcv_psi(cmcv_S_b10_low, cmcv_S_b2_low)

cmcv_sum_K_sq_low = cmcv_K_b10_low**2 + cmcv_K_b2_low**2
cmcv_cross_bucket_low = 2 * cmcv_gamma_low * cmcv_S_b10_low * cmcv_S_b2_low * cmcv_psi_cross_low
cmcv_K_low = np.sqrt(max(0, cmcv_sum_K_sq_low + cmcv_cross_bucket_low))


# ============================================================
# 13) CAPITAL
# ============================================================

cmcv_capital_df = pd.DataFrame([
    {'scenario': 'medium', 'capital': cmcv_K_medium},
    {'scenario': 'high', 'capital': cmcv_K_high},
    {'scenario': 'low', 'capital': cmcv_K_low},
])


# ============================================================
# 14) PRINTING
# ============================================================

# --- 14.1 Gross curvature positions --- #
cmcv_gross_cvr_print = format_table(
    cmcv_gross_cvr,
    columns=['trade_id', 'bucket', 'commodity', 'cvr_plus', 'cvr_minus'],
    col_formats={'cvr_plus': ',.0f', 'cvr_minus': ',.0f'},
    alignments=('left', 'left', 'left', 'right', 'right')
)

# --- 14.2 Net curvature positions --- #
cmcv_net_cvr_print = format_table(
    cmcv_net_cvr,
    columns=['bucket', 'commodity', 'cvr_plus', 'cvr_minus'],
    col_formats={'cvr_plus': ',.0f', 'cvr_minus': ',.0f'},
    alignments=('left', 'left', 'right', 'right')
)

# --- 14.3 Intra-bucket correlation --- #
cmcv_intra_corr_print = format_table(
    pd.DataFrame([
        {'bucket': '10', 'delta_rho': 'n/a', 'rho': 'n/a'},
        {'bucket': '2', 'delta_rho': CMCV_DELTA_RHO, 'rho': cmcv_rho_b2},
    ]),
    columns=['bucket', 'delta_rho', 'rho'],
    col_formats={'delta_rho': '.4f', 'rho': '.4f'},
    alignments=('left', 'right', 'right')
)

# --- 14.4 Intra-bucket safeguard function (ψ) --- #
cmcv_psi_intra_print = format_table(
    pd.DataFrame([
        {'bucket': '10', 'cvr_k+': cmcv_cvr_plus_b10[0], 'cvr_l+': 'n/a', 'ψ_up': 'n/a', 'cvr_k-': cmcv_cvr_minus_b10[0], 'cvr_l-': 'n/a', 'ψ_down': 'n/a'},
        {'bucket': '2', 'cvr_k+': cmcv_cvr_plus_b2[0], 'cvr_l+': cmcv_cvr_plus_b2[1], 'ψ_up': cmcv_psi_b2_plus, 'cvr_k-': cmcv_cvr_minus_b2[0], 'cvr_l-': cmcv_cvr_minus_b2[1], 'ψ_down': cmcv_psi_b2_minus},
    ]),
    columns=['bucket', 'cvr_k+', 'cvr_l+', 'ψ_up', 'cvr_k-', 'cvr_l-', 'ψ_down'],
    col_formats={'cvr_k+': ',.0f', 'cvr_l+': ',.0f', 'cvr_k-': ',.0f', 'cvr_l-': ',.0f'},
    alignments=('left', 'right', 'right', 'center', 'right', 'right', 'center')
)

# --- 14.5 Upward scenario (K_b+) --- #
cmcv_upward_print = format_table(
    pd.DataFrame([
        {'bucket': '10', 'cvr_k+': cmcv_cvr_plus_b10[0], 'cvr_l+': 'n/a', 'rho': 'n/a', 'ψ': 'n/a', 'K_b+': cmcv_K_b10_plus},
        {'bucket': '2', 'cvr_k+': cmcv_cvr_plus_b2[0], 'cvr_l+': cmcv_cvr_plus_b2[1], 'rho': cmcv_rho_b2, 'ψ': cmcv_psi_b2_plus, 'K_b+': cmcv_K_b2_plus},
    ]),
    columns=['bucket', 'cvr_k+', 'cvr_l+', 'rho', 'ψ', 'K_b+'],
    col_formats={'cvr_k+': ',.0f', 'cvr_l+': ',.0f', 'rho': '.4f', 'K_b+': ',.2f'},
    alignments=('left', 'right', 'right', 'right', 'center', 'right')
)

# Formula String
cmcv_Kb10_plus_formula_print = f"K_b10+ = sqrt( max({cmcv_cvr_plus_b10[0]},0)^2 )"

# Formula String
cmcv_Kb2_plus_formula_print = (
    f"K_b2+ = sqrt( (max({cmcv_cvr_plus_b2[0]},0)^2 + max({cmcv_cvr_plus_b2[1]},0)^2) + "
    f"(2 * {cmcv_rho_b2:.4f} * {cmcv_cvr_plus_b2[0]} * {cmcv_cvr_plus_b2[1]} * {cmcv_psi_b2_plus}) )"
)

# --- 14.6 Downward scenario (K_b-) --- #
cmcv_downward_print = format_table(
    pd.DataFrame([
        {'bucket': '10', 'cvr_k-': cmcv_cvr_minus_b10[0], 'cvr_l-': 'n/a', 'rho': 'n/a', 'ψ': 'n/a', 'K_b-': cmcv_K_b10_minus},
        {'bucket': '2', 'cvr_k-': cmcv_cvr_minus_b2[0], 'cvr_l-': cmcv_cvr_minus_b2[1], 'rho': cmcv_rho_b2, 'ψ': cmcv_psi_b2_minus, 'K_b-': cmcv_K_b2_minus},
    ]),
    columns=['bucket', 'cvr_k-', 'cvr_l-', 'rho', 'ψ', 'K_b-'],
    col_formats={'cvr_k-': ',.0f', 'cvr_l-': ',.0f', 'rho': '.4f', 'K_b-': ',.2f'},
    alignments=('left', 'right', 'right', 'right', 'center', 'right')
)

# Formula String
cmcv_Kb10_minus_formula_print = f"K_b10- = sqrt( max({cmcv_cvr_minus_b10[0]},0)^2 )"

# Formula String
cmcv_Kb2_minus_formula_print = (
    f"K_b2- = sqrt( (max({cmcv_cvr_minus_b2[0]},0)^2 + max({cmcv_cvr_minus_b2[1]},0)^2) + "
    f"(2 * {cmcv_rho_b2:.4f} * {cmcv_cvr_minus_b2[0]} * {cmcv_cvr_minus_b2[1]} * {cmcv_psi_b2_minus}) )"
)


# --- 14.7 Intra-bucket aggregation (K_b) --- #
cmcv_intra_agg_print = format_table(
    pd.DataFrame([
        {'bucket': '10', 'K_b+': cmcv_K_b10_plus, 'K_b-': cmcv_K_b10_minus, 'K_b': cmcv_K_b10, 'selected': cmcv_scenario_b10},
        {'bucket': '2', 'K_b+': cmcv_K_b2_plus, 'K_b-': cmcv_K_b2_minus, 'K_b': cmcv_K_b2, 'selected': cmcv_scenario_b2},
    ]),
    columns=['bucket', 'K_b+', 'K_b-', 'K_b', 'selected'],
    col_formats={'K_b+': ',.2f', 'K_b-': ',.2f', 'K_b': ',.2f'},
    alignments=('left', 'right', 'right', 'right', 'left')
)

# --- 14.8 Bucket sums (S_b) --- #
cmcv_bucket_sums_print = format_table(
    pd.DataFrame([
        {'bucket': '10', 'selected': cmcv_scenario_b10, 'cvr_k': cmcv_cvr_minus_b10[0] if cmcv_scenario_b10 == "downward" else cmcv_cvr_plus_b10[0], 'cvr_l': 'n/a', 'S_b': cmcv_S_b10},
        {'bucket': '2', 'selected': cmcv_scenario_b2, 'cvr_k': cmcv_cvr_minus_b2[0] if cmcv_scenario_b2 == "downward" else cmcv_cvr_plus_b2[0], 'cvr_l': cmcv_cvr_minus_b2[1] if cmcv_scenario_b2 == "downward" else cmcv_cvr_plus_b2[1], 'S_b': cmcv_S_b2},
    ]),
    columns=['bucket', 'selected', 'cvr_k', 'cvr_l', 'S_b'],
    col_formats={'cvr_k': ',.0f', 'cvr_l': ',.0f', 'S_b': ',.2f'},
    alignments=('left', 'left', 'right', 'right', 'right')
)

# --- 14.9 Cross-bucket correlation --- #
cmcv_cross_corr_print = format_table(
    pd.DataFrame([
        {'buckets': '10 vs 2', 'delta_gamma': CMCV_DELTA_GAMMA, 'gamma': cmcv_gamma},
    ]),
    columns=['buckets', 'delta_gamma', 'gamma'],
    col_formats={'delta_gamma': '.4f', 'gamma': '.4f'},
    alignments=('left', 'right', 'right')
)

# --- 14.10 Cross-bucket safeguard function (ψ) --- #
cmcv_psi_cross_print = format_table(
    pd.DataFrame([
        {'S_b10': cmcv_S_b10, 'S_b2': cmcv_S_b2, 'ψ': cmcv_psi_cross},
    ]),
    columns=['S_b10', 'S_b2', 'ψ'],
    col_formats={'S_b10': ',.2f', 'S_b2': ',.2f'},
    alignments=('right', 'right', 'center')
)

# --- 14.11 Cross-bucket aggregation --- #
cmcv_cross_agg_print = format_table(
    pd.DataFrame([
        {'component': 'K_b10', 'value': cmcv_K_b10},
        {'component': 'K_b2', 'value': cmcv_K_b2},
        {'component': 'S_b10', 'value': cmcv_S_b10},
        {'component': 'S_b2', 'value': cmcv_S_b2},
        {'component': 'gamma', 'value': f"{cmcv_gamma:.4f}"},
        {'component': 'ψ(S_b10, S_b2)', 'value': cmcv_psi_cross},
        {'component': 'capital', 'value': cmcv_K_medium},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# Formula String
cmcv_cross_agg_formula_print = (
    f"Capital = sqrt( ({cmcv_K_b10:,.2f}^2 + {cmcv_K_b2:,.2f}^2) + "
    f"(2 * {cmcv_gamma:.4f} * {cmcv_S_b10:,.2f} * {cmcv_S_b2:,.2f} * {cmcv_psi_cross}) )"
)

# --- 14.12 Correlation scenarios --- #
cmcv_corr_scenarios_print = format_table(
    pd.DataFrame([
        {'parameter': 'rho', 'medium': cmcv_rho, 'high': cmcv_rho_high, 'low': cmcv_rho_low},
        {'parameter': 'gamma', 'medium': cmcv_gamma, 'high': cmcv_gamma_high, 'low': cmcv_gamma_low},
    ]),
    columns=['parameter', 'medium', 'high', 'low'],
    col_formats={'medium': '.4f', 'high': '.4f', 'low': '.4f'},
    alignments=('left', 'right', 'right', 'right')
)

# --- 14.13 Capital --- #
cmcv_capital_print = format_table(
    cmcv_capital_df,
    columns=['scenario', 'capital'],
    col_formats={'capital': ',.2f'},
    alignments=('left', 'right')
)


# ============================================================
# DISPLAY ALL OUTPUTS
# ============================================================

print("=" * 60)
print("CMCV")
print("=" * 60)

print("\n1) GROSS CURVATURE POSITIONS")
print(cmcv_gross_cvr_print)

print("\n2) NET CURVATURE POSITIONS")
print(cmcv_net_cvr_print)

print("\n3) INTRA-BUCKET CORRELATION")
print(cmcv_intra_corr_print)

print("\n4) INTRA-BUCKET SAFEGUARD FUNCTION (ψ)")
print(cmcv_psi_intra_print)

print("\n5) UPWARD SCENARIO (K_b+)")
print("\n--- Bucket 10 ---")
print(cmcv_Kb10_plus_formula_print)
print("\n--- Bucket 2 ---")
print(cmcv_Kb2_plus_formula_print)
print(cmcv_upward_print)

print("\n6) DOWNWARD SCENARIO (K_b-)")
print("\n--- Bucket 10 ---")
print(cmcv_Kb10_minus_formula_print)
print("\n--- Bucket 2 ---")
print(cmcv_Kb2_minus_formula_print)
print(cmcv_downward_print)

print("\n7) INTRA-BUCKET AGGREGATION (K_b)")
print(cmcv_intra_agg_print)

print("\n8) BUCKET SUMS (S_b)")
print(cmcv_bucket_sums_print)

print("\n9) CROSS-BUCKET CORRELATION (gamma_bc)")
print(cmcv_cross_corr_print)

print("\n10) CROSS-BUCKET SAFEGUARD FUNCTION (ψ)")
print(cmcv_psi_cross_print)

print("\n11) CROSS-BUCKET AGGREGATION")
print(cmcv_cross_agg_formula_print)
print(cmcv_cross_agg_print)

print("\n12) CORRELATION SCENARIOS")
print(cmcv_corr_scenarios_print)

print("\n13) CAPITAL")
print(cmcv_capital_print)

CMCV

1) GROSS CURVATURE POSITIONS
┌────────────┬──────────┬─────────────┬────────────┬─────────────┐
│ trade_id   │ bucket   │ commodity   │   cvr_plus │   cvr_minus │
├────────────┼──────────┼─────────────┼────────────┼─────────────┤
│ 1          │ 10       │ sugar       │      2,600 │       1,400 │
│ 2          │ 10       │ sugar       │      9,100 │       4,900 │
│ 3          │ 2        │ brent       │      1,200 │      -1,520 │
│ 4          │ 2        │ wti         │     -3,500 │       6,300 │
└────────────┴──────────┴─────────────┴────────────┴─────────────┘

2) NET CURVATURE POSITIONS
┌──────────┬─────────────┬────────────┬─────────────┐
│ bucket   │ commodity   │   cvr_plus │   cvr_minus │
├──────────┼─────────────┼────────────┼─────────────┤
│ 10       │ sugar       │     11,700 │       6,300 │
│ 2        │ brent       │      1,200 │      -1,520 │
│ 2        │ wti         │     -3,500 │       6,300 │
└──────────┴─────────────┴────────────┴─────────────┘

3) INTRA-BUCKET CORREL

### FXCV

In [24]:
# ============================================================
# FXCV
# ============================================================


# TABLE OF CONTENTS
#   1) Gross curvature positions
#   2) Net curvature positions (CVR+/CVR-)
#   3) Intra-bucket correlation (rho_kl)
#   4) Intra-bucket safeguard function (ψ)
#   5) Upward scenario (K_b+)
#   6) Downward scenario (K_b-)
#   7) Intra-bucket aggregation (K_b)
#   8) Bucket sums (S_b)
#   9) Cross-bucket correlation (gamma_bc)
#  10) Cross-bucket safeguard function (ψ)
#  11) Cross-bucket aggregation
#  12) Correlation scenarios (high/low)
#  13) Capital
#  14) Printing
# ============================================================


# ============================================================
# 1) GROSS CURVATURE POSITIONS
# ============================================================
# Position-level curvature values from upward (+) and downward (-) shocks.

data = [
    {'trade_id': 1, 'bucket': 'KRW^USD', 'cvr_plus': -3300, 'cvr_minus':  4100},
    {'trade_id': 2, 'bucket': 'KRW^USD', 'cvr_plus':  4300, 'cvr_minus': -4300},
    {'trade_id': 3, 'bucket': 'HKD^USD', 'cvr_plus': -1500, 'cvr_minus': -2000},
    {'trade_id': 4, 'bucket': 'HKD^USD', 'cvr_plus':  2500, 'cvr_minus':  2800},
]

crcv_fx_gross_cvr = pd.DataFrame(data)


# ============================================================
# 2) NET CURVATURE POSITIONS (CVR+/CVR-)
# ============================================================
# Net CVR = Σ gross CVR for each unique risk factor.
# Risk factor defined by: bucket (currency pair).
# FX has single risk factor per bucket.

crcv_fx_risk_factor_cols = ['bucket']

crcv_fx_net_cvr = (
    crcv_fx_gross_cvr
    .groupby(crcv_fx_risk_factor_cols)
    .agg(cvr_plus=('cvr_plus', 'sum'), cvr_minus=('cvr_minus', 'sum'))
    .reset_index()
)

# Extract CVR vectors per bucket
crcv_fx_cvr_krw = crcv_fx_net_cvr[crcv_fx_net_cvr['bucket'] == 'KRW^USD']
crcv_fx_cvr_plus_krw = crcv_fx_cvr_krw['cvr_plus'].values
crcv_fx_cvr_minus_krw = crcv_fx_cvr_krw['cvr_minus'].values

crcv_fx_cvr_hkd = crcv_fx_net_cvr[crcv_fx_net_cvr['bucket'] == 'HKD^USD']
crcv_fx_cvr_plus_hkd = crcv_fx_cvr_hkd['cvr_plus'].values
crcv_fx_cvr_minus_hkd = crcv_fx_cvr_hkd['cvr_minus'].values


# ============================================================
# 3) INTRA-BUCKET CORRELATION (rho_kl)
# ============================================================
# For curvature: ρ_kl = (delta_ρ_kl)²

# Intra-bucket: Not applicable for FX (single RF per bucket)
# We calculate cross-bucket delta only.
CRCV_FX_DELTA_GAMMA = 0.60
crcv_fx_gamma = CRCV_FX_DELTA_GAMMA ** 2

# No intra-bucket correlation needed
crcv_fx_rho_krw = None
crcv_fx_rho_hkd = None


# ============================================================
# 4) INTRA-BUCKET SAFEGUARD FUNCTION (ψ)
# ============================================================
# ψ(x, y) = 0 if both x < 0 and y < 0, else 1

def crcv_fx_psi(x, y):
    """Safeguard function: returns 0 if both x and y are negative, else 1"""
    return 0 if x < 0 and y < 0 else 1

# Single risk factor per bucket -> No intra-bucket psi needed
crcv_fx_psi_krw_plus = None
crcv_fx_psi_krw_minus = None
crcv_fx_psi_hkd_plus = None
crcv_fx_psi_hkd_minus = None


# ============================================================
# 5) UPWARD SCENARIO (K_b+)
# ============================================================
# K_b+ = √( Σ max(CVR_k+, 0)² ) -- (No cross terms for single RF)

# --- KRW Bucket --- #
crcv_fx_sum_sq_plus_krw = np.sum(np.maximum(crcv_fx_cvr_plus_krw, 0)**2)
crcv_fx_cross_plus_krw = 0
crcv_fx_K_krw_plus = np.sqrt(max(0, crcv_fx_sum_sq_plus_krw + crcv_fx_cross_plus_krw))


# --- HKD Bucket --- #
crcv_fx_sum_sq_plus_hkd = np.sum(np.maximum(crcv_fx_cvr_plus_hkd, 0)**2)
crcv_fx_cross_plus_hkd = 0
crcv_fx_K_hkd_plus = np.sqrt(max(0, crcv_fx_sum_sq_plus_hkd + crcv_fx_cross_plus_hkd))


# ============================================================
# 6) DOWNWARD SCENARIO (K_b-)
# ============================================================
# K_b- = √( Σ max(CVR_k-, 0)² ) -- (No cross terms for single RF)

# --- KRW Bucket --- #
crcv_fx_sum_sq_minus_krw = np.sum(np.maximum(crcv_fx_cvr_minus_krw, 0)**2)
crcv_fx_cross_minus_krw = 0
crcv_fx_K_krw_minus = np.sqrt(max(0, crcv_fx_sum_sq_minus_krw + crcv_fx_cross_minus_krw))


# --- HKD Bucket --- #
crcv_fx_sum_sq_minus_hkd = np.sum(np.maximum(crcv_fx_cvr_minus_hkd, 0)**2)
crcv_fx_cross_minus_hkd = 0
crcv_fx_K_hkd_minus = np.sqrt(max(0, crcv_fx_sum_sq_minus_hkd + crcv_fx_cross_minus_hkd))


# ============================================================
# 7) INTRA-BUCKET AGGREGATION (K_b)
# ============================================================
# K_b = max(K_b+, K_b-)

# --- KRW Bucket --- #
if crcv_fx_K_krw_plus > crcv_fx_K_krw_minus:
    crcv_fx_K_krw = crcv_fx_K_krw_plus
    crcv_fx_scenario_krw = "upward"
elif crcv_fx_K_krw_minus > crcv_fx_K_krw_plus:
    crcv_fx_K_krw = crcv_fx_K_krw_minus
    crcv_fx_scenario_krw = "downward"
else:
    crcv_fx_K_krw = crcv_fx_K_krw_plus
    crcv_fx_scenario_krw = "upward" if np.sum(crcv_fx_cvr_plus_krw) >= np.sum(crcv_fx_cvr_minus_krw) else "downward"

# --- HKD Bucket --- #
if crcv_fx_K_hkd_plus > crcv_fx_K_hkd_minus:
    crcv_fx_K_hkd = crcv_fx_K_hkd_plus
    crcv_fx_scenario_hkd = "upward"
elif crcv_fx_K_hkd_minus > crcv_fx_K_hkd_plus:
    crcv_fx_K_hkd = crcv_fx_K_hkd_minus
    crcv_fx_scenario_hkd = "downward"
else:
    crcv_fx_K_hkd = crcv_fx_K_hkd_plus
    crcv_fx_scenario_hkd = "upward" if np.sum(crcv_fx_cvr_plus_hkd) >= np.sum(crcv_fx_cvr_minus_hkd) else "downward"


# ============================================================
# 8) BUCKET SUMS (S_b)
# ============================================================
# S_b = Σ CVR+ if upward scenario, Σ CVR- if downward scenario

crcv_fx_S_krw = (
    crcv_fx_cvr_plus_krw.sum() if crcv_fx_scenario_krw == "upward"
    else crcv_fx_cvr_minus_krw.sum()
)

crcv_fx_S_hkd = (
    crcv_fx_cvr_plus_hkd.sum() if crcv_fx_scenario_hkd == "upward"
    else crcv_fx_cvr_minus_hkd.sum()
)


# ============================================================
# 9) CROSS-BUCKET CORRELATION (gamma_bc)
# ============================================================
# γ_bc = (delta_γ_bc)²

CRCV_FX_DELTA_GAMMA = 0.60
crcv_fx_gamma = CRCV_FX_DELTA_GAMMA ** 2


# ============================================================
# 10) CROSS-BUCKET SAFEGUARD FUNCTION (ψ)
# ============================================================
# ψ(S_b, S_c) = 0 if both S_b < 0 and S_c < 0, else 1

crcv_fx_psi_cross = crcv_fx_psi(crcv_fx_S_krw, crcv_fx_S_hkd)


# ============================================================
# 11) CROSS-BUCKET AGGREGATION
# ============================================================
# K = √( Σ K_b² + Σ Σ γ_bc × S_b × S_c × ψ(S_b, S_c) )

crcv_fx_sum_K_sq = crcv_fx_K_krw**2 + crcv_fx_K_hkd**2
crcv_fx_cross_bucket = (
    2 * crcv_fx_gamma
    * crcv_fx_S_krw
    * crcv_fx_S_hkd
    * crcv_fx_psi_cross
)
crcv_fx_K_medium = np.sqrt(max(0, crcv_fx_sum_K_sq + crcv_fx_cross_bucket))


# ============================================================
# 12) CORRELATION SCENARIOS (HIGH/LOW)
# ============================================================
# High: γ_high = min(1.25 × γ_medium, 1.0)
# Low:  γ_low  = max(2 × γ_medium - 1, 0.75 × γ_medium)
# Note: Intra-bucket K_b does not change because there are no intra-correlations to shock.

# --- High scenario correlations --- #
crcv_fx_gamma_high = min(crcv_fx_gamma * 1.25, 1.0)

# --- Low scenario correlations --- #
crcv_fx_gamma_low = max(2 * crcv_fx_gamma - 1.0, 0.75 * crcv_fx_gamma)

# --- High scenario K (Scenarios/Sums identical to medium for single RF) --- #
crcv_fx_S_krw_high = crcv_fx_S_krw
crcv_fx_S_hkd_high = crcv_fx_S_hkd
crcv_fx_psi_cross_high = crcv_fx_psi(crcv_fx_S_krw_high, crcv_fx_S_hkd_high)

crcv_fx_sum_K_sq_high = crcv_fx_K_krw**2 + crcv_fx_K_hkd**2
crcv_fx_cross_bucket_high = 2 * crcv_fx_gamma_high * crcv_fx_S_krw_high * crcv_fx_S_hkd_high * crcv_fx_psi_cross_high
crcv_fx_K_high = np.sqrt(max(0, crcv_fx_sum_K_sq_high + crcv_fx_cross_bucket_high))

# --- Low scenario K --- #
crcv_fx_S_krw_low = crcv_fx_S_krw
crcv_fx_S_hkd_low = crcv_fx_S_hkd
crcv_fx_psi_cross_low = crcv_fx_psi(crcv_fx_S_krw_low, crcv_fx_S_hkd_low)

crcv_fx_sum_K_sq_low = crcv_fx_K_krw**2 + crcv_fx_K_hkd**2
crcv_fx_cross_bucket_low = 2 * crcv_fx_gamma_low * crcv_fx_S_krw_low * crcv_fx_S_hkd_low * crcv_fx_psi_cross_low
crcv_fx_K_low = np.sqrt(max(0, crcv_fx_sum_K_sq_low + crcv_fx_cross_bucket_low))


# ============================================================
# 13) CAPITAL
# ============================================================
# Final capital = max(K_medium, K_high, K_low)

crcv_fx_capital = max(crcv_fx_K_medium, crcv_fx_K_high, crcv_fx_K_low)
crcv_fx_binding_scenario = (
    "medium" if crcv_fx_capital == crcv_fx_K_medium else
    "high" if crcv_fx_capital == crcv_fx_K_high else "low"
)

crcv_fx_capital_df = pd.DataFrame([
    {'scenario': 'medium', 'capital': crcv_fx_K_medium},
    {'scenario': 'high', 'capital': crcv_fx_K_high},
    {'scenario': 'low', 'capital': crcv_fx_K_low},
])


# ============================================================
# 14) PRINTING
# ============================================================

# --- 14.1 Gross curvature positions --- #
crcv_fx_gross_cvr_print = format_table(
    crcv_fx_gross_cvr,
    columns=['trade_id', 'bucket', 'cvr_plus', 'cvr_minus'],
    col_formats={'cvr_plus': ',.0f', 'cvr_minus': ',.0f'},
    alignments=('left', 'left', 'right', 'right')
)

# --- 14.2 Net curvature positions --- #
crcv_fx_net_cvr_print = format_table(
    crcv_fx_net_cvr,
    columns=['bucket', 'cvr_plus', 'cvr_minus'],
    col_formats={'cvr_plus': ',.0f', 'cvr_minus': ',.0f'},
    alignments=('left', 'right', 'right')
)

# --- 14.3 Intra-bucket correlation --- #
crcv_fx_intra_corr_print = format_table(
    pd.DataFrame([
        {'bucket': 'KRW^USD', 'rho': 'n/a (single RF)'},
        {'bucket': 'HKD^USD', 'rho': 'n/a (single RF)'},
    ]),
    columns=['bucket', 'rho'],
    col_formats={},
    alignments=('left', 'right')
)

# --- 14.4 Intra-bucket safeguard function (ψ) --- #
crcv_fx_psi_intra_print = format_table(
    pd.DataFrame([
        {'bucket': 'KRW^USD', 'cvr_k+': crcv_fx_cvr_plus_krw[0], 'ψ_up': 'n/a', 'cvr_k-': crcv_fx_cvr_minus_krw[0], 'ψ_down': 'n/a'},
        {'bucket': 'HKD^USD', 'cvr_k+': crcv_fx_cvr_plus_hkd[0], 'ψ_up': 'n/a', 'cvr_k-': crcv_fx_cvr_minus_hkd[0], 'ψ_down': 'n/a'},
    ]),
    columns=['bucket', 'cvr_k+', 'ψ_up', 'cvr_k-', 'ψ_down'],
    col_formats={'cvr_k+': ',.0f', 'cvr_k-': ',.0f'},
    alignments=('left', 'right', 'center', 'right', 'center')
)

# --- 14.5 Upward scenario (K_b+) --- #
crcv_fx_upward_print = format_table(
    pd.DataFrame([
        {'bucket': 'KRW^USD', 'cvr_k+': crcv_fx_cvr_plus_krw[0], 'rho': 'n/a', 'ψ': 'n/a', 'K_b+': crcv_fx_K_krw_plus},
        {'bucket': 'HKD^USD', 'cvr_k+': crcv_fx_cvr_plus_hkd[0], 'rho': 'n/a', 'ψ': 'n/a', 'K_b+': crcv_fx_K_hkd_plus},
    ]),
    columns=['bucket', 'cvr_k+', 'rho', 'ψ', 'K_b+'],
    col_formats={'cvr_k+': ',.0f', 'K_b+': ',.2f'},
    alignments=('left', 'right', 'right', 'center', 'right')
)

# Formula String
crcv_fx_Kb_krw_plus_formula_print = f"K_b_krw+ = sqrt( max({crcv_fx_cvr_plus_krw[0]:,.2f}, 0)^2 )"

# Formula String
crcv_fx_Kb_hkd_plus_formula_print = f"K_b_hkd+ = sqrt( max({crcv_fx_cvr_plus_hkd[0]:,.2f}, 0)^2 )"

# --- 14.6 Downward scenario (K_b-) --- #
crcv_fx_downward_print = format_table(
    pd.DataFrame([
        {'bucket': 'KRW^USD', 'cvr_k-': crcv_fx_cvr_minus_krw[0], 'rho': 'n/a', 'ψ': 'n/a', 'K_b-': crcv_fx_K_krw_minus},
        {'bucket': 'HKD^USD', 'cvr_k-': crcv_fx_cvr_minus_hkd[0], 'rho': 'n/a', 'ψ': 'n/a', 'K_b-': crcv_fx_K_hkd_minus},
    ]),
    columns=['bucket', 'cvr_k-', 'rho', 'ψ', 'K_b-'],
    col_formats={'cvr_k-': ',.0f', 'K_b-': ',.2f'},
    alignments=('left', 'right', 'right', 'center', 'right')
)

# Formula String
crcv_fx_Kb_krw_minus_formula_print = f"K_b_krw- = sqrt( max({crcv_fx_cvr_minus_krw[0]:,.2f}, 0)^2 )"

# Formula String
crcv_fx_Kb_hkd_minus_formula_print = f"K_b_hkd- = sqrt( max({crcv_fx_cvr_minus_hkd[0]:,.2f}, 0)^2 )"

# --- 14.7 Intra-bucket aggregation (K_b) --- #
crcv_fx_intra_agg_print = format_table(
    pd.DataFrame([
        {'bucket': 'KRW^USD', 'K_b+': crcv_fx_K_krw_plus, 'K_b-': crcv_fx_K_krw_minus, 'K_b': crcv_fx_K_krw, 'selected': crcv_fx_scenario_krw},
        {'bucket': 'HKD^USD', 'K_b+': crcv_fx_K_hkd_plus, 'K_b-': crcv_fx_K_hkd_minus, 'K_b': crcv_fx_K_hkd, 'selected': crcv_fx_scenario_hkd},
    ]),
    columns=['bucket', 'K_b+', 'K_b-', 'K_b', 'selected'],
    col_formats={'K_b+': ',.2f', 'K_b-': ',.2f', 'K_b': ',.2f'},
    alignments=('left', 'right', 'right', 'right', 'left')
)

# --- 14.8 Bucket sums (S_b) --- #
crcv_fx_bucket_sums_print = format_table(
    pd.DataFrame([
        {'bucket': 'KRW^USD', 'selected': crcv_fx_scenario_krw, 'cvr_k': crcv_fx_cvr_minus_krw[0] if crcv_fx_scenario_krw == "downward" else crcv_fx_cvr_plus_krw[0], 'S_b': crcv_fx_S_krw},
        {'bucket': 'HKD^USD', 'selected': crcv_fx_scenario_hkd, 'cvr_k': crcv_fx_cvr_minus_hkd[0] if crcv_fx_scenario_hkd == "downward" else crcv_fx_cvr_plus_hkd[0], 'S_b': crcv_fx_S_hkd},
    ]),
    columns=['bucket', 'selected', 'cvr_k', 'S_b'],
    col_formats={'cvr_k': ',.0f', 'S_b': ',.2f'},
    alignments=('left', 'left', 'right', 'right')
)

# --- 14.9 Cross-bucket correlation --- #
crcv_fx_cross_corr_print = format_table(
    pd.DataFrame([
        {'buckets': 'KRW vs HKD', 'delta_gamma': CRCV_FX_DELTA_GAMMA, 'gamma': crcv_fx_gamma},
    ]),
    columns=['buckets', 'delta_gamma', 'gamma'],
    col_formats={'delta_gamma': '.4f', 'gamma': '.4f'},
    alignments=('left', 'right', 'right')
)

# --- 14.10 Cross-bucket safeguard function (ψ) --- #
crcv_fx_psi_cross_print = format_table(
    pd.DataFrame([
        {'S_krw': crcv_fx_S_krw, 'S_hkd': crcv_fx_S_hkd, 'ψ': crcv_fx_psi_cross},
    ]),
    columns=['S_krw', 'S_hkd', 'ψ'],
    col_formats={'S_krw': ',.2f', 'S_hkd': ',.2f'},
    alignments=('right', 'right', 'center')
)

# --- 14.11 Cross-bucket aggregation --- #
crcv_fx_cross_agg_print = format_table(
    pd.DataFrame([
        {'component': 'K_krw', 'value': crcv_fx_K_krw},
        {'component': 'K_hkd', 'value': crcv_fx_K_hkd},
        {'component': 'S_krw', 'value': crcv_fx_S_krw},
        {'component': 'S_hkd', 'value': crcv_fx_S_hkd},
        {'component': 'gamma', 'value': f"{crcv_fx_gamma:.4f}"},
        {'component': 'ψ(S_krw, S_hkd)', 'value': crcv_fx_psi_cross},
        {'component': 'capital', 'value': crcv_fx_K_medium},
    ]),
    columns=['component', 'value'],
    col_formats={'value': ',.2f'},
    alignments=('left', 'right')
)

# Formula String
crcv_fx_cross_agg_formula_print = (
    f"Capital = sqrt( ({crcv_fx_K_krw:,.2f}^2 + {crcv_fx_K_hkd:,.2f}^2) + "
    f"(2 * {crcv_fx_gamma:.4f} * {crcv_fx_S_krw:,.2f} * {crcv_fx_S_hkd:,.2f} * {crcv_fx_psi_cross}) )"
)

# --- 14.12 Correlation scenarios --- #
crcv_fx_corr_scenarios_print = format_table(
    pd.DataFrame([
        {'parameter': 'rho', 'medium': 'n/a', 'high': 'n/a', 'low': 'n/a'},
        {'parameter': 'gamma', 'medium': crcv_fx_gamma, 'high': crcv_fx_gamma_high, 'low': crcv_fx_gamma_low},
    ]),
    columns=['parameter', 'medium', 'high', 'low'],
    col_formats={'medium': '.4f', 'high': '.4f', 'low': '.4f'},
    alignments=('left', 'right', 'right', 'right')
)

# --- 14.13 Capital --- #
crcv_fx_capital_print = format_table(
    crcv_fx_capital_df,
    columns=['scenario', 'capital'],
    col_formats={'capital': ',.2f'},
    alignments=('left', 'right')
)


# ============================================================
# DISPLAY ALL OUTPUTS
# ============================================================

print("=" * 60)
print("FRTB — FX Curvature Capital Calculation (KRW^USD & HKD^USD)")
print("=" * 60)

print("\n1) GROSS CURVATURE POSITIONS")
print(crcv_fx_gross_cvr_print)

print("\n2) NET CURVATURE POSITIONS")
print(crcv_fx_net_cvr_print)

print("\n3) INTRA-BUCKET CORRELATION (rho_kl)")
print(crcv_fx_intra_corr_print)

print("\n4) INTRA-BUCKET SAFEGUARD FUNCTION (ψ)")
print(crcv_fx_psi_intra_print)

print("\n5) UPWARD SCENARIO (K_b+)")
print("\n--- KRW Bucket ---")
print(crcv_fx_Kb_krw_plus_formula_print)
print("\n--- HKD Bucket ---")
print(crcv_fx_Kb_hkd_plus_formula_print)
print(crcv_fx_upward_print)

print("\n6) DOWNWARD SCENARIO (K_b-)")
print("\n--- KRW Bucket ---")
print(crcv_fx_Kb_krw_minus_formula_print)
print("\n--- HKD Bucket ---")
print(crcv_fx_Kb_hkd_minus_formula_print)
print(crcv_fx_downward_print)

print("\n7) INTRA-BUCKET AGGREGATION (K_b)")
print(crcv_fx_intra_agg_print)

print("\n8) BUCKET SUMS (S_b)")
print(crcv_fx_bucket_sums_print)

print("\n9) CROSS-BUCKET CORRELATION (gamma_bc)")
print(crcv_fx_cross_corr_print)

print("\n10) CROSS-BUCKET SAFEGUARD FUNCTION (ψ)")
print(crcv_fx_psi_cross_print)

print("\n11) CROSS-BUCKET AGGREGATION")
print(crcv_fx_cross_agg_formula_print)
print(crcv_fx_cross_agg_print)

print("\n12) CORRELATION SCENARIOS")
print(crcv_fx_corr_scenarios_print)

print("\n13) CAPITAL")
print(crcv_fx_capital_print)

FRTB — FX Curvature Capital Calculation (KRW^USD & HKD^USD)

1) GROSS CURVATURE POSITIONS
┌────────────┬──────────┬────────────┬─────────────┐
│ trade_id   │ bucket   │   cvr_plus │   cvr_minus │
├────────────┼──────────┼────────────┼─────────────┤
│ 1          │ KRW^USD  │     -3,300 │       4,100 │
│ 2          │ KRW^USD  │      4,300 │      -4,300 │
│ 3          │ HKD^USD  │     -1,500 │      -2,000 │
│ 4          │ HKD^USD  │      2,500 │       2,800 │
└────────────┴──────────┴────────────┴─────────────┘

2) NET CURVATURE POSITIONS
┌──────────┬────────────┬─────────────┐
│ bucket   │   cvr_plus │   cvr_minus │
├──────────┼────────────┼─────────────┤
│ HKD^USD  │      1,000 │         800 │
│ KRW^USD  │      1,000 │        -200 │
└──────────┴────────────┴─────────────┘

3) INTRA-BUCKET CORRELATION (rho_kl)
┌──────────┬─────────────────┐
│ bucket   │             rho │
├──────────┼─────────────────┤
│ KRW^USD  │ n/a (single RF) │
│ HKD^USD  │ n/a (single RF) │
└──────────┴─────────────